<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.es/cap04/cap04_aluno.ipynb"><img src="imagenes/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagenes/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 4 Morfología Matemática y Segmentación de Imágenes

Este capítulo presenta dos temas fundamentales del Procesamiento Digital de Imágenes (PDI): la **morfología matemática** y la **segmentación de imágenes**. La morfología matemática proporciona un marco teórico basado en la teoría de conjuntos para analizar, refinar y cuantificar la forma de objetos en imágenes binarias y en tonos de gris, mediante operadores fundamentales como la **erosión** y la **dilatación**. La segmentación, por su parte, tiene como objetivo particionar la imagen en regiones de interés, separando objetos del fondo y produciendo representaciones adecuadas para el análisis y la interpretación.

El capítulo comienza con la **umbralización**, una de las técnicas más importantes de segmentación, introduciendo el método automático de **Otsu** y revisitando el análisis de histogramas mediante la varianza interclases, presentada en el Capítulo 1. A continuación, se estudian los principales operadores de la morfología matemática, incluyendo erosión, dilatación, apertura, cierre y reconstrucción morfológica, que permiten refinar máscaras binarias y preservar estructuras relevantes de los objetos. Finalmente, se presentan técnicas de segmentación basada en regiones, como el **etiquetado de componentes conexos**, la **transformada de distancia** y el algoritmo ***watershed*** basado en marcadores, culminando en la extracción de descriptores geométricos y en la generación de *bounding boxes* compatibles con sistemas modernos de detección de objetos.

## 4.1 Objetivos

Al final de este capítulo, usted será capaz de:

* **Aplicar umbralización:** Comprender el criterio automático de Otsu por maximización de la varianza interclases ($\sigma_B^2$) y seleccionar estrategias adecuadas de preprocesamiento para facilitar la segmentación;
* **Dominar la morfología binaria:** Comprender y aplicar erosión ($A\ominus B$) y dilatación ($A\oplus B$) como operadores fundamentales, derivando apertura ($A\circ B$), cierre ($A\bullet B$) y operaciones basadas en reconstrucción morfológica, como `mm::clohole` y `mm::edgeoff`;
* **Aplicar morfología en tonos de gris:** Utilizar gradiente morfológico y filtros *top-hat* para realce y análisis de estructuras locales;
* **Etiquetar componentes conexos:** Identificar y separar regiones conectadas en imágenes binarias mediante algoritmos de etiquetado;
* **Aplicar transformada de distancia:** Interpretar y calcular distancias al fondo utilizando enfoques morfológicos y métricas geométricas;
* **Segmentar por regiones:** Construir *pipelines* de segmentación basados en marcadores utilizando Transformada de Distancia y el algoritmo ***watershed***;
* **Extraer descriptores geométricos:** Calcular propiedades como área, perímetro, centroide, circularidad y *bounding boxes* mediante `mm::label0` y extracción de contornos;
* **Relacionar PDI y visión computacional:** Comprender cómo los descriptores extraídos por segmentación pueden convertirse a formatos utilizados por detectores modernos, como YOLO.

In [1]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefactos de build de la trilha C++ (.cpp, binário, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# O kernel é Python mesmo na trilha C++: `mm` (morph.py) é usado pelos
# simuladores, pela exibição das figuras que o binário C++ gera e pelo
# estado mm::Image entre células. cpp=True baixa também a trilha compilada
# (morph.hpp + stb_image*.h), usada no #include das células %%writefile *.cpp.
# As células C++ deste capítulo compilam COM OpenCV (-DMM_USE_OPENCV +
# pkg-config opencv4): mm::dil/ero → cv::dilate/erode, então open/close/asf/
# gradm/tophat/blackhat rodam full-res e batem bit a bit com a trilha py.
import config
config.setup(cpp=True)
from morph import mm
import numpy as np

✅ Entorno listo. Morph: 1.1.9 | OpenCV: 5.0.0


## 4.2 Umbralización

La **umbralización** (*thresholding*) es una de las formas más simples y eficientes de segmentación de imágenes. Su objetivo es clasificar cada píxel en dos clases de intensidad, normalmente asociadas a *objeto* y *fondo*:

<a id="eq-04-limiar"></a>
$$
g(x,y) =
\begin{cases}
255, & \text{si } f(x,y) > T \\
0,   & \text{en caso contrario}
\end{cases} \tag{4.1}
$$


donde $f(x,y)$ representa la intensidad del píxel en la imagen original y $g(x,y)$ la imagen binaria resultante.

La elección del umbral $T$ es importante para la calidad de la segmentación. El método de **Otsu** determina automáticamente el umbral óptimo al maximizar la **varianza entre clases** $\sigma_B^2$ definida por:

<a id="eq-04-otsu"></a>
$$
\sigma_B^2(T) =
w_0(T)\,w_1(T)\,
\bigl[\mu_0(T)-\mu_1(T)\bigr]^2 \tag{4.2}
$$


donde:

- $w_0(T)$ y $w_1(T)$ son las probabilidades acumuladas de las clases fondo y objeto;
- $\mu_0(T)$ y $\mu_1(T)$ son las medias de intensidad de esas clases;
- $\sigma_B^2(T)$ representa la varianza entre clases para un umbral dado $T$.

El método funciona mejor cuando el histograma presenta dos grupos de intensidades relativamente separados. Para ello, el algoritmo evalúa todos los umbrales posibles de la imagen — típicamente en el intervalo $[0,255]$ para imágenes de 8 bits — y selecciona el valor que maximiza la varianza entre clases, denotada por $\sigma_B^2$:

$$
T^* =
\arg\max_{T \in [0,255]}
\sigma_B^2(T)
$$

> ### 📝 Otsu asume histogramas bimodales
>
> El método de Otsu produce mejores resultados cuando el histograma presenta dos picos bien definidos (*bimodalidad*), correspondientes al fondo y al objeto. Cuanto mayor sea la separación entre estos picos y más pronunciado sea el máximo de $\sigma_B^2$, más confiable tiende a ser el umbral obtenido.
>
> En imágenes con iluminación no uniforme o múltiples regiones de intensidad, las técnicas de **umbralización adaptativa** — en las cuales el umbral se calcula localmente — suelen producir segmentaciones más robustas.
>
> El subíndice $B$ en $\sigma_B^2$ significa ***between classes*** (*entre clases*). Así, $\sigma_B^2$ representa la **varianza entre las clases** (*between-class variance*).

### 4.2.1 Imagen de Monedas

La imagen utilizada para practicar la segmentación es una fotografía de una colección de monedas de diferentes países y épocas ([Figura 4.1](#fig-04-coins)). Crédito: GAZI.MD.AHAD (CC BY-SA 4.0). Presenta objetos circulares con bordes bien definidos, siendo ideal para demostrar umbralización, operadores morfológicos, transformada de distancia, *watershed* y descriptores de forma.

In [2]:
%%writefile tmp/fig_04_coins.cpp
#define MM_OUT "tmp/fig_04_coins.png"
//| label: fig-04-coins
//| fig-cap: "Imagem com moedas de vários tipos. Crédito: GAZI.MD.AHAD (CC BY-SA 4.0)."
//| echo: true

#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
    // A imagem já está no diretório do capítulo (imagens/coins.png); a trilha
    // Python cuida do download/cache quando o notebook roda avulso.
    mm::Image img_coins_color = mm::read("imagens/coins.png");
    mm::Image img_coins_gray  = mm::gray(img_coins_color);

    std::cout << "Dimensões [y,x,c]: [" << img_coins_color.h << ", " 
              << img_coins_color.w << ", " << img_coins_color.channels << "]" << std::endl;
    mm::show(img_coins_color, MM_OUT);

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_coins_gray, "tmp/state/img_coins_gray_8.png");
// [pdi:state-io:end]
return 0;
}

Overwriting tmp/fig_04_coins.cpp


In [3]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_coins.cpp -o tmp/fig_04_coins -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_coins \
  && test -f "tmp/fig_04_coins.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_coins.png"

Dimensões [y,x,c]: [2560, 1920, 3]


In [4]:
try:
    mm.show(mm.read("tmp/fig_04_coins.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_coins.png (ver a versao Python)")

<Figure size 960x1280 with 1 Axes>

**Figura 4.1:** Imagem com moedas de vários tipos. Crédito: GAZI.MD.AHAD (CC BY-SA 4.0).


### 4.2.2 Preprocesamiento para Otsu

El método de Otsu depende de un histograma bien **bimodal**. La imagen de las monedas tiene iluminación no uniforme y monedas oscuras cerca del fondo, lo cual dificulta el proceso. En la ruta C++ se aplica **ecualización global del histograma** (`mm::equalize`) antes de la umbralización — la comparación CLAHE × Gaussiano y las curvas $\sigma_B^2(T)$ se encuentran en la ruta Python ([Figura 4.2](#fig-04-otsu-comparacao-histogramas)).

In [5]:
%%writefile tmp/fig_04_otsu_comparacao_histogramas.cpp
#define MM_OUT "tmp/fig_04_otsu_comparacao_histogramas.png"
//| label: fig-04-otsu-comparacao-histogramas
//| fig-cap: "CLAHE (realce adaptativo com limite de contraste) antes da limiarização de Otsu: original, realçado, histograma e binarização. (A trilha Python também traça as curvas de $\\sigma^2_B(T)$.)"
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
// [pdi:state-io:end]

    mm::Image img_clahe = mm::clahe(img_coins_gray, 2.0, 8);   // realce adaptativo com limite de contraste
    mm::Image img_gauss = mm::gaussian(img_clahe, 5, 0);

    mm::show(
        std::vector<mm::Image>{img_coins_gray, img_clahe, mm::histImg(img_clahe), mm::threshold(img_clahe)},
        MM_OUT,
        std::vector<std::string>{"Original", "CLAHE", "Histograma (CLAHE)", "Otsu"},
        4
    );

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_clahe, "tmp/state/img_clahe_12.png");
// [pdi:state-io:end]
return 0;
}

Overwriting tmp/fig_04_otsu_comparacao_histogramas.cpp


In [6]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_otsu_comparacao_histogramas.cpp -o tmp/fig_04_otsu_comparacao_histogramas -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_otsu_comparacao_histogramas \
  && test -f "tmp/fig_04_otsu_comparacao_histogramas.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_otsu_comparacao_histogramas.png"

[1] Original
[2] CLAHE
[3] Histograma (CLAHE)
[4] Otsu


In [7]:
try:
    mm.show(mm.read("tmp/fig_04_otsu_comparacao_histogramas.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_otsu_comparacao_histogramas.png (ver a versao Python)")

<Figure size 1044x450 with 1 Axes>

**Figura 4.2:** CLAHE (realce adaptativo com limite de contraste) antes da limiarização de Otsu: original, realçado, histograma e binarização. (A trilha Python também traça as curvas de $\\sigma^2_B(T)$.)


### 4.2.3 Resultado: CLAHE como Mejor Preprocesamiento

El análisis de la [Figura 4.2](#fig-04-otsu-comparacao-histogramas) indica que el **CLAHE** obtuvo el mayor valor de la varianza entre clases ($\sigma_B^2 \approx 2{,}47 \times 10^3$), con umbral óptimo $T^* = 122$. Aunque la combinación **CLAHE+Gaussiano** haya producido un resultado muy similar ($\sigma_B^2 \approx 2{,}43 \times 10^3$, $T^* = 123$), el criterio cuantitativo del método de Otsu favorece ligeramente el uso del CLAHE de forma aislada.

En términos visuales, las imágenes binarizadas obtenidas con CLAHE y CLAHE+Gaussiano son prácticamente equivalentes. La diferencia entre ambos enfoques se hace más evidente en el análisis de los histogramas y de los valores de $\sigma_B^2(T)$ que en la inspección directa de las segmentaciones resultantes. Así, la elección del CLAHE se basa principalmente en la maximización de la separación estadística entre las clases de fondo y objeto.


> ### 💡 Interpretación de los resultados
>
> Observe que los preprocesamientos con CLAHE y CLAHE+Gaussiano producen histogramas y umbrales óptimos muy próximos ($T^*=122$ y $T^*=123$). En consecuencia, las imágenes binarizadas resultantes también son bastante similares. En este caso, la decisión no se basa en diferencias visuales marcadas, sino en el criterio objetivo del método de Otsu: el mayor valor de $\sigma_B^2$ indica la mejor separación entre las clases.

## 4.3 Morfología Matemática

La **morfología matemática** es una teoría basada en conjuntos utilizada para analizar la forma y la estructura de objetos en imágenes. A diferencia de los filtros lineales presentados en el Capítulo 3, los operadores morfológicos son **no lineales**, pues se basan en operaciones de mínimo, máximo e inclusión espacial, en lugar de combinaciones lineales de intensidades. Estos operadores actúan sobre la vecindad de cada píxel mediante un **elemento estructurante** $\mathbb{B}$, responsable de definir la forma y el tamaño de la región analizada.

En imágenes binarias y en tonos de gris con elementos planos, el elemento estructurante trasladado a la posición $x$ se define espacialmente como:

$$
\mathbb{B}_x = \{ x + b \mid b \in \mathbb{B} \}
$$

En las regiones de borde de la imagen, parte del conjunto $\mathbb{B}_x$ puede extrapolar el dominio físico de la escena ($\mathbb{E}$). Para garantizar la consistencia matemática de los operadores primitivos en esas fronteras, se asume teóricamente que el espacio exterior al dominio de la imagen se rellena con el **elemento neutro** de la operación correspondiente (infinito positivo para la erosión e infinito negativo para la dilatación), impidiendo que el entorno externo corrompa las estructuras internas del objeto.

Cuando el elemento estructurante asocia pesos a sus elementos — es decir, $b: \mathbb{B} \to \mathbb{Z}$ — se denomina **función estructurante** o **elemento estructurante no plano**.

Desarrollada por Matheron y Serra en la década de 1960 para imágenes binarias y posteriormente extendida a tonos de gris, la morfología matemática fundamenta operadores como el gradiente morfológico, el *top-hat*, el *watershed* y la transformada de distancia, todos derivados de dos primitivos: la **erosión** y la **dilatación** [@matheron1975random; Serra (1982)].

### 4.3.1 Erosión y Dilatación

Los dos operadores primitivos se definen de manera unificada para imágenes en tonos de gris ($f: \mathbb{E} \to \mathbb{Z}$) y, por restricción al dominio $\{0,1\}$, también para imágenes binarias.

#### 4.3.1.1 Erosión

La **Erosión** de una imagen $f$ por una función estructurante $b: \mathbb{B} \to \mathbb{Z}$ se define formalmente por:

<a id="eq-04-erosao"></a>
$$
\varepsilon_b(f)(x) = (f \ominus b)(x) = \min_{z \in \mathbb{B}}\{\, f(x + z) - b(z) \,\}, \quad \forall\, x \in \mathbb{E} \tag{4.3}
$$


En la práctica, la erosión sustituye la intensidad del píxel $x$ por el valor mínimo resultante de la diferencia entre la imagen y el elemento estructurante en la vecindad definida por el dominio $\mathbb{B}$. Los valores positivos en los pesos de $b(z)$ fuerzan el resultado local hacia abajo, "excavando" el relieve de la imagen más profundamente e intensificando la erosión.

En el caso **plano** (donde los pesos son nulos dentro del dominio, es decir, $b \equiv 0$), la expresión se simplifica al mínimo local puro:

$$
\varepsilon_B(f)(x) = \min\{\, f(y) : y \in \mathbb{B}_x \,\}
$$

En imágenes binarias, esta operación equivale a exigir que el conjunto $\mathbb{B}$, trasladado a la coordenada $x$, esté **completamente contenido** en el objeto $A$:

$$
A \ominus \mathbb{B} = \{\, z \in \mathbb{E} \mid \mathbb{B}_z \subseteq A \,\}
$$

**Efecto Visual:** *Encoge* objetos y estructuras claras, eliminando protuberancias, picos brillantes o ruidos que sean geométricamente menores que el dominio $\mathbb{B}$.

#### 4.3.1.2 Implementación de la erosión

La versión didáctica `mm::ero0` implementa el caso particular de erosión con **elemento estructurante plano**. Para cada píxel $(y,x)$, la función recorre los vecinos espaciales permitidos por $B$ y almacena el menor valor encontrado en la imagen de entrada $f$, reproduciendo directamente la operación de mínimo local descrita en la [Equação 4.3](#eq-04-erosao) para $b \equiv 0$.

Observe que los valores de los vecinos siempre se leen de forma estática de la imagen original $f$; la matriz de salida $g$ se utiliza exclusivamente para registrar el mínimo acumulado de la vecindad actual. De esta forma, el resultado final es invariante respecto al orden de barrido de los píxeles (ya sea por filas o columnas).

La función auxiliar `_viz` calcula las coordenadas de los vecinos válidos dentro de los límites físicos de la imagen. En los bordes, la inicialización del acumulador en `255` emula con exactitud el relleno por elemento neutro exigido por la teoría. Por su parte, la función de interfaz `mm::ero` recurre a la implementación nativa y optimizada de OpenCV (`mm::ero`) cuando el elemento estructurante es plano, cambiando a la rutina general `mm::ero1` en caso de que el elemento posea pesos topográficos.

El ejemplo computacional siguiente ilustra la aplicación de un elemento estructurante en cruz (`mm::secross()`) destacado en la [Figura 4.3](#fig-04-elemento-cruz), comparando la ejecución de la variante didáctica en bucle (`mm::ero0`) con el motor computacional de OpenCV (`mm::ero`).

In [8]:
%%writefile tmp/fig_04_elemento_cruz.cpp
#define MM_OUT "tmp/fig_04_elemento_cruz.png"
//| label: fig-04-elemento-cruz
//| fig-cap: "Elemento estruturante em formato de cruz ($B_{\\text{cruz}}$) utilizado para conectividade-4."
//| echo: true
//| output: true

#include "morph.hpp"

int main() {
    mm::Image B_cruz = mm::secross();
    mm::drawImgPlt(B_cruz, MM_OUT, 40);
    return 0;
}

Overwriting tmp/fig_04_elemento_cruz.cpp


In [9]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_elemento_cruz.cpp -o tmp/fig_04_elemento_cruz -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_elemento_cruz \
  && test -f "tmp/fig_04_elemento_cruz.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_elemento_cruz.png"

0 1 0 
1 1 1 
0 1 0 


In [10]:
try:
    mm.show(mm.read("tmp/fig_04_elemento_cruz.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_elemento_cruz.png (ver a versao Python)")

<Figure size 450x450 with 1 Axes>

**Figura 4.3:** Elemento estruturante em formato de cruz ($B_{\\text{cruz}}$) utilizado para conectividade-4.


In [11]:
# La morph.hpp es header-only; a continuación, el helper _viz y el cuerpo de mm::ero0().
import re, pathlib
hpp = pathlib.Path("morph.hpp").read_text()
for nome, pat in [("_viz", r"template <class F>\ninline void _viz\(.*?\n\}"),
                  ("ero0", r"inline Image ero0\(.*?\n\}")]:
    m = re.search(pat, hpp, re.S)
    print(f"// ---- mm::{nome} ----")
    print(m.group(0) if m else f"({nome} não encontrada)")
    print()

// ---- mm::_viz ----
template <class F>
inline void _viz(const Image& f, const SE& B, int y, int x, F&& cb) {
    double oh = -B.h / 2.0 + 0.5;
    double ow = -B.w / 2.0 + 0.5;
    for (int by = 0; by < B.h; ++by)
        for (int bx = 0; bx < B.w; ++bx) {
            int vy = (int)(y + by + oh);
            int vx = (int)(x + bx + ow);
            if (vy >= 0 && vy < f.h && vx >= 0 && vx < f.w)
                cb(vy, vx, B.at(by, bx));
        }
}

// ---- mm::ero0 ----
inline Image ero0(const Image& f, SE Bc = SE::box(3)) {
    _require_gray(f, "ero0");
    Image g(f.h, f.w, 1);
    for (int y = 0; y < f.h; ++y)
        for (int x = 0; x < f.w; ++x) {
            int mn = 255;
            _viz(f, Bc, y, x, [&](int vy, int vx, int bv) {
                if (bv != 0 && (int)f.at(vy, vx) < mn) mn = f.at(vy, vx);
            });
            g.at(y, x) = (unsigned char)mn;
        }
    return g;
}



#### 4.3.1.3 Dilatación

La **Dilatación** de una imagen $f$ mediante una función estructurante $b: \mathbb{B} \to \mathbb{Z}$ se define formalmente como:

<a id="eq-04-dilatacao"></a>
$$
\delta_b(f)(x) = (f \oplus b)(x) = \max_{z \in \mathbb{B}}\{\, f(x - z) + b(z) \,\}, \quad \forall\, x \in \mathbb{E} \tag{4.4}
$$


En la práctica, la dilatación reemplaza la intensidad del píxel $x$ por el mayor valor resultante de la suma entre la imagen y el elemento estructurante en la vecindad definida. El argumento de inversión espacial ($x - z$) indica que la dilatación evalúa implícitamente el elemento transpuesto (reflejado) $\hat{b}$, propiedad fundamental para asegurar la dualidad matemática respecto a la erosión.

En el caso **plano** (donde los pesos son nulos dentro del dominio, es decir, $b \equiv 0$), la expresión se reduce al máximo local puro:

$$
\delta_B(f)(x) = \max\{\, f(y) : y \in \mathbb{B}_x \,\}
$$

En imágenes binarias, esta operación equivale a exigir que el conjunto reflejado $\hat{\mathbb{B}}$, trasladado a la coordenada $x$, tenga una **intersección no vacía** con el objeto $A$:

$$
A \oplus \mathbb{B} = \{\, z \in \mathbb{E} \mid \hat{\mathbb{B}}_z \cap A \neq \varnothing \,\}
$$

**Efecto Visual:** *Expande* las estructuras claras de la imagen, aumentando el relleno de objetos, conectando componentes cercanos y eliminando canales, fosas oscuras o valles que sean geométricamente menores que el dominio $\mathbb{B}$.

#### 4.3.1.4 Implementación de la dilatación

La versión didáctica `mm::dil0` implementa el caso particular de dilatación con **elemento estructurante plano**. Para cada píxel $(y,x)$, la función recorre los vecinos espaciales permitidos por $B$ y almacena el mayor valor encontrado en la imagen de entrada $f$, reproduciendo directamente la operación de máximo local para $b \equiv 0$.

Antes de iniciar el barrido espacial, el elemento estructurante sufre una reflexión geométrica mediante la instrucción `np.flip(Bc)` para construir explícitamente la matriz transpuesta $\hat{B}$ exigida por la teoría. En máscaras perfectamente simétricas (como cruces, cuadrados y discos centrados en el origen), esta reflexión no altera la disposición de los píxeles; no obstante, para elementos asimétricos, tal etapa es estrictamente necesaria para garantizar la equivalencia con las definiciones formales y salvaguardar las leyes de dualidad.

Así como se verificó en el operador de erosión, los valores de los vecinos siempre se leen de forma estática a partir de la matriz original $f$, mientras que la matriz de salida $g$ actúa puramente como el registrador del máximo acumulado de la vecindad. En las fronteras de la imagen, la inicialización del acumulador en `0` emula con exactitud el relleno externo por elemento neutro ($-\infty$, o cero en representaciones de 8 bits), garantizando que los bordes físicos de la escena sean dilatados en perfecta conformidad con el estándar adoptado por OpenCV.

El ejemplo computacional siguiente ilustra la aplicación práctica de un elemento en cruz (`mm::secross()`), validando la consistencia entre la lógica en bucles (`mm::dil0`) y el método nativo industrial (`mm::dil`).

In [12]:
# Cuerpo de mm::dil0() en morph.hpp (refleja el SE antes de barrer, como np.flip).
import re, pathlib
hpp = pathlib.Path("morph.hpp").read_text()
m = re.search(r"inline Image dil0\(.*?\n\}", hpp, re.S)
print(m.group(0) if m else "(dil0 não encontrada)")

inline Image dil0(const Image& f, SE Bc = SE::box(3)) {
    _require_gray(f, "dil0");
    SE B = Bc.reflected();
    Image g(f.h, f.w, 1);
    for (int y = 0; y < f.h; ++y)
        for (int x = 0; x < f.w; ++x) {
            int mx = 0;
            _viz(f, B, y, x, [&](int vy, int vx, int bv) {
                if (bv != 0 && (int)f.at(vy, vx) > mx) mx = f.at(vy, vx);
            });
            g.at(y, x) = (unsigned char)mx;
        }
    return g;
}


> ### 📝 Nota: El Enfrentamiento de los Signos ($f(x+z)$ vs $f(x-z)$)
>
> Compare las definiciones formales de la erosión ([Equação 4.3](#eq-04-erosao)) y de la dilatación ([Equação 4.4](#eq-04-dilatacao)). Considere un elemento estructurante asimétrico a la derecha $\mathbb{B}=\{0,1\}$ (origen y un píxel a la derecha) aplicado en la posición $x=10$.
>
> 1. **En la Erosión** ([Equação 4.3](#eq-04-erosao)):
>
>    $$
>    \min\{f(x+z)-b(z)\}
>    $$
>
>    * $z=0 \Rightarrow f(10+0)=\mathbf{f(10)}$
>    * $z=1 \Rightarrow f(10+1)=\mathbf{f(11)}$
>
>    El operador consulta el píxel actual ($10$) y el píxel a la derecha ($11$), preservando la orientación original de $\mathbb{B}$.
>
> 2. **En la Dilatación** ([Equação 4.4](#eq-04-dilatacao)):
>
>    $$
>    \max\{f(x-z)+b(z)\}
>    $$
>
>    * $z=0 \Rightarrow f(10-0)=\mathbf{f(10)}$
>    * $z=1 \Rightarrow f(10-1)=\mathbf{f(9)}$
>
>    Debido al signo negativo ($-z$), avanzar en el elemento estructurante corresponde a retroceder en la imagen, haciendo que la dilatación consulte el píxel a la izquierda ($9$).
>
> La función `_viz`, utilizada en `morph.py`, genera vecinos mediante desplazamientos aditivos de la forma $x+z$. Por ese motivo, la implementación de `mm::dil0` refleja previamente el elemento estructurante mediante `np.flip(B)`. Tras la reflexión, el barrido basado en $x+z$ pasa a acceder exactamente a los mismos puntos definidos por la expresión teórica $f(x-z)$ de la dilatación en [Equação 4.4](#eq-04-dilatacao).
>
> Para elementos estructurantes simétricos (como discos, cuadrados y cruces centradas), la reflexión no altera la máscara. En cambio, para elementos asimétricos, esta etapa es indispensable para que la implementación reproduzca correctamente la definición matemática de la dilatación y preserve la dualidad erosión–dilatación.

> ### 📝 Dualidad erosión–dilatación
>
> La erosión y la dilatación son **duales por complemento**. Esto significa que un operador puede obtenerse completamente a partir del otro, siempre que se actúe sobre el complemento de la imagen utilizando el elemento estructurante reflejado $\hat{B}$:
>
> $$
> (A \ominus B)^c = A^c \oplus \hat{B} \quad \Longleftrightarrow \quad A \ominus B = (A^c \oplus \hat{B})^c
> $$
>
> De manera análoga, la **dilatación también puede obtenerse a partir de la erosión**:
>
> $$
> (A \oplus B)^c = A^c \ominus \hat{B} \quad \Longleftrightarrow \quad A \oplus B = (A^c \ominus \hat{B})^c
> $$
>
> En términos prácticos, la erosión de un objeto puede obtenerse mediante la dilatación de su complemento, seguida de la complementación del resultado (y viceversa). En la implementación del paquete `morph.py`, las versiones didácticas `mm::ero0` y `mm::dil0` hacen explícita esta estructura mediante bucles (*loops*), mientras que `mm::ero` y `mm::dil` delegan las operaciones a OpenCV buscando una mayor eficiencia computacional.

> ### 📝 Condiciones de contorno e imágenes finitas
>
> En la morfología matemática clásica, definida sobre un dominio infinito (típicamente $\mathbb{Z}^2$), esta dualidad es exacta. En imágenes digitales, sin embargo, se trabaja con matrices finitas, y el resultado pasa a depender de la forma en que se tratan los píxeles ubicados fuera de la imagen.
>
> Para que las identidades de dualidad permanezcan válidas, el complemento debe definirse con respecto al mismo universo y las condiciones de contorno adoptadas para la erosión y para la dilatación deben ser complementarias entre sí. Por ejemplo, si la erosión asume que los píxeles externos pertenecen al objeto ($255$), entonces la dilatación aplicada al complemento debe asumir que esos mismos píxeles externos pertenecen al fondo ($0$).
>
> Cuando se utilizan diferentes estrategias de relleno (replicación, reflexión, valor constante, etc.), la dualidad teórica puede dejar de satisfacerse exactamente en las regiones cercanas a los bordes de la imagen.

Para ilustrar numéricamente los operadores morfológicos y la dualidad erosión–dilatación, la [Figura 4.4](#fig-04-ero-dil-didatico) presenta una imagen binaria de 10×10 procesada con un elemento estructurante en forma de "L". En la implementación de `morph.py`, el origen de $B$ se fija en el centro geométrico de la máscara —posición $(1,1)$ para un *kernel* de 3×3— y debe corresponder a un elemento activo para que la erosión se comporte correctamente (según lo discutido anteriormente). El elemento estructurante $B_L$ definido a continuación satisface esa condición. La [Figura 4.5](#fig-04-sim-04-erosao) complementa el análisis con un simulador interactivo de la erosión, permitiendo visualizar el desplazamiento del elemento estructurante sobre la imagen e identificar las posiciones en las que este permanece completamente contenido en el objeto.

In [13]:
%%writefile tmp/fig_04_ero_dil_didatico.cpp
#define MM_OUT "tmp/fig_04_ero_dil_didatico.png"
// Compile with: g++ -std=c++17 -O2 snippet.cpp -o snippet $(pkg-config --cflags --libs opencv4) -DMM_USE_OPENCV -I../morph/cpp && ./snippet
#include "morph.hpp"
#include <opencv2/opencv.hpp>
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
    //| label: fig-04-ero-dil-didatico
    //| fig-cap: "Erosão e dilatação em imagem binária 10×10 com elemento estruturante 'L' assimétrico 3×3. Validação da dualidade erosão–dilatação."
    //| echo: true
    //| output: true

    // A = np.array([...], dtype=np.uint8) * 255
    mm::Image A(10, 10, 1);
    int A_data[10][10] = {
        {0,0,0,0,0,0,0,0,0,0},
        {0,0,0,1,1,1,0,0,0,0},
        {0,0,1,1,1,1,1,0,0,0},
        {0,1,1,1,1,1,1,1,0,0},
        {0,1,1,1,1,1,1,1,0,0},
        {0,1,1,1,1,1,1,0,0,0},
        {0,0,1,1,1,1,1,1,0,0},
        {0,0,0,1,1,1,1,0,0,0},
        {0,0,0,0,1,0,0,0,0,0},
        {0,0,0,0,0,0,0,0,0,0}
    };
    for (int y = 0; y < 10; y++) {
        for (int x = 0; x < 10; x++) {
            A.at(y, x) = A_data[y][x] * 255;
        }
    }

    // B_L = np.array([[1,0,0],[1,1,0],[1,1,0]], dtype=np.uint8)   # 'L', origem no centro
    mm::SE B_L{{1,0,0},{1,1,0},{1,1,0}};
    // B_hat = np.array([[0,1,1],[0,1,1],[0,0,1]], dtype=np.uint8)   # B_L girado 180° (B̂)
    mm::SE B_hat{{0,1,1},{0,1,1},{0,0,1}};

    std::cout << "Elemento estruturante B_L:\n"; 
    // B_L is an SE; convert to Image for drawing
    mm::Image B_L_img = mm::sebox(0); // placeholder, will fix below
    B_L_img = mm::dil0(mm::Image(3, 3, 1), B_L); // get binary SE shape
    std::cout << mm::drawImg(B_L_img);
    std::cout << "Elemento estruturante refletido B̂:\n";
    mm::Image B_hat_img = mm::dil0(mm::Image(3, 3, 1), B_hat); // get binary SE shape
    std::cout << mm::drawImg(B_hat_img);

    mm::Image img_ero0 = mm::ero0(A, B_L);

    // Dualidade: (A ⊖ B)ᶜ == Aᶜ ⊕ B̂
    mm::Image A_c = mm::neg(A);
    mm::Image ero_c = mm::neg(img_ero0);
    mm::Image dil_Ac = mm::dil0(A_c, B_hat);

    // Check if ero_c and dil_Ac are equal
    bool equal = true;
    for (int i = 0; i < (int)ero_c.data.size() && equal; i++) {
        if (ero_c.data[i] != dil_Ac.data[i]) equal = false;
    }
    std::cout << "Dualidade (A ⊖ B)ᶜ == Aᶜ ⊕ B̂ : " << (equal ? "True" : "False") << "\n";

    mm::show(
        std::vector<mm::Image>{A, A_c, img_ero0, ero_c, dil_Ac},
        MM_OUT,
        std::vector<std::string>{"A", "Aᶜ", "A ⊖ B", "(A ⊖ B)ᶜ", "Aᶜ ⊕ B̂"},
        5
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(A, "tmp/fig_04_ero_dil_didatico_0.png");
mm::write(A_c, "tmp/fig_04_ero_dil_didatico_1.png");
mm::write(img_ero0, "tmp/fig_04_ero_dil_didatico_2.png");
mm::write(ero_c, "tmp/fig_04_ero_dil_didatico_3.png");
mm::write(dil_Ac, "tmp/fig_04_ero_dil_didatico_4.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_ero_dil_didatico.cpp


In [14]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_ero_dil_didatico.cpp -o tmp/fig_04_ero_dil_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_ero_dil_didatico \
  && test -f "tmp/fig_04_ero_dil_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_ero_dil_didatico.png"

Elemento estruturante B_L:
0 0 0 
0 0 0 
0 0 0 
Elemento estruturante refletido B̂:
0 0 0 
0 0 0 
0 0 0 
Dualidade (A ⊖ B)ᶜ == Aᶜ ⊕ B̂ : True
[1] A
[2] Aᶜ
[3] A ⊖ B
[4] (A ⊖ B)ᶜ
[5] Aᶜ ⊕ B̂


In [15]:
try:
    mm.show(
        [
            mm.read("tmp/fig_04_ero_dil_didatico_0.png"),
            mm.read("tmp/fig_04_ero_dil_didatico_1.png"),
            mm.read("tmp/fig_04_ero_dil_didatico_2.png"),
            mm.read("tmp/fig_04_ero_dil_didatico_3.png"),
            mm.read("tmp/fig_04_ero_dil_didatico_4.png"),
        ],
        titles=[
            'A',
            'Aᶜ',
            'A ⊖ B',
            '(A ⊖ B)ᶜ',
            'Aᶜ ⊕ B̂',
        ],
        cols=5,
        figsize=(15, 3),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_ero_dil_didatico_0.png (ver a versao Python)")

<Figure size 2250x450 with 5 Axes>

**Figura 4.4:** Erosão e dilatação em imagem binária 10×10 com elemento estruturante 


In [16]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-erosao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪨 Simulador: Erosión Morfológica</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">A ⊖ B_L · offsets via _viz</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Haz clic en una celda del lienzo para mover el kernel B_L o usa los controles deslizantes para probar la inclusión de contenidos.</p>

    <!-- Estatísticas Principais -->
    <div style="display:grid;grid-template-columns:repeat(3, minmax(0, 1fr));gap:10px;margin-bottom:16px;text-align:center;">
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:9.5px;color:#8a8371;text-transform:uppercase;margin-bottom:2px;">Posición X (col)</div>
        <div id="sim-04-erosao_mX" style="font-size:16px;font-weight:700;font-family:monospace;color:#2980b9;">4</div>
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:9.5px;color:#8a8371;text-transform:uppercase;margin-bottom:2px;">Posición Y (fila)</div>
        <div id="sim-04-erosao_mY" style="font-size:16px;font-weight:700;font-family:monospace;color:#2980b9;">4</div>
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:9.5px;color:#8a8371;text-transform:uppercase;margin-bottom:2px;">Píxel Erosión</div>
        <div id="sim-04-erosao_mEro" style="font-size:16px;font-weight:700;font-family:monospace;color:#27ae60;">255</div>
      </div>
    </div>

    <!-- Área Gráfica e Controles Lado a Lado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(260px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Canvas Interativo -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <canvas id="sim-04-erosao_Canvas" style="display:block;max-width:100%;height:auto;border-radius:8px;border:1px solid #e4dcc8;cursor:crosshair;background:#ffffff;margin:0 auto;"></canvas>
        <div style="font-size:10px;color:#8a8371;margin-top:8px;">🖱️ Haz clic en una celda para mover el kernel B_L</div>
      </div>

      <!-- Painel de Controles e Status -->
      <div style="display:flex;flex-direction:column;gap:12px;">
        
        <!-- Status de Inclusão -->
        <div id="sim-04-erosao_stBox" style="border-radius:10px;border:1.5px solid #27ae60;padding:10px 12px;background:#eafaf1;transition:all 0.15s ease;">
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:4px;">
            <span style="font-size:16px;">🟢</span>
            <div id="sim-04-erosao_stTitle" style="font-size:11px;font-weight:700;color:#27ae60;">Éxito: ¡contenido!</div>
          </div>
          <div id="sim-04-erosao_stDesc" style="font-size:10.5px;color:#27ae60;line-height:1.4;">El píxel recibe 1 (255) en la imagen erosionada.</div>
        </div>

        <!-- Sliders de Posição -->
        <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 12px;">
          <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Controles de Coordenada</span>
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:6px;">
            <label for="sim-04-erosao_slX" style="font-size:10.5px;color:#8a8371;width:40px;flex-shrink:0;">X (col)</label>
            <input type="range" id="sim-04-erosao_slX" min="0" max="9" step="1" value="4" style="flex:1;cursor:pointer;">
            <span id="sim-04-erosao_vX" style="font-size:11px;font-family:monospace;font-weight:700;min-width:14px;text-align:right;">4</span>
          </div>
          <div style="display:flex;align-items:center;gap:8px;">
            <label for="sim-04-erosao_slY" style="font-size:10.5px;color:#8a8371;width:40px;flex-shrink:0;">Y (fila)</label>
            <input type="range" id="sim-04-erosao_slY" min="0" max="9" step="1" value="4" style="flex:1;cursor:pointer;">
            <span id="sim-04-erosao_vY" style="font-size:11px;font-family:monospace;font-weight:700;min-width:14px;text-align:right;">4</span>
          </div>
        </div>

        <button id="sim-04-erosao_rstBtn" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;width:100%;transition:all 0.15s ease;">↩ Restablecer Posición (4, 4)</button>

        <!-- Kernel B_L Informativo -->
        <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 12px;text-align:center;">
          <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Kernel B_L (3×3)</span>
          <div style="display:inline-grid;grid-template-columns:repeat(3, 30px);gap:2px;margin-bottom:6px;justify-content:center;">
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ebf4fd;border:1px solid #2980b9;border-radius:4px;color:#2980b9;">★</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          </div>
          <div id="sim-04-erosao_offsetInfo" style="font-size:10px;font-family:monospace;color:#5e5a4a;line-height:1.5;background:#ffffff;border:1px solid #e4dcc8;border-radius:6px;padding:6px 8px;margin-top:6px;text-align:left;"></div>
        </div>

      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSim04Ero(root){
    if (!root || root.dataset.sim04EroInit) return;
    root.dataset.sim04EroInit = "1";

    const ero_COLS = 10, ero_ROWS = 10, ero_CELL = 32, ero_PAD = 14;
    const ero_W = ero_COLS * ero_CELL + ero_PAD * 2, ero_H = ero_ROWS * ero_CELL + ero_PAD * 2;
    const ero_cv = root.querySelector('#sim-04-erosao_Canvas');
    ero_cv.width = ero_W; 
    ero_cv.height = ero_H;
    const ero_ctx = ero_cv.getContext('2d');

    const ero_A = [
      [0,0,0,0,0,0,0,0,0,0],
      [0,0,0,1,1,1,0,0,0,0],
      [0,0,1,1,1,1,1,0,0,0],
      [0,1,1,1,1,1,1,1,0,0],
      [0,1,1,1,1,1,1,1,0,0],
      [0,1,1,1,1,1,1,0,0,0],
      [0,0,1,1,1,1,1,1,0,0],
      [0,0,0,1,1,1,1,0,0,0],
      [0,0,0,0,1,0,0,0,0,0],
      [0,0,0,0,0,0,0,0,0,0]
    ];

    const ero_B = [
      [1,0,0],
      [1,1,0],
      [1,1,0]
    ];
    const ero_Bh = 3, ero_Bw = 3;

    function ero_vizOffsets(){
      const offs = [];
      for(let by=0; by<ero_Bh; by++) {
        for(let bx=0; bx<ero_Bw; bx++){
          const dr = by - Math.floor(ero_Bh/2);
          const dc = bx - Math.floor(ero_Bw/2);
          offs.push({dr, dc, bv: ero_B[by][bx], by, bx});
        }
      }
      return offs;
    }
    
    const ero_OFFSETS = ero_vizOffsets();
    const ero_ACTIVE  = ero_OFFSETS.filter(o => o.bv === 1);
    const ero_INACTIVE= ero_OFFSETS.filter(o => o.bv === 0);

    function ero_computeEro(){
      const e = Array.from({length:ero_ROWS}, () => new Array(ero_COLS).fill(0));
      for(let y=0; y<ero_ROWS; y++) {
        for(let x=0; x<ero_COLS; x++) {
          let ok = true;
          for(const {dr,dc,bv} of ero_OFFSETS){
            if(!bv) continue;
            const vy = y + dr, vx = x + dc;
            if(vy < 0 || vy >= ero_ROWS || vx < 0 || vx >= ero_COLS || !ero_A[vy][vx]) { 
              ok = false; 
              break; 
            }
          }
          e[y][x] = ok ? 1 : 0;
        }
      }
      return e;
    }
    
    const ero_ERO = ero_computeEro();
    let ero_cx = 4, ero_cy = 4;

    function ero_updateOffsetInfo(){
      const lines = ero_ACTIVE.map(o => {
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        const inBounds = vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS;
        const inA = inBounds && ero_A[vy][vx];
        const mark = inA ? '✓' : '✗';
        return 'B[' + o.by + '][' + o.bx + '] → (' + vy + ',' + vx + ') ' + mark;
      });
      
      root.querySelector('#sim-04-erosao_offsetInfo').innerHTML =
        '<span style="color:#8a8371; font-size:9.5px">offsets ativos @ (y=' + ero_cy + ', x=' + ero_cx + '):</span><br>' +
        lines.map(l => '<span style="color:' + (l.endsWith('✓') ? '#27ae60' : '#c0392b') + '">' + l + '</span>').join('<br>');
    }

    function ero_draw(){
      ero_ctx.clearRect(0, 0, ero_W, ero_H);

      const activeMap = new Map();
      for(const o of ero_ACTIVE){
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        if(vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS) activeMap.set(vy + ',' + vx, ero_A[vy][vx]);
      }
      const inactiveSet = new Set();
      for(const o of ero_INACTIVE){
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        if(vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS) inactiveSet.add(vy + ',' + vx);
      }

      const contained = ero_ACTIVE.every(o => {
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        return vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS && ero_A[vy][vx];
      });

      for(let r=0; r<ero_ROWS; r++) {
        for(let c=0; c<ero_COLS; c++) {
          const x = ero_PAD + c * ero_CELL, y = ero_PAD + r * ero_CELL;
          const key = r + ',' + c;
          const isActive = activeMap.has(key);
          const isInactive = !isActive && inactiveSet.has(key);
          const isCenter = r === ero_cy && c === ero_cx;
          const inA = ero_A[r][c];
          const inE = ero_ERO[r][c];
          
          let fill, stroke, lw=0.5, dash=[];

          if(isActive){
            const inside = activeMap.get(key);
            fill = inside ? 'rgba(243, 156, 18, 0.35)' : 'rgba(192, 57, 43, 0.35)';
            stroke = inside ? '#b9770e' : '#c0392b';
            lw = 2;
          } else if(isCenter){
            fill = 'rgba(41, 128, 185, 0.25)';
            stroke = '#2980b9'; lw = 2; dash = [4,3];
          } else if(isInactive){
            fill = 'rgba(235, 244, 253, 0.4)';
            stroke = '#b9770e'; lw = 1; dash = [3,3];
          } else if(inE){
            fill = 'rgba(39, 174, 96, 0.15)';
            stroke = '#27ae60';
          } else if(inA){
            fill = 'rgba(142, 68, 173, 0.15)';
            stroke = '#8e44ad';
          } else {
            fill = '#fafaf7';
            stroke = '#e4dcc8';
          }

          ero_ctx.setLineDash(dash);
          ero_ctx.fillStyle = fill; 
          ero_ctx.fillRect(x+1, y+1, ero_CELL-2, ero_CELL-2);
          ero_ctx.strokeStyle = stroke; 
          ero_ctx.lineWidth = lw; 
          ero_ctx.strokeRect(x+0.5, y+0.5, ero_CELL-1, ero_CELL-1);
          ero_ctx.setLineDash([]);

          ero_ctx.textAlign = 'center'; 
          ero_ctx.textBaseline = 'middle';
          
          if(isCenter && !isActive){
            ero_ctx.fillStyle = '#2980b9';
            ero_ctx.font = '13px monospace';
            ero_ctx.fillText('★', x+ero_CELL/2, y+ero_CELL/2);
          } else if(isActive){
            const inside = activeMap.get(key);
            ero_ctx.fillStyle = inside ? '#7d5a00' : '#78281f';
            ero_ctx.font = 'bold 11px monospace';
            ero_ctx.fillText(inA ? '1' : '0', x+ero_CELL/2, y+ero_CELL/2);
          } else if(inA){
            ero_ctx.fillStyle = '#4a235a';
            ero_ctx.font = '11px monospace';
            ero_ctx.fillText('1', x+ero_CELL/2, y+ero_CELL/2);
          } else {
            ero_ctx.fillStyle = '#8a8371';
            ero_ctx.font = '11px monospace';
            ero_ctx.fillText('0', x+ero_CELL/2, y+ero_CELL/2);
          }

          if(r===0){ero_ctx.fillStyle='#8a8371'; ero_ctx.font='9px monospace'; ero_ctx.textAlign='center'; ero_ctx.fillText(c, x+ero_CELL/2, ero_PAD/2);}
          if(c===0){ero_ctx.fillStyle='#8a8371'; ero_ctx.font='9px monospace'; ero_ctx.textAlign='center'; ero_ctx.fillText(r, ero_PAD/2, y+ero_CELL/2);}
        }
      }

      root.querySelector('#sim-04-erosao_mX').textContent = ero_cx;
      root.querySelector('#sim-04-erosao_mY').textContent = ero_cy;
      const ev = ero_ERO[ero_cy][ero_cx];
      const mEro = root.querySelector('#sim-04-erosao_mEro');
      mEro.textContent = ev ? '255' : '0';
      mEro.style.color = ev ? '#27ae60' : '#c0392b';
      root.querySelector('#sim-04-erosao_vX').textContent = ero_cx;
      root.querySelector('#sim-04-erosao_vY').textContent = ero_cy;
      root.querySelector('#sim-04-erosao_slX').value = ero_cx;
      root.querySelector('#sim-04-erosao_slY').value = ero_cy;

      const box = root.querySelector('#sim-04-erosao_stBox');
      const title = root.querySelector('#sim-04-erosao_stTitle');
      const desc = root.querySelector('#sim-04-erosao_stDesc');
      
      if(contained){
        box.style.background = '#eafaf1'; box.style.borderColor = '#27ae60';
        title.textContent = 'Sucesso: contido!'; title.style.color = '#27ae60';
        desc.textContent = 'Pixel recebe 1 (255) na imagem erodida.'; desc.style.color = '#27ae60';
      } else {
        box.style.background = '#fdecea'; box.style.borderColor = '#c0392b';
        title.textContent = 'Aviso: B_L sai do objeto!'; title.style.color = '#c0392b';
        desc.textContent = 'Pixel recebe 0 na imagem erodida.'; desc.style.color = '#c0392b';
      }
      
      ero_updateOffsetInfo();
    }

    ero_cv.addEventListener('click', function(e){
      const rect = ero_cv.getBoundingClientRect();
      const sx = ero_W / rect.width, sy = ero_H / rect.height;
      const c = Math.floor(((e.clientX - rect.left) * sx - ero_PAD) / ero_CELL);
      const r = Math.floor(((e.clientY - rect.top) * sy - ero_PAD) / ero_CELL);
      if(c >= 0 && c < ero_COLS && r >= 0 && r < ero_ROWS){
        ero_cx = c; ero_cy = r; 
        ero_draw();
      }
    });
    
    root.querySelector('#sim-04-erosao_slX').addEventListener('input', function(){ ero_cx = +this.value; ero_draw(); });
    root.querySelector('#sim-04-erosao_slY').addEventListener('input', function(){ ero_cy = +this.value; ero_draw(); });
    root.querySelector('#sim-04-erosao_rstBtn').addEventListener('click', function(){ ero_cx = 4; ero_cy = 4; ero_draw(); });
    
    ero_draw();
  }

  function tryInitSim04Ero(){
    var root = document.getElementById('sim-04-erosao');
    if (root) initSim04Ero(root); else setTimeout(tryInitSim04Ero, 200);
  }
  tryInitSim04Ero();
})();
</script>
</div>
""")

**Figura 4.5:** Simulador: Erosión Morfológica (A ⊖ B_L)


<figure id="fig-04-sim-04-erosao">
  <img src="imagens/fig-04-sim-04-erosao.png" alt=" Simulador: Erosión Morfológica (A ⊖ B_L) " style="max-width:80%" />
  <figcaption><strong>Figura 4.5:</strong>  Simulador: Erosión Morfológica (A ⊖ B_L) </figcaption>
</figure>

### 4.3.2 Apertura y Cierre

Combinando erosión y dilatación se obtienen dos operadores de gran utilidad práctica: la **apertura** y el **cierre**, definidos por las Ecuaciones [Equação 4.5](#eq-04-abertura) y [Equação 4.6](#eq-04-fechamento). Sus principales efectos se resumen en la [Tabela 4.1](#tbl-04-open-close).

**Apertura** (*opening*) — erosión seguida de dilatación por el mismo $B$:

<a id="eq-04-abertura"></a>
$$
A \circ B = (A \ominus B) \oplus B \tag{4.5}
$$


**Cierre** (*closing*) — dilatación seguida de erosión por el mismo $B$:

<a id="eq-04-fechamento"></a>
$$
A \bullet B = (A \oplus B) \ominus B \tag{4.6}
$$


En la práctica, `mm::open` y `mm::close` aplican el mismo elemento estructurante en las dos etapas. Para elementos estructurantes simétricos (los más comunes), esta implementación coincide con la definición matemática presentada anteriormente.

<a id="tbl-04-open-close"></a>

**Tabela 4.1:** Propiedades de apertura y cierre.

| Operador | Secuencia | Efecto principal |
|:--------:|:----------|:-----------------|
| Apertura $A \circ B$ | erosión → dilatación | Elimina estructuras incapaces de contener el elemento estructurante; suaviza contornos externos |
| Cierre $A \bullet B$ | dilatación → erosión | Rellena agujeros más pequeños que $B$; suaviza contornos internos |


**Propiedad importante:** ambos son **idempotentes**. Por ejemplo,

$$
(A \circ B) \circ B = A \circ B,
$$

es decir, después de la primera aplicación, nuevas aplicaciones del mismo operador no alteran más el resultado.

#### 4.3.2.1 Implementación de la apertura y el cierre

A diferencia de la erosión y la dilatación, la apertura y el cierre no introducen
nuevos mecanismos computacionales. Ambos se obtienen mediante la composición secuencial
de los operadores primitivos ya presentados:

```python
def open0(f, B):
    return mm.dil0(mm.ero0(f, B), B)

def close0(f, B):
    return mm.ero0(mm.dil0(f, B), B)
```

La función `mm::open` delega la operación a `mm::open(f, B)`,
mientras que `mm::close` utiliza `mm::close(f, B)`, produciendo
el mismo resultado de forma más eficiente.

La apertura hereda de la erosión la capacidad de eliminar estructuras más pequeñas que el
elemento estructurante y de la dilatación la restauración parcial de las regiones
preservadas. El cierre realiza el proceso inverso: primero expande los
objetos y luego restaura sus dimensiones originales, rellenando huecos y
agujeros más pequeños que el elemento estructurante.

#### 4.3.2.2 Filtro Secuencial Alternado

En la práctica, la apertura y el cierre se aplican frecuentemente en secuencia para eliminar simultáneamente el ruido externo y rellenar los huecos internos. La función `mm::asf` (*Filtro Secuencial Alternado*) generaliza esta estrategia al aplicar aperturas y cierres alternadamente con elementos estructurantes progresivamente más grandes. Las secuencias disponibles se presentan en la [Tabela 4.2](#tbl-04-asf).

<a id="tbl-04-asf"></a>

**Tabela 4.2:** Secuencias del filtro secuencial alternado mm.asf.

| Secuencia | Orden                              | Uso típico                                               |
| :-------: | :--------------------------------- | :------------------------------------------------------- |
|   `'OC'`  | apertura → cierre                  | elimina el ruido externo antes de rellenar pequeños huecos |
|   `'CO'`  | cierre → apertura                  | rellena pequeños huecos antes de eliminar el ruido externo |
|  `'OCO'`  | apertura → cierre → apertura       | enfatiza la eliminación del ruido externo               |
|  `'COC'`  | cierre → apertura → cierre         | enfatiza el relleno de huecos y lagunas                 |


El parámetro `n` controla el número de escalas utilizadas por el filtro. En cada iteración $i$, el elemento estructurante se amplía mediante la suma de Minkowski (`mm::sesum(b, i)`), produciendo una secuencia de filtros morfológicos cada vez más abarcativos. A diferencia de una única apertura o cierre con un elemento estructurante grande, el ASF realiza una suavización progresiva en múltiples escalas, preservando mejor la geometría de los objetos relevantes mientras elimina estructuras menores. La [Figura 4.6](#fig-04-open-close) presenta un ejemplo de aplicación de la apertura, el cierre y el ASF en la imagen de las monedas.

In [17]:
%%writefile tmp/fig_04_open_close.cpp
#define MM_OUT "tmp/fig_04_open_close.png"
#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_clahe = mm::_read_state("tmp/state/img_clahe_12.png");
// [pdi:state-io:end]

    mm::Image img_bin = mm::threshold(img_clahe);
    mm::Image B_disk = mm::sedisk(19);

    mm::Image img_open = mm::open(img_bin, B_disk);             // erosão → dilatação
    mm::Image img_close = mm::close(img_bin, B_disk);            // dilatação → erosão
    mm::Image img_oc = mm::close(img_open, B_disk);           // abertura seguida de fechamento
    mm::Image img_asf = mm::asf(img_bin, "OC", mm::sedisk(3), 7);  
    // ASF: disco base 3×3, cresce a cada iteração

    mm::show(
        std::vector<mm::Image>{img_bin, img_open, img_close, img_oc, img_asf},
        MM_OUT,
        std::vector<std::string>{"Binarização Otsu", "Abertura (A∘B)", "Fechamento (A∙B)",
            "Abertura→Fechamento", "ASF-OC (n=7)"},
        5
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_bin, "tmp/fig_04_open_close_0.png");
mm::write(img_open, "tmp/fig_04_open_close_1.png");
mm::write(img_close, "tmp/fig_04_open_close_2.png");
mm::write(img_oc, "tmp/fig_04_open_close_3.png");
mm::write(img_asf, "tmp/fig_04_open_close_4.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_open_close.cpp


In [18]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_open_close.cpp -o tmp/fig_04_open_close -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_open_close \
  && test -f "tmp/fig_04_open_close.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_open_close.png"

[1] Binarização Otsu
[2] Abertura (A∘B)
[3] Fechamento (A∙B)
[4] Abertura→Fechamento
[5] ASF-OC (n=7)


In [19]:
try:
    mm.show(
        [
            mm.read("tmp/fig_04_open_close_0.png"),
            mm.read("tmp/fig_04_open_close_1.png"),
            mm.read("tmp/fig_04_open_close_2.png"),
            mm.read("tmp/fig_04_open_close_3.png"),
            mm.read("tmp/fig_04_open_close_4.png"),
        ],
        titles=[
            'Binarização Otsu',
            'Abertura (A∘B)',
            'Fechamento (A∙B)',
            'Abertura→Fechamento',
            'ASF-OC (n=7)',
        ],
        cols=5,
        figsize=(15, 12),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_open_close_0.png (ver a versao Python)")

<Figure size 2250x1800 with 5 Axes>

**Figura 4.6:** Abertura, fechamento, composição e filtro sequencial alternado aplicados à binarização Otsu das moedas (pré-processadas por realce de contraste). Elemento estruturante: disco 13×13.


### 4.3.3 Operadores Geodésicos

Los operadores geodésicos introducen una restricción adicional a los operadores morfológicos clásicos mediante una imagen de control denominada **máscara** $g$. En lugar de permitir que la erosión o la dilatación se propaguen libremente por la imagen, el resultado de cada iteración se limita punto a punto por los valores de la máscara, restringiendo la evolución de la operación a las regiones permitidas.

#### 4.3.3.1 Dilatación Geodésica

La **dilatación geodésica** de una imagen marcador $f$ bajo una imagen máscara $g$, utilizando un elemento estructurante plano $b$, se define por:

<a id="eq-04-cdil"></a>
$$
f \oplus_g b = (f \oplus b) \wedge g, \tag{4.7}
$$


donde $\wedge$ representa el mínimo punto a punto.

En otras palabras, se realiza inicialmente una dilatación convencional sobre el marcador y, a continuación, el resultado se restringe mediante la máscara $g$. De esta manera, la propagación nunca puede superar las regiones permitidas por la máscara.

La formulación clásica de la dilatación geodésica presupone que el marcador esté contenido en la máscara, es decir, $f \le g$, garantizando que la evolución de la operación permanezca siempre limitada por la máscara.

#### 4.3.3.2 Implementación de la dilatación geodésica

La función `mm::cdil` implementa directamente este operador y permite ejecutar múltiples iteraciones consecutivas:

```python
def cdil(f, g, b=np.zeros((3,3),dtype='uint8'), n=1):
    """Dilatação geodésica do marcador f sob a máscara g."""
    y = f.copy()
    for _ in range(n):
        y = np.minimum(mm.dil(y, b), g)
    return y
```

La instrucción `np.minimum(mm::dil(y, b), g)` implementa exactamente la definición matemática de la dilatación geodésica, es decir, $(y \oplus b)\wedge g$.

Cuando $n=1$, la función ejecuta una única dilatación geodésica. Para $n>1$, el resultado de cada etapa se convierte en el marcador de la etapa siguiente, produciendo una propagación progresiva controlada por la máscara.

El simulador interactivo [Figura 4.7](#fig-04-sim-04-cdil) permite seguir, paso a paso, la propagación del marcador $f$ a lo largo de los corredores del laberinto. En cada iteración de `mm::cdil`, el frente de dilatación avanza hacia las celdas vecinas libres —aquellas en las que $g = 1$—, mientras que las paredes ($g = 0$) permanecen intransitables. El número de pasos necesarios para que el marcador alcance la salida corresponde exactamente a la longitud geodésica del camino más corto dentro de la máscara, evidenciando la conexión directa entre la dilatación geodésica iterada y la noción de distancia en grafos.

In [20]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-cdil" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🗺️ Simulador: Dilatación Geodésica en el Laberinto</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">δ_g^(n)(f)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    
    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:14px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:14px;">
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#2c2c2a;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Pared</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#eafaf1;border:1px solid #a3e4d7;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Camino libre (g)</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#27ae60;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Marcador f</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#a3e4d7;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Propagación</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#fef9e7;border:1px solid #f8c471;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Salida</span></div>
    </div>

    <!-- Canvas Centralizado -->
    <div style="display:flex;justify-content:center;margin-bottom:14px;">
      <canvas id="sim-04-cdil_Canvas" width="435" height="435" style="width:100%;max-width:435px;display:block;border-radius:12px;border:1px solid #e4dcc8;background:#ffffff;"></canvas>
    </div>

    <!-- Controles e Informações do Passo -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;gap:6px;flex-wrap:wrap;">
        <button id="sim-04-cdil_btnReset" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">↺ Reiniciar</button>
        <button id="sim-04-cdil_btnPrev" disabled style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">‹ Anterior</button>
        <button id="sim-04-cdil_btnNext" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">Siguiente ›</button>
        <button id="sim-04-cdil_btnPlay" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">▶ Animar</button>
      </div>
      <div id="sim-04-cdil_info" style="font-size:11px;font-family:monospace;color:#26241d;flex:1;min-width:180px;text-align:right;">Paso 0 — marcador inicial f (entrada)</div>
    </div>

  </div>
</div>

<script>
(function() {
  function initSim04Cdil(root){
    if (!root || root.dataset.sim04CdilInit) return;
    root.dataset.sim04CdilInit = "1";

    const G = [
      [0,0,0,0,0,0,0,0,0,0,0,0],
      [0,1,1,1,0,1,1,1,1,1,1,0],
      [0,0,0,1,0,1,0,0,0,0,1,0],
      [0,1,0,1,0,1,0,1,1,0,1,0],
      [0,1,0,1,1,1,0,1,0,0,1,0],
      [0,1,0,0,0,0,0,1,0,1,1,0],
      [0,1,1,1,1,1,1,1,0,1,0,0],
      [0,0,0,0,0,0,1,0,0,1,0,0],
      [0,1,1,1,1,0,1,1,1,1,0,0],
      [0,1,0,0,1,0,0,0,0,1,0,0],
      [0,1,1,1,1,1,1,1,1,1,0,0],
      [0,0,0,0,0,0,0,0,0,0,0,0]
    ];
    const ROWS = G.length, COLS = G[0].length;
    const START = [1,1];
    const EXIT  = [10,9];

    function dilate(prev, mask) {
      const next = prev.map(r => [...r]);
      const dirs = [[-1,0],[1,0],[0,-1],[0,1]];
      for (let r=0; r<ROWS; r++)
        for (let c=0; c<COLS; c++)
          if (prev[r][c]) {
            for (const [dr,dc] of dirs) {
              const nr=r+dr, nc=c+dc;
              if (nr>=0&&nr<ROWS&&nc>=0&&nc<COLS&&mask[nr][nc])
                next[nr][nc]=1;
            }
          }
      return next;
    }

    function emptyGrid() { return Array.from({length:ROWS},()=>new Array(COLS).fill(0)); }

    let steps = [];
    function buildSteps() {
      steps = [];
      let cur = emptyGrid();
      cur[START[0]][START[1]] = 1;
      steps.push(cur.map(r=>[...r]));
      let prev = null;
      while (JSON.stringify(cur) !== JSON.stringify(prev)) {
        prev = cur.map(r=>[...r]);
        cur = dilate(cur, G);
        steps.push(cur.map(r=>[...r]));
        if (steps.length > 80) break;
      }
    }

    buildSteps();
    let idx = 0, playing = false, timer = null;

    const cv = root.querySelector('#sim-04-cdil_Canvas');
    const ctx = cv.getContext('2d');

    const C = {
      wall:    '#2c2c2a',
      free:    '#eafaf1',
      freeBd:  '#a3e4d7',
      marker:  '#27ae60',
      trail:   '#a3e4d7',
      exit:    '#fef9e7',
      exitBd:  '#f8c471',
      bg:      '#fafaf7',
      text:    '#5e5a4a',
      grid:    '#e4dcc8',
    };

    function draw() {
      const W = cv.width, H = cv.height;
      const cw = W / COLS, ch = H / ROWS;
      ctx.clearRect(0,0,W,H);
      ctx.fillStyle = C.bg;
      ctx.fillRect(0,0,W,H);

      const cur = steps[idx];
      const prev = idx > 0 ? steps[idx-1] : null;

      for (let r=0; r<ROWS; r++) {
        for (let c=0; c<COLS; c++) {
          const x = c*cw, y = r*ch;
          const isExit = r===EXIT[0]&&c===EXIT[1];

          if (!G[r][c]) {
            ctx.fillStyle = C.wall;
            ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
          } else {
            if (isExit) {
              ctx.fillStyle = C.exit;
              ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
              ctx.strokeStyle = C.exitBd;
              ctx.lineWidth = 1.5;
              ctx.strokeRect(x+1.5,y+1.5,cw-3,ch-3);
            } else {
              ctx.fillStyle = C.free;
              ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
              ctx.strokeStyle = C.freeBd;
              ctx.lineWidth = 0.5;
              ctx.strokeRect(x+0.5,y+0.5,cw-1,ch-1);
            }
            if (prev && prev[r][c]) {
              ctx.fillStyle = C.trail;
              ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
            }
            if (cur[r][c]) {
              ctx.fillStyle = C.marker;
              ctx.fillRect(x+2,y+2,cw-4,ch-4);
            }
          }
        }
      }

      ctx.strokeStyle = C.grid;
      ctx.lineWidth = 0.5;
      for (let r=0; r<=ROWS; r++) { ctx.beginPath(); ctx.moveTo(0,r*ch); ctx.lineTo(W,r*ch); ctx.stroke(); }
      for (let c=0; c<=COLS; c++) { ctx.beginPath(); ctx.moveTo(c*cw,0); ctx.lineTo(c*cw,H); ctx.stroke(); }

      ctx.fillStyle = '#ffffff';
      ctx.font = 'bold 11px monospace';
      ctx.textAlign = 'center';
      ctx.textBaseline = 'middle';
      const [sr,sc] = START;
      ctx.fillText('E', (sc+0.5)*cw, (sr+0.5)*ch);
      const [er,ec] = EXIT;
      ctx.fillStyle = '#b9770e';
      ctx.fillText('S', (ec+0.5)*cw, (er+0.5)*ch);

      const reached = steps[idx][EXIT[0]][EXIT[1]];
      let msg = '';
      if (idx === 0) msg = 'Passo 0 — marcador inicial f (entrada)';
      else if (reached) msg = 'Passo ' + idx + ' — saída alcançada! 🎉';
      else msg = 'Passo ' + idx + ' — propagação geodésica';

      root.querySelector('#sim-04-cdil_info').textContent = msg;
      root.querySelector('#sim-04-cdil_btnPrev').disabled = idx === 0;
      root.querySelector('#sim-04-cdil_btnNext').disabled = idx === steps.length-1;
    }

    function step(d) {
      idx = Math.max(0, Math.min(steps.length-1, idx+d));
      draw();
    }

    function reset() {
      stopPlay();
      idx = 0;
      draw();
    }

    function togglePlay() {
      playing ? stopPlay() : startPlay();
    }

    function startPlay() {
      playing = true;
      root.querySelector('#sim-04-cdil_btnPlay').textContent = '⏸ Pausar';
      timer = setInterval(() => {
        if (idx >= steps.length-1) { stopPlay(); return; }
        idx++;
        draw();
      }, 350);
    }

    function stopPlay() {
      playing = false;
      clearInterval(timer);
      root.querySelector('#sim-04-cdil_btnPlay').textContent = '▶ Animar';
    }

    root.querySelector('#sim-04-cdil_btnReset').addEventListener('click', reset);
    root.querySelector('#sim-04-cdil_btnPrev').addEventListener('click', () => step(-1));
    root.querySelector('#sim-04-cdil_btnNext').addEventListener('click', () => step(1));
    root.querySelector('#sim-04-cdil_btnPlay').addEventListener('click', togglePlay);

    draw();
  }

  function tryInitSim04Cdil(){
    var root = document.getElementById('sim-04-cdil');
    if (root) initSim04Cdil(root); else setTimeout(tryInitSim04Cdil, 200);
  }
  tryInitSim04Cdil();
})();
</script>
""")

**Figura 4.7:** Simulador interactivo de dilatación geodésica: el marcador f (verde) se propaga paso a paso por los caminos libres de la máscara g, sin atravesar paredes.


<figure id="fig-04-sim-04-cdil">
  <img src="imagens/fig-04-sim-04-cdil.png" alt=" Simulador interactivo de dilatación geodésica: el marcador f (verde) se propaga paso a paso por los caminos libres de la máscara g, sin atravesar paredes. " style="max-width:80%" />
  <figcaption><strong>Figura 4.7:</strong>  Simulador interactivo de dilatación geodésica: el marcador f (verde) se propaga paso a paso por los caminos libres de la máscara g, sin atravesar paredes. </figcaption>
</figure>

#### 4.3.3.3 Erosión Geodésica

De forma dual, la **erosión geodésica** de una imagen marcador $f$ bajo una imagen máscara $g$ se define por:

<a id="eq-04-cero"></a>
$$
f \ominus_g b = (f \ominus b) \vee g, \tag{4.8}
$$


donde $\vee$ representa el operador de máximo punto a punto.

En este caso, la erosión convencional del marcador es seguida por una restricción inferior impuesta por la máscara. Así, ningún píxel del resultado puede asumir un valor inferior al correspondiente píxel de la máscara.

La formulación clásica de la erosión geodésica presupone la condición dual

$$
f \ge g,
$$

de modo que la máscara actúe como límite inferior durante todo el proceso.

#### 4.3.3.4 Implementación de la erosión geodésica

La función estática `mm::cero` materializa este operador:

```python
staticmethod
def cero(f, g, b=np.zeros((3,3),dtype='uint8'), n=1):
    """Erosión geodésica del marcador f bajo la máscara g."""
    y = f.copy()
    for _ in range(n):
        y = np.maximum(mm.ero(y, b), g)
    return y
```

La instrucción `np.maximum(mm::ero(y, b), g)` implementa directamente la expresión $(y \ominus b)\vee g$.

Al igual que en la dilatación geodésica, el parámetro $n$ define cuántas erosiones geodésicas sucesivas se calcularán antes de devolver la imagen final.

> ### 📝 Relación con la reconstrucción morfológica
>
> La reconstrucción morfológica presentada en la siguiente sección se obtiene mediante la aplicación iterativa de la dilatación geodésica (`mm::cdil`) hasta alcanzar un punto fijo, es decir, hasta que ninguna celda cambie de valor entre dos iteraciones consecutivas. En otras palabras, la reconstrucción consiste en una secuencia de dilataciones geodésicas sucesivas que se propagan dentro de la máscara hasta que no haya más alteraciones.
>
> De forma dual, también es posible definir reconstrucciones basadas en erosión geodésica mediante aplicaciones sucesivas de `mm::cero`.

#### 4.3.3.5 Ejemplo: propagación en un laberinto mediante dualidad

La [Figura 4.8](#fig-04-cero-labirinto) ilustra la resolución del problema de conectividad de un laberinto utilizando la dualidad morfológica por medio de las funciones `mm::cero` y `mm::suprec`.

En lugar de propagar un marcador por los corredores libres mediante dilataciones geodésicas, el problema se formula en el dominio complementario. Inicialmente, la máscara original se invierte,

$$
g = 1 - g_{orig},
$$

de modo que las paredes pasan a tomar el valor 1 y los corredores el valor 0. De manera análoga, el marcador se construye en ese mismo dominio complementario, conteniendo un único valor 0 en la posición de entrada del laberinto y valor 1 en los demás píxeles.

Utilizando el elemento estructurante en cruz (`mm::secross()`), la erosión geodésica actúa sobre el marcador complementado. En cada iteración de `mm::cero`, la región conectada al marcador inicial sufre erosiones sucesivas, mientras que la máscara impone un límite inferior que impide la propagación a través de las paredes del laberinto.

Las imágenes intermediarias muestran estados de la evolución después de diferentes números de iteraciones (`n=5`, `n=12` y `n=22`). El resultado final se obtiene mediante la reconstrucción geodésica por erosión (`mm::suprec`), que aplica erosiones geodésicas sucesivas hasta alcanzar un punto fijo, es decir, una situación en la que no se produce ninguna alteración adicional entre dos iteraciones consecutivas.

En el dominio complementario, la región reconstruida corresponde exactamente al conjunto de corredores conectados a la entrada del laberinto. Así, la conectividad entre la entrada y la salida puede determinarse directamente a partir de la imagen reconstruida.

In [21]:
%%writefile tmp/fig_04_cero_labirinto.cpp
#define MM_OUT "tmp/fig_04_cero_labirinto.png"
#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
    // Máscara já invertida (255 = parede, 0 = corredor)
    mm::Image g(10, 10);
    unsigned char g_data[10][10] = {
        {255,  0,255,255,255,255,255,255,255,255},
        {255,  0,  0,  0,  0,  0,255,  0,  0,  0},
        {255,255,255,255,255,  0,255,  0,255,  0},
        {255,  0,  0,  0,255,  0,  0,  0,255,  0},
        {255,  0,255,  0,255,255,255,255,255,  0},
        {255,  0,255,  0,  0,  0,  0,  0,  0,  0},
        {255,  0,255,255,255,255,255,255,  0,255},
        {255,  0,  0,  0,  0,  0,  0,255,  0,255},
        {255,255,255,255,255,255,  0,  0,  0,255},
        {255,255,255,255,255,255,255,255,  0,255}
    };
    for (int y = 0; y < 10; y++)
        for (int x = 0; x < 10; x++)
            g.at(y, x) = g_data[y][x];

    mm::Image f(10, 10);
    std::fill(f.data.begin(), f.data.end(), 255);
    f.at(0, 1) = 0;                      // semente (0) no corredor de entrada
    mm::Image B_cruz = mm::secross();

    mm::Image passo_5    = mm::cero(f, g, B_cruz, 5);
    mm::Image passo_12   = mm::cero(f, g, B_cruz, 12);
    mm::Image passo_22   = mm::cero(f, g, B_cruz, 22);
    mm::Image ponto_fixo = mm::suprec(f, g, B_cruz);

    mm::show(std::vector<mm::Image>{g, f, passo_5, passo_12, passo_22, ponto_fixo},
             MM_OUT,
             std::vector<std::string>{"Mascara (~g)", "Marcador (~f)", "n=5", "n=12", "n=22", "~mm.suprec"},
             6);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(g, "tmp/fig_04_cero_labirinto_0.png");
mm::write(f, "tmp/fig_04_cero_labirinto_1.png");
mm::write(passo_5, "tmp/fig_04_cero_labirinto_2.png");
mm::write(passo_12, "tmp/fig_04_cero_labirinto_3.png");
mm::write(passo_22, "tmp/fig_04_cero_labirinto_4.png");
mm::write(ponto_fixo, "tmp/fig_04_cero_labirinto_5.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_cero_labirinto.cpp


In [22]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_cero_labirinto.cpp -o tmp/fig_04_cero_labirinto -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_cero_labirinto \
  && test -f "tmp/fig_04_cero_labirinto.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_cero_labirinto.png"

[1] Mascara (~g)
[2] Marcador (~f)
[3] n=5
[4] n=12
[5] n=22
[6] ~mm.suprec


In [23]:
try:
    mm.show(
        [
            mm.read("tmp/fig_04_cero_labirinto_0.png"),
            mm.read("tmp/fig_04_cero_labirinto_1.png"),
            mm.read("tmp/fig_04_cero_labirinto_2.png"),
            mm.read("tmp/fig_04_cero_labirinto_3.png"),
            mm.read("tmp/fig_04_cero_labirinto_4.png"),
            mm.read("tmp/fig_04_cero_labirinto_5.png"),
        ],
        titles=[
            'Mascara (~g)',
            'Marcador (~f)',
            'n=5',
            'n=12',
            'n=22',
            '~mm.suprec',
        ],
        cols=6,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_cero_labirinto_0.png (ver a versao Python)")

<Figure size 4500x750 with 6 Axes>

**Figura 4.8:** Resolução de labirinto no domínio complementar: com máscara e marcador invertidos, *mm::cero* e *mm::suprec* propagam a onda geodésica pelos corredores.


### 4.3.4 Reconstrucción Morfológica

La **reconstrucción morfológica** propaga una imagen **marcadora** $f$ dentro de una imagen **máscara** $g$, garantizando que el resultado nunca supere los valores de intensidad impuestos por la máscara. El operador fundamental que posibilita esta propagación contenida es la **dilatación geodésica**, definida por la [Equação 4.7](#eq-04-cdil).

La reconstrucción se obtiene mediante la aplicación iterativa de esta dilatación condicionada. Inicialmente, el marcador está limitado por la máscara para establecer el estado inicial:

<a id="eq-04-infrec"></a>
$$
X^{(0)} = f \wedge g,
$$

y las iteraciones subsiguientes se definen de forma recursiva mediante:

$$
X^{(k)} = (X^{(k-1)} \oplus b) \wedge g.
$$

La secuencia crece de manera monótona hasta alcanzar un punto fijo, produciendo la **reconstrucción morfológica por dilatación** (también conocida en la literatura como *inf-reconstrucción*):

$$
R_g^\delta(f) = \lim_{k\to\infty} X^{(k)} = X^{(k)} \quad \text{cuando} \quad X^{(k)} = X^{(k-1)}. \tag{4.9}
$$


El ascenso iterativo se detiene tan pronto como se logra la estabilidad, es decir, cuando dos iteraciones consecutivas producen matrices con valores absolutamente idénticos.

#### 4.3.4.1 Implementação da reconstrucción morfológica

La rutina didáctica `mm::infrec` implementa directamente el algoritmo iterativo de punto fijo. Inicialmente, el marcador efectivo inicial $X^{(0)}$ se determina mediante la operación `np.minimum(f, g)`. Para garantizar que el bucle de verificación ejecute la primera pasada sin disparar falsas convergencias prematuras, la variable de control de la iteración anterior (`y1`) se inicializa rellenada con un valor centinela fuera del dominio de los datos (o simplemente con una matriz que fuerce la primera ejecución).

```python
def infrec(f, g, b=np.zeros((3,3), dtype='uint8')):
    """Inf-reconstrucción: dilata el marcador (f ∧ g) hasta converger bajo la máscara g."""
    y = np.minimum(f, g)
    # Inicializa y1 con valores imposibles para forzar la entrada en el bucle
    y1 = np.full_like(f, 256, dtype=np.int16) 
    while not np.array_equal(y, y1):
        y1 = y.copy()
        # Aplica la dilatación geodésica: (y ⊕ b) ∧ g
        y = np.minimum(mm.dil(y, b), g)
    return y.astype('uint8')
```

En el interior del bucle `while`, la variable `y` almacena la estimación corriente de la reconstrucción $X^{(k)}$, mientras que `y1` preserva la imagen del estadio inmediatamente anterior $X^{(k-1)}$. La instrucción de control condicional `np.minimum(mm::dil(y, b), g)` traduce fielmente la dilatación geodésica teórica, donde la expansión morfológica convencional comandada por OpenCV se "poda" inmediatamente y queda limitada por las barreras de intensidad de la máscara $g$. El bucle cesa cuando no se registra ninguna modificación de píxel entre los pasos.

#### 4.3.4.2 Ventajas de la reconstrucción morfológica

La reconstrucción morfológica es significativamente más robusta que la apertura convencional porque es capaz de eliminar estructuras no deseadas sin distorsionar ni alterar la morfología de los objetos que deben conservarse.

Mientras que la apertura clásica suaviza esquinas, elimina puntas y deforma contornos debido a la imposición geométrica rígida del elemento estructurante, la reconstrucción geodésica utiliza la máscara para recuperar con exactitud los límites y formatos originales de los objetos que poseen conectividad con el marcador original.

En términos intuitivos, el marcador actúa como una semilla de contagio que se expande progresivamente, pero solo transitando por las regiones permitidas por la máscara. Los componentes que no poseen ninguna intersección con el marcador jamás serán reconstruidos (siendo eliminados), mientras que los componentes tocados por la semilla se expanden hasta restaurar integralmente su geometría original.

Este comportamiento discriminatorio y conservativo se ilustra en la [Figura 4.9](#fig-04-reconstrucao-didatica), utilizando el elemento estructurante en cruz presentado en la [Figura 4.3](#fig-04-elemento-cruz)..

In [24]:
%%writefile tmp/fig_04_reconstrucao_didatica.cpp
#define MM_OUT "tmp/fig_04_reconstrucao_didatica.png"
// g++ -std=c++17 -DMORPH_USE_OPENCV snippet.cpp -o snippet $(pkg-config --cflags --libs opencv4) -lm

#include "morph.hpp"
#include <vector>
#include <string>
#include <iostream>
#include <algorithm>

int main() {
    //| label: fig-04-reconstrucao-didatica
    //| fig-cap: "*Pipeline* de Reconstrução Morfológica por Dilatação Condicionada: a máscara contém dois objetos, o marcador isola apenas o núcleo do objeto principal, e as iterações reconstroem sua forma exata até a convergência."
    //| echo: true
    //| output: true

    mm::Image g(10, 10);
    // Initialize g with the 0/1 pattern scaled by 255
    int vals_g[10][10] = {
        {0,0,0,0,0,0,0,0,0,0},
        {0,1,1,1,0,0,0,1,1,0},
        {0,1,1,1,1,0,0,1,1,0},
        {0,1,1,1,1,0,0,0,0,0},
        {0,1,1,1,1,0,0,0,0,0},
        {0,1,1,1,0,0,0,0,0,0},
        {0,0,1,1,1,0,0,1,0,0},
        {0,0,1,1,1,0,0,1,1,0},
        {0,0,0,1,0,0,0,0,0,0},
        {0,0,0,0,0,0,0,0,0,0}
    };
    for (int y = 0; y < 10; y++) {
        for (int x = 0; x < 10; x++) {
            g.at(y, x) = (unsigned char)(vals_g[y][x] * 255);
        }
    }

    mm::Image B_cruz = mm::secross();
    mm::Image f = mm::ero(g, mm::sebox(0));          // marcador: núcleo do objeto principal

    std::vector<mm::Image> iteracoes;
    std::vector<std::string> titulos;
    mm::Image img_atual = f;
    for (int i = 1; i <= 5; i++) {
        img_atual = mm::cdil(img_atual, g, B_cruz);
        iteracoes.push_back(img_atual);
        titulos.push_back("Dilatação Cond. (n=" + std::to_string(i) + ")");
    }

    mm::Image img_reconstruida = mm::infrec(f, g, B_cruz);

    // Check if img_reconstruida equals iteracoes[-1]
    bool igual = true;
    if (img_reconstruida.h != iteracoes.back().h || img_reconstruida.w != iteracoes.back().w) {
        igual = false;
    } else {
        for (size_t i = 0; i < img_reconstruida.data.size(); i++) {
            if (img_reconstruida.data[i] != iteracoes.back().data[i]) {
                igual = false;
                break;
            }
        }
    }
    std::cout << "✅ Estabilidade na iteração 5: " << (igual ? "True" : "False") << std::endl;

    std::vector<mm::Image> todas_imagens;
    todas_imagens.push_back(g);
    todas_imagens.push_back(f);
    for (auto& img : iteracoes) {
        todas_imagens.push_back(img);
    }
    todas_imagens.push_back(img_reconstruida);

    std::vector<std::string> todos_titulos = {"Máscara (g)", "Marcador (f)"};
    for (auto& t : titulos) {
        todos_titulos.push_back(t);
    }
    todos_titulos.push_back("Reconstrução R_g(f)");

    mm::show(todas_imagens, MM_OUT, todos_titulos, 8);

    return 0;
}

Overwriting tmp/fig_04_reconstrucao_didatica.cpp


In [25]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_reconstrucao_didatica.cpp -o tmp/fig_04_reconstrucao_didatica -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_reconstrucao_didatica \
  && test -f "tmp/fig_04_reconstrucao_didatica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_reconstrucao_didatica.png"

✅ Estabilidade na iteração 5: True
[1] Máscara (g)
[2] Marcador (f)
[3] Dilatação Cond. (n=1)
[4] Dilatação Cond. (n=2)
[5] Dilatação Cond. (n=3)
[6] Dilatação Cond. (n=4)
[7] Dilatação Cond. (n=5)
[8] Reconstrução R_g(f)


In [26]:
try:
    mm.show(mm.read("tmp/fig_04_reconstrucao_didatica.png"), figsize=(18, 3))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_reconstrucao_didatica.png (ver a versao Python)")

<Figure size 2700x450 with 1 Axes>

**Figura 4.9:** *Pipeline* de Reconstrução Morfológica por Dilatação Condicionada: a máscara contém dois objetos, o marcador isola apenas o núcleo do objeto principal, e as iterações reconstroem sua forma exata até a convergência.


### 4.3.5 Relleno de Agujeros y Eliminación de Bordes

Dos operadores basados en reconstrucción morfológica completan el *pipeline* de limpieza binaria. Sus características principales se resumen en la [Tabela 4.3](#tbl-04-clohole-edgeoff).

**Relleno de agujeros** (`mm::clohole`) elimina cavidades completamente rodeadas por el objeto, independientemente del tamaño, sin alterar los contornos externos. El procedimiento actúa sobre el complemento de la imagen, utilizando como marcador una restricción del marco (*frame*) al fondo:

<a id="eq-04-clohole"></a>
$$
\text{clohole}(f) = \bigl(R_{f^c}^\delta(\text{frame}(f) \wedge f^c)\bigr)^c \tag{4.10}
$$



En términos operativos, primero se reconstruye el fondo externo y, a continuación, se aplica la complementación para recuperar los objetos con los agujeros rellenos.

**Eliminación de objetos de borde** (`mm::edgeoff`) elimina todos los objetos que tocan el borde de la imagen, preservando únicamente los componentes totalmente internos. El marcador se obtiene mediante la intersección entre el marco (*frame*) y los objetos de la imagen:

<a id="eq-04-edgeoff"></a>
$$
\text{edgeoff}(f) = f \setminus R_f^\delta(\text{frame}(f) \wedge f) \tag{4.11}
$$


<a id="tbl-04-clohole-edgeoff"></a>

**Tabela 4.3:** Comparación entre los operadores *clohole* y *edgeoff*.

| Operador | Marcador | Máscara | Efecto |
|:---------|:---------|:--------|:-------|
| `mm::clohole` | *frame* restringido al fondo ($f^c$) | $f^c$ | Rellena agujeros internos |
| `mm::edgeoff` | *frame* restringido al objeto ($f$) | $f$ | Elimina objetos conectados al borde |


La evolución paso a paso de estas transformaciones geodésicas puede seguirse en las figuras siguientes. La [Figura 4.10](#fig-04-clohole-didatico) ilustra el mecanismo de inundación controlada del operador `mm::clohole`, en el cual la reconstrucción se produce a partir del fondo externo e impide la propagación hacia regiones internas no conectadas al exterior, dando como resultado el relleno consistente de las cavidades internas. En cambio, la [Figura 4.11](#fig-04-edgeoff-didatico) detalla la dinámica del operador `mm::edgeoff`, en la que solo los componentes conectados al borde se reconstruyen y posteriormente se eliminan, preservando exclusivamente los objetos totalmente contenidos en el interior de la imagen.

La conectividad de la propagación geodésica se controla mediante el elemento estructurante: `mm::sebox()` (vecindad de 8) incluye conexiones diagonales, mientras que `mm::secross()` (vecindad de 4) las excluye. En consecuencia, la elección del elemento estructurante afecta a qué componentes se alcanzan mediante la reconstrucción y, por tanto, cuáles se preservarán o eliminarán.

#### 4.3.5.1 Conformidad con la implementación

Las definiciones anteriores están directamente alineadas con la implementación en `morph.py`, reproducida a continuación:

```python
staticmethod
def clohole(f, b=np.ones((3,3),dtype='uint8')):
    # marcador restrito ao fundo da imagem
    marcador = mm.frame(f, border=1) & mm.neg(f)
    return mm.neg(mm.infrec(marcador, mm.neg(f), b))

staticmethod
def edgeoff(f, b=np.ones((3,3),dtype='uint8')):
    # marcador restrito aos objetos da imagem
    marcador = mm.frame(f, border=1) & f
    return mm.subm(f, mm.infrec(marcador, f, b))
```

Estas implementaciones dejan explícito que ambos operadores son instancias directas de reconstrucción morfológica por dilatación geodésica con `mm::infrec`, diferenciándose únicamente en la elección del marcador y de la máscara: `clohole` actúa sobre el complemento de la imagen, mientras que `edgeoff` actúa directamente en el dominio de los objetos.

In [27]:
%%writefile tmp/fig_04_clohole_didatico.cpp
#define MM_OUT "tmp/fig_04_clohole_didatico.png"
//| label: fig-04-clohole-didatico
//| fig-cap: "*Pipeline* de preenchimento de buracos (*clohole*): o marcador vem da borda da imagem, restrito ao complemento $f^c$. A dilatação geodésica reconstrói o fundo externo; após a complementação, os buracos internos ficam preenchidos."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>

int main() {
    mm::Image f(10, 10);
    // Matriz binária definida manualmente
    int dados[10][10] = {
        {0,0,0,0,0,0,0,0,0,0},
        {0,1,1,1,1,0,0,0,0,1},
        {0,1,0,0,1,0,0,1,0,1},
        {0,1,0,0,1,0,0,1,0,1},
        {0,1,1,1,1,0,0,1,0,0},
        {0,0,0,0,0,0,0,0,0,0},
        {0,0,1,1,1,0,1,1,1,0},
        {0,0,1,1,1,0,1,0,1,0},
        {0,0,1,1,1,0,1,1,1,0},
        {0,0,0,0,0,0,0,0,0,1}
    };
    for (int y = 0; y < 10; y++)
        for (int x = 0; x < 10; x++)
            f.at(y, x) = dados[y][x] * 255;

    mm::Image B_cruz = mm::secross();
    mm::Image f_c = mm::neg(f);
    mm::Image marcador_ch = mm::frame(f, 1);  // borda externa

    std::vector<mm::Image> iteracoes;
    std::vector<std::string> titulos;
    mm::Image img_atual = marcador_ch;
    for (int i = 1; i <= 5; i++) {
        img_atual = mm::cdil(img_atual, f_c, B_cruz);
        iteracoes.push_back(img_atual);
        titulos.push_back("Iter. (n=" + std::to_string(i) + ")");
    }

    mm::Image img_clohole = mm::neg(mm::infrec(marcador_ch, f_c, B_cruz));

    // Verificação
    mm::Image esperado = mm::clohole(f);
    bool igual = true;
    for (int y = 0; y < f.h && igual; y++)
        for (int x = 0; x < f.w; x++)
            if (img_clohole.at(y,x) != esperado.at(y,x)) { igual = false; break; }
    std::cout << "✅ Validação clohole: " << (igual ? "true" : "false") << "\n";

    std::vector<mm::Image> todas = {f, marcador_ch};
    todas.insert(todas.end(), iteracoes.begin(), iteracoes.end());
    todas.push_back(img_clohole);

    std::vector<std::string> nomes = {"f original", "Marcador (borda)"};
    nomes.insert(nomes.end(), titulos.begin(), titulos.end());
    nomes.push_back("clohole(f)");

    mm::show(todas, MM_OUT, nomes, 8);
    return 0;
}

Overwriting tmp/fig_04_clohole_didatico.cpp


In [28]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_clohole_didatico.cpp -o tmp/fig_04_clohole_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_clohole_didatico \
  && test -f "tmp/fig_04_clohole_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_clohole_didatico.png"

✅ Validação clohole: true
[1] f original
[2] Marcador (borda)
[3] Iter. (n=1)
[4] Iter. (n=2)
[5] Iter. (n=3)
[6] Iter. (n=4)
[7] Iter. (n=5)
[8] clohole(f)


In [29]:
try:
    mm.show(mm.read("tmp/fig_04_clohole_didatico.png"), figsize=(18, 3))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_clohole_didatico.png (ver a versao Python)")

<Figure size 2700x450 with 1 Axes>

**Figura 4.10:** *Pipeline* de preenchimento de buracos (*clohole*): o marcador vem da borda da imagem, restrito ao complemento $f^c$. A dilatação geodésica reconstrói o fundo externo; após a complementação, os buracos internos ficam preenchidos.


In [30]:
%%writefile tmp/fig_04_edgeoff_didatico.cpp
#define MM_OUT "tmp/fig_04_edgeoff_didatico.png"
//| label: fig-04-edgeoff-didatico
//| fig-cap: "*Pipeline* de eliminação de estruturas de borda (*edgeoff*): o marcador captura as raízes conectadas às extremidades, a reconstrução delimita esses elementos e a subtração preserva só os objetos totalmente internos."
//| echo: true
//| output: true

#include "morph.hpp"
#include <vector>
#include <string>

int main() {
    mm::Image f(10, 10);
    int data[10][10] = {
        {0,0,0,0,0,0,0,0,0,0},
        {0,1,1,1,1,0,0,0,0,1},
        {0,1,0,0,1,0,0,1,0,1},
        {0,1,0,0,1,0,0,1,0,1},
        {0,1,1,1,1,0,0,1,0,0},
        {0,0,0,0,0,0,0,0,0,0},
        {0,0,1,1,1,0,1,1,1,0},
        {0,0,1,1,1,0,1,0,1,0},
        {0,0,1,1,1,0,1,1,1,0},
        {0,0,0,0,0,0,0,0,0,1}
    };
    for (int y = 0; y < 10; y++)
        for (int x = 0; x < 10; x++)
            f.at(y, x) = data[y][x] * 255;

    mm::Image B_box = mm::sebox();
    mm::Image marcador_eo = mm::band(mm::frame(f, 1), f);   // borda ∩ f

    std::vector<mm::Image> iteracoes;
    std::vector<std::string> titulos;
    mm::Image img_atual = marcador_eo;
    for (int i = 1; i <= 5; i++) {
        img_atual = mm::cdil(img_atual, f, B_box);
        iteracoes.push_back(img_atual);
        titulos.push_back("Iter. (n=" + std::to_string(i) + ")");
    }

    mm::Image img_edgeoff = mm::edgeoff(f, B_box);

    std::vector<mm::Image> imgs = {f, marcador_eo};
    imgs.insert(imgs.end(), iteracoes.begin(), iteracoes.end());
    imgs.push_back(img_edgeoff);

    std::vector<std::string> titles = {"f original", "Marcador (borda)"};
    titles.insert(titles.end(), titulos.begin(), titulos.end());
    titles.push_back("edgeoff(f)");

    mm::show(imgs, MM_OUT, titles, 8);
    return 0;
}

Overwriting tmp/fig_04_edgeoff_didatico.cpp


In [31]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_edgeoff_didatico.cpp -o tmp/fig_04_edgeoff_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_edgeoff_didatico \
  && test -f "tmp/fig_04_edgeoff_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_edgeoff_didatico.png"

[1] f original
[2] Marcador (borda)
[3] Iter. (n=1)
[4] Iter. (n=2)
[5] Iter. (n=3)
[6] Iter. (n=4)
[7] Iter. (n=5)
[8] edgeoff(f)


In [32]:
try:
    mm.show(mm.read("tmp/fig_04_edgeoff_didatico.png"), figsize=(18, 3))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_edgeoff_didatico.png (ver a versao Python)")

<Figure size 2700x450 with 1 Axes>

**Figura 4.11:** *Pipeline* de eliminação de estruturas de borda (*edgeoff*): o marcador captura as raízes conectadas às extremidades, a reconstrução delimita esses elementos e a subtração preserva só os objetos totalmente internos.


### 4.3.6 *Pipeline* de Limpieza Binaria con CLAHE

Con base en el análisis anterior —en el cual el CLAHE produjo el mayor valor de la varianza interclases ($\sigma_B^2 \approx 2{,}47 \times 10^3$), ver [Figura 4.2](#fig-04-otsu-comparacao-histogramas), y el operador `mm::clohole` mostró ser eficaz en el relleno de las cavidades internas—, el *pipeline* final de segmentación, ilustrado en la [Figura 4.12](#fig-04-pipeline-clahe), se estructura mediante el siguiente flujo computacional:

$$
\text{gray}
\xrightarrow{\text{CLAHE}}
\xrightarrow{\text{Otsu}}
\xrightarrow{\text{open}}
\xrightarrow{\text{clohole}}
\xrightarrow{\text{open}}
\xrightarrow{\text{edgeoff}}
\text{segmentación}
$$

Tras la etapa de `mm::clohole`, se aplica una segunda apertura morfológica con un elemento estructurante mayor (`mm::sedisk(33)`, disco de diámetro 33). Esta operación elimina pequeñas regiones residuales y artefactos que podrían haber permanecido después de la segmentación. En particular, el relleno geodésico puede transformar pequeñas cavidades aisladas en componentes conectados al objeto, haciendo conveniente una etapa adicional de filtrado basada en tamaño. El diámetro se eligió de modo que las monedas sigan siendo capaces de contener el elemento estructurante, mientras que los componentes significativamente más pequeños sean eliminados.

En esta imagen, ninguna moneda está conectada al borde de la matriz. En consecuencia, la aplicación de `mm::edgeoff` no altera el resultado obtenido tras la segunda apertura. Aun así, esta etapa se mantiene en el *pipeline* por robustez, pues en otras imágenes pueden existir objetos parcialmente visibles o conectados a los bordes, que deben eliminarse antes de la etapa de análisis.

> ### 💡 ¿Por qué la apertura después del clohole?
>
> El operador `mm::clohole` rellena todas las cavidades cerradas presentes en los objetos segmentados. En algunas situaciones, pequeñas regiones no deseadas pueden permanecer después de esta etapa o volverse conectadas a los objetos principales. La apertura morfológica subsiguiente elimina componentes más pequeños que el elemento estructurante, preservando las monedas debido a su tamaño significativamente mayor.

In [33]:
%%writefile tmp/fig_04_pipeline_clahe.cpp
#define MM_OUT "tmp/fig_04_pipeline_clahe.png"
// Compile: g++ -std=c++17 -O2 -o program program.cpp -I/usr/include/opencv4 $(pkg-config --cflags --libs opencv4) -DMM_USE_OPENCV -DMM_OUT="\"output.png\""
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
// [pdi:state-io:end]

    mm::Image img_clahe0 = mm::clahe(img_coins_gray, 2.0, 8);
    mm::Image img_bin = mm::threshold(img_clahe0);
    mm::Image img_open = mm::open(img_bin, mm::sedisk(9));
    mm::Image img_hole = mm::clohole(img_open);
    mm::Image img_limpo = mm::open(img_hole, mm::sedisk(33));
    mm::Image img_final = mm::edgeoff(img_limpo, mm::SE::box(3), 1);

    mm::show(
        std::vector<mm::Image>{img_clahe0, img_bin, img_open, img_hole, img_limpo, img_final},
        MM_OUT,
        std::vector<std::string>{"CLAHE", "Otsu", "Abertura (r=9)", "clohole", "Abertura (r=33)", "Final (edgeoff)"},
        6
    );
    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_final, "tmp/state/img_final_55.png");
// [pdi:state-io:end]

// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_clahe0, "tmp/fig_04_pipeline_clahe_0.png");
mm::write(img_bin, "tmp/fig_04_pipeline_clahe_1.png");
mm::write(img_open, "tmp/fig_04_pipeline_clahe_2.png");
mm::write(img_hole, "tmp/fig_04_pipeline_clahe_3.png");
mm::write(img_limpo, "tmp/fig_04_pipeline_clahe_4.png");
mm::write(img_final, "tmp/fig_04_pipeline_clahe_5.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_pipeline_clahe.cpp


In [34]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_pipeline_clahe.cpp -o tmp/fig_04_pipeline_clahe -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_pipeline_clahe \
  && test -f "tmp/fig_04_pipeline_clahe.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_pipeline_clahe.png"

[1] CLAHE
[2] Otsu
[3] Abertura (r=9)
[4] clohole
[5] Abertura (r=33)
[6] Final (edgeoff)


In [35]:
try:
    mm.show(
        [
            mm.read("tmp/fig_04_pipeline_clahe_0.png"),
            mm.read("tmp/fig_04_pipeline_clahe_1.png"),
            mm.read("tmp/fig_04_pipeline_clahe_2.png"),
            mm.read("tmp/fig_04_pipeline_clahe_3.png"),
            mm.read("tmp/fig_04_pipeline_clahe_4.png"),
            mm.read("tmp/fig_04_pipeline_clahe_5.png"),
        ],
        titles=[
            'CLAHE',
            'Otsu',
            'Abertura (r=9)',
            'clohole',
            'Abertura (r=33)',
            'Final (edgeoff)',
        ],
        cols=6,
        figsize=(18, 6),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_pipeline_clahe_0.png (ver a versao Python)")

<Figure size 2700x900 with 6 Axes>

**Figura 4.12:** *Pipeline* completo de segmentação com CLAHE: Otsu → abertura (r=9) → clohole → abertura (r=33) → edgeoff.


### 4.3.7 Morfología en Tonos de Gris

Los operadores morfológicos se extienden naturalmente a imágenes en tonos de gris. En esta formulación, la erosión y la dilatación pasan a actuar directamente sobre los niveles de intensidad de la imagen. Para elementos estructurantes planos ($b \equiv 0$), la erosión corresponde al **mínimo local** y la dilatación al **máximo local** dentro de la vecindad definida por el elemento estructurante.

La interpretación intuitiva es sencilla: la erosión **oscurece** regiones al reemplazar cada píxel por el menor valor presente en su vecindad, mientras que la dilatación **aclara** regiones al utilizar el mayor valor disponible. La combinación de estos operadores permite construir transformaciones capaces de resaltar bordes, eliminar tendencias de iluminación y destacar estructuras locales.

Tres operadores derivados son especialmente útiles:

**Gradiente morfológico** — resalta bordes como la diferencia entre dilatación y erosión:

<a id="eq-04-gradiente-morf"></a>
$$
\text{grad}_B(f) = (f \oplus B) - (f \ominus B) \tag{4.12}
$$


***Top-hat*** — resalta estructuras brillantes más pequeñas que el elemento estructurante (diferencia entre la imagen original y su apertura):

<a id="eq-04-tophat"></a>
$$
\text{top-hat}_B(f) = f - (f \circ B) \tag{4.13}
$$


***Black-hat*** — resalta estructuras oscuras más pequeñas que el elemento estructurante (diferencia entre el cierre y la imagen original):

<a id="eq-04-blackhat"></a>
$$
\text{black-hat}_B(f) = (f \bullet B) - f \tag{4.14}
$$


El *Top-hat* extrae detalles brillantes que no sobreviven a la apertura, mientras que el *Black-hat* evidencia detalles oscuros eliminados por el cierre. Por su parte, el gradiente morfológico resalta transiciones abruptas de intensidad, produciendo una representación similar a la de un detector de bordes.

Para comprender el mecanismo de estos operadores a nivel local, la [Figura 4.13](#fig-04-sim-04-operadores-av) presenta un simulador interactivo de morfología en tonos de gris. El simulador permite editar libremente el elemento estructurante, visualizar su desplazamiento sobre la imagen y acompañar simultáneamente el perfil unidimensional de las intensidades. De esta forma, se hace posible observar directamente cómo la erosión selecciona mínimos locales, cómo la dilatación selecciona máximos locales y cómo el gradiente morfológico emerge de la diferencia entre estos dos operadores.

El botón ubicado en la esquina superior derecha permite alternar entre la visualización original en tonos de gris y una representación pseudocoloreada (*colormap*) solo en los tres tipos de gradientes. La versión coloreada facilita la percepción visual de las variaciones de intensidad, haciendo más evidente la acción de los operadores morfológicos sobre máximos, mínimos y transiciones locales de la imagen.

Los operadores presentados están disponibles en `morph.py` mediante las funciones `mm::gradm`, `mm::tophat` y `mm::blackhat`.

In [36]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-operadores-av" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-operadores-av * { box-sizing: border-box; }
  #sim-04-operadores-av canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-04-operadores-av button { font-size: 11px; padding: 6px 10px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 4px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-operadores-av button:hover { background: #e8dfcf; }
  #sim-04-operadores-av button.gc_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  
  #sim-04-operadores-av button.gc_toggle_btn { background: #ebf4fd; border-color: #2980b9; color: #2980b9; font-weight: 700; }
  #sim-04-operadores-av button.gc_toggle_btn:hover { background: #d4e6fc; }
  
  .gc_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .gc_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 8px; padding: 8px; text-align: center; }
  .gc_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .gc_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  
  #sim04_seCanvas { cursor: pointer; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎛️ Simulador Avanzado de Morfología Matemática</span>
  <button id="sim04_btn_toggle_view" class="gc_toggle_btn" onclick="sim04_toggleViewMode()">
    Visualización: Gris Real
  </button>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Barra de Modos de Operação -->
  <div style="display:flex; gap:6px; flex-wrap:wrap; margin-bottom:14px; justify-content:center;">
    <button id="sim04_btn_orig" class="gc_active" onclick="sim04_setMode('orig')">Original f</button>
    <button id="sim04_btn_ero" onclick="sim04_setMode('ero')">Erosión</button>
    <button id="sim04_btn_dil" onclick="sim04_setMode('dil')">Dilatación</button>
    <button id="sim04_btn_open" onclick="sim04_setMode('open')">Apertura (∘)</button>
    <button id="sim04_btn_close" onclick="sim04_setMode('close')">Cierre (•)</button>
    <button id="sim04_btn_grad" onclick="sim04_setMode('grad')">Gradiente</button>
    <button id="sim04_btn_tophat" onclick="sim04_setMode('tophat')">Top-hat</button>
    <button id="sim04_btn_bhat" onclick="sim04_setMode('bhat')">Black-hat</button>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(260px, 1fr)); gap:16px; align-items:start; margin-bottom:14px;">
    
    <!-- Canvas Principal -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
      <canvas id="sim04_Canvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Mueva el mouse para actualizar el perfil 1D de la línea correspondiente
      </div>
    </div>

    <!-- Painel Lateral de Estatísticas, Descrição e Kernel B -->
    <div style="display:flex; flex-direction:column; gap:12px;">
      
      <!-- Estatísticas / Valores do Pixel -->
      <div style="display:grid; grid-template-columns:repeat(2, 1fr); gap:8px;">
        <div class="gc_stat_box"><div class="gc_stat_label">X (col)</div><div id="sim04_sX" class="gc_stat_value">—</div></div>
        <div class="gc_stat_box"><div class="gc_stat_label">Y (fil)</div><div id="sim04_sY" class="gc_stat_value">—</div></div>
        <div class="gc_stat_box"><div class="gc_stat_label">f(x,y)</div><div id="sim04_sOrig" class="gc_stat_value" style="color:#27ae60;">—</div></div>
        <div class="gc_stat_box"><div class="gc_stat_label" id="sim04_sLabel">valor</div><div id="sim04_sVal" class="gc_stat_value" style="color:#2980b9;">—</div></div>
      </div>

      <!-- Descrição Teórica -->
      <div id="sim04_desc" class="gc_panel" style="font-size:11px; color:#5e5a4a; line-height:1.5; min-height:45px;"></div>

      <!-- Editor do Elemento Estruturante B -->
      <div class="gc_panel" style="text-align:center;">
        <div style="font-size:10.5px; font-weight:700; color:#5e5a4a; margin-bottom:8px; text-transform:uppercase; letter-spacing:.3px;">
          Elemento B (Haga clic para Editar)
        </div>
        <canvas id="sim04_seCanvas" width="114" height="114" style="margin:0 auto; display:block; border-radius:6px; border:1px solid #e4dcc8; background:#ffffff;"></canvas>
      </div>

      <!-- Fórmulas Auxiliares -->
      <div class="gc_panel" style="font-size:9.5px; font-family:monospace; color:#5e5a4a; line-height:1.5;">
        <strong style="color:#26241d;">Apertura (f∘B):</strong> dil(ero(f))<br>
        <strong style="color:#26241d;">Cierre (f•B):</strong> ero(dil(f))<br>
        <strong style="color:#26241d;">Gradiente:</strong> dil(f) − ero(f)<br>
        <strong style="color:#26241d;">Top-hat:</strong> f − (f∘B)<br>
        <strong style="color:#26241d;">Black-hat:</strong> (f•B) − f
      </div>

    </div>

  </div>

  <!-- Perfil 1D na Parte Inferior -->
  <div style="background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
    <canvas id="sim04_profileCanvas" width="660" height="110" style="border-radius:8px; border:1px solid #e4dcc8; width:100%; height:auto; background:#ffffff;"></canvas>
    <div id="sim04_profileLabel" style="font-size:10.5px; color:#8a8371; margin-top:6px; font-family:monospace;">Perfil 1D de la línea: Ninguno (pase el mouse sobre la imagen)</div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04Advanced(root){
    if (!root || root.dataset.sim04AdvancedInit) return;
    root.dataset.sim04AdvancedInit = "1";

    const ROWS = 40, COLS = 40, CELL = 8, PAD = 6;
    const W = COLS * CELL + PAD * 2, H = ROWS * CELL + PAD * 2;

    const cv = root.querySelector('#sim04_Canvas');
    cv.width = W; cv.height = H;
    const ctx = cv.getContext('2d');

    let gc_viewMode = 'gray';
    let gc_mode = 'orig';
    let gc_hx = -1, gc_hy = 19;

    const f = Array.from({length: ROWS}, ()=>new Array(COLS).fill(0));
    const cx1 = 13, cy1 = 14, r1 = 9, cx2 = 28, cy2 = 14, r2 = 7, cx3 = 20, cy3 = 28, r3 = 8;
    for (let r=0; r<ROWS; r++) {
      for (let c=0; c<COLS; c++) {
        let v = 30;
        const d1 = Math.hypot(c - cx1, r - cy1), d2 = Math.hypot(c - cx2, r - cy2), d3 = Math.hypot(c - cx3, r - cy3);
        if (d1 < r1) v = Math.round(200 - 90 * (d1 / r1));
        if (d2 < r2) v = Math.max(v, Math.round(180 - 70 * (d2 / r2)));
        if (d3 < r3) v = Math.max(v, Math.round(160 - 50 * (d3 / r3)));
        if (Math.hypot(c - 33, r - 8) < 2.2) v = 230;
        if (Math.hypot(c - 6, r - 32) < 2.2) v = Math.min(v, 20);
        f[r][c] = Math.max(0, Math.min(255, v));
      }
    }

    const B_SIZE = 5;
    const B = Array.from({length: B_SIZE}, () => new Array(B_SIZE).fill(0));
    const rad = 2;
    for (let r=0; r<B_SIZE; r++) {
      for (let c=0; c<B_SIZE; c++) {
        if (Math.hypot(c - rad, r - rad) <= rad) B[r][c] = 1;
      }
    }

    const MORPH_DATA = {};

    function recalculateMorphology() {
      const Boff = [];
      const center = Math.floor(B_SIZE / 2);
      for (let r=0; r<B_SIZE; r++) {
        for (let c=0; c<B_SIZE; c++) {
          if (B[r][c]) Boff.push([r - center, c - center]);
        }
      }
      if (Boff.length === 0) Boff.push([0, 0]);

      function e_op(img) {
        const res = Array.from({length: ROWS}, ()=>new Array(COLS).fill(0));
        for (let r=0; r<ROWS; r++) {
          for (let c=0; c<COLS; c++) {
            let mn = 255;
            for (const [dr, dc] of Boff) {
              const vr = r + dr, vc = c + dc;
              const val = (vr >= 0 && vr < ROWS && vc >= 0 && vc < COLS) ? img[vr][vc] : 255;
              if (val < mn) mn = val;
            }
            res[r][c] = mn;
          }
        }
        return res;
      }

      function d_op(img) {
        const res = Array.from({length: ROWS}, ()=>new Array(COLS).fill(0));
        for (let r=0; r<ROWS; r++) {
          for (let c=0; c<COLS; c++) {
            let mx = 0;
            for (const [dr, dc] of Boff) {
              const vr = r + dr, vc = c + dc;
              const val = (vr >= 0 && vr < ROWS && vc >= 0 && vc < COLS) ? img[vr][vc] : 0;
              if (val > mx) mx = val;
            }
            res[r][c] = mx;
          }
        }
        return res;
      }

      function diff(a, b) {
        return Array.from({length: ROWS}, (_, r) => Array.from({length: COLS}, (_, c) => Math.max(0, Math.min(255, a[r][c] - b[r][c]))));
      }

      MORPH_DATA.orig = f;
      MORPH_DATA.ero = e_op(f);
      MORPH_DATA.dil = d_op(f);
      MORPH_DATA.open = d_op(MORPH_DATA.ero);
      MORPH_DATA.close = e_op(MORPH_DATA.dil);
      MORPH_DATA.grad = diff(MORPH_DATA.dil, MORPH_DATA.ero);
      MORPH_DATA.tophat = diff(f, MORPH_DATA.open);
      MORPH_DATA.bhat = diff(MORPH_DATA.close, f);
    }

    const MODES_CONFIG = {
      orig: {label: 'f(x,y)', color: '#27ae60', desc: 'Imagem original f.'},
      ero: {label: 'ero(x,y)', color: '#c0392b', desc: 'Erosão: Encolhe estruturas claras de acordo com a geometria de B.'},
      dil: {label: 'dil(x,y)', color: '#2980b9', desc: 'Dilatação: Expande estruturas claras preenchendo falhas.'},
      open: {label: 'open(x,y)', color: '#8e44ad', desc: 'Abertura: Suaviza contornos, elimina pequenos ruídos e picos brilhantes isolados.'},
      close: {label: 'close(x,y)', color: '#d35400', desc: 'Fechamento: Preenche pequenos canais ou buracos escuros interiores.'},
      grad: {label: 'grad(x,y)', color: '#16a085', desc: 'Gradiente morfológico: Destaca as bordas físicas dos objetos.'},
      tophat: {label: 'th(x,y)', color: '#b9770e', desc: 'Top-hat: Isola elementos brilhantes menores que B.'},
      bhat: {label: 'bh(x,y)', color: '#2c3e50', desc: 'Black-hat: Isola fossas escuras ou vales menores que B.'}
    };

    window.sim04_toggleViewMode = function() {
      if (gc_viewMode === 'gray') {
        gc_viewMode = 'colormap';
        root.querySelector('#sim04_btn_toggle_view').innerHTML = 'Visualização: Falsa Cor';
      } else {
        gc_viewMode = 'gray';
        root.querySelector('#sim04_btn_toggle_view').innerHTML = 'Visualização: Cinza Real';
      }
      gc_draw();
    };

    window.sim04_setMode = function(m) {
      gc_mode = m;
      Object.keys(MODES_CONFIG).forEach(k => {
        const btn = root.querySelector('#sim04_btn_' + k);
        if (btn) btn.classList.toggle('gc_active', k === m);
      });
      const cfg = MODES_CONFIG[m];
      root.querySelector('#sim04_sLabel').textContent = cfg.label;
      root.querySelector('#sim04_sVal').style.color = cfg.color;
      root.querySelector('#sim04_desc').textContent = cfg.desc;
      gc_draw();
      gc_drawProfile();
    };

    function gc_imgToGray(data) {
      const id = ctx.createImageData(W, H);
      for (let r=0; r<ROWS; r++) {
        for (let c=0; c<COLS; c++) {
          const v = data[r][c];
          const x = PAD + c * CELL, y = PAD + r * CELL;
          let rc = v, gc = v, bc = v;

          if (gc_viewMode === 'colormap' && (gc_mode === 'grad' || gc_mode === 'tophat' || gc_mode === 'bhat')) {
            if (v > 0) {
              rc = Math.min(255, v * 7);       
              gc = Math.min(255, v * 3.5);     
              bc = Math.max(40, 255 - v * 4.5); 
            } else { rc = 250; gc = 250; bc = 247; }
          }

          for (let dy=0; dy<CELL; dy++) {
            for (let dx=0; dx<CELL; dx++) {
              const idx = 4 * ((y + dy) * W + (x + dx));
              id.data[idx] = rc; id.data[idx+1] = gc; id.data[idx+2] = bc; id.data[idx+3] = 255;
            }
          }
        }
      }
      return id;
    }

    function gc_draw() {
      const data = MORPH_DATA[gc_mode];
      const id = gc_imgToGray(data);
      ctx.putImageData(id, 0, 0);

      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 0.5;
      for (let c=0; c<=COLS; c++) { ctx.beginPath(); ctx.moveTo(PAD + c * CELL, PAD); ctx.lineTo(PAD + c * CELL, PAD + ROWS * CELL); ctx.stroke(); }
      for (let r=0; r<=ROWS; r++) { ctx.beginPath(); ctx.moveTo(PAD, PAD + r * CELL); ctx.lineTo(PAD + COLS * CELL, PAD + r * CELL); ctx.stroke(); }

      if (gc_hy >= 0 && gc_hy < ROWS) {
        ctx.strokeStyle = 'rgba(192, 57, 43, 0.4)'; ctx.lineWidth = 1;
        ctx.beginPath(); ctx.moveTo(PAD, PAD + gc_hy * CELL + CELL / 2); ctx.lineTo(PAD + COLS * CELL, PAD + gc_hy * CELL + CELL / 2); ctx.stroke();
      }

      if (gc_hx >= 0 && gc_hy >= 0) {
        ctx.strokeStyle = 'rgba(41, 128, 185, 0.9)'; ctx.lineWidth = 1.5;
        ctx.strokeRect(PAD + gc_hx * CELL + 0.5, PAD + gc_hy * CELL + 0.5, CELL - 1, CELL - 1);
      }

      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 1;
      ctx.strokeRect(PAD, PAD, COLS * CELL, ROWS * CELL);
    }

    const seCanvas = root.querySelector('#sim04_seCanvas');
    const seCtx = seCanvas.getContext('2d');
    const SE_CELL = 20, SE_PAD = 7;

    function gc_drawSE() {
      seCtx.clearRect(0, 0, 114, 114);
      const center = Math.floor(B_SIZE / 2);
      for (let r=0; r<B_SIZE; r++) {
        for (let c=0; c<B_SIZE; c++) {
          const x = SE_PAD + c * SE_CELL, y = SE_PAD + r * SE_CELL;
          seCtx.fillStyle = B[r][c] ? '#2980b9' : '#fafaf7';
          seCtx.fillRect(x, y, SE_CELL - 2, SE_CELL - 2);
          seCtx.strokeStyle = '#e4dcc8';
          seCtx.strokeRect(x, y, SE_CELL - 2, SE_CELL - 2);
          if (r === center && c === center) {
            seCtx.fillStyle = '#ffffff'; seCtx.font = 'bold 11px monospace';
            seCtx.textAlign = 'center'; seCtx.textBaseline = 'middle';
            seCtx.fillText('★', x + (SE_CELL - 2) / 2, y + (SE_CELL - 2) / 2);
          }
        }
      }
    }

    seCanvas.addEventListener('click', function(e) {
      const rect = seCanvas.getBoundingClientRect();
      const scaleX = seCanvas.width / rect.width;
      const scaleY = seCanvas.height / rect.height;
      const cx = (e.clientX - rect.left) * scaleX;
      const cy = (e.clientY - rect.top) * scaleY;
      const c = Math.floor((cx - SE_PAD) / SE_CELL);
      const r = Math.floor((cy - SE_PAD) / SE_CELL);
      if (c >= 0 && c < B_SIZE && r >= 0 && r < B_SIZE) {
        B[r][c] = B[r][c] ? 0 : 1;
        recalculateMorphology();
        gc_drawSE();
        gc_draw();
        gc_drawProfile();
      }
    });

    function gc_drawProfile() {
      const pc = root.querySelector('#sim04_profileCanvas');
      const pctx = pc.getContext('2d');
      pctx.clearRect(0, 0, 660, 110);
      
      if (gc_hy < 0 || gc_hy >= ROWS) return;

      root.querySelector('#sim04_profileLabel').innerHTML = `<span style='color:#c0392b; font-weight:bold;'>Perfil 1D da Linha ${gc_hy}</span> — Tracejado: Original f(x) | Cor da Aba: Operação Atual`;

      const dataOrig = MORPH_DATA.orig[gc_hy];
      const dataCurrent = MORPH_DATA[gc_mode][gc_hy];
      const currentColor = MODES_CONFIG[gc_mode].color;
      
      const stepX = 660 / (COLS - 1);
      
      function drawLine(arrayData, color, width, isDash = false) {
        pctx.strokeStyle = color; pctx.lineWidth = width;
        pctx.beginPath();
        if (isDash) pctx.setLineDash([4, 4]); else pctx.setLineDash([]);
        for (let c=0; c<COLS; c++) {
          const x = c * stepX;
          const y = 95 - (arrayData[c] / 255) * 85; 
          if (c === 0) pctx.moveTo(x, y); else pctx.lineTo(x, y);
        }
        pctx.stroke();
      }

      pctx.setLineDash([]);
      pctx.strokeStyle = '#e4dcc8'; pctx.lineWidth = 0.5;
      for (let h=0; h<=4; h++) {
        let yVal = 10 + h * 21.25;
        pctx.beginPath(); pctx.moveTo(0, yVal); pctx.lineTo(660, yVal); pctx.stroke();
      }

      if (gc_mode !== 'orig') {
        drawLine(dataOrig, 'rgba(39, 174, 96, 0.4)', 1.5, true);
      }
      drawLine(dataCurrent, currentColor, 2.5, false);

      pctx.setLineDash([]);
      pctx.fillStyle = '#8a8371'; pctx.font = '9px monospace';
      pctx.fillText('Intensidade (255)', 5, 10);
      pctx.fillText('Fundo (0)', 5, 104);
    }

    cv.addEventListener('mousemove', function(e) {
      const rect = cv.getBoundingClientRect();
      const sx = W / rect.width, sy = H / rect.height;
      const c = Math.floor(((e.clientX - rect.left) * sx - PAD) / CELL);
      const r = Math.floor(((e.clientY - rect.top) * sy - PAD) / CELL);
      if (c >= 0 && c < COLS && r >= 0 && r < ROWS) {
        gc_hx = c; gc_hy = r;
        root.querySelector('#sim04_sX').textContent = c;
        root.querySelector('#sim04_sY').textContent = r;
        root.querySelector('#sim04_sOrig').textContent = f[r][c];
        root.querySelector('#sim04_sVal').textContent = MORPH_DATA[gc_mode][r][c];
      }
      gc_draw();
      gc_drawProfile();
    });

    recalculateMorphology();
    sim04_setMode('orig');
    gc_drawSE();
  }

  function tryInitSim04Advanced(){
    var root = document.getElementById('sim-04-operadores-av');
    if (root) initSim04Advanced(root); else setTimeout(tryInitSim04Advanced, 200);
  }
  tryInitSim04Advanced();
})();
</script>
""")

**Figura 4.13:** Simulador interactivo avanzado de morfología con elemento estructurante editable y perfil 1D.


<figure id="fig-04-sim-04-operadores-av">
  <img src="imagens/fig-04-sim-04-operadores-av.png" alt=" Simulador interactivo avanzado de morfología con elemento estructurante editable y perfil 1D. " style="max-width:80%" />
  <figcaption><strong>Figura 4.13:</strong>  Simulador interactivo avanzado de morfología con elemento estructurante editable y perfil 1D. </figcaption>
</figure>

La [Figura 4.14](#fig-04-morf-gc-histogramas) ilustra los efectos de estos operadores sobre la imagen de las monedas y sus respectivos histogramas. Observe que la erosión desplaza la distribución hacia intensidades más bajas, mientras que la dilatación la desplaza hacia intensidades más altas. El gradiente concentra valores en las regiones de contorno, y los operadores *Top-hat* y *Black-hat* producen histogramas fuertemente concentrados en bajos niveles de intensidad, ya que solo se resaltan pequeñas estructuras locales.

In [37]:
%%writefile tmp/fig_04_morf_gc_histogramas.cpp
#define MM_OUT "tmp/fig_04_morf_gc_histogramas.png"
//| label: fig-04-morf-gc-histogramas
//| fig-cap: "Morfologia em tons de cinza e seus histogramas: erosão, dilatação, gradiente, top-hat e black-hat. Elemento estruturante: disco de diâmetro 19."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
// [pdi:state-io:end]

    // img_coins_gray is already available as mm::Image

    mm::Image B = mm::sedisk(19);
    mm::Image img_ero = mm::ero(img_coins_gray, B);
    mm::Image img_dil = mm::dil(img_coins_gray, B);
    mm::Image img_grad = mm::gradm(img_coins_gray, B);
    mm::Image img_th = mm::tophat(img_coins_gray, B);
    mm::Image img_bh = mm::blackhat(img_coins_gray, B);

    mm::show(
        std::vector<mm::Image>{img_coins_gray, mm::histImg(img_coins_gray), img_ero, mm::histImg(img_ero),
                               img_dil, mm::histImg(img_dil), img_grad, mm::histImg(img_grad),
                               img_th, mm::histImg(img_th), img_bh, mm::histImg(img_bh)},
        MM_OUT,
        std::vector<std::string>{"Original", "Hist", "Erosão", "Hist", "Dilatação", "Hist",
                                 "Gradiente", "Hist", "Top-hat", "Hist", "Black-hat", "Hist"},
        4
    );

    return 0;
}

Overwriting tmp/fig_04_morf_gc_histogramas.cpp


In [38]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_morf_gc_histogramas.cpp -o tmp/fig_04_morf_gc_histogramas -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_morf_gc_histogramas \
  && test -f "tmp/fig_04_morf_gc_histogramas.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_morf_gc_histogramas.png"

[1] Original
[2] Hist
[3] Erosão
[4] Hist
[5] Dilatação
[6] Hist
[7] Gradiente
[8] Hist
[9] Top-hat
[10] Hist
[11] Black-hat
[12] Hist


In [39]:
try:
    mm.show(mm.read("tmp/fig_04_morf_gc_histogramas.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_morf_gc_histogramas.png (ver a versao Python)")

<Figure size 1044x784 with 1 Axes>

**Figura 4.14:** Morfologia em tons de cinza e seus histogramas: erosão, dilatação, gradiente, top-hat e black-hat. Elemento estruturante: disco de diâmetro 19.


Los operadores morfológicos presentados anteriormente se utilizarán ahora como herramientas de refinamiento y generación de marcadores para métodos de segmentación más avanzados, que se presentan a continuación.

## 4.4 Segmentación de Imágenes: Fundamentación y Taxonomía

La **segmentación de imágenes** consiste en dividir la imagen en regiones asociadas a objetos o estructuras de interés. En PDI, representa la transición entre el procesamiento de bajo nivel —como filtrado y realce— y etapas de análisis más avanzadas, como extracción de características, reconocimiento e interpretación de la escena.

Formalmente, el objetivo de la segmentación consiste en descomponer el dominio espacial completo de una imagen, denotado por $\Omega$, en una partición de subconjuntos $\{R_1, R_2, \ldots, R_n\}$ que satisfaga simultáneamente los criterios de **completitud** y **disyunción**:

<a id="eq-04-particao"></a>
$$
\bigcup_{i=1}^{n} R_i = \Omega, \qquad R_i \cap R_j = \emptyset \quad \forall\, i \neq j \tag{4.15}
$$


Además de las propiedades de completitud y disyunción expresadas en la [Equação 4.15](#eq-04-particao), cada subregión $R_i$ debe constituir un dominio **homogéneo** según un predicado de similitud definido sobre propiedades locales —intensidad, color o textura— y, simultáneamente, ser **distinta** de las regiones adyacentes.

Las técnicas de segmentación pueden organizarse en diferentes familias. En este capítulo se enfatizarán los enfoques resumidos en la [Tabela 4.4](#tbl-04-segmentacao), fundamentados principalmente en criterios de intensidad, conectividad y proximidad espacial.

<a id="tbl-04-segmentacao"></a>

**Tabela 4.4:** Taxonomía simplificada de los principales enfoques de segmentación y refinamiento estudiados en este capítulo.

| Enfoque | Criterio de Segmentación | Operadores de Referencia |
| :--- | :--- | :--- |
| **Umbralización** | Particionamiento del espacio de intensidades | Criterio de Otsu, umbralización global y local |
| **Morfología Matemática** | Relaciones espaciales definidas por elementos/funciones estructurantes | Erosión, dilatación, apertura, cierre y reconstrucción |
| **Basada en Regiones** | Homogeneidad local y conectividad espacial | Etiquetado de componentes conexos, Transformada de Distancia y *Watershed* |


Hasta este punto, el desarrollo práctico se ha centrado en la **umbralización**, mediante la combinación entre ecualización adaptativa CLAHE y el método global de Otsu. Esta etapa se complementó con operadores de **reconstrucción morfológica** basados en dilataciones geodésicas, implementados por las funciones `mm::infrec`, `mm::clohole` y `mm::edgeoff`, produciendo una máscara binaria limpia y adecuada para el análisis.

Sin embargo, en escenarios donde objetos distintos aparecen conectados en la máscara binaria —ya sea por contacto físico, superposición parcial o por puentes estrechos de píxeles producidos por la segmentación—, la umbralización deja de ser suficiente para individualizar cada objeto. En estos casos, múltiples objetos pasan a componer un único componente conexo, dificultando etapas posteriores de medición e interpretación.

Para superar esta limitación, las próximas secciones introducen tres herramientas complementarias: el **Etiquetado de Componentes Conexos**, la **Transformada de Distancia** y el algoritmo de segmentación por ***Watershed*** basado en marcadores. En conjunto, estas técnicas permiten separar objetos adyacentes, identificar regiones individualmente y extraer descriptores geométricos consistentes para el análisis cuantitativo.

### 4.4.1 Etiquetado

El **etiquetado de componentes conexas** (*connected component labeling*) es el operador que asigna un identificador entero único a cada conjunto de píxeles pertenecientes a la misma componente conexa en una imagen binaria.

> ### 📝 Definición formal
>
> Dada una imagen binaria $f$ y una relación de conectividad definida por un elemento estructurante $B$ (típicamente conectividad-4 o conectividad-8), el algoritmo de etiquetado de la [Figura 4.15](#fig-04-sim-alg-rotulagem2) produce una imagen $g$ en la cual todos los píxeles pertenecientes a la misma componente conexa reciben la misma etiqueta entera positiva, mientras que los píxeles pertenecientes a componentes distintas reciben etiquetas diferentes.

La conectividad define qué píxeles se consideran vecinos directos de un píxel $(x,y)$. Las definiciones más utilizadas son:

* **Conectividad-4:** considera solo los cuatro vecinos ortogonales (norte, sur, este y oeste).
* **Conectividad-8:** considera los cuatro vecinos ortogonales y los cuatro diagonales, totalizando ocho vecinos.

La elección de la conectividad influye directamente en la formación de las componentes conexas y, consecuentemente, en el resultado del etiquetado, como se ilustra en la [Figura 4.17](#fig-04-rotulacao-didatico). Un ejemplo adicional puede explorarse de forma interactiva en el simulador presentado en la [Figura 4.16](#fig-04-sim-04-rotulacao).

La implementación en `morph.py` proporciona dos versiones de este operador. La función `mm::label0` reproduce explícitamente el algoritmo de *flood-fill* utilizando una pila y permite controlar la conectividad mediante el elemento estructurante adoptado. En cambio, `mm::label` delega la operación a la implementación optimizada de OpenCV (`mm::label0`). En ambos casos, el resultado es una imagen etiquetada en la cual cada componente conexa recibe un identificador entero distinto.

In [40]:
# @title { display-mode: "form" }
import numpy as np, cv2, os  # [pdi] passthrough: garante imports desta trilha
# Prefijo exclusivo para evitar conflictos con otras celdas del cuaderno
PREFIX = "lbl2"

from IPython.display import HTML
HTML(f'''
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:var(--font-sans)}}
.{PREFIX}-tabs{{display:flex;gap:6px;margin-bottom:20px;flex-wrap:wrap}}
.{PREFIX}-tab{{padding:6px 14px;border-radius:20px;font-size:13px;cursor:pointer;border:0.5px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all .15s;white-space:nowrap}}
.{PREFIX}-tab.active{{background:var(--color-text-primary);color:var(--color-background-primary);border-color:transparent}}
.{PREFIX}-panel{{display:none}}.{PREFIX}-panel.active{{display:block}}

.algo-wrap{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.algo-header{{padding:14px 20px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;align-items:center;gap:10px}}
.algo-icon{{width:32px;height:32px;border-radius:var(--border-radius-md);display:flex;align-items:center;justify-content:center;font-size:16px;background:#EEEDFE;color:#534AB7}}
.algo-title{{font-size:14px;font-weight:500;color:var(--color-text-primary);text-align:left}}
.algo-sub{{font-size:12px;color:var(--color-text-secondary);margin-top:1px;text-align:left}}
.algo-body{{padding:20px;text-align:left}}

.step{{display:flex;gap:12px;margin-bottom:14px;align-items:flex-start}}
.step:last-child{{margin-bottom:0}}
.step-num{{min-width:24px;height:24px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0;margin-top:1px}}
.step-text{{font-size:13.5px;line-height:1.65;color:var(--color-text-primary);text-align:left}}
.step-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 6px;border-radius:4px;color:var(--color-text-primary)}}
.step-sub{{margin-top:8px;border-left:2px solid var(--color-border-secondary);padding-left:12px;display:flex;flex-direction:column;gap:5px}}
.step-sub-item{{font-size:13px;color:var(--color-text-secondary);line-height:1.55;text-align:left}}
.step-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.while-box{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:11px 13px;margin-top:8px;text-align:left}}
.while-head{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.06em;margin-bottom:7px}}
.note{{margin-top:16px;padding-top:12px;border-top:0.5px solid var(--color-border-tertiary);font-size:12px;color:var(--color-text-secondary);font-style:italic;line-height:1.5;text-align:left}}

.num-teal{{background:#E1F5EE;color:#0F6E56}}
.num-purple{{background:#EEEDFE;color:#534AB7}}
.num-amber{{background:#FAEEDA;color:#854F0B}}
.num-coral{{background:#FAECE7;color:#993C1D}}
.num-blue{{background:#E6F1FB;color:#185FA5}}
.num-gray{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}

.card-step{{display:flex;border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);overflow:hidden;margin-bottom:8px}}
.card-step:last-child{{margin-bottom:0}}
.card-badge{{min-width:48px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0}}
.card-content{{padding:10px 14px;flex:1;text-align:left}}
.card-label{{font-size:10px;font-weight:500;letter-spacing:.08em;text-transform:uppercase;margin-bottom:3px}}
.card-text{{font-size:13.5px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.card-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.card-sub{{margin-top:6px;display:flex;flex-direction:column;gap:3px}}
.card-sub-item{{font-size:12.5px;color:var(--color-text-secondary);padding-left:10px;border-left:2px solid var(--color-border-secondary);line-height:1.5;text-align:left}}
.card-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 4px;border-radius:3px;color:var(--color-text-primary)}}

.tl{{position:relative;padding:4px 0 4px 36px;text-align:left}}
.tl-line{{position:absolute;left:11px;top:20px;bottom:20px;width:1.5px;background:var(--color-border-secondary);border-radius:2px}}
.tl-step{{display:flex;gap:0;margin-bottom:18px;position:relative}}
.tl-step:last-child{{margin-bottom:0}}
.tl-dot-wrap{{position:absolute;left:-36px;top:2px;display:flex;flex-direction:column;align-items:center;gap:3px}}
.tl-dot{{width:14px;height:14px;border-radius:50%;border:2px solid var(--color-border-secondary);background:var(--color-background-primary);transition:all .2s;z-index:1}}
.tl-step:hover .tl-dot{{background:var(--color-text-primary);border-color:var(--color-text-primary)}}
.tl-num{{font-size:9px;color:var(--color-text-secondary);font-weight:500;letter-spacing:.04em}}
.tl-title{{font-size:13.5px;font-weight:500;color:var(--color-text-primary);margin-bottom:3px;text-align:left}}
.tl-desc{{font-size:13px;color:var(--color-text-secondary);line-height:1.6;text-align:left}}
.tl-desc code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.code-wrap{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.code-bar{{background:var(--color-background-secondary);padding:10px 16px;display:flex;align-items:center;justify-content:space-between;border-bottom:0.5px solid var(--color-border-tertiary)}}
.code-lang{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.08em}}
.code-body{{padding:18px 20px;overflow-x:auto;text-align:left}}
.code-body pre{{font-family:var(--font-mono);font-size:13px;line-height:1.75;color:var(--color-text-primary);margin:0;white-space:pre;text-align:left}}
.kw{{color:#7C3AED}} .fn{{color:#0369A1}} .cm{{color:#6B7280;font-style:italic}} .st{{color:#059669}} .num-lit{{color:#DC2626}}

.ann-line{{display:flex;align-items:flex-start;gap:10px;margin-bottom:8px;padding:10px 12px;background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);text-align:left}}
.ann-badge{{min-width:20px;height:20px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;flex-shrink:0;margin-top:1px}}
.ann-text{{font-size:13px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.ann-text code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
</style>

<h2 class="sr-only" style="position:absolute;left:-9999px">Algoritmo de rotulação por flood-fill com pilha — painel interativo com HTML e SVG</h2>

<div class="{PREFIX}-tabs" id="{PREFIX}-tabs-container">
  <div class="{PREFIX}-tab" data-idx="0"><i class="ti ti-list-numbers" aria-hidden="true"></i> Passo a passo</div>
  <div class="{PREFIX}-tab" data-idx="1"><i class="ti ti-cards" aria-hidden="true"></i> Cards</div>
  <div class="{PREFIX}-tab" data-idx="2"><i class="ti ti-timeline" aria-hidden="true"></i> Linha do tempo</div>
  <div class="{PREFIX}-tab" data-idx="3"><i class="ti ti-code" aria-hidden="true"></i> Código Python</div>
  <div class="{PREFIX}-tab active" data-idx="4"><i class="ti ti-git-branch" aria-hidden="true"></i> Fluxograma</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-0">
<div class="algo-wrap">
  <div class="algo-header">
    <div class="algo-icon"><i class="ti ti-stack-2" aria-hidden="true"></i></div>
    <div><div class="algo-title">Flood-fill com pilha</div><div class="algo-sub">Rotulagem de componentes conexas</div></div>
  </div>
  <div class="algo-body">
    <div class="step"><div class="step-num num-teal">1</div><div class="step-text">Criar imagem de saída <em>g</em>, inicializada com <code>zeros</code>, mesma dimensão de <em>f</em>.</div></div>
    <div class="step"><div class="step-num num-teal">2</div><div class="step-text">Inicializar contador de rótulos (cor) <code>cor ← 1</code>.</div></div>
    <div class="step"><div class="step-num num-purple">3</div><div class="step-text">Percorrer <em>f</em> em <strong>ordem raster</strong> (coordenadas <code>x</code> e <code>y</code>) até encontrar uma semente: pixel ativo (<code>f[x,y] ≠ 0</code>) ainda não rotulado (<code>g[x,y] = 0</code>).</div></div>
    <div class="step"><div class="step-num num-amber">4</div><div class="step-text">Inserir a semente encontrada na pilha <code>pilha ← [[x,y]]</code>.</div></div>
    <div class="step">
      <div class="step-num num-coral">5</div>
      <div class="step-text">Enquanto a pilha contiver coordenadas (<code>while pilha</code>):
        <div class="step-sub">
          <div class="step-sub-item">Desempilhar pixel atual: <code>i, j ← pilha.pop()</code> e atribuir o rótulo: <code>g[i,j] ← cor</code>.</div>
          <div class="step-sub-item">Buscar vizinhos usando o iterador <code>mm._viz(f,b,i,j)</code>. Se o vizinho for ativo no elemento estruturante (<code>bv ≠ 0</code>), ativo na imagem (<code>f[vy,vx] ≠ 0</code>) e não rotulado (<code>g[vy,vx] = 0</code>), empilhá-lo.</div>
        </div>
      </div>
    </div>
    <div class="step"><div class="step-num num-blue">6</div><div class="step-text">Pilha vazia ⟹ Toda a componente conexa atual foi explorada e rotulada com sucesso.</div></div>
    <div class="step"><div class="step-num num-blue">7</div><div class="step-text">Incrementar o rótulo para a próxima componente: <code>cor ← cor + 1</code> e continuar a varredura raster.</div></div>
    <div class="note">A conectividade (4 ou 8 vizinhos) é definida unicamente pela matriz morfológica <code>b</code> passada como parâmetro, alterando os pixels retornados em <code>mm._viz</code>.</div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-1">
  <div class="card-step">
    <div class="card-badge num-teal">01</div>
    <div class="card-content">
      <div class="card-label" style="color:#0F6E56">Inicialização</div>
      <div class="card-text">Criar matriz de rótulos <code>g</code> preenchida com zeros (fundo). Definir rótulo inicial <code>cor ← 1</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-purple">02</div>
    <div class="card-content">
      <div class="card-label" style="color:#534AB7">Varredura Raster</div>
      <div class="card-text">Percorrer a matriz bidimensional linha por linha, localizando pixels pertencentes ao objeto que ainda não possuem rótulo.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-amber">03</div>
    <div class="card-content">
      <div class="card-label" style="color:#854F0B">Semente inicial</div>
      <div class="card-text">Ao achar um pixel válido, inicializar a estrutura LIFO de busca: <code>pilha = [[x, y]]</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">04</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Expansão por Flood-Fill</div>
      <div class="card-text">Enquanto houver elementos na pilha:</div>
      <div class="card-sub">
        <div class="card-sub-item">Extrair <code>(i, j)</code> via <code>pop()</code> e marcar <code>g[i, j] = cor</code>.</div>
        <div class="card-sub-item">Inspecionar vizinhança geométrica e adicionar novos candidatos à pilha.</div>
      </div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-blue">05</div>
    <div class="card-content">
      <div class="card-label" style="color:#185FA5">Próxima Componente</div>
      <div class="card-text">Pilha esvaziada ⟹ Incrementar indexador <code>cor ← cor + 1</code> para diferenciar o próximo objeto isolado.</div>
    </div>
  </div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-2">
<div class="tl">
  <div class="tl-line"></div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">01</span></div>
    <div><div class="tl-title">Alocação Espacial</div><div class="tl-desc"><code>g ← zeros_like(f)</code> e definição do primeiro identificador: <code>cor ← 1</code>.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">02</span></div>
    <div><div class="tl-title">Varredura Bidimensional</div><div class="tl-desc">Laços encadeados varrendo as dimensões <code>h</code> e <code>w</code> da imagem.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">03</span></div>
    <div><div class="tl-title">Descoberta de Objeto</div><div class="tl-desc">Filtro condicional localiza pixel ativo não indexado e cria a <code>pilha</code> semente.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">04–05</span></div>
    <div>
      <div class="tl-title">Preenchimento por Região (Flood-fill)</div>
      <div class="while-box">
        <div class="while-head">while pilha</div>
        <div class="step-sub" style="border-color:var(--color-border-tertiary)">
          <div class="step-sub-item">Remover último da pilha <code>(i,j)</code> e aplicar rótulo atual.</div>
          <div class="step-sub-item">Empilhar vizinhos conectados que atendam aos critérios morfológicos de <code>b</code>.</div>
        </div>
      </div>
    </div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">06</span></div>
    <div><div class="tl-title">Fechamento do Objeto</div><div class="tl-desc">Pilha vazia determina o fim do isolamento daquela componente.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">07</span></div>
    <div><div class="tl-title">Atualização do Rótulo</div><div class="tl-desc">Incremento linear: <code>cor ← cor + 1</code>. A varredura raster continua do ponto onde parou.</div></div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-3">
<div class="code-wrap">
  <div class="code-bar">
    <span class="code-lang"><i class="ti ti-brand-python" aria-hidden="true" style="font-size:14px;vertical-align:-2px;margin-right:5px"></i>label0.py</span>
    <span style="font-size:11px;color:var(--color-text-secondary)">flood-fill com pilha</span>
  </div>
  <div class="code-body">
<pre><span class="kw">def</span> <span class="fn">label0</span>(f, b=np.ones((<span class="num-lit">3</span>,<span class="num-lit">3</span>),dtype=<span class="st">'uint8'</span>)):
    <span class="cm">"""Rotulagem por flood-fill com pilha."""</span>
    h, w = f.shape
    g = np.zeros(f.shape, dtype=<span class="kw">int</span>)
    cor = <span class="num-lit">1</span>
    <span class="kw">for</span> x <span class="kw">in</span> <span class="fn">range</span>(h):
        <span class="kw">for</span> y <span class="kw">in</span> <span class="fn">range</span>(w):
            <span class="kw">if</span> f[x,y] <span class="kw">and not</span> g[x,y]:
                pilha = [[x,y]]
                <span class="kw">while</span> pilha:
                    i,j = pilha.pop(); g[i,j] = cor
                    <span class="kw">for</span> vy,vx,bv <span class="kw">in</span> mm._viz(f,b,i,j):
                        <span class="kw">if</span> bv <span class="kw">and</span> f[vy,vx] <span class="kw">and not</span> g[vy,vx]:
                            pilha.append([vy,vx])
                cor += <span class="num-lit">1</span>
    <span class="kw">return</span> g</pre>
  </div>
</div>
<div style="margin-top:16px;display:flex;flex-direction:column;gap:8px;text-align:left">
  <div class="ann-line"><div class="ann-badge num-teal">1</div><div class="ann-text"><code>g = np.zeros(f.shape, dtype=int)</code> — Inicializa a matriz de saída com zeros. Zeros representam o fundo invariável.</div></div>
  <div class="ann-line"><div class="ann-badge num-purple">2</div><div class="ann-text"><code>mm._viz(f, b, i, j)</code> — O iterador morfológico avalia a conectividade. Passando <code>B_cruz</code> a busca expande em 4-vizinhança; passando quadrado (<code>ones</code>) expande em 8-vizinhança.</div></div>
  <div class="ann-line"><div class="ann-badge num-amber">3</div><div class="ann-text"><code>pilha.pop()</code> — Remove o último par de coordenadas inserido, caracterizando um comportamento LIFO de busca em profundidade (DFS) para varrer o objeto de forma contígua.</div></div>
  <div class="ann-line"><div class="ann-badge num-coral">4</div><div class="ann-text"><code>cor += 1</code> — O incremento ocorre estritamente fora do laço <code>while</code>, garantindo que o mesmo número marque toda a extensão da componente concluída antes de passar para a próxima semente raster.</div></div>
</div>
</div>

<div class="{PREFIX}-panel active" id="{PREFIX}-p-4">
<svg viewBox="0 0 580 820" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:580px;display:block;margin:0 auto">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#888780"/></marker>
    <marker id="arr-b" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#534AB7"/></marker>
    <marker id="arr-r" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#993C1D"/></marker>
    <marker id="arr-g" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#0F6E56"/></marker>
  </defs>
  
  <ellipse cx="280" cy="36" rx="60" ry="22" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/>
  <text x="280" y="41" text-anchor="middle" font-size="13" font-weight="500" fill="#085041">início</text>
  
  <rect x="160" y="80" width="240" height="48" rx="6" fill="#E1F5EE" stroke="#9FE1CB" stroke-width="1"/>
  <text x="280" y="99" text-anchor="middle" font-size="12" font-weight="500" fill="#0F6E56">inicializar saída</text>
  <text x="280" y="116" text-anchor="middle" font-size="12" fill="#085041">g ← zeros(f.shape);  cor ← 1</text>
  
  <rect x="160" y="150" width="240" height="48" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="280" y="169" text-anchor="middle" font-size="12" font-weight="500" fill="#534AB7">varredura raster</text>
  <text x="280" y="186" text-anchor="middle" font-size="12" fill="#3C3489">próximo pixel (x, y) em f</text>
  
  <polygon points="280,224 370,252 280,280 190,252" fill="#F1EFE8" stroke="#B4B2A9" stroke-width="1"/>
  <text x="280" y="248" text-anchor="middle" font-size="11.5" fill="#444441">imagem toda</text>
  <text x="280" y="264" text-anchor="middle" font-size="11.5" fill="#444441">varrida?</text>
  
  <ellipse cx="450" cy="252" rx="52" ry="22" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="450" y="257" text-anchor="middle" font-size="13" font-weight="500" fill="#27500A">fim</text>
  
  <polygon points="280,304 380,334 280,364 180,334" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="280" y="329" text-anchor="middle" font-size="11.5" fill="#3C3489">f[x,y] ≠ 0  e</text>
  <text x="280" y="344" text-anchor="middle" font-size="11.5" fill="#3C3489">g[x,y] = 0?</text>
  
  <rect x="170" y="388" width="220" height="44" rx="6" fill="#FAEEDA" stroke="#FAC775" stroke-width="1"/>
  <text x="280" y="406" text-anchor="middle" font-size="12" font-weight="500" fill="#854F0B">inserir semente</text>
  <text x="280" y="422" text-anchor="middle" font-size="12" fill="#633806">pilha ← [[x, y]]</text>
  
  <polygon points="280,456 370,486 280,516 190,486" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="491" text-anchor="middle" font-size="11.5" fill="#712B13">pilha vazia?</text>
  
  <rect x="160" y="542" width="240" height="48" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="561" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">i, j ← pilha.pop()</text>
  <text x="280" y="578" text-anchor="middle" font-size="12" fill="#712B13">g[i, j] ← cor</text>
  
  <rect x="150" y="614" width="260" height="60" rx="6" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="634" text-anchor="middle" font-size="12" font-weight="500" fill="#185FA5">inspecionar vizinhos</text>
  <text x="280" y="650" text-anchor="middle" font-size="12" fill="#0C447C">se ativo e não rotulado</text>
  <text x="280" y="666" text-anchor="middle" font-size="12" fill="#0C447C">→ pilha.append([vy, vx])</text>
  
  <rect x="435" y="466" width="110" height="40" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="490" y="482" text-anchor="middle" font-size="12" font-weight="500" fill="#534AB7">cor ← cor + 1</text>
  <text x="490" y="496" text-anchor="middle" font-size="10.5" fill="#3C3489">próximo rótulo</text>

  <line x1="280" y1="58" x2="280" y2="80" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="128" x2="280" y2="150" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="198" x2="280" y2="224" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <line x1="370" y1="252" x2="398" y2="252" stroke="#3B6D11" stroke-width="1.2" marker-end="url(#arr-g)"/>
  <text x="384" y="245" text-anchor="middle" font-size="11" fill="#3B6D11">sim</text>
  
  <line x1="280" y1="280" x2="280" y2="304" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <path d="M 180,334 L 130,334 L 130,174 L 160,174" fill="none" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-b)"/>
  <text x="152" y="327" text-anchor="middle" font-size="11" fill="#534AB7">não</text>
  
  <line x1="280" y1="364" x2="280" y2="388" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <text x="292" y="378" font-size="11" fill="#534AB7">sim</text>
  
  <line x1="280" y1="432" x2="280" y2="456" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <line x1="280" y1="516" x2="280" y2="542" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="292" y="531" font-size="11" fill="#993C1D">não</text>
  
  <line x1="280" y1="590" x2="280" y2="614" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <path d="M 280,674 L 280,694 L 110,694 L 110,486 L 190,486" fill="none" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  
  <line x1="370" y1="486" x2="435" y2="486" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="402" y="479" text-anchor="middle" font-size="11" fill="#993C1D">sim</text>
  
  <path d="M 490,466 L 490,174 L 400,174" fill="none" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-b)"/>
</svg>
</div>

<script>
(function() {{
  var container = document.getElementById('{PREFIX}-tabs-container');
  if (!container) return;
  
  var tabs = container.querySelectorAll('.{PREFIX}-tab');
  var panels = [
    document.getElementById('{PREFIX}-p-0'),
    document.getElementById('{PREFIX}-p-1'),
    document.getElementById('{PREFIX}-p-2'),
    document.getElementById('{PREFIX}-p-3'),
    document.getElementById('{PREFIX}-p-4')
  ];

  tabs.forEach(function(tab) {{
    tab.onclick = function() {{
      var idx = parseInt(this.getAttribute('data-idx'));
      
      tabs.forEach(function(t) {{ t.classList.remove('active'); }});
      panels.forEach(function(p) {{ if(p) p.classList.remove('active'); }});
      
      this.classList.add('active');
      if(panels[idx]) panels[idx].classList.add('active');
    }};
  }});
}})();
</script>
''')

**Figura 4.15:** Algoritmo de etiquetado por *flood-fill* con pila.


<figure id="fig-04-sim-alg-rotulagem2">
  <img src="imagens/fig-04-sim-alg-rotulagem2.png" alt=" Algoritmo de etiquetado por *flood-fill* con pila. " style="max-width:80%" />
  <figcaption><strong>Figura 4.15:</strong>  Algoritmo de etiquetado por *flood-fill* con pila. </figcaption>
</figure>

El auxiliar `_viz` itera sobre la ventana estructurante `b` centrada en $(i,j)$, generando únicamente los vecinos válidos dentro de los límites de la imagen — la conectividad deseada está enteramente determinada por la forma de `b` pasada al algoritmo.

**Ejemplo didáctico — efecto de la conectividad:**

In [41]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-rotulacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-rotulacao * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-rotulacao canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; }
  #sim-04-rotulacao button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-rotulacao button:hover { background: #e8dfcf; }
  #sim-04-rotulacao button.rt_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-rotulacao .rt_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .rt_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .rt_grid_stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 14px; }
  .rt_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .rt_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .rt_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🪙 Simulador: Etiquetado de Componentes Conexas</span>
  <span class="rt_pill">flood-fill con pila</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="rt_grid_stats">
    <div class="rt_stat_box">
      <div class="rt_stat_label">Píxeles Activos</div>
      <div id="rt_statPx" class="rt_stat_value" style="color:#2980b9;">0</div>
    </div>
    <div class="rt_stat_box">
      <div class="rt_stat_label">Componentes</div>
      <div id="rt_statCC" class="rt_stat_value" style="color:#27ae60;">0</div>
    </div>
    <div class="rt_stat_box">
      <div class="rt_stat_label">Conectividad</div>
      <div id="rt_statConn" class="rt_stat_value" style="color:#b9770e;">4</div>
    </div>
    <div class="rt_stat_box">
      <div class="rt_stat_label">Paso Raster</div>
      <div id="rt_statStep" class="rt_stat_value" style="color:#c0392b;">–</div>
    </div>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start;">
    
    <!-- Canvas e Legenda -->
    <div style="flex:2; min-width:260px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
      <canvas id="rt_ccCanvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Haga clic para activar/desactivar píxeles · Arrastre para pintar
      </div>
      <div style="display:flex; flex-wrap:wrap; gap:8px; margin-top:10px; justify-content:center;" id="rt_legendBox"></div>
    </div>

    <!-- Painel de Controles Lateral -->
    <div style="flex:1; min-width:220px; display:flex; flex-direction:column; gap:12px;">
      
      <!-- Seletor de Conectividade -->
      <div class="rt_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Conectividad
        </div>
        <div style="display:flex; gap:6px;">
          <button id="rt_btn4" class="rt_active" onclick="window.rt_setConn(4)" style="flex:1; justify-content:center;">C-4</button>
          <button id="rt_btn8" onclick="window.rt_setConn(8)" style="flex:1; justify-content:center;">C-8</button>
        </div>
        <div id="rt_connDesc" style="font-size:9.5px; color:#8a8371; margin-top:6px; line-height:1.4;">
          4 vecinos ortogonales: N, S, E, O
        </div>
      </div>

      <!-- Seletor de Modo de Visualização -->
      <div class="rt_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Visualización
        </div>
        <div style="display:flex; gap:6px;">
          <button id="rt_btnLabel" class="rt_active" onclick="window.rt_setMode('label')" style="flex:1; justify-content:center;">Etiquetas</button>
          <button id="rt_btnAnim" onclick="window.rt_setMode('anim')" style="flex:1; justify-content:center;">Animado</button>
        </div>
      </div>

      <!-- Controles de Animação -->
      <div id="rt_animCtrl" class="rt_panel" style="display:none;">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Paso a Paso
        </div>
        <div style="display:flex; gap:6px; margin-bottom:8px;">
          <button onclick="window.rt_animStep(-1)" style="flex:1; justify-content:center;">◀</button>
          <button id="rt_btnPlay" onclick="window.rt_togglePlay()" style="flex:1; justify-content:center;">▶</button>
          <button onclick="window.rt_animStep(1)" style="flex:1; justify-content:center;">▶▶</button>
        </div>
        <div style="display:flex; align-items:center; gap:8px;">
          <label style="font-size:9.5px; color:#8a8371; font-weight:700;">Vel.</label>
          <input type="range" id="rt_speedSlider" min="1" max="10" value="5" style="flex:1; max-width:160px; height:4px; cursor:pointer;">
        </div>
      </div>

      <!-- Status da Animação -->
      <div id="rt_animStatus" class="rt_panel" style="display:none;">
        <div id="rt_animTitle" style="font-size:11px; font-weight:700; margin-bottom:3px; color:#26241d;">–</div>
        <div id="rt_animDesc" style="font-size:9.5px; color:#8a8371; line-height:1.4;">–</div>
      </div>

      <!-- Exemplos / Presets -->
      <div class="rt_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Ejemplos Predefinidos
        </div>
        <div style="display:flex; flex-direction:column; gap:5px;">
          <button onclick="window.rt_loadPreset('diagonal')" style="justify-content:flex-start;">■ Diagonal (C-4 vs C-8)</button>
          <button onclick="window.rt_loadPreset('letters')" style="justify-content:flex-start;">Letras Separadas</button>
          <button onclick="window.rt_loadPreset('ring')" style="justify-content:flex-start;">◎ Anillo con Agujero</button>
        </div>
      </div>

      <!-- Botão de Limpeza -->
      <button onclick="window.rt_clearGrid()" style="justify-content:center; border-color:#f5b7b1; color:#c0392b; background:#fdecea;">
        🗑️ Limpiar Cuadrícula
      </button>

    </div>

  </div>

</div>
</div>

<script>
(function(){
  function initSim04RotulacaoCC(root){
    if (!root || root.dataset.sim04RotulacaoCCInit) return;
    root.dataset.sim04RotulacaoCCInit = "1";

    const rt_COLS = 12, rt_ROWS = 12, rt_CELL = 22, rt_PAD = 10;
    const rt_W = rt_COLS * rt_CELL + rt_PAD * 2, rt_H = rt_ROWS * rt_CELL + rt_PAD * 2;
    const rt_cv = root.querySelector('#rt_ccCanvas');
    rt_cv.width = rt_W; 
    rt_cv.height = rt_H;
    const rt_ctx = rt_cv.getContext('2d');

    const rt_PALETTE = [
      ['#2980b9', '#ebf4fd', '#a9cce3', '#042c53'],
      ['#27ae60', '#eafaf1', '#a3e4d7', '#04342C'],
      ['#b9770e', '#fef5e7', '#f8c471', '#412402'],
      ['#c0392b', '#fdecea', '#f5b7b1', '#4A1B0C'],
      ['#8e44ad', '#f5eef8', '#d7bde2', '#4B1528'],
      ['#16a085', '#e8f8f5', '#a3e4d7', '#0e6251'],
      ['#d35400', '#fbeee6', '#f5cba7', '#7e5109'],
      ['#2c3e50', '#ebedef', '#bdc3c7', '#17202a']
    ];

    let rt_grid = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
    let rt_connectivity = 4;
    let rt_mode = 'label';
    let rt_painting = false;
    let rt_paintVal = 1;

    let rt_animSteps = [];
    let rt_animIdx = 0;
    let rt_playing = false;
    let rt_playTimer = null;

    function rt_neighbors(r, c, conn) {
      const n = [[r-1,c],[r+1,c],[r,c-1],[r,c+1]];
      if (conn === 8) n.push([r-1,c-1],[r-1,c+1],[r+1,c-1],[r+1,c+1]);
      return n.filter(([nr,nc]) => nr>=0 && nr<rt_ROWS && nc>=0 && nc<rt_COLS);
    }

    function rt_label(g, conn) {
      const lbl = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
      let c = 1;
      for (let r=0; r<rt_ROWS; r++) {
        for (let cc=0; cc<rt_COLS; cc++) {
          if (g[r][cc] && !lbl[r][cc]) {
            const stack = [[r,cc]];
            while (stack.length) {
              const [i,j] = stack.pop();
              if (lbl[i][j]) continue;
              lbl[i][j] = c;
              for (const [ni,nj] of rt_neighbors(i, j, conn))
                if (g[ni][nj] && !lbl[ni][nj]) stack.push([ni,nj]);
            }
            c++;
          }
        }
      }
      return lbl;
    }

    function rt_buildAnimSteps(g, conn) {
      const steps = [];
      const lbl = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
      let c = 1;
      for (let r=0; r<rt_ROWS; r++) {
        for (let cc=0; cc<rt_COLS; cc++) {
          steps.push({type:'raster', r, cc, lbl:lbl.map(a => [...a]), c});
          if (g[r][cc] && !lbl[r][cc]) {
            steps.push({type:'found', r, cc, c, lbl:lbl.map(a => [...a])});
            const stack = [[r,cc]];
            while (stack.length) {
              const [i,j] = stack.pop();
              if (lbl[i][j]) continue;
              lbl[i][j] = c;
              steps.push({type:'fill', i, j, c, lbl:lbl.map(a => [...a])});
              for (const [ni,nj] of rt_neighbors(i, j, conn))
                if (g[ni][nj] && !lbl[ni][nj]) stack.push([ni,nj]);
            }
            c++;
          }
        }
      }
      steps.push({type:'done', lbl:lbl.map(a => [...a]), c:c-1});
      return steps;
    }

    function rt_getColor(c) {
      const p = rt_PALETTE[(c-1) % rt_PALETTE.length];
      return { fill: p[1], stroke: p[0], text: p[0] };
    }

    function rt_drawLabel() {
      const lbl = rt_label(rt_grid, rt_connectivity);
      const numCC = Math.max(0, ...lbl.flat());
      rt_ctx.clearRect(0, 0, rt_W, rt_H);

      for (let r=0; r<rt_ROWS; r++) {
        for (let c=0; c<rt_COLS; c++) {
          const x = rt_PAD + c * rt_CELL, y = rt_PAD + r * rt_CELL;
          const l = lbl[r][c];
          const active = rt_grid[r][c];

          if (active && l) {
            const col = rt_getColor(l);
            rt_ctx.fillStyle = col.fill; rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = col.stroke; rt_ctx.lineWidth = 1.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.fillStyle = col.text;
            rt_ctx.font = 'bold 11px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText(l, x+rt_CELL/2, y+rt_CELL/2);
          } else {
            rt_ctx.fillStyle = '#fafaf7'; rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = '#e4dcc8'; rt_ctx.lineWidth = 0.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.fillStyle = '#8a8371';
            rt_ctx.font = '9px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText('0', x+rt_CELL/2, y+rt_CELL/2);
          }

          if (r===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(c, x+rt_CELL/2, rt_PAD/2); }
          if (c===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(r, rt_PAD/2, y+rt_CELL/2); }
        }
      }

      const numPx = rt_grid.flat().filter(Boolean).length;
      root.querySelector('#rt_statPx').textContent = numPx;
      root.querySelector('#rt_statCC').textContent = numCC;
      root.querySelector('#rt_statConn').textContent = rt_connectivity;
      root.querySelector('#rt_statStep').textContent = '–';
      rt_buildLegend(numCC);
    }

    function rt_drawAnimFrame() {
      if (!rt_animSteps.length) return;
      const step = rt_animSteps[Math.min(rt_animIdx, rt_animSteps.length-1)];
      const lbl = step.lbl;
      const numCC = step.c-1 || 0;

      rt_ctx.clearRect(0, 0, rt_W, rt_H);

      for (let r=0; r<rt_ROWS; r++) {
        for (let c=0; c<rt_COLS; c++) {
          const x = rt_PAD + c * rt_CELL, y = rt_PAD + r * rt_CELL;
          const l = lbl[r][c];
          const active = rt_grid[r][c];

          if (active && l) {
            const col = rt_getColor(l);
            rt_ctx.fillStyle = col.fill; rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = col.stroke; rt_ctx.lineWidth = 1.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.fillStyle = col.text;
            rt_ctx.font = 'bold 11px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText(l, x+rt_CELL/2, y+rt_CELL/2);
          } else if (active) {
            rt_ctx.fillStyle = '#fef5e7';
            rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = '#b9770e'; rt_ctx.lineWidth = 1; rt_ctx.setLineDash([2,2]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.setLineDash([]);
            rt_ctx.fillStyle = '#b9770e';
            rt_ctx.font = '9px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText('?', x+rt_CELL/2, y+rt_CELL/2);
          } else {
            rt_ctx.fillStyle = '#fafaf7';
            rt_ctx.strokeStyle = '#e4dcc8'; rt_ctx.lineWidth = 0.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
          }

          if (r===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(c, x+rt_CELL/2, rt_PAD/2); }
          if (c===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(r, rt_PAD/2, y+rt_CELL/2); }
        }
      }

      if (step.type==='raster' || step.type==='found') {
        const x = rt_PAD + step.cc * rt_CELL, y = rt_PAD + step.r * rt_CELL;
        rt_ctx.strokeStyle = '#c0392b'; rt_ctx.lineWidth = 2; rt_ctx.setLineDash([]);
        rt_ctx.strokeRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
      }
      if (step.type==='fill') {
        const x = rt_PAD + step.j * rt_CELL, y = rt_PAD + step.i * rt_CELL;
        rt_ctx.strokeStyle = '#b9770e'; rt_ctx.lineWidth = 2; rt_ctx.setLineDash([]);
        rt_ctx.strokeRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
      }

      const descriptions = {
        raster: 'Varredura: (' + step.r + ',' + (step.cc||0) + ') — buscando não rotulado',
        found: 'Pixel em (' + step.r + ',' + (step.cc||0) + ')! Flood-fill: C' + step.c,
        fill: 'Preenchimento: (' + (step.i||0) + ',' + (step.j||0) + ') rotulado C' + step.c,
        done: 'Concluído! ' + step.c + ' componente(s).'
      };
      const titles = { raster: 'Varredura', found: 'Semente!', fill: 'Preenchendo...', done: 'Pronto!' };
      
      root.querySelector('#rt_animTitle').textContent = titles[step.type] || '–';
      root.querySelector('#rt_animDesc').textContent = descriptions[step.type] || '–';
      root.querySelector('#rt_statStep').textContent = (rt_animIdx+1) + '/' + rt_animSteps.length;
      root.querySelector('#rt_statCC').textContent = numCC;
      root.querySelector('#rt_statPx').textContent = rt_grid.flat().filter(Boolean).length;
      rt_buildLegend(numCC);
    }

    function rt_buildLegend(n) {
      const box = root.querySelector('#rt_legendBox');
      box.innerHTML = '';
      if (n === 0) {
        box.innerHTML = '<span style="font-size:9.5px; color:#8a8371;">Nenhum píxel ativo</span>';
        return;
      }
      for (let i=1; i<=n; i++) {
        const p = rt_PALETTE[(i-1) % rt_PALETTE.length];
        const span = document.createElement('span');
        span.style.cssText = 'display:flex; align-items:center; gap:4px; font-size:9.5px; color:#5e5a4a;';
        span.innerHTML = '<span style="width:12px; height:12px; border-radius:3px; background:' + p[1] + '; border:1px solid ' + p[0] + '; display:inline-block"></span><span style="color:' + p[0] + '; font-weight:bold">C' + i + '</span>';
        box.appendChild(span);
      }
    }

    function rt_redraw() {
      if (rt_mode === 'label') rt_drawLabel();
      else rt_drawAnimFrame();
    }

    window.rt_setConn = function(c) {
      rt_connectivity = c;
      root.querySelector('#rt_btn4').classList.toggle('rt_active', c===4);
      root.querySelector('#rt_btn8').classList.toggle('rt_active', c===8);
      root.querySelector('#rt_statConn').textContent = c;
      root.querySelector('#rt_connDesc').textContent = c===4 ? '4 vizinhos ortogonais: N, S, L, O' : '8 vizinhos (inclui diagonais)';
      if (rt_mode==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    };

    window.rt_setMode = function(m) {
      rt_mode = m;
      window.rt_stopPlay();
      root.querySelector('#rt_btnLabel').classList.toggle('rt_active', m==='label');
      root.querySelector('#rt_btnAnim').classList.toggle('rt_active', m==='anim');
      root.querySelector('#rt_animCtrl').style.display = m==='anim' ? 'block' : 'none';
      root.querySelector('#rt_animStatus').style.display = m==='anim' ? 'block' : 'none';
      if (m==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    };

    window.rt_animStep = function(d) {
      rt_animIdx = Math.max(0, Math.min(rt_animSteps.length-1, rt_animIdx+d));
      rt_drawAnimFrame();
    };

    window.rt_togglePlay = function() {
      if (rt_playing) window.rt_stopPlay(); else window.rt_startPlay();
    };

    window.rt_startPlay = function() {
      rt_playing = true;
      root.querySelector('#rt_btnPlay').textContent = '⏸';
      function tick() {
        if (rt_animIdx >= rt_animSteps.length-1) { window.rt_stopPlay(); return; }
        rt_animIdx++;
        rt_drawAnimFrame();
        const spd = +root.querySelector('#rt_speedSlider').value;
        const delay = Math.round(1100 - spd * 100);
        rt_playTimer = setTimeout(tick, delay);
      }
      tick();
    };

    window.rt_stopPlay = function() {
      rt_playing = false;
      if (rt_playTimer) { clearTimeout(rt_playTimer); rt_playTimer = null; }
      root.querySelector('#rt_btnPlay').textContent = '▶';
    };

    window.rt_clearGrid = function() {
      rt_grid = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
      rt_animIdx = 0; rt_animSteps = [];
      window.rt_stopPlay();
      rt_redraw();
    };

    const rt_PRESETS = {
      diagonal: () => {
        const g = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
        for (let i=1; i<=5; i++) g[i][i]=1;
        [[2,8],[2,9],[3,8],[3,9],[3,10], [6,1],[6,2],[7,1],[7,2]].forEach(([r,c]) => g[r][c]=1);
        return g;
      },
      letters: () => {
        const g = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
        [[1,1],[2,1],[3,1],[4,1],[5,1],[5,2],[5,3], [1,5],[1,6],[1,7],[2,5],[2,7],[3,5],[3,7],[4,5],[4,7],[5,5],[5,6],[5,7], [1,9],[1,10],[2,9],[3,9],[4,9],[5,9],[5,10]].forEach(([r,c]) => g[r][c]=1);
        return g;
      },
      ring: () => {
        const g = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
        for (let c=2; c<=9; c++) { g[2][c]=1; g[8][c]=1; }
        for (let r=3; r<=7; r++) { g[r][2]=1; g[r][9]=1; }
        for (let c=4; c<=7; c++) { g[4][c]=1; g[6][c]=1; }
        for (let r=5; r<=5; r++) { g[r][4]=1; g[r][7]=1; }
        return g;
      }
    };

    window.rt_loadPreset = function(name) {
      rt_grid = rt_PRESETS[name]();
      rt_animIdx = 0; window.rt_stopPlay();
      if (rt_mode==='anim') rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity);
      rt_redraw();
    };

    function rt_cellAt(e) {
      const rect = rt_cv.getBoundingClientRect();
      const sx = rt_W / rect.width, sy = rt_H / rect.height;
      const raw = e.touches ? e.touches[0] : e;
      const c = Math.floor(((raw.clientX - rect.left) * sx - rt_PAD) / rt_CELL);
      const r = Math.floor(((raw.clientY - rect.top) * sy - rt_PAD) / rt_CELL);
      return {r, c};
    }

    rt_cv.addEventListener('mousedown', function(e){
      const {r, c} = rt_cellAt(e);
      if (r<0 || r>=rt_ROWS || c<0 || c>=rt_COLS) return;
      rt_painting = true;
      rt_paintVal = rt_grid[r][c] ? 0 : 1;
      rt_grid[r][c] = rt_paintVal;
      if (rt_mode==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    });

    rt_cv.addEventListener('mousemove', function(e){
      if (!rt_painting) return;
      const {r, c} = rt_cellAt(e);
      if (r<0 || r>=rt_ROWS || c<0 || c>=rt_COLS || rt_grid[r][c] === rt_paintVal) return;
      rt_grid[r][c] = rt_paintVal;
      if (rt_mode==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    });
    
    document.addEventListener('mouseup', () => rt_painting = false);

    rt_grid = rt_PRESETS.diagonal();
    rt_drawLabel();
  }

  function tryInitSim04RotulacaoCC(){
    var root = document.getElementById('sim-04-rotulacao');
    if (root) initSim04RotulacaoCC(root); else setTimeout(tryInitSim04RotulacaoCC, 200);
  }
  tryInitSim04RotulacaoCC();
})();
</script>
""")

**Figura 4.16:** Simulador interactivo de etiquetado de componentes conexas (*connected component labeling*): visualización de la expansión flood-fill, conectividad-4 y conectividad-8.


<figure id="fig-04-sim-04-rotulacao">
  <img src="imagens/fig-04-sim-04-rotulacao.png" alt=" Simulador interactivo de etiquetado de componentes conexas (*connected component labeling*): visualización de la expansión flood-fill, conectividad-4 y conectividad-8. " style="max-width:80%" />
  <figcaption><strong>Figura 4.16:</strong>  Simulador interactivo de etiquetado de componentes conexas (*connected component labeling*): visualización de la expansión flood-fill, conectividad-4 y conectividad-8. </figcaption>
</figure>

In [42]:
%%writefile tmp/fig_04_rotulacao_didatico.cpp
#define MM_OUT "tmp/fig_04_rotulacao_didatico.png"
//| label: fig-04-rotulacao-didatico
//| fig-cap: "Efeito da conectividade na rotulação (*mm::label0*): pixels diagonalmente adjacentes formam componentes distintas em conectividade-4 e se fundem em conectividade-8."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
    mm::Image f(10, 10);
    unsigned char vals[10][10] = {
        {0,0,0,0,0,0,0,0,0,0},
        {0,1,0,0,0,0,0,0,0,0},
        {0,0,1,0,0,0,0,0,0,0},
        {0,0,0,1,0,0,1,1,0,0},
        {0,0,0,0,0,0,1,1,0,0},
        {0,0,0,0,0,0,0,0,0,0},
        {0,1,1,1,0,0,0,0,0,0},
        {0,1,0,1,0,0,0,1,0,0},
        {0,1,1,1,0,0,0,0,1,0},
        {0,0,0,0,0,0,0,0,0,0}
    };
    for (int y = 0; y < 10; y++)
        for (int x = 0; x < 10; x++)
            f.at(y, x) = vals[y][x] * 255;

    mm::Image B4 = mm::secross();   // conectividade-4
    mm::Image B8 = mm::sebox();     // conectividade-8

    mm::Image lbl4 = mm::label0(f, B4);
    mm::Image lbl8 = mm::label0(f, B8);
    int n4 = 0, n8 = 0;
    for (int i = 0; i < (int)lbl4.data.size(); i++)
        n4 = std::max(n4, (int)lbl4.data[i]);
    for (int i = 0; i < (int)lbl8.data.size(); i++)
        n8 = std::max(n8, (int)lbl8.data[i]);
    std::cout << "Componentes C4: " << n4 << "  |  C8: " << n8 << "\n";

    auto norm_label = [](const mm::Image& lbl, int mx) {
        mm::Image out = lbl;
        for (int y = 0; y < lbl.h; y++) {
            for (int x = 0; x < lbl.w; x++) {
                if (lbl.at(y, x))
                    out.at(y, x) = (int)(lbl.at(y, x) * 255 / mx);
            }
        }
        return out;
    };

    mm::show(
        std::vector<mm::Image>{f, norm_label(lbl4, std::max(n4, 1)), norm_label(lbl8, std::max(n8, 1))},
        MM_OUT,
        std::vector<std::string>{"f original", "C4 (" + std::to_string(n4) + " comp.)", "C8 (" + std::to_string(n8) + " comp.)"},
        3
    );

    return 0;
}

Overwriting tmp/fig_04_rotulacao_didatico.cpp


In [43]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_rotulacao_didatico.cpp -o tmp/fig_04_rotulacao_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_rotulacao_didatico \
  && test -f "tmp/fig_04_rotulacao_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_rotulacao_didatico.png"

Componentes C4: 7  |  C8: 4
[1] f original
[2] C4 (7 comp.)
[3] C8 (4 comp.)


In [44]:
try:
    mm.show(mm.read("tmp/fig_04_rotulacao_didatico.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_rotulacao_didatico.png (ver a versao Python)")

<Figure size 784x450 with 1 Axes>

**Figura 4.17:** Efeito da conectividade na rotulação (*mm::label0*): pixels diagonalmente adjacentes formam componentes distintas em conectividade-4 e se fundem em conectividade-8.


### 4.4.2 Transformada de Distancia

La **Transformada de Distancia** (TD) es un operador que, aplicado a una imagen binaria $f$, produce una imagen en niveles de gris $D$ en la cual cada píxel perteneciente al objeto ($f(x,y)\neq 0$) recibe como valor la distancia geométrica hasta el píxel de fondo ($f(x',y')=0$) más cercano:

$$
D(x,y) = \min_{(x',y') \,:\, f(x',y')=0} \; d\bigl((x,y),\,(x',y')\bigr)
$$

donde $d(\cdot,\cdot)$ es una métrica de distancia — típicamente la distancia Euclidiana ($L_2$). El resultado es una representación topográfica de los objetos: los píxeles ubicados en el interior asumen valores elevados, mientras que los píxeles próximos a los bordes presentan bajos valores de distancia. Los **máximos locales** de $D$ corresponden a los puntos más alejados del borde del objeto, frecuentemente próximos a sus centros geométricos o centros de máxima inscripción — propiedad particularmente útil para la generación automática de marcadores en el algoritmo *watershed*.

> ### 📝 Definición formal mediante erosiones
>
> La TD admite además una interpretación morfológica iterativa, conforme al algoritmo de la [Figura 4.18](#fig-04-sim-alg-distancia2). Considérese una función estructurante $b$ cuyo valor central es nulo y cuyos vecinos poseen costos negativos asociados al desplazamiento. Al aplicar erosiones sucesivas con esta función estructurante particular, los valores de los píxeles de los objetos (que deben asumir la distancia máxima posible de la imagen) se reducen progresivamente según los costos definidos por $b$. El valor acumulado de esta propagación pasa entonces a representar la distancia al fondo según la métrica inducida por la función estructurante.

Esta interpretación se implementa en `mm::dist1()`, que acumula erosiones sucesivas utilizando la operación `mm::ero1()`. Por su parte, `mm::dist()` delega el cálculo de la distancia Euclidiana al operador optimizado de OpenCV `mm::dist(f)`, donde `f` es la imagen binaria de entrada, la distancia L2 especifica la métrica Euclidiana ($L_2$) y `5` indica el uso de una máscara 5×5 para aproximar la distancia con elevada precisión.

La función `dist1` produce una transformada de distancia discreta cuya métrica está determinada por la geometría y los pesos de la función estructurante utilizada. Por ejemplo, utilizando una función estructurante en cruz con costo unitario para los cuatro vecinos ortogonales, se obtiene la distancia de ***Manhattan*** ($L_1$). Otras elecciones de vecindad y pesos inducen métricas diferentes. En cambio, `mm::dist()` calcula una aproximación eficiente de la distancia Euclidiana ($L_2$).

Por exigir sucesivas erosiones sobre toda la imagen, el enfoque `dist1` posee un costo computacional significativamente mayor que `mm::dist()`, siendo empleado en este libro principalmente con fines didácticos y para evidenciar la relación entre morfología matemática y transformadas de distancia.

In [45]:
# @title { display-mode: "form" }
import numpy as np, cv2, os  # [pdi] passthrough: garante imports desta trilha
# Prefijo exclusivo para evitar conflicto con otras celdas del cuaderno
PREFIX = "dist2"

from IPython.display import HTML
HTML(f'''
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:var(--font-sans)}}
.{PREFIX}-tabs{{display:flex;gap:6px;margin-bottom:20px;flex-wrap:wrap}}
.{PREFIX}-tab{{padding:6px 14px;border-radius:20px;font-size:13px;cursor:pointer;border:0.5px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all .15s;white-space:nowrap}}
.{PREFIX}-tab.active{{background:var(--color-text-primary);color:var(--color-background-primary);border-color:transparent}}
.{PREFIX}-panel{{display:none}}.{PREFIX}-panel.active{{display:block}}

.algo-wrap{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.algo-header{{padding:14px 20px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;align-items:center;gap:10px}}
.algo-icon{{width:32px;height:32px;border-radius:var(--border-radius-md);display:flex;align-items:center;justify-content:center;font-size:16px;background:#E6F1FB;color:#185FA5}}
.algo-title{{font-size:14px;font-weight:500;color:var(--color-text-primary);text-align:left}}
.algo-sub{{font-size:12px;color:var(--color-text-secondary);margin-top:1px;text-align:left}}
.algo-body{{padding:20px;text-align:left}}

.step{{display:flex;gap:12px;margin-bottom:14px;align-items:flex-start}}
.step:last-child{{margin-bottom:0}}
.step-num{{min-width:24px;height:24px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0;margin-top:1px}}
.step-text{{font-size:13.5px;line-height:1.65;color:var(--color-text-primary);text-align:left}}
.step-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 6px;border-radius:4px;color:var(--color-text-primary)}}
.step-sub{{margin-top:8px;border-left:2px solid var(--color-border-secondary);padding-left:12px;display:flex;flex-direction:column;gap:5px}}
.step-sub-item{{font-size:13px;color:var(--color-text-secondary);line-height:1.55;text-align:left}}
.step-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.while-box{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:11px 13px;margin-top:8px;text-align:left}}
.while-head{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.06em;margin-bottom:7px}}
.note{{margin-top:16px;padding-top:12px;border-top:0.5px solid var(--color-border-tertiary);font-size:12px;color:var(--color-text-secondary);font-style:italic;line-height:1.5;text-align:left}}

.num-teal{{background:#E1F5EE;color:#0F6E56}}
.num-purple{{background:#EEEDFE;color:#534AB7}}
.num-amber{{background:#FAEEDA;color:#854F0B}}
.num-coral{{background:#FAECE7;color:#993C1D}}
.num-blue{{background:#E6F1FB;color:#185FA5}}
.num-gray{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}

.card-step{{display:flex;border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);overflow:hidden;margin-bottom:8px}}
.card-step:last-child{{margin-bottom:0}}
.card-badge{{min-width:48px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0}}
.card-content{{padding:10px 14px;flex:1;text-align:left}}
.card-label{{font-size:10px;font-weight:500;letter-spacing:.08em;text-transform:uppercase;margin-bottom:3px}}
.card-text{{font-size:13.5px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.card-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.card-sub{{margin-top:6px;display:flex;flex-direction:column;gap:3px}}
.card-sub-item{{font-size:12.5px;color:var(--color-text-secondary);padding-left:10px;border-left:2px solid var(--color-border-secondary);line-height:1.5;text-align:left}}
.card-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 4px;border-radius:3px;color:var(--color-text-primary)}}

.tl{{position:relative;padding:4px 0 4px 36px;text-align:left}}
.tl-line{{position:absolute;left:11px;top:20px;bottom:20px;width:1.5px;background:var(--color-border-secondary);border-radius:2px}}
.tl-step{{display:flex;gap:0;margin-bottom:18px;position:relative}}
.tl-step:last-child{{margin-bottom:0}}
.tl-dot-wrap{{position:absolute;left:-36px;top:2px;display:flex;flex-direction:column;align-items:center;gap:3px}}
.tl-dot{{width:14px;height:14px;border-radius:50%;border:2px solid var(--color-border-secondary);background:var(--color-background-primary);transition:all .2s;z-index:1}}
.tl-step:hover .tl-dot{{background:var(--color-text-primary);border-color:var(--color-text-primary)}}
.tl-num{{font-size:9px;color:var(--color-text-secondary);font-weight:500;letter-spacing:.04em}}
.tl-title{{font-size:13.5px;font-weight:500;color:var(--color-text-primary);margin-bottom:3px;text-align:left}}
.tl-desc{{font-size:13px;color:var(--color-text-secondary);line-height:1.6;text-align:left}}
.tl-desc code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.code-wrap{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.code-bar{{background:var(--color-background-secondary);padding:10px 16px;display:flex;align-items:center;justify-content:space-between;border-bottom:0.5px solid var(--color-border-tertiary)}}
.code-lang{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.08em}}
.code-body{{padding:18px 20px;overflow-x:auto;text-align:left}}
.code-body pre{{font-family:var(--font-mono);font-size:13px;line-height:1.75;color:var(--color-text-primary);margin:0;white-space:pre;text-align:left}}
.kw{{color:#7C3AED}} .fn{{color:#0369A1}} .cm{{color:#6B7280;font-style:italic}} .st{{color:#059669}} .num-lit{{color:#DC2626}}

.ann-line{{display:flex;align-items:flex-start;gap:10px;margin-bottom:8px;padding:10px 12px;background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);text-align:left}}
.ann-badge{{min-width:20px;height:20px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;flex-shrink:0;margin-top:1px}}
.ann-text{{font-size:13px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.ann-text code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.metric-grid{{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin-top:16px}}
.metric-card{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:12px 14px;text-align:left}}
.metric-title{{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.07em;margin-bottom:6px}}
.metric-formula{{font-family:var(--font-mono);font-size:12px;color:var(--color-text-primary);margin-bottom:4px}}
.metric-desc{{font-size:12px;color:var(--color-text-secondary);line-height:1.5}}
.metric-elem{{display:inline-grid;grid-template-columns:repeat(3,16px);grid-template-rows:repeat(3,16px);gap:2px;margin-top:6px}}
.mc{{width:16px;height:16px;border-radius:2px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:500}}
.mc-on{{background:#E6F1FB;color:#185FA5}}
.mc-off{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}
.mc-ctr{{background:#185FA5;color:#fff}}
</style>

<h2 class="sr-only" style="position:absolute;left:-9999px">Transformada de distância por erosões numéricas sucessivas — painel interativo</h2>

<div class="{PREFIX}-tabs" id="{PREFIX}-tabs-container">
  <div class="{PREFIX}-tab" data-idx="0"><i class="ti ti-list-numbers" aria-hidden="true"></i> Passo a passo</div>
  <div class="{PREFIX}-tab" data-idx="1"><i class="ti ti-cards" aria-hidden="true"></i> Cards</div>
  <div class="{PREFIX}-tab" data-idx="2"><i class="ti ti-timeline" aria-hidden="true"></i> Linha do tempo</div>
  <div class="{PREFIX}-tab" data-idx="3"><i class="ti ti-code" aria-hidden="true"></i> Código Python</div>
  <div class="{PREFIX}-tab active" data-idx="4"><i class="ti ti-git-branch" aria-hidden="true"></i> Fluxograma</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-0">
<div class="algo-wrap">
  <div class="algo-header">
    <div class="algo-icon"><i class="ti ti-ripple" aria-hidden="true"></i></div>
    <div><div class="algo-title">Transformada de distância por erosão numérica</div><div class="algo-sub">Propagação matemática de distâncias via elemento estruturante com pesos</div></div>
  </div>
  <div class="algo-body">
    <div class="step"><div class="step-num num-teal">1</div><div class="step-text">Inicializar a imagem de trabalho fazendo uma cópia da original: <code>g ← f.copy()</code>. Os pixels de fundo (0) servem como fontes de distância nula.</div></div>
    <div class="step">
      <div class="step-num num-coral">2</div>
      <div class="step-text">
        Entrar em um laço infinito de erosões com pesos (ponto fixo):
        <div class="step-sub">
          <div class="step-sub-item">Salvar estado anterior: <code>f ← g.copy()</code>.</div>
          <div class="step-sub-item">Erodir: <code>g ← ero1(g, b)</code>, aplicando a subtração local de pesos e computando o valor mínimo para cada vizinhança.</div>
        </div>
      </div>
    </div>
    <div class="step"><div class="step-num num-blue">3</div><div class="step-text">Verificar convergência: se <code>f</code> for idêntica a <code>g</code> (<code>array_equal</code>), a frente de onda de distâncias se estabilizou. Romper o laço (<code>break</code>).</div></div>
    <div class="step"><div class="step-num num-blue">4</div><div class="step-text">Retornar a matriz modificada <code>g</code> contendo o mapa exato de distâncias.</div></div>
    <div class="note">Nesta abordagem morfológica numérica, não há incremento artificial ou contador. A distância propaga-se de fora para dentro porque a erosão contínua puxa o valor <code>0</code> do fundo e o decrementa matematicamente (subtraindo os pesos negativos como <code>-1</code>), fazendo com que os valores escalem radialmente.</div>
  </div>
</div>

<div class="metric-grid">
  <div class="metric-card">
    <div class="metric-title" style="color:#185FA5">Cruz — L₁ (Manhattan)</div>
    <div class="metric-formula">B_cruz [y,x]</div>
    <div class="metric-desc">Pesos: Centro=0, Lados=-1, Cantos=-inf</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#854F0B">Erosão de Cinzas</div>
    <div class="metric-formula">f[vy,vx] - bv</div>
    <div class="metric-desc">Subtrai o peso e busca o valor mínimo local</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#0F6E56">Convergência</div>
    <div class="metric-formula">f == g</div>
    <div class="metric-desc">Para quando nenhum pixel muda de valor</div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-1">
  <div class="card-step">
    <div class="card-badge num-teal">01</div>
    <div class="card-content">
      <div class="card-label" style="color:#0F6E56">Inicialização</div>
      <div class="card-text">Clonar imagem de entrada: <code>g ← f.copy()</code>. O objeto possui intensidade alta (255) e o fundo possui intensidade 0.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">02</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Mapeamento Local (ero1)</div>
      <div class="card-text">Para cada coordenada <code>(y, x)</code>, buscar o mínimo valor da operação <code>f[vy, vx] - bv</code> aplicada à sua vizinhança estruturante.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">03</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Loop Iterativo</div>
      <div class="card-text">Atualizar sequencialmente: <code>f = g.copy()</code> seguido de <code>g = ero1(g, b)</code>. Os valores nulos propagam-se para o interior do objeto.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-blue">04</div>
    <div class="card-content">
      <div class="card-label" style="color:#185FA5">Critério de Parada</div>
      <div class="card-text">Se <code>np.array_equal(f, g)</code>, significa que o mapa de distâncias atingiu o equilíbrio estável e a propagação terminou.</div>
    </div>
  </div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-2">
<div class="tl">
  <div class="tl-line"></div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">01</span></div>
    <div><div class="tl-title">Cópia de Trabalho</div><div class="tl-desc">Prepara a matriz inicial `g`.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">02</span></div>
    <div>
      <div class="tl-title">Loop de Erosão de Escala</div>
      <div class="while-box">
        <div class="while-head">while True</div>
        <div class="step-sub" style="border-color:var(--color-border-tertiary)">
          <div class="step-sub-item">Guarda estado: <code>f ← g.copy()</code></div>
          <div class="step-sub-item">Aplica erosão com pesos: <code>g ← ero1(g, b)</code></div>
        </div>
      </div>
    </div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">03</span></div>
    <div><div class="tl-title">Estabilização Espacial</div><div class="tl-desc">Condição de parada acionada assim que <code>np.array_equal(f, g)</code> se torna verdadeiro.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">04</span></div>
    <div><div class="tl-title">Retorno Numérico</div><div class="tl-desc">Retorna <code>g</code> contendo as distâncias calculadas pela subtração cumulativa dos pesos.</div></div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-3">
<div class="code-wrap">
  <div class="code-bar">
    <span class="code-lang"><i class="ti ti-brand-python" aria-hidden="true" style="font-size:14px;vertical-align:-2px;margin-right:5px"></i>morph_dist.py</span>
    <span style="font-size:11px;color:var(--color-text-secondary)">Erosão numérica iterativa</span>
  </div>
  <div class="code-body">
<pre><span class="kw">@staticmethod</span>
<span class="kw">def</span> <span class="fn">ero1</span>(f, b):
    g = np.empty_like(f)
    <span class="kw">for</span> y <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">0</span>]):
        <span class="kw">for</span> x <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">1</span>]):
            g[y,x] = <span class="num-lit">255</span>
            <span class="kw">for</span> vy,vx,bv <span class="kw">in</span> mm._viz(f,b,y,x):
                <span class="kw">if</span> np.isinf(bv): <span class="kw">continue</span> 
                val = int(f[vy,vx]) - int(bv)
                <span class="kw">if</span> g[y,x] > val: 
                    g[y,x] = max(0, val)
    <span class="kw">return</span> g

<span class="kw">@staticmethod</span>
<span class="kw">def</span> <span class="fn">dist1</span>(f, b):
    g = f.copy()
    <span class="kw">while</span> <span class="fn">True</span>:
        f = g.copy()
        g = mm.ero1(g, b)
        <span class="kw">if</span> np.array_equal(f, g): 
            <span class="kw">break</span>
    <span class="kw">return</span> g</pre>
  </div>
</div>

<div style="margin-top:16px;display:flex;flex-direction:column;gap:8px;text-align:left">
  <div class="ann-line"><div class="ann-badge num-teal">1</div><div class="ann-text"><code>g[y,x] = 255</code> — Inicializa o elemento com o valor máximo antes de computar o operador de mínimo da erosão.</div></div>
  <div class="ann-line"><div class="ann-badge num-coral">2</div><div class="ann-text"><code>f[vy,vx] - bv</code> — Subtrai o peso associado da vizinhança. Como os pesos da cruz externa são negativos (ex: <code>-1</code>), a operação torna-se uma adição matemática (<code>f[vy,vx] - (-1) = f[vy,vx] + 1</code>) propagando a distância a partir das bordas zeradas.</div></div>
  <div class="ann-line"><div class="ann-badge num-blue">3</div><div class="ann-text"><code>np.array_equal(f, g)</code> — Critério de convergência exato por estabilização de ponto fixo.</div></div>
</div>
</div>

<div class="{PREFIX}-panel active" id="{PREFIX}-p-4">
<svg viewBox="0 0 560 720" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:560px;display:block">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#888780"/></marker>
    <marker id="arr-b" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#185FA5"/></marker>
    <marker id="arr-r" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#993C1D"/></marker>
    <marker id="arr-g" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#0F6E56"/></marker>
  </defs>

  <ellipse cx="280" cy="36" rx="60" ry="22" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/>
  <text x="280" y="41" text-anchor="middle" font-size="13" font-weight="500" fill="#085041">início</text>

  <rect x="150" y="85" width="260" height="46" rx="6" fill="#E1F5EE" stroke="#9FE1CB" stroke-width="1"/>
  <text x="280" y="104" text-anchor="middle" font-size="12" font-weight="500" fill="#0F6E56">inicializar mapa</text>
  <text x="280" y="120" text-anchor="middle" font-size="12" fill="#085041">g ← f.copy()</text>

  <rect x="150" y="165" width="260" height="46" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="184" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">salvar estado anterior</text>
  <text x="280" y="200" text-anchor="middle" font-size="12" fill="#712B13">f ← g.copy()</text>

  <rect x="150" y="245" width="260" height="46" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="264" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">executar erosão com pesos</text>
  <text x="280" y="280" text-anchor="middle" font-size="12" fill="#712B13">g ← ero1(g, b)</text>

  <polygon points="280,325 390,355 280,385 170,355" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="351" text-anchor="middle" font-size="12" fill="#0C447C">array_equal(f, g)</text>
  <text x="280" y="367" text-anchor="middle" font-size="12" fill="#0C447C">estabilizou?</text>

  <line x1="170" y1="355" x2="100" y2="355" stroke="#993C1D" stroke-width="1.2"/>
  <line x1="100" y1="355" x2="100" y2="188" stroke="#993C1D" stroke-width="1.2"/>
  <line x1="100" y1="188" x2="150" y2="188" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="120" y="345" text-anchor="middle" font-size="11" fill="#993C1D">não</text>

  <line x1="280" y1="385" x2="280" y2="430" stroke="#0F6E56" stroke-width="1.2" marker-end="url(#arr-g)"/>
  <text x="295" y="405" font-size="11" fill="#0F6E56">sim</text>

  <rect x="150" y="430" width="260" height="46" rx="6" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="280" y="458" text-anchor="middle" font-size="12" font-weight="500" fill="#27500A">retornar g</text>

  <ellipse cx="280" cy="525" rx="50" ry="20" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="280" y="529" text-anchor="middle" font-size="13" font-weight="500" fill="#27500A">fim</text>
  <line x1="280" y1="476" x2="280" y2="505" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>

  <line x1="280" y1="58" x2="280" y2="85" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="131" x2="280" y2="165" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="211" x2="280" y2="245" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="291" x2="280" y2="325" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>

  <text x="280" y="600" text-anchor="middle" font-size="11.5" fill="#888780" font-style="italic">Ponto Fixo Numérico: A distância emerge da propagação matemática do valor zero.</text>
</svg>
</div>

<script>
(function() {{
  var container = document.getElementById('{PREFIX}-tabs-container');
  if (!container) return;
  
  var tabs = container.querySelectorAll('.{PREFIX}-tab');
  var panels = [
    document.getElementById('{PREFIX}-p-0'),
    document.getElementById('{PREFIX}-p-1'),
    document.getElementById('{PREFIX}-p-2'),
    document.getElementById('{PREFIX}-p-3'),
    document.getElementById('{PREFIX}-p-4')
  ];

  tabs.forEach(function(tab) {{
    tab.onclick = function() {{
      var idx = parseInt(this.getAttribute('data-idx'));
      
      tabs.forEach(function(t) {{ t.classList.remove('active'); }});
      panels.forEach(function(p) {{ if(p) p.classList.remove('active'); }});
      
      this.classList.add('active');
      if(panels[idx]) panels[idx].classList.add('active');
    }};
  }});
}})();
</script>
''')

**Figura 4.18:** Algoritmo de la Transformada de Distancia.


<figure id="fig-04-sim-alg-distancia2">
  <img src="imagens/fig-04-sim-alg-distancia2.png" alt=" Algoritmo de la Transformada de Distancia. " style="max-width:80%" />
  <figcaption><strong>Figura 4.18:</strong>  Algoritmo de la Transformada de Distancia. </figcaption>
</figure>

La [Figura 4.19](#fig-04-sim-04-distancia) presenta un simulador interactivo de la TD: es posible posicionar el cursor sobre diferentes píxeles del objeto y observar, en tiempo real, el valor de la distancia asociado a esa posición, es decir, la distancia hasta el píxel de fondo más cercano. La [Figura 4.20](#fig-04-distancia-didatico) presenta un ejemplo práctico de esta ejecución en entorno Python.

In [46]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-distancia" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-distancia * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-distancia canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; }
  #sim-04-distancia button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-distancia button:hover { background: #e8dfcf; }
  #sim-04-distancia button.sim04_td_act { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-distancia .sim04_td_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim04_td_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim04_td_grid_stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 14px; }
  .sim04_td_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim04_td_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim04_td_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  
  .sim04_td_se_grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 4px; max-width: 120px; margin: 6px auto; }
  .sim04_td_se_btn { padding: 6px!important; font-size: 10px!important; font-family: monospace; font-weight: 700; }
  .sim04_td_se_btn.sim04_td_inf { color: #8a8371!important; background: #fafaf7!important; border-color: #e4dcc8!important; font-weight: normal; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🗺️ Simulador: Transformada de Distancia (TD)</span>
  <span class="sim04_td_pill">Fronteras en +∞ (144)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="sim04_td_grid_stats">
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Píxeles Activos</div>
      <div id="sim04_td_statPx" class="sim04_td_stat_value" style="color:#27ae60;">0</div>
    </div>
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Dist. Máx.</div>
      <div id="sim04_td_statMax" class="sim04_td_stat_value" style="color:#2980b9;">0</div>
    </div>
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Métrica Actual</div>
      <div id="sim04_td_statMetric" class="sim04_td_stat_value" style="color:#b9770e; font-size:11px;">L∞ (Chebyshev)</div>
    </div>
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Iteración (k)</div>
      <div id="sim04_td_statStep" class="sim04_td_stat_value" style="color:#c0392b;">–</div>
    </div>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start;">
    
    <!-- Canvas e Legenda -->
    <div style="flex:2; min-width:260px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
      <canvas id="sim04_td_Canvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Clic para activar/desactivar píxeles · Arrastra para pintar
      </div>
      <div style="display:flex; flex-wrap:wrap; gap:6px; margin-top:10px; align-items:center; justify-content:center;" id="sim04_td_legendBox"></div>
    </div>

    <!-- Painel de Controles Lateral -->
    <div style="flex:1; min-width:200px; display:flex; flex-direction:column; gap:12px;">
      
      <!-- Editor do Elemento Estruturante (b) -->
      <div class="sim04_td_panel" style="text-align:center;">
        <div style="font-size:11px; font-weight:700; margin-bottom:2px; text-align:left; color:#5e5a4a;">
          Elemento Estructurante (b)
        </div>
        <div style="font-size:9.5px; color:#8a8371; margin-bottom:6px; text-align:left;">Clic para cambiar los pesos:</div>
        
        <div class="sim04_td_se_grid">
          <button id="sim04_td_se_0_0" class="sim04_td_se_btn sim04_td_active" data-se="0,0">-1</button>
          <button id="sim04_td_se_0_1" class="sim04_td_se_btn sim04_td_active" data-se="0,1">-1</button>
          <button id="sim04_td_se_0_2" class="sim04_td_se_btn sim04_td_active" data-se="0,2">-1</button>
          
          <button id="sim04_td_se_1_0" class="sim04_td_se_btn sim04_td_active" data-se="1,0">-1</button>
          <button id="sim04_td_se_1_1" class="sim04_td_se_btn" style="background:#e4dcc8; cursor:not-allowed; color:#8a8371;" disabled>0</button>
          <button id="sim04_td_se_1_2" class="sim04_td_se_btn sim04_td_active" data-se="1,2">-1</button>
          
          <button id="sim04_td_se_2_0" class="sim04_td_se_btn sim04_td_active" data-se="2,0">-1</button>
          <button id="sim04_td_se_2_1" class="sim04_td_se_btn sim04_td_active" data-se="2,1">-1</button>
          <button id="sim04_td_se_2_2" class="sim04_td_se_btn sim04_td_active" data-se="2,2">-1</button>
        </div>
        
        <div style="display:flex; flex-wrap:wrap; gap:4px; margin-top:8px;">
          <button data-se-preset="cross" style="flex:1; font-size:9.5px;">L1 (Cruz)</button>
          <button data-se-preset="square" style="flex:1; font-size:9.5px;">L∞ (Cuad.)</button>
          <button data-se-preset="chamfer" style="flex:1; font-size:9.5px; min-width:90px;">Chamfer 3-4</button>
        </div>
      </div>

      <!-- Modo de Visualização -->
      <div class="sim04_td_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Visualización
        </div>
        <div style="display:flex; gap:6px;">
          <button id="sim04_td_btnLabel" class="sim04_td_active" data-mode="label" style="flex:1; justify-content:center;">Final (TD)</button>
          <button id="sim04_td_btnAnim" data-mode="anim" style="flex:1; justify-content:center;">Animado</button>
        </div>
      </div>

      <!-- Controles de Animação -->
      <div id="sim04_td_animCtrl" class="sim04_td_panel" style="display:none;">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Propagación Paso a Paso
        </div>
        <div style="display:flex; gap:6px; margin-bottom:8px;">
          <button data-action="prev" style="flex:1; justify-content:center;">◀</button>
          <button id="sim04_td_btnPlay" data-action="play" style="flex:1; justify-content:center;">▶</button>
          <button data-action="next" style="flex:1; justify-content:center;">▶▶</button>
        </div>
        <div style="display:flex; align-items:center; gap:8px;">
          <label style="font-size:9.5px; color:#8a8371; font-weight:700;">Vel.</label>
          <input type="range" id="sim04_td_speedSlider" min="1" max="10" value="5" style="flex:1; max-width:160px; height:4px; cursor:pointer;">
        </div>
      </div>

      <!-- Exemplos f -->
      <div class="sim04_td_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Ejemplos Predefinidos
        </div>
        <div style="display:flex; flex-direction:column; gap:5px;">
          <button data-preset="square" style="justify-content:flex-start;">■ Cuadrado Lleno</button>
          <button data-preset="lshape" style="justify-content:flex-start;">╚ Forma en L</button>
          <button data-preset="ring" style="justify-content:flex-start;">◎ Anillo con Agujero</button>
          <button data-preset="corner0" style="justify-content:flex-start;">↘ Fondo en (0,0)</button>
        </div>
      </div>

      <!-- Botão de Limpeza -->
      <button data-action="clear" style="justify-content:center; border-color:#f5b7b1; color:#c0392b; background:#fdecea;">
        🗑️ Limpiar Cuadrícula
      </button>

    </div>

  </div>

</div>
</div>

<script>
(function(){
  function initSim04TD(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const td_COLS = 12, td_ROWS = 12, td_CELL = 22, td_PAD = 10;
    const td_W = td_COLS * td_CELL + td_PAD * 2, td_H = td_ROWS * td_CELL + td_PAD * 2;
    const td_cv = root.querySelector('#sim04_td_Canvas');
    td_cv.width = td_W; 
    td_cv.height = td_H;
    const td_ctx = td_cv.getContext('2d');
    
    const td_MAX_DIST = 144;

    const td_COLORS = ['#ebf4fd', '#a9cce3', '#73c6b6', '#52be80', '#27ae60', '#2980b9', '#1b4f72', '#5b2c6f', '#4a235a'];
    function td_getColor(d, maxD) {
      if(d === 0) return { fill: 'transparent', text: '#8a8371', stroke: '#e4dcc8' };
      if(d >= td_MAX_DIST) return { fill: '#fdecea', text: '#c0392b', stroke: '#f5b7b1' };
      
      let ratio = maxD > 1 ? (d - 1) / (maxD - 1) : 0;
      let idx = Math.max(0, Math.min(td_COLORS.length - 1, Math.floor(ratio * (td_COLORS.length - 1))));
      
      const bg = td_COLORS[idx];
      const text = (idx > 4) ? '#ffffff' : '#26241d';
      return { fill: bg, text: text, stroke: '#2980b9' };
    }

    let td_grid = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
    
    const td_VALS = [-1, -2, -3, -4, -5, -6, -7, -8, -9, -Infinity];
    let td_b = [
      [-1, -1, -1],
      [-1,  0, -1],
      [-1, -1, -1]
    ];
    
    let td_mode = 'label';
    let td_painting = false;
    let td_paintVal = 1;

    let td_animSteps = [];
    let td_animIdx = 0;
    let td_playing = false;
    let td_playTimer = null;

    function td_compute() {
      let f = td_grid;
      let g = Array.from({length: td_ROWS}, (_, r) => 
        Array.from({length: td_COLS}, (_, c) => f[r][c] ? td_MAX_DIST : 0)
      );
      let steps = [];
      let k = 0;

      const copy = arr => arr.map(row => [...row]);
      steps.push({ g: copy(g), eroded: Array.from({length:td_ROWS}, ()=>new Array(td_COLS).fill(0)), k: k });

      let activePx = f.flat().reduce((a,b)=>a+b, 0);
      if(activePx === 0) return { D: g, steps, max: 0 };

      while (true) {
        k++;
        let nextG = Array.from({length: td_ROWS}, () => new Array(td_COLS).fill(0));
        let changed = false;
        let erodedCells = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));

        for (let r=0; r<td_ROWS; r++) {
          for (let c=0; c<td_COLS; c++) {
            if (f[r][c] === 1) {
              let min_val = g[r][c];
              
              for (let dr=-1; dr<=1; dr++) {
                for (let dc=-1; dc<=1; dc++) {
                  let b_val = td_b[dr+1][dc+1];
                  if (b_val !== -Infinity && !(dr===0 && dc===0)) {
                    let nr = r + dr, nc = c + dc;
                    let neighbor_val = td_MAX_DIST;
                    
                    if (nr >= 0 && nr < td_ROWS && nc >= 0 && nc < td_COLS) {
                      neighbor_val = g[nr][nc];
                    }
                    
                    let val = neighbor_val - b_val; 
                    if (val < min_val) {
                      min_val = val;
                    }
                  }
                }
              }
              nextG[r][c] = min_val;
              if (min_val !== g[r][c]) {
                changed = true;
                erodedCells[r][c] = 1;
              }
            } else {
              nextG[r][c] = 0;
            }
          }
        }
        
        if (!changed) {
          steps.push({ g: copy(nextG), eroded: erodedCells, k: k, done: true });
          break;
        }
        
        steps.push({ g: copy(nextG), eroded: erodedCells, k: k });
        g = nextG;
      }
      
      let maxD = 0;
      for(let r=0; r<td_ROWS; r++){
        for(let c=0; c<td_COLS; c++){
           if(f[r][c] === 1 && g[r][c] < td_MAX_DIST && g[r][c] > maxD) maxD = g[r][c];
        }
      }
      return { D: g, steps, max: maxD };
    }

    function td_drawFinal() {
      const res = td_compute();
      const D = res.D;
      td_ctx.clearRect(0, 0, td_W, td_H);

      for (let r=0; r<td_ROWS; r++) {
        for (let c=0; c<td_COLS; c++) {
          const x = td_PAD + c * td_CELL, y = td_PAD + r * td_CELL;
          const d = D[r][c];
          const active = td_grid[r][c];

          if (active) {
            const col = td_getColor(d, res.max);
            td_ctx.fillStyle = col.fill; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            td_ctx.strokeStyle = col.stroke; td_ctx.lineWidth = 1; td_ctx.setLineDash([]);
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = col.text;
            td_ctx.font = d >= td_MAX_DIST ? 'bold 9px monospace' : 'bold 11px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText(d >= td_MAX_DIST ? '144' : d, x+td_CELL/2, y+td_CELL/2);
          } else {
            td_ctx.fillStyle = '#fafaf7'; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            td_ctx.strokeStyle = '#e4dcc8'; td_ctx.lineWidth = 0.5; td_ctx.setLineDash([]);
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = '#8a8371';
            td_ctx.font = '9px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText('0', x+td_CELL/2, y+td_CELL/2);
          }

          if (r===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(c, x+td_CELL/2, td_PAD/2); }
          if (c===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(r, td_PAD/2, y+td_CELL/2); }
        }
      }

      root.querySelector('#sim04_td_statPx').textContent = td_grid.flat().filter(Boolean).length;
      root.querySelector('#sim04_td_statMax').textContent = res.max;
      root.querySelector('#sim04_td_statStep').textContent = 'Finalizado';
      td_buildLegend(res.D, res.max);
    }

    function td_drawAnimFrame() {
      if (!td_animSteps.length) return;
      const step = td_animSteps[Math.min(td_animIdx, td_animSteps.length-1)];
      const D = step.g;
      const eroded = step.eroded;
      
      let currentMaxD = 0;
      for(let r=0; r<td_ROWS; r++) {
        for(let c=0; c<td_COLS; c++) {
          if(td_grid[r][c] && D[r][c] < td_MAX_DIST && D[r][c] > currentMaxD) currentMaxD = D[r][c];
        }
      }

      td_ctx.clearRect(0, 0, td_W, td_H);

      for (let r=0; r<td_ROWS; r++) {
        for (let c=0; c<td_COLS; c++) {
          const x = td_PAD + c * td_CELL, y = td_PAD + r * td_CELL;
          const d = D[r][c];
          const active = td_grid[r][c];

          if (active) {
            let maxValArray = td_animSteps[td_animSteps.length-1].g.flat().filter(v=>v<td_MAX_DIST);
            const globalMax = maxValArray.length > 0 ? Math.max(...maxValArray) : 1;
            const col = td_getColor(d, globalMax);
            td_ctx.fillStyle = col.fill; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            
            if(eroded[r][c] === 1) {
               td_ctx.strokeStyle = '#b9770e'; td_ctx.lineWidth = 2; td_ctx.setLineDash([2,2]);
               td_ctx.fillStyle = 'rgba(185, 119, 14, 0.15)'; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            } else {
               td_ctx.strokeStyle = col.stroke; td_ctx.lineWidth = 1; td_ctx.setLineDash([]);
            }
            
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = col.text;
            td_ctx.font = d >= td_MAX_DIST ? 'bold 9px monospace' : 'bold 11px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText(d >= td_MAX_DIST ? '144' : d, x+td_CELL/2, y+td_CELL/2);
            td_ctx.setLineDash([]);
          } else {
            td_ctx.fillStyle = '#fafaf7';
            td_ctx.strokeStyle = '#e4dcc8'; td_ctx.lineWidth = 0.5; td_ctx.setLineDash([]);
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = '#8a8371';
            td_ctx.font = '9px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText('0', x+td_CELL/2, y+td_CELL/2);
          }

          if (r===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(c, x+td_CELL/2, td_PAD/2); }
          if (c===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(r, td_PAD/2, y+td_CELL/2); }
        }
      }

      root.querySelector('#sim04_td_statMax').textContent = currentMaxD;
      root.querySelector('#sim04_td_statPx').textContent = td_grid.flat().filter(Boolean).length;
      root.querySelector('#sim04_td_statStep').textContent = step.done ? 'Concluído' : step.k;
      td_buildLegend(D, currentMaxD);
    }

    function td_buildLegend(D_matrix, maxD) {
      const box = root.querySelector('#sim04_td_legendBox');
      box.innerHTML = '';
      
      let uniqueD = new Set();
      D_matrix.forEach(row => row.forEach(val => { 
        if(val > 0 && val < td_MAX_DIST) uniqueD.add(val); 
      }));
      let sortedD = Array.from(uniqueD).sort((a,b) => a-b);
      
      if (sortedD.length === 0) {
        box.innerHTML = '<span style="font-size:9.5px; color:#8a8371;">Sem propagação visível</span>';
        return;
      }
      
      let displayArr = sortedD;
      if (sortedD.length > 12) {
          displayArr = sortedD.filter((_, i) => i === 0 || i === sortedD.length-1 || i%Math.ceil(sortedD.length/10) === 0);
      }
      
      box.innerHTML = '<span style="font-size:9.5px; color:#8a8371; width:100%;">Cores (exceto 144):</span>';
      
      displayArr.forEach(d => {
        const col = td_getColor(d, maxD);
        const span = document.createElement('span');
        span.style.cssText = 'display:flex; align-items:center; gap:4px; font-size:9.5px; color:#5e5a4a;';
        span.innerHTML = '<span style="width:12px; height:12px; border-radius:3px; background:' + col.fill + '; border:1px solid ' + col.stroke + '; display:inline-block"></span><span>' + d + '</span>';
        box.appendChild(span);
      });
    }

    function td_updateSEUI() {
      let typeName = "Personalizada";
      let isCross = td_b[0][1]===-1 && td_b[1][0]===-1 && td_b[1][2]===-1 && td_b[2][1]===-1 && td_b[0][0]===-Infinity && td_b[0][2]===-Infinity && td_b[2][0]===-Infinity && td_b[2][2]===-Infinity;
      let isSquare = td_b.every((r, i) => r.every((v, j) => (i===1 && j===1) ? true : v===-1));
      let isChamfer = td_b[0][1]===-3 && td_b[1][0]===-3 && td_b[1][2]===-3 && td_b[2][1]===-3 && td_b[0][0]===-4 && td_b[0][2]===-4 && td_b[2][0]===-4 && td_b[2][2]===-4;

      if (isCross) typeName = "L1 (City-Block)";
      else if (isSquare) typeName = "L∞ (Chebyshev)";
      else if (isChamfer) typeName = "Chamfer 3-4";

      root.querySelector('#sim04_td_statMetric').textContent = typeName;

      for (let r=0; r<3; r++) {
        for (let c=0; c<3; c++) {
          if(r===1 && c===1) continue;
          const btn = root.querySelector('#sim04_td_se_' + r + '_' + c);
          const val = td_b[r][c];
          
          if(val !== -Infinity) {
            btn.textContent = String(val);
            btn.className = 'sim04_td_se_btn sim04_td_active';
          } else {
            btn.textContent = '-∞';
            btn.className = 'sim04_td_se_btn sim04_td_inf';
          }
        }
      }
    }

    function td_toggleSE(r, c) {
      if(r===1 && c===1) return;
      
      let currentVal = td_b[r][c];
      let idx = td_VALS.indexOf(currentVal);
      let nextVal = td_VALS[(idx + 1) % td_VALS.length];
      
      td_b[r][c] = nextVal;
      td_updateSEUI();
      td_refresh();
    };

    function td_setPresetSE(type) {
      for (let r=0; r<3; r++) for (let c=0; c<3; c++) td_b[r][c] = -Infinity;
      td_b[1][1] = 0;

      if (type === 'cross') {
        td_b[0][1] = td_b[1][0] = td_b[1][2] = td_b[2][1] = -1;
      } else if (type === 'square') {
        for (let r=0; r<3; r++) for (let c=0; c<3; c++) if(!(r===1 && c===1)) td_b[r][c] = -1;
      } else if (type === 'chamfer') {
        td_b[0][1] = td_b[1][0] = td_b[1][2] = td_b[2][1] = -3;
        td_b[0][0] = td_b[0][2] = td_b[2][0] = td_b[2][2] = -4;
      }
      
      td_updateSEUI();
      td_refresh();
    };

    function td_refresh() {
      if (td_mode === 'anim') {
        td_animSteps = td_compute().steps;
        td_animIdx = 0;
        td_drawAnimFrame();
      } else {
        td_drawFinal();
      }
    }

    function td_setMode(m) {
      td_mode = m;
      td_stopPlay();
      root.querySelector('#sim04_td_btnLabel').classList.toggle('sim04_td_active', m==='label');
      root.querySelector('#sim04_td_btnAnim').classList.toggle('sim04_td_active', m==='anim');
      root.querySelector('#sim04_td_animCtrl').style.display = m==='anim' ? 'block' : 'none';
      td_refresh();
    }

    function td_animStep(d) {
      td_animIdx = Math.max(0, Math.min(td_animSteps.length-1, td_animIdx+d));
      td_drawAnimFrame();
    }

    function td_togglePlay() {
      if (td_playing) td_stopPlay(); else td_startPlay();
    }

    function td_startPlay() {
      td_playing = true;
      root.querySelector('#sim04_td_btnPlay').textContent = '⏸';
      function tick() {
        if (td_animIdx >= td_animSteps.length-1) { td_stopPlay(); return; }
        td_animIdx++;
        td_drawAnimFrame();
        const spd = +root.querySelector('#sim04_td_speedSlider').value;
        const delay = Math.round(1200 - spd * 100);
        td_playTimer = setTimeout(tick, delay);
      }
      tick();
    }

    function td_stopPlay() {
      td_playing = false;
      if (td_playTimer) { clearTimeout(td_playTimer); td_playTimer = null; }
      root.querySelector('#sim04_td_btnPlay').textContent = '▶';
    }

    function td_clearGrid() {
      td_grid = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
      td_stopPlay();
      td_refresh();
    }

    const td_PRESETS = {
      square: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
        for(let r=2; r<=9; r++) for(let c=2; c<=9; c++) g[r][c] = 1;
        return g;
      },
      lshape: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
        for(let r=2; r<=9; r++) { g[r][2]=1; g[r][3]=1; g[r][4]=1; }
        for(let c=5; c<=9; c++) { g[7][c]=1; g[8][c]=1; g[9][c]=1; }
        return g;
      },
      ring: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
        for(let c=2; c<=9; c++) { g[2][c]=1; g[3][c]=1; g[8][c]=1; g[9][c]=1; }
        for(let r=4; r<=7; r++) { g[r][2]=1; g[r][3]=1; g[r][8]=1; g[r][9]=1; }
        return g;
      },
      corner0: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(1));
        g[0][0] = 0;
        return g;
      }
    };

    function td_loadPreset(name) {
      td_grid = td_PRESETS[name]();
      td_stopPlay();
      td_refresh();
    }

    function td_cellAt(e) {
      const rect = td_cv.getBoundingClientRect();
      const sx = td_W / rect.width, sy = td_H / rect.height;
      const raw = e.touches ? e.touches[0] : e;
      const c = Math.floor(((raw.clientX - rect.left) * sx - td_PAD) / td_CELL);
      const r = Math.floor(((raw.clientY - rect.top) * sy - td_PAD) / td_CELL);
      return {r, c};
    }

    td_cv.addEventListener('mousedown', function(e){
      const {r, c} = td_cellAt(e);
      if (r<0 || r>=td_ROWS || c<0 || c>=td_COLS) return;
      td_painting = true;
      td_paintVal = td_grid[r][c] ? 0 : 1;
      td_grid[r][c] = td_paintVal;
      td_refresh();
    });

    td_cv.addEventListener('mousemove', function(e){
      if (!td_painting) return;
      const {r, c} = td_cellAt(e);
      if (r<0 || r>=td_ROWS || c<0 || c>=td_COLS || td_grid[r][c] === td_paintVal) return;
      td_grid[r][c] = td_paintVal;
      td_refresh();
    });
    
    document.addEventListener('mouseup', () => td_painting = false);

    root.querySelectorAll('[data-se]').forEach(btn => {
      btn.addEventListener('click', function() {
        const parts = this.getAttribute('data-se').split(',');
        td_toggleSE(parseInt(parts[0], 10), parseInt(parts[1], 10));
      });
    });

    root.querySelectorAll('[data-se-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        td_setPresetSE(this.getAttribute('data-se-preset'));
      });
    });

    root.querySelectorAll('[data-mode]').forEach(btn => {
      btn.addEventListener('click', function() {
        td_setMode(this.getAttribute('data-mode'));
      });
    });

    root.querySelectorAll('[data-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        td_loadPreset(this.getAttribute('data-preset'));
      });
    });

    root.querySelectorAll('[data-action]').forEach(btn => {
      btn.addEventListener('click', function() {
        const act = this.getAttribute('data-action');
        if(act === 'clear') td_clearGrid();
        if(act === 'prev') td_animStep(-1);
        if(act === 'next') td_animStep(1);
        if(act === 'play') td_togglePlay();
      });
    });

    td_grid = td_PRESETS.corner0();
    td_updateSEUI();
    td_refresh();
  }

  function tryInitSim04TD(){
    var root = document.getElementById('sim-04-distancia');
    if (root) initSim04TD(root); else setTimeout(tryInitSim04TD, 200);
  }
  tryInitSim04TD();
})();
</script>
""")

**Figura 4.19:** Simulador interactivo de la Transformada de Distancia (TD) iterativa mediante erosión en tonos de gris. Los píxeles fuera de la imagen asumen el valor máximo (144), propagando los costos a partir del fondo interno.


<figure id="fig-04-sim-04-distancia">
  <img src="imagens/fig-04-sim-04-distancia.png" alt=" Simulador interactivo de la Transformada de Distancia (TD) iterativa mediante erosión en tonos de gris. Los píxeles fuera de la imagen asumen el valor máximo (144), propagando los costos a partir del fondo interno. " style="max-width:80%" />
  <figcaption><strong>Figura 4.19:</strong>  Simulador interactivo de la Transformada de Distancia (TD) iterativa mediante erosión en tonos de gris. Los píxeles fuera de la imagen asumen el valor máximo (144), propagando los costos a partir del fondo interno. </figcaption>
</figure>

In [47]:
%%writefile tmp/fig_04_distancia_didatico.cpp
#define MM_OUT "tmp/fig_04_distancia_didatico.png"
//| label: fig-04-distancia-didatico
//| fig-cap: "Transformada de Distância em imagem binária 10×10. Esquerda: original (*foreground* = 255). Centro: mm.dist1 iterativa (erosões com cruz). Direita: mm.dist (L2)."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <algorithm>
#include <vector>
#include <filesystem>

int main() {
    mm::Image f(10, 10);
    std::fill(f.data.begin(), f.data.end(), 255);
    f.at(0, 0) = 0;

    mm::SE B_cruz{{mm::SE_OUT, -1, mm::SE_OUT}, {-1, 0, -1}, {mm::SE_OUT, -1, mm::SE_OUT}};

    mm::Image d_iter = mm::dist1(f, B_cruz);
    mm::Image d_l2 = mm::dist(f);

    std::cout << "Máx. dist1 (erosões) : " << (int)*std::max_element(d_iter.data.begin(), d_iter.data.end()) << " px\n";
    std::cout << "Máx. dist  (L2)      : " << (int)*std::max_element(d_l2.data.begin(), d_l2.data.end()) << " px\n";

    mm::show(
        std::vector<mm::Image>{f, d_iter, d_l2},
        MM_OUT,
        std::vector<std::string>{"f original", "mm.dist1 (erosões com cruz)", "mm.dist (L2)"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(f, "tmp/fig_04_distancia_didatico_0.png");
mm::write(d_iter, "tmp/fig_04_distancia_didatico_1.png");
mm::write(d_l2, "tmp/fig_04_distancia_didatico_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_distancia_didatico.cpp


In [48]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_distancia_didatico.cpp -o tmp/fig_04_distancia_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_distancia_didatico \
  && test -f "tmp/fig_04_distancia_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_distancia_didatico.png"

Máx. dist1 (erosões) : 18 px
Máx. dist  (L2)      : 13 px
[1] f original
[2] mm.dist1 (erosões com cruz)
[3] mm.dist (L2)


In [49]:
try:
    mm.show(
        [
            mm.read("tmp/fig_04_distancia_didatico_0.png"),
            mm.read("tmp/fig_04_distancia_didatico_1.png"),
            mm.read("tmp/fig_04_distancia_didatico_2.png"),
        ],
        titles=[
            'f original',
            'mm.dist1 (erosões com cruz)',
            'mm.dist (L2)',
        ],
        cols=3,
        axis=True,
        figsize=(12, 4),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_distancia_didatico_0.png (ver a versao Python)")

<Figure size 1800x600 with 3 Axes>

**Figura 4.20:** Transformada de Distância em imagem binária 10×10. Esquerda: original (*foreground* = 255). Centro: mm.dist1 iterativa (erosões com cruz). Direita: mm.dist (L2).


La anotación de los valores numéricos directamente sobre los píxeles permite verificar cómo `dist1` propaga las distancias según la métrica inducida por la función estructural utilizada. En el caso del elemento cruz con costo unitario, los valores obtenidos corresponden a la distancia de *Manhattan* ($L_1$). Aunque `dist1` y `mm::dist` producen valores numéricos distintos por adoptar métricas diferentes, ambas transformadas preservan la estructura topográfica de los objetos, haciendo que sus máximos ocurran en regiones centrales similares. Esta propiedad justifica el uso de `mm::dist` en aplicaciones prácticas, debido a su elevada eficiencia computacional.

### 4.4.3 Transformada de Distancia Euclidiana en cuatro pasos

El simulador de la [Figura 4.21](#fig-04-sim-04-tde) implementa el algoritmo de la TDE de Lotufo (2001) en dos etapas. En la primera etapa, la función `edt1` realiza una transformación unidimensional vertical de forma secuencial (*in-place*), recorriendo cada columna en *raster* (↓) y *anti-raster* (↑) para calcular las distancias en la dirección vertical (dos pasos: Sur y Norte). En la segunda etapa, la función `edt2` utiliza ese resultado como entrada y realiza una propagación horizontal por colas: para cada fila de la matriz, dos colas de prioridad, `Eq` y `Wq`, se inicializan recorriendo los índices de columna en sentidos opuestos (`Eq` de `W-1` hasta `1`, `Wq` de `2` hasta `W`), de modo que cada píxel actualizado encola inmediatamente a sus vecinos para reprocesamiento dentro de la misma ronda (dos pasos más: Este y Oeste). Este mecanismo de cola permite aplicar erosiones sucesivas con pesos impares crecientes (`b = 1, 3, 5, ...`, incrementado en cada iteración del bucle externo) sin que la propagación quede atrapada en valores desactualizados, ya que cada ronda resuelve por completo la cadena de dependencias horizontales antes del siguiente incremento de `b`. La convergencia de esta propagación produce la Transformada de Distancia Euclidiana en toda la matriz, combinando la información vertical obtenida en `edt1` con la propagación horizontal en cola realizada en `edt2`.

In [50]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-tde" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-tde * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-tde canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; margin: 0 auto; }
  #sim-04-tde button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all .15s ease; display: inline-flex; align-items: center; gap: 5px; font-weight: 600; }
  #sim-04-tde button:hover { background: #e8dfcf; }
  #sim-04-tde button.sim04_edt_act { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-tde .sim04_edt_smtxt { font-size: 10.5px; color: #8a8371; line-height: 1.4; }
  #sim-04-tde .sim04_edt_mono { font-family: monospace; }
  #sim-04-tde .sim04_edt_statcard { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 8px 10px; text-align: center; min-width: 0; }
  #sim-04-tde .sim04_edt_statlabel { font-size: 9.5px; text-transform: uppercase; letter-spacing: .04em; color: #8a8371; margin-bottom: 2px; font-weight: 700; }
  #sim-04-tde .sim04_edt_statval { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim04_edt_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📐 EDT² 2D Corrigido · Matriz 4×4</span>
  <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Convergencia Exacta</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição Teórica -->
  <div class="sim04_edt_panel" style="margin-bottom:14px;">
    <div style="font-size:11.5px; font-weight:700; color:#26241d; margin-bottom:2px;">Paso 1: edt1 Vertical In-place · Paso 2: edt2 Horizontal In-place</div>
    <div class="sim04_edt_smtxt">Usa la misma estructura raster/anti-raster descrita en el artículo para propagación de distancias exactas (ejemplo del artículo, pág. 103).</div>
  </div>

  <!-- Cards de Estatísticas -->
  <div style="display:grid; grid-template-columns:repeat(3, 1fr); gap:10px; margin-bottom:14px;">
    <div class="sim04_edt_statcard"><div class="sim04_edt_statlabel">Paso Actual</div><div class="sim04_edt_statval" id="sim04_edt_sStep" style="color:#c0392b;">–</div></div>
    <div class="sim04_edt_statcard"><div class="sim04_edt_statlabel">Fase</div><div class="sim04_edt_statval" id="sim04_edt_sPhase" style="color:#2980b9; font-size:12px;">–</div></div>
    <div class="sim04_edt_statcard"><div class="sim04_edt_statlabel">b Actual</div><div class="sim04_edt_statval" id="sim04_edt_sB" style="color:#b9770e;">–</div></div>
  </div>

  <!-- Canvas da Matriz 4x4 -->
  <div style="margin-bottom:14px; text-align:center;">
    <canvas id="sim04_edt_cvs"></canvas>
  </div>

  <!-- Barra de Ações -->
  <div style="display:flex; gap:6px; flex-wrap:wrap; margin-bottom:12px; align-items:center; justify-content:center;">
    <button id="sim04_edt_btnFinal" class="sim04_edt_act" data-action="final">Resultado Final</button>
    <button id="sim04_edt_btnStep" data-action="step">Paso a Paso</button>
    <div style="width:1px; background:#e4dcc8; height:20px;"></div>
    <button data-action="clear" style="border-color:#f5b7b1; color:#c0392b; background:#fdecea;">× Limpiar</button>
  </div>

  <!-- Painel de Passo a Passo (Controles de Animação) -->
  <div id="sim04_edt_stepCtrl" style="display:none;" class="sim04_edt_panel">
    <div style="display:flex; gap:6px; align-items:center; flex-wrap:wrap;">
      <button data-action="prev">◀ Anterior</button>
      <button id="sim04_edt_btnPlay" data-action="play">▶ Reproducir</button>
      <button data-action="next">Siguiente ▶</button>
      <span class="sim04_edt_smtxt" style="margin-left:4px; font-weight:700;">Vel.:</span>
      <input type="range" id="sim04_edt_speedSlider" min="1" max="10" value="5" step="1" style="width:70px; cursor:pointer;">
      <span id="sim04_edt_stepLabel" class="sim04_edt_mono" style="font-size:10.5px; color:#5e5a4a; flex:1; text-align:right; min-width:0; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;"></span>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04EdtConvergencia(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const COLS=4, ROWS=4, CELL=50, PAD=20, INF=9999;
    const cv = root.querySelector('#sim04_edt_cvs');
    const CTX = cv.getContext('2d');

    cv.width = COLS * CELL + PAD * 2;
    cv.height = ROWS * CELL + PAD * 2;

    let grid = [
      [1, 1, 0, 1],
      [1, 1, 1, 1],
      [1, 0, 1, 1],
      [0, 1, 1, 1]
    ];

    let mode='final';
    let steps=[], stepIdx=0;
    let playing=false, playTimer=null;

    function getCol(d) {
      if(d === 0) return { fill: 'transparent', txt: '#8a8371', bdr: '#e4dcc8' };
      if(d >= INF) return { fill: '#fdecea', txt: '#c0392b', bdr: '#f5b7b1' };
      const colors = ['#ebf4fd', '#a9cce3', '#73c6b6', '#52be80', '#27ae60', '#2980b9', '#1b4f72'];
      let idx = Math.min(colors.length - 1, Math.floor(d / 1.5));
      return { fill: colors[idx], txt: idx > 3 ? '#ffffff' : '#26241d', bdr: colors[Math.min(idx+1, colors.length-1)] };
    }

    function drawMatrix(matrix, opts={}) {
      CTX.clearRect(0, 0, cv.width, cv.height);
      for(let r=0; r<ROWS; r++) {
        for(let c=0; c<COLS; c++) {
          const px = PAD + c * CELL; const py = PAD + r * CELL;
          const d = matrix[r][c]; const col = getCol(d);

          CTX.fillStyle = col.fill; CTX.fillRect(px+1, py+1, CELL-2, CELL-2);

          const isCur = opts.cursor && opts.cursor.r === r && opts.cursor.c === c;
          const isChg = opts.changed && opts.changed[r][c];

          CTX.strokeStyle = isCur ? '#c0392b' : isChg ? '#b9770e' : col.bdr;
          CTX.lineWidth = isCur ? 3 : isChg ? 2.5 : 0.5;
          CTX.strokeRect(px+0.5, py+0.5, CELL-1, CELL-1);

          CTX.fillStyle = col.txt; CTX.font = 'bold 14px monospace';
          CTX.textAlign = 'center'; CTX.textBaseline = 'middle';
          CTX.fillText(d >= INF ? '∞' : String(d), px + CELL/2, py + CELL/2);

          if (r === 0) { CTX.fillStyle='#8a8371'; CTX.font='9px monospace'; CTX.fillText('c'+c, px+CELL/2, PAD/2); }
          if (c === 0) { CTX.fillStyle='#8a8371'; CTX.font='9px monospace'; CTX.fillText('r'+r, PAD/2, py+CELL/2); }
        }
      }
    }

    function copyMat(m) { return m.map(row => [...row]); }

    function computeEdt1(gInit) {
      const all = [];
      let g = copyMat(gInit);
      all.push({ phase: 'edt1_init', g: copyMat(g), cursor: null, b: 0, changed: null, label: 'edt1 · Matriz de entrada inicializada' });

      for(let c=0; c<COLS; c++) {
        let b = 1;
        for(let r=1; r<ROWS; r++) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r-1][c] < INF) {
            if(g[r][c] > g[r-1][c] + b) {
              g[r][c] = g[r-1][c] + b; chg[r][c] = 1;
              all.push({ phase: 'edt1_raster', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt1 raster ↓ · f(' + r + ',' + c + ') = f(' + (r-1) + ',' + c + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt1_raster', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt1 raster ↓ · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }

      for(let c=0; c<COLS; c++) {
        let b = 1;
        for(let r=ROWS-2; r>=0; r--) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r+1][c] < INF) {
            if(g[r][c] > g[r+1][c] + b) {
              g[r][c] = g[r+1][c] + b; chg[r][c] = 1;
              all.push({ phase: 'edt1_anti', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt1 anti-raster ↑ · f(' + r + ',' + c + ') = f(' + (r+1) + ',' + c + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt1_anti', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt1 anti-raster ↑ · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }
      return { steps: all, g: copyMat(g) };
    }

    function computeEdt2(gIn) {
      const all = [];
      let g = copyMat(gIn);
      all.push({ phase: 'edt2_init', g: copyMat(g), cursor: null, b: 0, changed: null, label: 'edt2 · Estado inicial f_v antes da propagação horizontal' });

      for(let r=0; r<ROWS; r++) {
        let b = 1;
        for(let c=1; c<COLS; c++) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r][c-1] < INF) {
            if(g[r][c] > g[r][c-1] + b) {
              g[r][c] = g[r][c-1] + b; chg[r][c] = 1;
              all.push({ phase: 'edt2_raster', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt2 raster → · f(' + r + ',' + c + ') = f(' + r + ',' + (c-1) + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt2_raster', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt2 raster → · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }

      for(let r=0; r<ROWS; r++) {
        let b = 1;
        for(let c=COLS-2; c>=0; c--) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r][c+1] < INF) {
            if(g[r][c] > g[r][c+1] + b) {
              g[r][c] = g[r][c+1] + b; chg[r][c] = 1;
              all.push({ phase: 'edt2_anti', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt2 anti-raster ← · f(' + r + ',' + c + ') = f(' + r + ',' + (c+1) + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt2_anti', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt2 anti-raster ← · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }

      all.push({ phase: 'edt2_done', g: copyMat(g), cursor: null, b: 0, changed: null, label: 'Convergência final atingida!' });
      return { steps: all, g: copyMat(g) };
    }

    function computeAll() {
      const gInit = Array.from({length: ROWS}, (_, r) => Array.from({length: COLS}, (_, c) => grid[r][c] ? INF : 0));
      const r1 = computeEdt1(gInit); const r2 = computeEdt2(r1.g);
      return [...r1.steps, ...r2.steps];
    }

    function setMode(m) {
      mode = m; stopPlay();
      root.querySelector('#sim04_edt_btnFinal').className = m === 'final' ? 'sim04_edt_act' : '';
      root.querySelector('#sim04_edt_btnStep').className = m === 'step' ? 'sim04_edt_act' : '';
      root.querySelector('#sim04_edt_stepCtrl').style.display = m === 'step' ? 'block' : 'none';

      if(m === 'step') {
        steps = computeAll(); stepIdx = 0; drawFrame();
      } else {
        const finalSteps = computeAll();
        drawMatrix(finalSteps[finalSteps.length - 1].g);
        root.querySelector('#sim04_edt_sStep').textContent = 'Final';
        root.querySelector('#sim04_edt_sPhase').textContent = 'Concluído';
        root.querySelector('#sim04_edt_sB').textContent = '–';
      }
    }

    function drawFrame() {
      if(!steps.length) return;
      const s = steps[Math.min(stepIdx, steps.length - 1)];
      drawMatrix(s.g, { cursor: s.cursor, changed: s.changed });
      root.querySelector('#sim04_edt_sStep').textContent = (stepIdx + 1) + '/' + steps.length;
      root.querySelector('#sim04_edt_sPhase').textContent = s.phase;
      root.querySelector('#sim04_edt_sB').textContent = s.b > 0 ? s.b : '–';
      root.querySelector('#sim04_edt_stepLabel').textContent = s.label;
    }

    function goStep(d) {
      stepIdx = Math.max(0, Math.min(steps.length - 1, stepIdx + d));
      drawFrame();
    }

    function togglePlay() { playing ? stopPlay() : startPlay(); }
    function startPlay() {
      playing = true; root.querySelector('#sim04_edt_btnPlay').textContent = '⏸ Pausar';
      function tick() {
        if(stepIdx >= steps.length - 1) { stopPlay(); return; }
        stepIdx++; drawFrame();
        playTimer = setTimeout(tick, Math.round(1300 - (+root.querySelector('#sim04_edt_speedSlider').value) * 110));
      }
      tick();
    }
    function stopPlay() { playing = false; if(playTimer) clearTimeout(playTimer); const b = root.querySelector('#sim04_edt_btnPlay'); if(b) b.textContent = '▶ Play'; }

    function clearGrid() { grid = Array.from({length: ROWS}, () => new Array(COLS).fill(0)); refresh(); }
    function refresh() { stopPlay(); if(mode === 'step') { steps = computeAll(); stepIdx = Math.min(stepIdx, steps.length - 1); drawFrame(); } else { setMode('final'); } }

    cv.addEventListener('mousedown', e => {
      const rect = cv.getBoundingClientRect();
      const c = Math.floor(((e.clientX - rect.left) * (cv.width / rect.width) - PAD) / CELL);
      const r = Math.floor(((e.clientY - rect.top) * (cv.height / rect.height) - PAD) / CELL);
      if(r >= 0 && r < ROWS && c >= 0 && c < COLS) { grid[r][c] = grid[r][c] ? 0 : 1; refresh(); }
    });

    root.querySelectorAll('[data-action]').forEach(btn => {
      btn.addEventListener('click', function() {
        const act = this.getAttribute('data-action');
        if(act === 'final') setMode('final');
        if(act === 'step') setMode('step');
        if(act === 'clear') clearGrid();
        if(act === 'prev') goStep(-1);
        if(act === 'next') goStep(1);
        if(act === 'play') togglePlay();
      });
    });

    setMode('final');
  }

  function tryInit(){
    var root = document.getElementById('sim-04-tde');
    if (root) initSim04EdtConvergencia(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
""")

**Figura 4.21:** Simulador interactivo 2D (4x4) con sincronización estricta de colas de propagación horizontal (b) para obtener la convergencia exacta descrita en el artículo.


<figure id="fig-04-sim-04-tde">
  <img src="imagens/fig-04-sim-04-tde.png" alt=" Simulador interactivo 2D (4x4) con sincronización estricta de colas de propagación horizontal (b) para obtener la convergencia exacta descrita en el artículo. " style="max-width:80%" />
  <figcaption><strong>Figura 4.21:</strong>  Simulador interactivo 2D (4x4) con sincronización estricta de colas de propagación horizontal (b) para obtener la convergencia exacta descrita en el artículo. </figcaption>
</figure>

#### 4.4.3.1 Transformada de Distancia Geodésica

La transformada de distancia geodésica asocia a cada píxel la menor distancia hasta un marcador, bajo la restricción impuesta por una máscara. De esta forma, la propagación ocurre exclusivamente por los píxeles permitidos, preservando la conectividad del dominio.

La [Figura 4.22](#fig-04-gdist-menor-caminho) ilustra este proceso en un laberinto: (a) la máscara `g`; (b) la distancia geodésica `D1` calculada a partir de la entrada; (c) la distancia `D2` calculada a partir de la salida; y (d) el camino mínimo obtenido a partir de estas dos transformadas.

El camino óptimo se determina mediante la suma de las distancias (`D1 + D2`). Los píxeles pertenecientes a la trayectoria mínima son aquellos para los cuales esta suma asume su menor valor, definiendo una conexión entre entrada y salida con longitud geodésica mínima.

Este principio permite resolver laberintos sin la necesidad de explorar explícitamente todas las posibilidades de recorrido. La solución emerge directamente de la propagación de distancias en un dominio restringido. Este enfoque es particularmente relevante en laberintos de elevada complejidad, como los construidos a partir de estructuras cuasicristalinas y ciclos hamiltonianos descritos por Singh (2024). En Zampirolli (2025), este mismo formalismo se emplea para resolver un laberinto complejo; a continuación, el método se ilustra en una versión simplificada del problema.

In [51]:
%%writefile tmp/fig_04_gdist_menor_caminho.cpp
#define MM_OUT "tmp/fig_04_gdist_menor_caminho.png"
// Compile with: g++ -std=c++17 -I. program.cpp -o program $(pkg-config --cflags --libs opencv4) -DMM_USE_OPENCV
//| label: fig-04-gdist-menor-caminho
//| fig-cap: "Menor caminho geodésico em um labirinto. As distâncias geodésicas são calculadas a partir da entrada e da saída usando mm.gdist. Os pixels cujo somatório das duas distâncias é igual à distância mínima entre os marcadores pertencem a um caminho ótimo."
//| echo: true
//| output: true
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

int main() {
    // 1 = corredor, 0 = parede
    mm::Image g(10, 10);
    unsigned char g_data[10][10] = {
        {0,1,0,0,0,0,0,0,0,0},
        {0,1,1,1,1,1,0,1,1,1},
        {0,0,0,0,0,1,0,1,0,1},
        {0,1,1,1,0,1,1,1,0,1},
        {0,1,0,1,0,0,0,0,0,1},
        {0,1,0,1,1,1,1,1,1,1},
        {0,1,0,0,0,0,0,0,1,0},
        {0,1,1,1,1,1,1,0,1,0},
        {0,0,0,0,0,0,1,1,1,0},
        {0,0,0,0,0,0,0,0,1,0}
    };
    for (int y = 0; y < 10; y++)
        for (int x = 0; x < 10; x++)
            g.at(y, x) = g_data[y][x];

    // marcador da entrada
    mm::Image entrada(10, 10);
    entrada.at(0, 1) = 1;

    // marcador da saída
    mm::Image saida(10, 10);
    saida.at(9, 8) = 1;

    // distâncias geodésicas
    mm::Image D1 = mm::gdist(g, entrada);
    mm::Image D2 = mm::gdist(g, saida);

    // soma das distâncias
    mm::Image S = mm::addm(D1, D2);

    // menor valor válido da soma
    int dmin = 255;
    for (int i = 0; i < S.h * S.w; i++)
        if (S.data[i] > 0 && S.data[i] < dmin)
            dmin = S.data[i];

    // pixels pertencentes a um caminho ótimo
    mm::Image caminho(S.h, S.w);
    for (int i = 0; i < S.h * S.w; i++)
        caminho.data[i] = (S.data[i] == dmin) ? 255 : 0;

    std::cout << "Distância geodésica mínima: " << dmin << "\n";

    mm::show(
        std::vector<mm::Image>{g, D1, D2, caminho},
        MM_OUT,
        std::vector<std::string>{
            "Labirinto",
            "Distância da Entrada",
            "Distância da Saída",
            "Menor Caminho\n(d=" + std::to_string(dmin) + ")"
        },
        4
    );

    return 0;
}

Overwriting tmp/fig_04_gdist_menor_caminho.cpp


In [52]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_gdist_menor_caminho.cpp -o tmp/fig_04_gdist_menor_caminho -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_gdist_menor_caminho \
  && test -f "tmp/fig_04_gdist_menor_caminho.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_gdist_menor_caminho.png"

Distância geodésica mínima: 15
[1] Labirinto
[2] Distância da Entrada
[3] Distância da Saída
[4] Menor Caminho
(d=15)


In [53]:
try:
    mm.show(mm.read("tmp/fig_04_gdist_menor_caminho.png"), figsize=(14, 4))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_gdist_menor_caminho.png (ver a versao Python)")

<Figure size 2100x600 with 1 Axes>

**Figura 4.22:** Menor caminho geodésico em um labirinto. As distâncias geodésicas são calculadas a partir da entrada e da saída usando mm.gdist. Os pixels cujo somatório das duas distâncias é igual à distância mínima entre os marcadores pertencem a um caminho ótimo.


### 4.4.4 Segmentación por *Watershed*

El algoritmo ***Watershed*** interpreta una imagen en niveles de gris como una superficie topográfica, en la cual valores elevados corresponden a montañas y valores bajos corresponden a valles o cuencas de drenaje (*catchment basins*). En el contexto de la segmentación basada en marcadores, los máximos de la Transformada de Distancia se utilizan frecuentemente para identificar regiones internas de los objetos, proporcionando semillas confiables para el proceso de inundación.

La segmentación se realiza entonces mediante una simulación conceptual de inundación progresiva a partir de estos marcadores. A medida que las cuencas asociadas a diferentes semillas se expanden, las regiones vecinas eventualmente entran en contacto. En ese instante se construyen barreras virtuales, denominadas *watershed lines*, que pasan a delimitar los objetos de la escena. Este mecanismo permite separar objetos adyacentes o parcialmente superpuestos, incluso cuando forman una única componente conexa tras la umbralización.

La implementación didáctica presentada en este capítulo explora inicialmente el concepto de crecimiento de regiones (*region growing*) confinado por una máscara binaria, según se detalla en el algoritmo interactivo de la [Figura 4.23](#fig-04-sim-alg-watershed2).

> ### 📝 Versión didáctica versus implementación clásica
>
> La función `mm::watershed0` no implementa el algoritmo *watershed* clásico. Su objetivo es ilustrar, de forma simplificada, la propagación de marcadores por crecimiento de regiones (*region growing*), permitiendo visualizar cómo diferentes semillas compiten por la ocupación del espacio disponible. El crecimiento está delimitado por una máscara binaria de soporte y monitoreado por un control de estancamiento, generando un resultado similar a una partición de Voronoi restringida a la geometría de los objetos de entrada.
>
> En cambio, la función `mm::watershed` utiliza la implementación optimizada de OpenCV (`mm::watershed`), que realiza la inundación sobre una superficie topográfica definida por la imagen de entrada. En este caso, la propagación de los marcadores está influenciada por los valores de los píxeles, haciendo que las líneas de separación se formen naturalmente sobre las crestas del relieve.

In [54]:
# @title { display-mode: "form" }
import numpy as np, cv2, os  # [pdi] passthrough: garante imports desta trilha
# Prefijo exclusivo para evitar conflicto con otras celdas del cuaderno
PREFIX = "wat0"

from IPython.display import HTML
HTML(f'''
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:var(--font-sans)}}
.{PREFIX}-tabs{{display:flex;gap:6px;margin-bottom:20px;flex-wrap:wrap}}
.{PREFIX}-tab{{padding:6px 14px;border-radius:20px;font-size:13px;cursor:pointer;border:0.5px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all .15s;white-space:nowrap}}
.{PREFIX}-tab.active{{background:var(--color-text-primary);color:var(--color-background-primary);border-color:transparent}}
.{PREFIX}-panel{{display:none}}.{PREFIX}-panel.active{{display:block}}

.algo-wrap{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.algo-header{{padding:14px 20px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;align-items:center;gap:10px}}
.algo-icon{{width:32px;height:32px;border-radius:var(--border-radius-md);display:flex;align-items:center;justify-content:center;font-size:16px;background:#EBF7F2;color:#0F6E56}}
.algo-title{{font-size:14px;font-weight:500;color:var(--color-text-primary);text-align:left}}
.algo-sub{{font-size:12px;color:var(--color-text-secondary);margin-top:1px;text-align:left}}
.algo-body{{padding:20px;text-align:left}}

.step{{display:flex;gap:12px;margin-bottom:14px;align-items:flex-start}}
.step:last-child{{margin-bottom:0}}
.step-num{{min-width:24px;height:24px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0;margin-top:1px}}
.step-text{{font-size:13.5px;line-height:1.65;color:var(--color-text-primary);text-align:left}}
.step-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 6px;border-radius:4px;color:var(--color-text-primary)}}
.step-sub{{margin-top:8px;border-left:2px solid var(--color-border-secondary);padding-left:12px;display:flex;flex-direction:column;gap:5px}}
.step-sub-item{{font-size:13px;color:var(--color-text-secondary);line-height:1.55;text-align:left}}
.step-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.while-box{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:11px 13px;margin-top:8px;text-align:left}}
.while-head{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.06em;margin-bottom:7px}}
.note{{margin-top:16px;padding-top:12px;border-top:0.5px solid var(--color-border-tertiary);font-size:12px;color:var(--color-text-secondary);font-style:italic;line-height:1.5;text-align:left}}

.num-teal{{background:#E1F5EE;color:#0F6E56}}
.num-purple{{background:#EEEDFE;color:#534AB7}}
.num-amber{{background:#FAEEDA;color:#854F0B}}
.num-coral{{background:#FAECE7;color:#993C1D}}
.num-blue{{background:#E6F1FB;color:#185FA5}}
.num-gray{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}

.card-step{{display:flex;border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);overflow:hidden;margin-bottom:8px}}
.card-step:last-child{{margin-bottom:0}}
.card-badge{{min-width:48px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0}}
.card-content{{padding:10px 14px;flex:1;text-align:left}}
.card-label{{font-size:10px;font-weight:500;letter-spacing:.08em;text-transform:uppercase;margin-bottom:3px}}
.card-text{{font-size:13.5px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.card-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.card-sub{{margin-top:6px;display:flex;flex-direction:column;gap:3px}}
.card-sub-item{{font-size:12.5px;color:var(--color-text-secondary);padding-left:10px;border-left:2px solid var(--color-border-secondary);line-height:1.5;text-align:left}}
.card-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 4px;border-radius:3px;color:var(--color-text-primary)}}

.tl{{position:relative;padding:4px 0 4px 36px;text-align:left}}
.tl-line{{position:absolute;left:11px;top:20px;bottom:20px;width:1.5px;background:var(--color-border-secondary);border-radius:2px}}
.tl-step{{display:flex;gap:0;margin-bottom:18px;position:relative}}
.tl-step:last-child{{margin-bottom:0}}
.tl-dot-wrap{{position:absolute;left:-36px;top:2px;display:flex;flex-direction:column;align-items:center;gap:3px}}
.tl-dot{{width:14px;height:14px;border-radius:50%;border:2px solid var(--color-border-secondary);background:var(--color-background-primary);transition:all .2s;z-index:1}}
.tl-step:hover .tl-dot{{background:var(--color-text-primary);border-color:var(--color-text-primary)}}
.tl-num{{font-size:9px;color:var(--color-text-secondary);font-weight:500;letter-spacing:.04em}}
.tl-title{{font-size:13.5px;font-weight:500;color:var(--color-text-primary);margin-bottom:3px;text-align:left}}
.tl-desc{{font-size:13px;color:var(--color-text-secondary);line-height:1.6;text-align:left}}
.tl-desc code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.code-wrap{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.code-bar{{background:var(--color-background-secondary);padding:10px 16px;display:flex;align-items:center;justify-content:space-between;border-bottom:0.5px solid var(--color-border-tertiary)}}
.code-lang{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.08em}}
.code-body{{padding:18px 20px;overflow-x:auto;text-align:left}}
.code-body pre{{font-family:var(--font-mono);font-size:13px;line-height:1.75;color:var(--color-text-primary);margin:0;white-space:pre;text-align:left}}
.kw{{color:#7C3AED}} .fn{{color:#0369A1}} .cm{{color:#6B7280;font-style:italic}} .st{{color:#059669}} .num-lit{{color:#DC2626}}

.ann-line{{display:flex;align-items:flex-start;gap:10px;margin-bottom:8px;padding:10px 12px;background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);text-align:left}}
.ann-badge{{min-width:20px;height:20px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;flex-shrink:0;margin-top:1px}}
.ann-text{{font-size:13px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.ann-text code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.metric-grid{{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin-top:16px}}
.metric-card{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:12px 14px;text-align:left}}
.metric-title{{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.07em;margin-bottom:6px}}
.metric-formula{{font-family:var(--font-mono);font-size:12px;color:var(--color-text-primary);margin-bottom:4px}}
.metric-desc{{font-size:12px;color:var(--color-text-secondary);line-height:1.5}}
</style>

<h2 class="sr-only" style="position:absolute;left:-9999px">Algoritmo Didático do Watershed Limitado por Máscara — painel interativo</h2>

<div class="{PREFIX}-tabs" id="{PREFIX}-tabs-container">
  <div class="{PREFIX}-tab" data-idx="0"><i class="ti ti-list-numbers" aria-hidden="true"></i> Passo a passo</div>
  <div class="{PREFIX}-tab" data-idx="1"><i class="ti ti-cards" aria-hidden="true"></i> Cards</div>
  <div class="{PREFIX}-tab" data-idx="2"><i class="ti ti-timeline" aria-hidden="true"></i> Linha do tempo</div>
  <div class="{PREFIX}-tab" data-idx="3"><i class="ti ti-code" aria-hidden="true"></i> Código Python</div>
  <div class="{PREFIX}-tab active" data-idx="4"><i class="ti ti-git-branch" aria-hidden="true"></i> Fluxograma</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-0">
<div class="algo-wrap">
  <div class="algo-header">
    <div class="algo-icon"><i class="ti ti-bucket" aria-hidden="true"></i></div>
    <div><div class="algo-title">Crescimento de Regiões Confinado por Máscara</div><div class="algo-sub">Inundação concorrente com restrição geométrica de suporte e sincronização síncrona por malha</div></div>
  </div>
  <div class="algo-body">
    <div class="step"><div class="step-num num-teal">1</div><div class="step-text">Rotular os marcadores sementes em <code>f</code> via <code>mm.label0(f, b)</code>, instanciar a malha dinâmica <code>g ← f.copy()</code> e binarizar a <code>mask</code>.</div></div>
    <div class="step">
      <div class="step-num num-coral">2</div>
      <div class="step-text">
        Enquanto houver pixels não rotulados (<code>while True</code>), reiniciar o controle de atividade <code>mudou ← False</code> e varrer a imagem:
        <div class="step-sub">
          <div class="step-sub-item">Identificar se a coordenada atual é um vazio contido no escopo: <code>g[x,y] == 0 and mask[x,y]</code>.</div>
          <div class="step-sub-item">Avaliar a vizinhança estrutural em <code>mm._viz(f, b, x, y)</code> baseada no estado síncrono estável <code>f</code>.</div>
          <div class="step-sub-item">Se um vizinho possuir rótulo dominante (<code>g[x,y] < f[vy,vx]</code>), a célula em <code>g</code> absorve esse identificador e marca-se <code>mudou ← True</code>.</div>
        </div>
      </div>
    </div>
    <div class="step"><div class="step-num num-purple">3</div><div class="step-text">Verificar ponto fixo: caso uma varredura completa não expanda nenhuma fronteira (<code>not mudou</code>), interrompe-se o laço (<code>break</code>).</div></div>
    <div class="step"><div class="step-num num-blue">4</div><div class="step-text">Atualizar o estado de referência de forma síncrona para a próxima iteração: <code>f ← g.copy()</code>.</div></div>
    <div class="step"><div class="step-num num-blue">5</div><div class="step-text">Se <code>op == 'region'</code>, retornar o mapa de bacias <code>g</code>; caso contrário, extrair as cristas divisórias via <code>mm.gradm(g)</code>.</div></div>
    <div class="note">A sincronização <code>f = g.copy()</code> ao final de cada ciclo impede o crescimento assimétrico ou dependente da ordem da varredura raster (propagação em estilo Jacobi).</div>
  </div>
</div>

<div class="metric-grid">
  <div class="metric-card">
    <div class="metric-title" style="color:#0F6E56">Escopo Geométrico</div>
    <div class="metric-formula">mask[x,y] > 0</div>
    <div class="metric-desc">Restrição binária rígida impedindo o avanço periférico de rótulos.</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#993C1D">Estabilização</div>
    <div class="metric-formula">if not mudou: break</div>
    <div class="metric-desc">Evita loops infinitos interrompendo ao saturar o domínio da máscara.</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#185FA5">Mapeamento Jacobi</div>
    <div class="metric-formula">f = g.copy()</div>
    <div class="metric-desc">Sincronização em bloco após inspeção de todas as coordenadas.</div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-1">
  <div class="card-step">
    <div class="card-badge num-teal">01</div>
    <div class="card-content">
      <div class="card-label" style="color:#0F6E56">Inicialização</div>
      <div class="card-text">Geração dos identificadores iniciais pelo mapeamento de componentes conexas e binarização da máscara de suporte.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">02</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Expansão Concorrente</div>
      <div class="card-text">Varredura 2D inspecionando vazios internos autorizados. A malha de trabalho <code>g</code> absorve os rótulos lidos da referência estável <code>f</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-purple">03</div>
    <div class="card-content">
      <div class="card-label" style="color:#534AB7">Ponto Fixo Local</div>
      <div class="card-text">A flag <code>mudou</code> monitora mudanças estruturais. Se nenhuma frente avançar, o laço de inundação é finalizado via <code>break</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-blue">04</div>
    <div class="card-content">
      <div class="card-label" style="color:#185FA5">Sincronização e Saída</div>
      <div class="card-text">Atualização em bloco do estado referencial. A saída pode ser moldada como partições regionais ou linhas de cristas (linhas de watershed).</div>
    </div>
  </div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-2">
<div class="tl">
  <div class="tl-line"></div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">01</span></div>
    <div><div class="tl-title">Condicionamento Prévio</div><div class="tl-desc">Rotulagem preliminar de marcadores e isolamento booleano do domínio.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">02</span></div>
    <div>
      <div class="tl-title">Laço Síncrono Iterativo</div>
      <div class="while-box">
        <div class="while-head">while True</div>
        <div class="step-sub" style="border-color:var(--color-border-tertiary)">
          <div class="step-sub-item">Redefinição de flag: <code>mudou = False</code></div>
          <div class="step-sub-item">Crescimento condicional: se <code>g[x,y] == 0</code> e estiver na máscara, expande lendo <code>f</code></div>
          <div class="step-sub-item">Controle de estabilidade: <code>if not mudou: break</code></div>
          <div class="step-sub-item">Atualização síncrona: <code>f = g.copy()</code></div>
        </div>
      </div>
    </div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">03</span></div>
    <div><div class="tl-title">Extração Topológica</div><div class="tl-desc">Retorno condicional das bacias preenchidas ou cálculo morfológico do gradiente de transição.</div></div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-3">
<div class="code-wrap">
  <div class="code-bar">
    <span class="code-lang"><i class="ti ti-brand-python" aria-hidden="true" style="font-size:14px;vertical-align:-2px;margin-right:5px"></i>mm_watershed.py</span>
    <span style="font-size:11px;color:var(--color-text-secondary)">Algoritmo com Restrição de Máscara</span>
  </div>
  <div class="code-body">
<pre><span class="kw">def</span> <span class="fn">watershed0</span>(f, mask=<span class="fn">None</span>, b=np.zeros((<span class="num-lit">3</span>,<span class="num-lit">3</span>),dtype=<span class="st">'uint8'</span>), op=<span class="st">'region'</span>):
    f = mm.label0(f, b)
    g = f.copy()
    mask = np.ones_like(f) <span class="kw">if</span> mask <span class="kw">is</span> <span class="fn">None</span> <span class="kw">else</span> (mask > <span class="num-lit">0</span>)
    
    <span class="kw">while</span> <span class="fn">True</span>:
        mudou = <span class="fn">False</span>
        <span class="kw">for</span> x <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">0</span>]):
            <span class="kw">for</span> y <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">1</span>]):
                <span class="kw">if</span> g[x,y] == <span class="num-lit">0</span> <span class="kw">and</span> mask[x,y]:
                    <span class="kw">for</span> vy,vx,bv <span class="kw">in</span> mm._viz(f, b, x, y):
                        <span class="kw">if</span> bv <span class="kw">and g[x,y] < f[vy,vx]</span>: 
                            g[x,y] = f[vy,vx]
                            mudou = <span class="fn">True</span>
        <span class="kw">if</span> <span class="kw">not</span> mudou: 
            <span class="kw">break</span>
        f = g.copy()
        
    <span class="kw">return</span> g <span class="kw">if</span> op == <span class="st">'region'</span> <span class="kw">else</span> mm.gradm(g, mm.secross())</pre>
  </div>
</div>

<div style="margin-top:16px;display:flex;flex-direction:column;gap:8px;text-align:left">
  <div class="ann-line"><div class="ann-badge num-teal">1</div><div class="ann-text"><code>mask = (mask &gt; 0)</code> — Converte a imagem de suporte informada para um mapa Booleano indexável.</div></div>
  <div class="ann-line"><div class="ann-badge num-coral">2</div><div class="ann-text"><code>g[x,y] == 0 and mask[x,y]</code> — Filtro ativo: pixels fora da máscara (fundo zero) são ignorados de imediato, confinando as frentes de expansão.</div></div>
  <div class="ann-line"><div class="ann-badge num-purple">3</div><div class="ann-text"><code>if not mudou: break</code> — Mecanismo de escape. Quando todos os espaços internos permitidos forem preenchidos ou estabilizados contra a barreira, o laço aborta de forma limpa.</div></div>
</div>
</div>

<div class="{PREFIX}-panel active" id="{PREFIX}-p-4">
<svg viewBox="0 0 580 840" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:580px;display:block;margin:0 auto">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#888780"/></marker>
    <marker id="arr-b" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#185FA5"/></marker>
    <marker id="arr-r" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#993C1D"/></marker>
    <marker id="arr-g" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#0F6E56"/></marker>
    <marker id="arr-p" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#534AB7"/></marker>
  </defs>

  <ellipse cx="280" cy="36" rx="60" ry="22" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/>
  <text x="280" y="41" text-anchor="middle" font-size="13" font-weight="500" fill="#085041">início</text>

  <rect x="130" y="85" width="300" height="52" rx="6" fill="#E1F5EE" stroke="#9FE1CB" stroke-width="1"/>
  <text x="280" y="104" text-anchor="middle" font-size="12" font-weight="500" fill="#0F6E56">rotular sementes e normalizar máscara</text>
  <text x="280" y="121" text-anchor="middle" font-size="11" fill="#085041">f ← mm.label0(f, b) ; g ← f.copy() ; mask ← mask > 0</text>

  <polygon points="280,166 390,196 280,226 170,196" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="201" text-anchor="middle" font-size="12" fill="#0C447C">True?</text>

  <rect x="150" y="256" width="260" height="40" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="281" text-anchor="middle" font-size="12" fill="#712B13">reiniciar ciclo: mudou ← False</text>

  <rect x="120" y="326" width="320" height="72" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="345" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">varredura espacial e expansão</text>
  <text x="280" y="364" text-anchor="middle" font-size="11.5" fill="#712B13">se g[x,y] == 0 e mask[x,y] e f[vy,vx] > g[x,y]:</text>
  <text x="280" y="382" text-anchor="middle" font-size="11.5" font-weight="500" fill="#712B13">g[x,y] ← f[vy,vx] ; mudou ← True</text>

  <polygon points="280,430 380,460 280,490 180,460" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="465" text-anchor="middle" font-size="11.5" fill="#712B13">not mudou?</text>

  <rect x="150" y="520" width="260" height="40" rx="6" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="545" text-anchor="middle" font-size="12" fill="#0C447C">sincronizar malha de ref: f ← g.copy()</text>

  <polygon points="280,600 390,630 280,660 170,630" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="280" y="635" text-anchor="middle" font-size="12" fill="#3C3489">op == 'region'?</text>

  <rect x="70" y="695" width="170" height="40" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="155" y="719" text-anchor="middle" font-size="11.5" fill="#3C3489">retornar g ( bacias )</text>

  <rect x="320" y="695" width="200" height="40" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="420" y="719" text-anchor="middle" font-size="11.5" fill="#3C3489">retornar mm.gradm(g)</text>

  <ellipse cx="280" cy="790" rx="55" ry="22" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="280" y="795" text-anchor="middle" font-size="13" font-weight="500" fill="#27500A">fim</text>

  <line x1="280" y1="58" x2="280" y2="85" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="137" x2="280" y2="166" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <line x1="280" y1="226" x2="280" y2="256" stroke="#185FA5" stroke-width="1.2" marker-end="url(#arr-b)"/>
  <text x="295" y="240" font-size="11" fill="#185FA5">sim</text>

  <line x1="280" y1="296" x2="280" y2="326" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <line x1="280" y1="398" x2="280" y2="430" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  
  <line x1="280" y1="490" x2="280" y2="520" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="295" y="504" font-size="11" fill="#993C1D">não</text>

  <path d="M 150,540 L 45,540 L 45,196 L 170,196" fill="none" stroke="#185FA5" stroke-width="1.2" marker-end="url(#arr-b)"/>

  <path d="M 380,460 L 460,460 L 460,580 L 280,580 L 280,600" fill="none" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="420" y="450" text-anchor="middle" font-size="11" fill="#993C1D">sim (break)</text>

  <path d="M 390,196 L 500,196 L 500,580 L 280,580" fill="none" stroke="#185FA5" stroke-width="1.2"/>
  <text x="445" y="186" font-size="11" fill="#185FA5">não</text>

  <line x1="170" y1="630" x2="155" y2="630" stroke="#534AB7" stroke-width="1.2"/>
  <line x1="155" y1="630" x2="155" y2="695" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-p)"/>
  <text x="140" y="620" font-size="11" fill="#534AB7">sim</text>

  <line x1="390" y1="630" x2="420" y2="630" stroke="#534AB7" stroke-width="1.2"/>
  <line x1="420" y1="630" x2="420" y2="695" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-p)"/>
  <text x="402" y="620" font-size="11" fill="#534AB7">não</text>

  <line x1="155" y1="735" x2="155" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="155" y1="755" x2="280" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="420" y1="735" x2="420" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="420" y1="755" x2="280" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="280" y1="755" x2="280" y2="768" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
</svg>
</div>

<script>
(function() {{
  var container = document.getElementById('{PREFIX}-tabs-container');
  if (!container) return;
  
  var tabs = container.querySelectorAll('.{PREFIX}-tab');
  var panels = [
    document.getElementById('{PREFIX}-p-0'),
    document.getElementById('{PREFIX}-p-1'),
    document.getElementById('{PREFIX}-p-2'),
    document.getElementById('{PREFIX}-p-3'),
    document.getElementById('{PREFIX}-p-4')
  ];

  tabs.forEach(function(tab) {{
    tab.onclick = function() {{
      var idx = parseInt(this.getAttribute('data-idx'));
      
      tabs.forEach(function(t) {{ t.classList.remove('active'); }});
      panels.forEach(function(p) {{ if(p) p.classList.remove('active'); }});
      
      this.classList.add('active');
      if(panels[idx]) panels[idx].classList.add('active');
    }};
  }});
}})();
</script>
''')

**Figura 4.23:** Algoritmo Didáctico de *Watershed* por Crecimiento de Regiones Limitado por Máscara.


<figure id="fig-04-sim-alg-watershed2">
  <img src="imagens/fig-04-sim-alg-watershed2.png" alt=" Algoritmo Didáctico de *Watershed* por Crecimiento de Regiones Limitado por Máscara. " style="max-width:80%" />
  <figcaption><strong>Figura 4.23:</strong>  Algoritmo Didáctico de *Watershed* por Crecimiento de Regiones Limitado por Máscara. </figcaption>
</figure>

A [Figura 4.24](#fig-04-sim-04-watershed) presenta un simulador iterativo que ilustra la propagación de los marcadores por la región de interés. Cada marcador actúa como una fuente de inundación que expande su área de influencia hasta encontrar regiones provenientes de otras semillas. En el algoritmo de OpenCV, los píxeles pertenecientes a las líneas divisorias se identifican por el valor `-1`, representando las fronteras entre cuencas adyacentes.

In [55]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-watershed" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-watershed * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-watershed canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; }
  #sim-04-watershed button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-watershed button:hover { background: #e8dfcf; }
  #sim-04-watershed button.ws_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-watershed .ws_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .ws_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .ws_grid_stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 14px; }
  .ws_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .ws_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .ws_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  
  .ws_se_grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 4px; max-width: 110px; margin: 6px auto; }
  .ws_se_btn { padding: 6px!important; font-size: 10px!important; font-family: monospace; font-weight: 700; }
  .ws_se_btn.ws_off { color: #8a8371!important; background: #fafaf7!important; border-color: #e4dcc8!important; font-weight: normal; }
  
  .ws_tool_grid { display: grid; grid-template-columns: 1fr 1fr; gap: 6px; margin-top: 6px; }
  .ws_color_dot { width: 10px; height: 10px; border-radius: 2px; display: inline-block; border: 1px solid rgba(0,0,0,0.2); }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">💧 Simulador: Segmentación por Watershed</span>
  <span class="ws_pill">Propagación con Elemento Estructurante (b)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="ws_grid_stats">
    <div class="ws_stat_box">
      <div class="ws_stat_label">Área (Máscara)</div>
      <div id="ws_statPx" class="ws_stat_value" style="color:#8a8371;">0</div>
    </div>
    <div class="ws_stat_box">
      <div class="ws_stat_label">Marcadores</div>
      <div id="ws_statSeeds" class="ws_stat_value" style="color:#27ae60;">0</div>
    </div>
    <div class="ws_stat_box">
      <div class="ws_stat_label">Relleno</div>
      <div id="ws_statFill" class="ws_stat_value" style="color:#2980b9;">0%</div>
    </div>
    <div class="ws_stat_box">
      <div class="ws_stat_label">Iteración (k)</div>
      <div id="ws_statStep" class="ws_stat_value" style="color:#c0392b;">–</div>
    </div>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start;">
    
    <!-- Canvas -->
    <div style="flex:2; min-width:260px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
      <canvas id="ws_Canvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Arrastre para dibujar/borrar la máscara o las semillas
      </div>
    </div>

    <!-- Painel Lateral de Ferramentas -->
    <div style="flex:1; min-width:200px; display:flex; flex-direction:column; gap:12px;">
      
      <!-- Ferramentas de Desenho -->
      <div class="ws_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Herramientas
        </div>
        <div class="ws_tool_grid">
          <button id="ws_tool_mask" class="ws_active" data-tool="mask">
            <span class="ws_color_dot" style="background:#e4dcc8;"></span> Máscara
          </button>
          <button id="ws_tool_erase" data-tool="erase">
            <span class="ws_color_dot" style="background:#fafaf7; border-style:dashed;"></span> Borrar
          </button>
          <button id="ws_tool_s2" data-tool="s2">
            <span class="ws_color_dot" style="background:#2980b9;"></span> Sem. 1
          </button>
          <button id="sim04_ws_tool_s3" data-tool="s3">
            <span class="ws_color_dot" style="background:#27ae60;"></span> Sem. 2
          </button>
          <button id="sim04_ws_tool_s4" data-tool="s4" style="grid-column: span 2;">
            <span class="ws_color_dot" style="background:#b9770e;"></span> Sem. 3
          </button>
        </div>
      </div>

      <!-- Elemento Estruturante (b) -->
      <div class="ws_panel" style="text-align:center;">
        <div style="font-size:11px; font-weight:700; margin-bottom:2px; text-align:left; color:#5e5a4a;">
          Elemento Estructurante (b)
        </div>
        <div class="ws_se_grid">
          <button id="ws_se_0_0" class="ws_se_btn ws_off" data-se="0,0">0</button>
          <button id="ws_se_0_1" class="ws_se_btn ws_active" data-se="0,1">1</button>
          <button id="sim04_ws_se_0_2" class="ws_se_btn ws_off" data-se="0,2">0</button>
          
          <button id="sim04_ws_se_1_0" class="ws_se_btn ws_active" data-se="1,0">1</button>
          <button class="ws_se_btn" style="background:#e4dcc8; cursor:not-allowed; color:#8a8371;" disabled>1</button>
          <button id="sim04_ws_se_1_2" class="ws_se_btn ws_active" data-se="1,2">1</button>
          
          <button id="sim04_ws_se_2_0" class="ws_se_btn ws_off" data-se="2,0">0</button>
          <button id="sim04_ws_se_2_1" class="ws_se_btn ws_active" data-se="2,1">1</button>
          <button id="sim04_ws_se_2_2" class="ws_se_btn ws_off" data-se="2,2">0</button>
        </div>
        <div style="display:flex; gap:4px; margin-top:8px;">
          <button data-se-preset="cross" style="flex:1; font-size:9.5px;">Cruz (C-4)</button>
          <button data-se-preset="square" style="flex:1; font-size:9.5px;">Cuad. (C-8)</button>
        </div>
      </div>

      <!-- Modo de Visualização -->
      <div class="ws_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Visualización
        </div>
        <div style="display:flex; gap:6px;">
          <button id="ws_btnLabel" class="ws_active" data-mode="label" style="flex:1; justify-content:center;">Final</button>
          <button id="ws_btnAnim" data-mode="anim" style="flex:1; justify-content:center;">Animado</button>
        </div>
      </div>

      <!-- Controles de Animação -->
      <div id="ws_animCtrl" class="ws_panel" style="display:none;">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Inundación Paso a Paso
        </div>
        <div style="display:flex; gap:6px; margin-bottom:8px;">
          <button data-action="prev" style="flex:1; justify-content:center;">◀</button>
          <button id="ws_btnPlay" data-action="play" style="flex:1; justify-content:center;">▶</button>
          <button data-action="next" style="flex:1; justify-content:center;">▶▶</button>
        </div>
        <div style="display:flex; align-items:center; gap:8px;">
          <label style="font-size:9.5px; color:#8a8371; font-weight:700;">Vel.</label>
          <input type="range" id="ws_speedSlider" min="1" max="10" value="6" style="flex:1; max-width:160px; height:4px; cursor:pointer;">
        </div>
      </div>

      <!-- Exemplos Iniciais -->
      <div class="ws_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Ejemplos Iniciales
        </div>
        <div style="display:flex; flex-direction:column; gap:5px;">
          <button data-preset="moedas" style="justify-content:flex-start;">🪙 Monedas Tocándose</button>
          <button data-preset="celulas" style="justify-content:flex-start;">🦠 Aglomerado de 3</button>
        </div>
      </div>

      <!-- Botão de Limpeza -->
      <button data-action="clear" style="justify-content:center; border-color:#f5b7b1; color:#c0392b; background:#fdecea;">
        🗑️ Limpiar Todo
      </button>

    </div>

  </div>

</div>
</div>

<script>
(function(){
  function initSim04Watershed(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const ws_COLS = 16, ws_ROWS = 14, ws_CELL = 20, ws_PAD = 10;
    const ws_W = ws_COLS * ws_CELL + ws_PAD * 2, ws_H = ws_ROWS * ws_CELL + ws_PAD * 2;
    const ws_cv = root.querySelector('#ws_Canvas');
    ws_cv.width = ws_W; 
    ws_cv.height = ws_H;
    const ws_ctx = ws_cv.getContext('2d');

    const ws_COLORS = {
      0:  { fill: 'transparent', text: '#8a8371', stroke: '#e4dcc8' },
      1:  { fill: '#fafaf7',     text: '#8a8371', stroke: '#e4dcc8' },
      2:  { fill: '#ebf4fd',     text: '#2980b9', stroke: '#a9cce3' },
      3:  { fill: '#eafaf1',     text: '#27ae60', stroke: '#a3e4d7' },
      4:  { fill: '#fef5e7',     text: '#b9770e', stroke: '#f8c471' }
    };

    let ws_grid = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
    let ws_markers = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
    
    let ws_b = [
      [false, true,  false],
      [true,  true,  true],
      [false, true,  false]
    ];
    
    let ws_currentTool = 'mask'; 
    let ws_mode = 'label';
    let ws_painting = false;

    let ws_animSteps = [];
    let ws_animIdx = 0;
    let ws_playing = false;
    let ws_playTimer = null;

    function ws_compute() {
      let steps = [];
      
      let current = Array.from({length:ws_ROWS}, (_, r) => 
        Array.from({length:ws_COLS}, (_, c) => {
          if (ws_markers[r][c] > 1) return ws_markers[r][c];
          if (ws_grid[r][c] === 1) return 1;
          return 0;
        })
      );
      
      const copy = arr => arr.map(row => [...row]);
      steps.push({ g: copy(current), k: 0 });

      let k = 0;
      while (true) {
        k++;
        let nextG = copy(current);
        let changed = false;

        for (let r = 0; r < ws_ROWS; r++) {
          for (let c = 0; c < ws_COLS; c++) {
            if (current[r][c] === 1) {
              let labels = new Set();
              
              for (let dr = -1; dr <= 1; dr++) {
                for (let dc = -1; dc <= 1; dc++) {
                  if (dr === 0 && dc === 0) continue;
                  if (ws_b[dr+1][dc+1]) { 
                    let nr = r + dr, nc = c + dc;
                    if (nr >= 0 && nr < ws_ROWS && nc >= 0 && nc < ws_COLS) {
                      let val = current[nr][nc];
                      if (val > 1) labels.add(val);
                    }
                  }
                }
              }

              if (labels.size === 1) {
                nextG[r][c] = [...labels][0];
                changed = true;
              } else if (labels.size > 1) {
                let labelsArr = [...labels];
                nextG[r][c] = labelsArr[Math.floor(Math.random() * labelsArr.length)];
                changed = true;
              }
            }
          }
        }

        if (!changed) {
          steps.push({ g: copy(nextG), k: k, done: true });
          break;
        }
        steps.push({ g: copy(nextG), k: k });
        current = nextG;
      }
      
      return { final: current, steps };
    }

    function ws_drawState(matrix, stepInfo) {
      ws_ctx.clearRect(0, 0, ws_W, ws_H);
      let area = 0, filled = 0, numSeeds = new Set();

      for (let r=0; r<ws_ROWS; r++) {
        for (let c=0; c<ws_COLS; c++) {
          const x = ws_PAD + c * ws_CELL, y = ws_PAD + r * ws_CELL;
          const val = matrix[r][c];
          
          if (ws_grid[r][c] === 1) area++;
          if (ws_markers[r][c] > 1) numSeeds.add(ws_markers[r][c]);
          if (val > 1) filled++;

          const col = ws_COLORS[val] || ws_COLORS[0];
          ws_ctx.fillStyle = col.fill; 
          ws_ctx.fillRect(x+1, y+1, ws_CELL-2, ws_CELL-2);
          
          if (val > 1) {
            ws_ctx.strokeStyle = col.stroke; ws_ctx.lineWidth = 1; ws_ctx.setLineDash([]);
            ws_ctx.strokeRect(x+0.5, y+0.5, ws_CELL-1, ws_CELL-1);
            ws_ctx.fillStyle = col.text;
            ws_ctx.font = 'bold 10px monospace';
            ws_ctx.textAlign = 'center'; ws_ctx.textBaseline = 'middle';
            ws_ctx.fillText('S' + (val-1), x+ws_CELL/2, y+ws_CELL/2);
          } else if (val === 1) {
            ws_ctx.strokeStyle = col.stroke; ws_ctx.lineWidth = 1; ws_ctx.setLineDash([2,2]);
            ws_ctx.strokeRect(x+0.5, y+0.5, ws_CELL-1, ws_CELL-1);
          } else {
            ws_ctx.strokeStyle = col.stroke; ws_ctx.lineWidth = 0.5; ws_ctx.setLineDash([]);
            ws_ctx.strokeRect(x+0.5, y+0.5, ws_CELL-1, ws_CELL-1);
          }

          if (r===0) { ws_ctx.fillStyle='#8a8371'; ws_ctx.font='8px monospace'; ws_ctx.textAlign='center'; ws_ctx.fillText(c, x+ws_CELL/2, ws_PAD/2); }
          if (c===0) { ws_ctx.fillStyle='#8a8371'; ws_ctx.font='8px monospace'; ws_ctx.textAlign='center'; ws_ctx.fillText(r, ws_PAD/2, y+ws_CELL/2); }
        }
      }

      let pct = area === 0 ? 0 : Math.round((filled / area) * 100);
      root.querySelector('#ws_statPx').textContent = area;
      root.querySelector('#ws_statSeeds').textContent = numSeeds.size;
      root.querySelector('#ws_statFill').textContent = pct + '%';
      
      if (stepInfo) {
        root.querySelector('#ws_statStep').textContent = stepInfo.done ? 'Concluído' : stepInfo.k;
      } else {
        root.querySelector('#ws_statStep').textContent = 'Finalizado';
      }
    }

    function ws_updateSEUI() {
      for (let r=0; r<3; r++) {
        for (let c=0; c<3; c++) {
          if(r===1 && c===1) continue;
          const btn = root.querySelector('#ws_se_' + r + '_' + c);
          if(!btn) continue;
          if(ws_b[r][c]) {
            btn.textContent = '1';
            btn.className = 'ws_se_btn ws_active';
          } else {
            btn.textContent = '0';
            btn.className = 'ws_se_btn ws_off';
          }
        }
      }
    }

    window.ws_toggleSE = function(r, c) {
      if(r===1 && c===1) return;
      ws_b[r][c] = !ws_b[r][c];
      ws_updateSEUI();
      ws_refresh();
    };

    window.ws_setPresetSE = function(type) {
      for (let r=0; r<3; r++) for (let c=0; c<3; c++) ws_b[r][c] = false;
      ws_b[1][1] = true;
      if (type === 'cross') {
        ws_b[0][1] = ws_b[1][0] = ws_b[1][2] = ws_b[2][1] = true;
      } else if (type === 'square') {
        for (let r=0; r<3; r++) for (let c=0; c<3; c++) ws_b[r][c] = true;
      }
      ws_updateSEUI();
      ws_refresh();
    };

    function ws_refresh() {
      const res = ws_compute();
      if (ws_mode === 'anim') {
        ws_animSteps = res.steps;
        ws_animIdx = 0;
        ws_drawState(ws_animSteps[0].g, ws_animSteps[0]);
      } else {
        ws_drawState(res.final, null);
      }
    }

    function ws_setTool(t) {
      ws_currentTool = t;
      ['mask','erase','s2','s3','s4'].forEach(id => {
        const el = root.querySelector('#ws_tool_' + id);
        if (el) el.classList.toggle('ws_active', id === t);
      });
    }

    function ws_setMode(m) {
      ws_mode = m;
      ws_stopPlay();
      root.querySelector('#ws_btnLabel').classList.toggle('ws_active', m==='label');
      root.querySelector('#ws_btnAnim').classList.toggle('ws_active', m==='anim');
      root.querySelector('#ws_animCtrl').style.display = m==='anim' ? 'block' : 'none';
      ws_refresh();
    }

    function ws_animStep(d) {
      ws_animIdx = Math.max(0, Math.min(ws_animSteps.length-1, ws_animIdx+d));
      ws_drawState(ws_animSteps[ws_animIdx].g, ws_animSteps[ws_animIdx]);
    }

    function ws_togglePlay() {
      if (ws_playing) ws_stopPlay(); else ws_startPlay();
    }

    function ws_startPlay() {
      if (ws_animSteps.length === 0) return;
      ws_playing = true;
      root.querySelector('#ws_btnPlay').textContent = '⏸';
      function tick() {
        if (ws_animIdx >= ws_animSteps.length-1) { ws_stopPlay(); return; }
        ws_animIdx++;
        ws_drawState(ws_animSteps[ws_animIdx].g, ws_animSteps[ws_animIdx]);
        const spd = +root.querySelector('#ws_speedSlider').value;
        const delay = Math.round(1100 - spd * 100);
        ws_playTimer = setTimeout(tick, delay);
      }
      tick();
    }

    function ws_stopPlay() {
      ws_playing = false;
      if (ws_playTimer) { clearTimeout(ws_playTimer); ws_playTimer = null; }
      root.querySelector('#ws_btnPlay').textContent = '▶';
    }

    function ws_clearGrid() {
      ws_grid = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
      ws_markers = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
      ws_stopPlay();
      ws_refresh();
    }

    const ws_PRESETS = {
      moedas: () => {
        let g = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        let m = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        for(let r=3; r<=10; r++) for(let c=2; c<=7; c++) g[r][c] = 1;
        g[3][2]=0; g[3][7]=0; g[10][2]=0; g[10][7]=0; 
        for(let r=4; r<=11; r++) for(let c=6; c<=12; c++) g[r][c] = 1;
        g[4][6]=0; g[4][12]=0; g[11][6]=0; g[11][12]=0;
        m[6][4] = 2;
        m[8][9] = 3;
        return {g, m};
      },
      celulas: () => {
        let g = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        let m = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        for(let r=2; r<=12; r++) for(let c=3; c<=12; c++) {
           if((r>=3 && r<=11) || (c>=4 && c<=11)) g[r][c] = 1;
        }
        g[2][3]=g[2][12]=g[12][3]=g[12][12]=0;
        m[4][6] = 2;
        m[10][5] = 3;
        m[7][10] = 4;
        return {g, m};
      }
    };

    function ws_loadPreset(name) {
      const {g, m} = ws_PRESETS[name]();
      ws_grid = g; ws_markers = m;
      ws_stopPlay();
      ws_refresh();
    }

    function ws_cellAt(e) {
      const rect = ws_cv.getBoundingClientRect();
      const sx = ws_W / rect.width, sy = ws_H / rect.height;
      const raw = e.touches ? e.touches[0] : e;
      const c = Math.floor(((raw.clientX - rect.left) * sx - ws_PAD) / ws_CELL);
      const r = Math.floor(((raw.clientY - rect.top) * sy - ws_PAD) / ws_CELL);
      return {r, c};
    }

    function ws_applyTool(r, c) {
      if (ws_currentTool === 'mask') {
        ws_grid[r][c] = 1;
        if (ws_markers[r][c] > 0) ws_markers[r][c] = 0;
      } else if (ws_currentTool === 'erase') {
        ws_grid[r][c] = 0;
        ws_markers[r][c] = 0;
      } else {
        let seedVal = parseInt(ws_currentTool.replace('s',''));
        ws_grid[r][c] = 1;
        ws_markers[r][c] = seedVal;
      }
    }

    ws_cv.addEventListener('mousedown', function(e){
      const {r, c} = ws_cellAt(e);
      if (r<0 || r>=ws_ROWS || c<0 || c>=ws_COLS) return;
      ws_painting = true;
      ws_applyTool(r, c);
      ws_refresh();
    });

    ws_cv.addEventListener('mousemove', function(e){
      if (!ws_painting) return;
      const {r, c} = ws_cellAt(e);
      if (r<0 || r>=ws_ROWS || c<0 || c>=ws_COLS) return;
      ws_applyTool(r, c);
      ws_refresh();
    });
    
    document.addEventListener('mouseup', () => ws_painting = false);

    root.querySelectorAll('[data-tool]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_setTool(this.getAttribute('data-tool'));
      });
    });

    root.querySelectorAll('[data-se]').forEach(btn => {
      btn.addEventListener('click', function() {
        const parts = this.getAttribute('data-se').split(',');
        ws_toggleSE(parseInt(parts[0], 10), parseInt(parts[1], 10));
      });
    });

    root.querySelectorAll('[data-se-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_setPresetSE(this.getAttribute('data-se-preset'));
      });
    });

    root.querySelectorAll('[data-mode]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_setMode(this.getAttribute('data-mode'));
      });
    });

    root.querySelectorAll('[data-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_loadPreset(this.getAttribute('data-preset'));
      });
    });

    root.querySelectorAll('[data-action]').forEach(btn => {
      btn.addEventListener('click', function() {
        const act = this.getAttribute('data-action');
        if(act === 'clear') ws_clearGrid();
        if(act === 'prev') ws_animStep(-1);
        if(act === 'next') ws_animStep(1);
        if(act === 'play') ws_togglePlay();
      });
    });

    ws_updateSEUI();
    ws_loadPreset('moedas');
  }

  function tryInitSim04Watershed(){
    var root = document.getElementById('sim-04-watershed');
    if (root) initSim04Watershed(root); else setTimeout(tryInitSim04Watershed, 200);
  }
  tryInitSim04Watershed();
})();
</script>
""")

**Figura 4.24:** Simulador interactivo del Algoritmo *Watershed* por propagación morfológica. Dibuja la máscara, coloca los marcadores y ajusta el Elemento Estructurante para observar la inundación. Cuando las cuencas se encuentran simultáneamente, el empate se resuelve asumiendo una de las regiones de manera aleatoria.


<figure id="fig-04-sim-04-watershed">
  <img src="imagens/fig-04-sim-04-watershed.png" alt=" Simulador interactivo del Algoritmo *Watershed* por propagación morfológica. Dibuja la máscara, coloca los marcadores y ajusta el Elemento Estructurante para observar la inundación. Cuando las cuencas se encuentran simultáneamente, el empate se resuelve asumiendo una de las regiones de manera aleatoria. " style="max-width:80%" />
  <figcaption><strong>Figura 4.24:</strong>  Simulador interactivo del Algoritmo *Watershed* por propagación morfológica. Dibuja la máscara, coloca los marcadores y ajusta el Elemento Estructurante para observar la inundación. Cuando las cuencas se encuentran simultáneamente, el empate se resuelve asumiendo una de las regiones de manera aleatoria. </figcaption>
</figure>

#### *Pipeline* Morfológico del *Watershed*

El *watershed* basado en marcadores normalmente integra un flujo más amplio de segmentación. En imágenes reales, etapas de preprocesamiento son frecuentemente necesarias para mejorar el contraste, reducir ruidos y generar marcadores confiables. Este flujo completo está resumido en la [Tabela 4.5](#tbl-04-watershed-pipeline-real).

<a id="tbl-04-watershed-pipeline-real"></a>

**Tabela 4.5:** *Pipeline* completo del *watershed* basado en marcadores para imágenes reales.

| Etapa | Operación | Finalidad |
| --- | --- | --- |
| 1 | CLAHE + Suavizado | Realce de contraste y reducción de ruido |
| 2 | Umbralización | Separación inicial entre objeto y fondo |
| 3 | Apertura/Cierre | Eliminación de ruidos y pequeñas imperfecciones |
| 4 | Dilatación de la Máscara | Identificación del Fondo Seguro (*Sure Background*) |
| 5 | Transformada de Distancia + Umbral | Identificación del Objeto Seguro (*Sure Foreground*) |
| 6 | Región Incierta | Diferencia entre Fondo Seguro y Objeto Seguro |
| 7 | `mm::watershed` | Propagación de los marcadores por la región incierta |


Para enfatizar exclusivamente los conceptos de Transformada de Distancia, marcadores e inundación topográfica, el ejemplo de la [Figura 4.25](#fig-04-watershed-didatico) utiliza una imagen binaria sintética y adopta un flujo simplificado, resumido en la [Tabela 4.6](#tbl-04-watershed-pipeline-simples).

<a id="tbl-04-watershed-pipeline-simples"></a>

**Tabela 4.6:** *Pipeline* simplificado utilizado en el ejemplo didáctico de la [Figura 4.25](#fig-04-watershed-didatico).

| Etapa | Operación | Finalidad |
| --- | --- | --- |
| 1 | Transformada de Distancia | Construcción de la superficie topográfica |
| 2 | Umbral de la TD | Extracción de los marcadores (*Sure Foreground*) |
| 3 | Dilatación | Determinación del Fondo Seguro (*Sure Background*) |
| 4 | Región Incierta | Diferencia entre fondo y marcadores |
| 5 | `mm::watershed` | Propagación de los marcadores y generación de las fronteras |


In [56]:
%%writefile tmp/fig_04_watershed_didatico.cpp
#define MM_OUT "tmp/fig_04_watershed_didatico.png"
// g++ -std=c++17 snippet.cpp -o snippet $(pkg-config --cflags --libs opencv4) -DMM_USE_OPENCV
#include "morph.hpp"
#include <vector>
#include <string>
#include <algorithm>
#include <filesystem>

int main() {
    //| label: fig-04-watershed-didatico
    //| fig-cap: "*Pipeline* *watershed* delimitado por máscara em imagem binária 20×20."
    //| echo: true
    //| output: true

    // 1. Imagem sintética e Transformada de Distância
    mm::Image f_sint(20, 20);
    f_sint = mm::circle(f_sint, 6, 10, 5, 255, -1);
    f_sint = mm::circle(f_sint, 14, 10, 5, 255, -1);
    mm::Image dist = mm::dist(f_sint);

    // 2. Marcadores (picos da distância)
    mm::Image m(20, 20);
    unsigned char max_dist = 0;
    for (int y = 0; y < dist.h; y++) {
        for (int x = 0; x < dist.w; x++) {
            if (dist.at(y, x) > max_dist) {
                max_dist = dist.at(y, x);
            }
        }
    }
    for (int y = 0; y < dist.h; y++) {
        for (int x = 0; x < dist.w; x++) {
            m.at(y, x) = (dist.at(y, x) > 0.8 * max_dist) ? 255 : 0;
        }
    }

    // 3. Execução do Watershed0 condicional (m=marcadores primeiro, mask=f_sint)
    mm::Image w_reg = mm::watershed(m, f_sint, "region");
    mm::Image w_line = mm::watershed(m, f_sint, "line");

    //w_reg  = mm.watershedB(m, mask=f_sint, op='region')
    //w_line = mm.watershedB(m, mask=f_sint, op='line')

    // 4. Exibição dos resultados
    mm::show(std::vector<mm::Image>{f_sint, dist, m, w_reg, w_line}, MM_OUT,
             std::vector<std::string>{"Original", "Distância", "Marcadores", "Regiões", "Linhas"}, 5);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(f_sint, "tmp/fig_04_watershed_didatico_0.png");
mm::write(dist, "tmp/fig_04_watershed_didatico_1.png");
mm::write(m, "tmp/fig_04_watershed_didatico_2.png");
mm::write(w_reg, "tmp/fig_04_watershed_didatico_3.png");
mm::write(w_line, "tmp/fig_04_watershed_didatico_4.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_watershed_didatico.cpp


In [57]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_watershed_didatico.cpp -o tmp/fig_04_watershed_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_watershed_didatico \
  && test -f "tmp/fig_04_watershed_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_watershed_didatico.png"

[1] Original
[2] Distância
[3] Marcadores
[4] Regiões
[5] Linhas


In [58]:
try:
    mm.show(
        [
            mm.read("tmp/fig_04_watershed_didatico_0.png"),
            mm.read("tmp/fig_04_watershed_didatico_1.png"),
            mm.read("tmp/fig_04_watershed_didatico_2.png"),
            mm.read("tmp/fig_04_watershed_didatico_3.png"),
            mm.read("tmp/fig_04_watershed_didatico_4.png"),
        ],
        titles=[
            'Original',
            'Distância',
            'Marcadores',
            'Regiões',
            'Linhas',
        ],
        cols=5,
        figsize=(16, 4),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_watershed_didatico_0.png (ver a versao Python)")

<Figure size 2400x600 with 5 Axes>

**Figura 4.25:** *Pipeline* *watershed* delimitado por máscara em imagem binária 20×20.


**Aplicación en monedas superpuestas:**

In [59]:
%%writefile tmp/fig_04_watershed_moedas.cpp
#define MM_OUT "tmp/fig_04_watershed_moedas.png"
#include "morph.hpp"
#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <iostream>
#include <set>
#include <algorithm>
#include <numeric>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
mm::Image img_final = mm::_read_state("tmp/state/img_final_55.png");
// [pdi:state-io:end]

    // Variables img_coins_gray (mm::Image) and img_final (mm::Image) are provided

    mm::Image img_base = img_coins_gray;

    // 1. Simular moedas sobrepostas/conectadas (usando a máscara binária base)
    mm::Image img_sobrepostas = mm::dil(img_final, mm::sebox(40));

    // 2. Abertura morfológica para limpar ruídos
    mm::Image opening = mm::open(img_sobrepostas, mm::sebox(2));

    // 3. Transformada de Distância
    mm::Image dist = mm::dist(opening);

    // Normalize dist to [0, 255] for visualization
    mm::Image dist_vis(dist.h, dist.w);
    int max_val = 0;
    for (int i = 0; i < (int)dist.data.size(); i++) {
        max_val = std::max(max_val, (int)dist.data[i]);
    }
    if (max_val > 0) {
        for (int i = 0; i < (int)dist.data.size(); i++) {
            dist_vis.data[i] = (unsigned char)(255 * (double)dist.data[i] / max_val);
        }
    } else {
        dist_vis = dist;
    }

    // 4. Picos seguros (Marcadores das moedas)
    mm::Image picos(dist.h, dist.w);
    for (int i = 0; i < (int)dist.data.size(); i++) {
        if ((double)dist.data[i] > 0.5 * max_val) {
            picos.data[i] = 255;
        }
    }

    // 5. Execução do Watershed (Ajustado para usar a nova assinatura)
    // Passamos 'opening' direto em mask, pois ela delimita o escopo de expansão das moedas
    mm::Image ws_region = mm::watershed(picos, opening, "region");
    mm::Image ws_line = mm::watershed(picos, opening, "line");

    // 6. Contagem de objetos (Ignora fundo 0)
    std::set<int> unique_labels;
    for (int i = 0; i < (int)ws_region.data.size(); i++) {
        int val = ws_region.data[i];
        if (val > 0) {
            unique_labels.insert(val);
        }
    }
    std::vector<int> labels(unique_labels.begin(), unique_labels.end());
    std::cout << "Objetos detectados: " << labels.size() << "\n";

    // 7. Anotação Final
    // Convert img_base (1-channel) to BGR for annotation
    cv::Mat img_base_mat(img_base.h, img_base.w, CV_8UC1, img_base.data.data());
    cv::Mat img_annotated_mat;
    cv::cvtColor(img_base_mat, img_annotated_mat, cv::COLOR_GRAY2BGR);

    for (size_t idx = 0; idx < labels.size(); idx++) {
        int label_id = labels[idx];

        // Build mask for this label
        cv::Mat mask_reg(img_base.h, img_base.w, CV_8UC1, cv::Scalar(0));
        int mask_sum = 0;
        for (int y = 0; y < img_base.h; y++) {
            for (int x = 0; x < img_base.w; x++) {
                if (ws_region.at(y, x) == label_id) {
                    mask_reg.at<unsigned char>(y, x) = 255;
                    mask_sum++;
                }
            }
        }
        if (mask_sum < 500) continue;

        // Compute centroid
        int sum_x = 0, sum_y = 0;
        for (int y = 0; y < img_base.h; y++) {
            for (int x = 0; x < img_base.w; x++) {
                if (ws_region.at(y, x) == label_id) {
                    sum_x += x;
                    sum_y += y;
                }
            }
        }
        int cx = sum_x / mask_sum;
        int cy = sum_y / mask_sum;

        cv::putText(img_annotated_mat, std::to_string(idx + 1), cv::Point(cx - 25, cy + 20),
                    cv::FONT_HERSHEY_SIMPLEX, 3.2, cv::Scalar(0, 255, 0), 8, cv::LINE_AA);
    }

    // Copy back to mm::Image
    mm::Image img_annotated(img_annotated_mat.rows, img_annotated_mat.cols, img_annotated_mat.channels());
    std::memcpy(img_annotated.data.data(), img_annotated_mat.data, img_annotated.data.size());

    // Desenha as linhas de separação em vermelho
    // Build 11x11 ones kernel as SE for dilation
    mm::SE kernel_11 = mm::SE::box(11);
    mm::Image mask_ann = mm::dil(ws_line, kernel_11);

    // Apply red color where mask has values > 0
    for (int y = 0; y < img_base.h; y++) {
        for (int x = 0; x < img_base.w; x++) {
            if (mask_ann.at(y, x) > 0) {
                img_annotated.at(y, x, 0) = 255;
                img_annotated.at(y, x, 1) = 0;
                img_annotated.at(y, x, 2) = 0;
            }
        }
    }

    // Exibição
    mm::show(
        std::vector<mm::Image>{img_base, img_sobrepostas, dist_vis, picos, ws_region, img_annotated},
        MM_OUT,
        std::vector<std::string>{"Original", "Sobrepostas", "Distância", "Marcadores", "Watershed", "Anotado"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_base, "tmp/fig_04_watershed_moedas_0.png");
mm::write(img_sobrepostas, "tmp/fig_04_watershed_moedas_1.png");
mm::write(dist_vis, "tmp/fig_04_watershed_moedas_2.png");
mm::write(picos, "tmp/fig_04_watershed_moedas_3.png");
mm::write(ws_region, "tmp/fig_04_watershed_moedas_4.png");
mm::write(img_annotated, "tmp/fig_04_watershed_moedas_5.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_watershed_moedas.cpp


In [60]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_watershed_moedas.cpp -o tmp/fig_04_watershed_moedas -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_watershed_moedas \
  && test -f "tmp/fig_04_watershed_moedas.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_watershed_moedas.png"

Objetos detectados: 12


[1] Original
[2] Sobrepostas
[3] Distância
[4] Marcadores
[5] Watershed
[6] Anotado


In [61]:
try:
    mm.show(
        [
            mm.read("tmp/fig_04_watershed_moedas_0.png"),
            mm.read("tmp/fig_04_watershed_moedas_1.png"),
            mm.read("tmp/fig_04_watershed_moedas_2.png"),
            mm.read("tmp/fig_04_watershed_moedas_3.png"),
            mm.read("tmp/fig_04_watershed_moedas_4.png"),
            mm.read("tmp/fig_04_watershed_moedas_5.png"),
        ],
        titles=[
            'Original',
            'Sobrepostas',
            'Distância',
            'Marcadores',
            'Watershed',
            'Anotado',
        ],
        cols=3,
        rows=2,
        figsize=(12, 8),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_watershed_moedas_0.png (ver a versao Python)")

<Figure size 1800x1200 with 6 Axes>

**Figura 4.26:** *Pipeline* *Watershed* para separação de moedas sobrepostas: da máscara binária dilatada até os contornos finais anotados sobre a imagem original.


## 4.5 Extracción de Componentes y Descriptores de Forma

Tras la segmentación y el refinamiento morfológico, el siguiente paso consiste en identificar individualmente cada objeto presente en la imagen y extraer sus propiedades geométricas. Esta etapa es fundamental para tareas de medición, clasificación y reconocimiento de patrones.

Un **componente conexo** es un conjunto maximal de píxeles pertenecientes al objeto que permanecen mutuamente conectados según una relación de conectividad previamente definida (conectividad 4 u 8). Tras la rotulación, cada componente recibe un identificador único, permitiendo que sus características sean analizadas individualmente.

OpenCV ofrece dos enfoques complementarios para este análisis, resumidos en la [Tabela 4.7](#tbl-04-componentes-vs-contornos).

<a id="tbl-04-componentes-vs-contornos"></a>

**Tabela 4.7:** Comparación entre los enfoques basados en componentes conexos y en contornos.

| | `connectedComponentsWithStats` | `findContours` |
|---|---|---|
| **Devuelve** | etiqueta por píxel y estadísticas por componente | secuencia de puntos que describe el borde |
| **Descriptores directos** | área, *bounding box* y centroide | perímetro, forma y jerarquía |
| **Objetos en contacto** | tiende a fusionar regiones conectadas | tiende a producir un único contorno externo |
| **Uso típico** | conteo, filtrado y rotulación | análisis geométrico y descriptores de forma |


### 4.5.1 Etiquetado y Estadísticas de Componentes

`mm::label0` asigna una etiqueta a cada componente conexo. A partir de la imagen etiquetada se obtienen, mediante un barrido, el **área** (conteo de píxeles) y la **caja delimitadora** de cada objeto ([Figura 4.27](#fig-04-componentes)). El `connectedComponentsWithStats` de OpenCV, que devuelve estas estadísticas ya calculadas, y las anotaciones coloreadas sobre la imagen quedan en el rastro de Python.

In [62]:
%%writefile tmp/fig_04_componentes.cpp
#define MM_OUT "tmp/fig_04_componentes.png"
//| label: fig-04-componentes
//| fig-cap: "Componentes conexos extraídos após o *pipeline* CLAHE → Otsu → limpeza morfológica. Cada objeto é colorido com cor distinta e anotado com sua área em pixels."
//| echo: true
//| output: true
#include "morph.hpp"
#include <opencv2/opencv.hpp>
#include <iostream>
#include <vector>
#include <iomanip>
#include <cstring>
#include <numeric>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
mm::Image img_final = mm::_read_state("tmp/state/img_final_55.png");
// [pdi:state-io:end]

    // img_coins_gray e img_final são fornecidos automaticamente

    // 1. Rotulagem e estatísticas
    cv::Mat mask_final(img_final.h, img_final.w, CV_8UC1, img_final.data.data());
    cv::Mat labels_mat, stats_mat, centroids_mat;
    int n = cv::connectedComponentsWithStats(mask_final, labels_mat, stats_mat, centroids_mat, 8, CV_32S);

    // Converter labels para mm::Image
    mm::Image labels_img(labels_mat.rows, labels_mat.cols);
    std::memcpy(labels_img.data.data(), labels_mat.data, labels_img.data.size());

    // 2. Coloração dos componentes — paleta FIXA (gerada uma vez com
    // np.random.seed(4)) para a trilha C++ reproduzir cor a cor sem
    // depender do gerador do numpy. Se n exceder len(PALETA), cicla.
    const std::vector<std::array<unsigned char, 3>> PALETA = {
        {0, 0, 0}, {224, 82, 193}, {233, 155, 73}, {190, 247, 244},
        {103, 94, 179}, {51, 154, 59}, {137, 96, 232}, {250, 243, 205},
        {100, 141, 208}, {228, 187, 163}, {202, 108, 214}, {131, 105, 104},
        {86, 153, 81}
    };

    mm::Image img_colored(img_final.h, img_final.w, 3);
    mm::Image img_annotated = img_colored;

    for (int y = 0; y < img_final.h; y++) {
        for (int x = 0; x < img_final.w; x++) {
            int label = labels_img.at(y, x);
            const auto& cor = PALETA[label % PALETA.size()];
            img_colored.at(y, x, 0) = cor[0];
            img_colored.at(y, x, 1) = cor[1];
            img_colored.at(y, x, 2) = cor[2];
        }
    }
    img_annotated = img_colored;

    // 3. Tabela de descritores
    std::cout << "Componentes detectados (excluindo fundo): " << (n - 1) << "\n";
    std::cout << std::setw(4) << "ID" << std::setw(8) << "Área" << std::setw(6) << "cx" 
              << std::setw(6) << "cy" << std::setw(6) << "w" << std::setw(6) << "h" << "\n";
    std::cout << std::string(42, '-') << "\n";

    for (int i = 1; i < n; i++) {
        int area = stats_mat.at<int>(i, cv::CC_STAT_AREA);
        int cx = static_cast<int>(centroids_mat.at<double>(i, 0));
        int cy = static_cast<int>(centroids_mat.at<double>(i, 1));
        int w = stats_mat.at<int>(i, cv::CC_STAT_WIDTH);
        int h = stats_mat.at<int>(i, cv::CC_STAT_HEIGHT);
        std::cout << std::setw(4) << i << std::setw(8) << area << std::setw(6) << cx 
                  << std::setw(6) << cy << std::setw(6) << w << std::setw(6) << h << "\n";

        // Anotar no image
        cv::Mat ann_mat(img_annotated.h, img_annotated.w, CV_8UC3, img_annotated.data.data());
        std::string texto = std::to_string(i) + ": " + std::to_string(area);
        for (const auto& [cor, esp] : std::vector<std::pair<cv::Scalar, int>>{
            {cv::Scalar(0, 0, 0), 10}, {cv::Scalar(255, 0, 0), 5}}) {
            cv::putText(ann_mat, texto, cv::Point(cx - 150, cy + 18),
                        cv::FONT_HERSHEY_SIMPLEX, 2.0, cor, esp, cv::LINE_AA);
        }
        std::memcpy(img_annotated.data.data(), ann_mat.data, img_annotated.data.size());
    }

    // Conversões cv::Mat → mm::Image para exibição
    mm::Image img_colored_mm(img_colored.h, img_colored.w, 3);
    std::memcpy(img_colored_mm.data.data(), img_colored.data.data(), img_colored.data.size());

    mm::show(std::vector<mm::Image>{img_coins_gray, img_final, img_colored_mm, img_annotated},
             MM_OUT,
             std::vector<std::string>{"Original", "Segmentação Final", "Componentes Conexos", "Áreas Anotadas"},
             4);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_coins_gray, "tmp/fig_04_componentes_0.png");
mm::write(img_final, "tmp/fig_04_componentes_1.png");
mm::write(img_colored, "tmp/fig_04_componentes_2.png");
mm::write(img_annotated, "tmp/fig_04_componentes_3.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_componentes.cpp


In [63]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_componentes.cpp -o tmp/fig_04_componentes -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_componentes \
  && test -f "tmp/fig_04_componentes.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_componentes.png"

Componentes detectados (excluindo fundo): 12
  ID   Área    cx    cy     w     h
------------------------------------------
   1  231969  1484   280   553   542
   2  158967   917   250   453   468
   3  139229   250   312   433   419
   4  175550   857   821   474   465
   5  222882  1442   889   539   531
   6  210043   316   977   527   516
   7  343376   934  1559   667   665
   8  213641  1555  1574   530   515
   9  147880   328  1543   432   438
  10  187280   363  2111   487   492
  11  150110  1487  2215   433   444
  12  215387   932  2292   528   522


[1] Original
[2] Segmentação Final
[3] Componentes Conexos
[4] Áreas Anotadas


In [64]:
try:
    mm.show(
        [
            mm.read("tmp/fig_04_componentes_0.png"),
            mm.read("tmp/fig_04_componentes_1.png"),
            mm.read("tmp/fig_04_componentes_2.png"),
            mm.read("tmp/fig_04_componentes_3.png"),
        ],
        titles=[
            'Original',
            'Segmentação Final',
            'Componentes Conexos',
            'Áreas Anotadas',
        ],
        cols=4,
        figsize=(18, 6),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_componentes_0.png (ver a versao Python)")

<Figure size 2700x900 with 4 Axes>

**Figura 4.27:** Componentes conexos extraídos após o *pipeline* CLAHE → Otsu → limpeza morfológica. Cada objeto é colorido com cor distinta e anotado com sua área em pixels.


### 4.5.2 Descriptores de Forma

Los descriptores basados en **contorno vectorial** — perímetro (`arcLength`), área del polígono (`contourArea`), circularidad, momentos y aproximación poligonal (`approxPolyDP`) — dependen del rastreo de frontera (`findContours`), ausente en `morph.hpp`. Solo permanecen en la ruta de Python. En la ruta de C++, el **gradiente morfológico** (`mm::gradm`) resalta el contorno de cada objeto ([Figura 4.28](#fig-04-contornos)).

In [65]:
%%writefile tmp/fig_04_contornos.cpp
#define MM_OUT "tmp/fig_04_contornos.png"
// Compile: g++ -std=c++17 -o programa programa.cpp -DMM_USE_OPENCV `pkg-config --cflags --libs opencv4`
#include "morph.hpp"
#include <opencv2/opencv.hpp>
#include <iostream>
#include <vector>
#include <cmath>
#include <iomanip>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
mm::Image img_final = mm::_read_state("tmp/state/img_final_55.png");
// [pdi:state-io:end]

    //| label: fig-04-contornos
    //| fig-cap: "Contornos extraídos com *findContours*. Cada moeda é anotada com sua circularidade — valores próximos de 1 confirmam forma circular."
    //| echo: true
    //| output: true

    // Bridge mm::Image to cv::Mat for OpenCV operations
    cv::Mat img_final_mat(img_final.h, img_final.w, 
                          img_final.channels == 1 ? CV_8UC1 : CV_8UC3,
                          img_final.data.data());

    std::vector<std::vector<cv::Point>> contornos;
    std::vector<cv::Vec4i> hierarquia;
    cv::findContours(img_final_mat, contornos, cv::RETR_EXTERNAL, cv::CHAIN_APPROX_SIMPLE);

    // Bridge mm::Image to cv::Mat for OpenCV operations
    cv::Mat img_coins_gray_mat(img_coins_gray.h, img_coins_gray.w, 
                               img_coins_gray.channels == 1 ? CV_8UC1 : CV_8UC3,
                               img_coins_gray.data.data());

    cv::Mat img_contornos_mat;
    cv::cvtColor(img_coins_gray_mat, img_contornos_mat, cv::COLOR_GRAY2BGR);
    cv::Mat img_circulares_mat = img_contornos_mat.clone();

    std::cout << "Contornos detectados: " << contornos.size() << "\n";
    std::cout << std::setw(4) << "ID" << std::setw(8) << "Área" 
              << std::setw(10) << "Perímetro" << std::setw(14) << "Circularidade" << "\n";
    std::cout << std::string(42, '-') << "\n";

    int id = 1;
    for (const auto& cnt : contornos) {
        double area = cv::contourArea(cnt);
        double perim = cv::arcLength(cnt, true);
        double circ = (perim > 0) ? (4 * M_PI * area / (perim * perim)) : 0;
        cv::Moments M = cv::moments(cnt);
        int cx = (M.m00 > 0) ? static_cast<int>(M.m10 / M.m00) : 0;
        int cy = (M.m00 > 0) ? static_cast<int>(M.m01 / M.m00) : 0;

        std::cout << std::setw(4) << id << std::setw(8) << static_cast<int>(area) 
                  << std::setw(10) << std::fixed << std::setprecision(1) << perim 
                  << std::setw(14) << std::setprecision(3) << circ << "\n";

        cv::drawContours(img_contornos_mat, std::vector<std::vector<cv::Point>>{cnt}, -1, cv::Scalar(0, 255, 0), 3);
        cv::drawContours(img_circulares_mat, std::vector<std::vector<cv::Point>>{cnt}, -1, cv::Scalar(0, 255, 0), 3);

        std::vector<std::pair<cv::Scalar, int>> cor_esp = {
            {cv::Scalar(0, 0, 0), 8}, {cv::Scalar(255, 0, 0), 3}
        };
        for (const auto& ce : cor_esp) {
            cv::putText(img_circulares_mat, std::to_string(circ).substr(0, std::to_string(circ).find('.') + 3),
                        cv::Point(cx - 80, cy + 15),
                        cv::FONT_HERSHEY_SIMPLEX, 3.6, ce.first, ce.second, cv::LINE_AA);
        }
        id++;
    }

    // Build result images and copy back to mm::Image
    mm::Image out_contornos(img_contornos_mat.rows, img_contornos_mat.cols, img_contornos_mat.channels());
    std::memcpy(out_contornos.data.data(), img_contornos_mat.data, out_contornos.data.size());

    mm::Image out_circulares(img_circulares_mat.rows, img_circulares_mat.cols, img_circulares_mat.channels());
    std::memcpy(out_circulares.data.data(), img_circulares_mat.data, out_circulares.data.size());

    mm::show(
        std::vector<mm::Image>{img_final, out_contornos, out_circulares},
        MM_OUT,
        std::vector<std::string>{"Segmentação Final", "Contornos", "Circularidade"},
        3
    );

    return 0;
}

Overwriting tmp/fig_04_contornos.cpp


In [66]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_contornos.cpp -o tmp/fig_04_contornos -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_contornos \
  && test -f "tmp/fig_04_contornos.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_contornos.png"

Contornos detectados: 12
  ID   ÁreaPerímetro Circularidade
------------------------------------------


   1  214626    1769.6         0.861
   2  149480    1465.1         0.875
   3  186570    1654.9         0.856
   4  147252    1458.6         0.870
   5  212885    1756.3         0.867
   6  342424    2232.5         0.863
   7  209282    1767.7         0.842
   8  222108    1809.7         0.852
   9  174854    1616.4         0.841
  10  138602    1471.5         0.804
  11  158306    1538.7         0.840
  12  231181    1847.4         0.851


[1] Segmentação Final
[2] Contornos
[3] Circularidade


In [67]:
try:
    mm.show(mm.read("tmp/fig_04_contornos.png"), figsize=(18, 6))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_contornos.png (ver a versao Python)")

<Figure size 2700x900 with 1 Axes>

**Figura 4.28:** Contornos extraídos com *findContours*. Cada moeda é anotada com sua circularidade — valores próximos de 1 confirmam forma circular.


### 4.5.3 Conexión con la Detección de Objetos Moderna

Los descriptores extraídos en las secciones anteriores — especialmente *bounding boxes*, centroides, áreas y medidas de forma — establecen un puente natural entre la segmentación morfológica clásica y los sistemas modernos de detección de objetos. Aunque las técnicas estudiadas en este capítulo utilizan operaciones sobre píxeles y regiones segmentadas, muchas de las representaciones producidas son directamente compatibles con los formatos empleados en modelos contemporáneos de visión por computadora.

Los detectores basados en aprendizaje profundo, como la familia YOLO (*You Only Look Once*) (REDMON, 2016), operan directamente sobre imágenes en color y producen, para cada objeto detectado, una *bounding box* descrita por el centro $(cx,cy)$ y las dimensiones $(w,h)$, además de una clase y una puntuación de confianza. Esta representación comparte la misma estructura geométrica básica obtenida por `connectedComponentsWithStats`, aunque es producida por un modelo aprendido y no mediante segmentación explícita.

La [Figura 4.29](#fig-04-bbox) ilustra cómo las *bounding boxes* obtenidas por morfología pueden exportarse en el formato YOLO para componer conjuntos de datos utilizados en el entrenamiento o la evaluación de detectores.

In [68]:
%%writefile tmp/fig_04_bbox.cpp
#define MM_OUT "tmp/fig_04_bbox.png"
// Compile with: g++ -std=c++17 -o program program.cpp -I. -L. -lmorph -lopencv_core -lopencv_imgproc -lopencv_imgcodecs -lopencv_highgui
#include "morph.hpp"
#include <opencv2/opencv.hpp>
#include <iostream>
#include <iomanip>
#include <string>
#include <vector>
#include <fstream>
#include <sstream>
#include <cstring>
#include <filesystem>

//| label: fig-04-bbox
//| fig-cap: "*Bounding boxes* derivadas dos componentes conexos sobrepostas à imagem original. As anotações são exportadas no formato YOLO (*classe cx cy w h*), com coordenadas normalizadas para o intervalo [0,1]."
//| echo: true
//| output: true

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
mm::Image img_final = mm::_read_state("tmp/state/img_final_55.png");
// [pdi:state-io:end]

    // Recalcula rótulos/estatísticas a partir da segmentação final
    cv::Mat labels, stats, centroids;
    int n = cv::connectedComponentsWithStats(
        cv::Mat(img_final.h, img_final.w, CV_8UC1, img_final.data.data()),
        labels, stats, centroids, 8, CV_32S
    );

    int H_img = img_coins_gray.h;
    int W_img = img_coins_gray.w;

    // Convert mm::Image to cv::Mat for operations
    cv::Mat gray_mat(H_img, W_img, CV_8UC1, img_coins_gray.data.data());
    cv::Mat img_bbox;
    cv::cvtColor(gray_mat, img_bbox, cv::COLOR_GRAY2BGR);

    int CLASSE = 0;   // 0 = moeda (única categoria neste exemplo)

    std::cout << std::setw(4) << "cls" << std::setw(9) << "cx_n" << std::setw(9) << "cy_n" 
              << std::setw(9) << "w_n" << std::setw(9) << "h_n" << "  ← formato YOLO\n";
    std::cout << std::string(54, '-') << "\n";

    std::vector<std::string> yolo_linhas;
    for (int i = 1; i < n; i++) {
        int x0 = stats.at<int>(i, cv::CC_STAT_LEFT);
        int y0 = stats.at<int>(i, cv::CC_STAT_TOP);
        int w  = stats.at<int>(i, cv::CC_STAT_WIDTH);
        int h  = stats.at<int>(i, cv::CC_STAT_HEIGHT);

        double cx_n = (x0 + w / 2.0) / W_img;
        double cy_n = (y0 + h / 2.0) / H_img;
        double w_n  = w / (double)W_img;
        double h_n  = h / (double)H_img;

        std::ostringstream oss;
        oss << CLASSE << " " << std::fixed << std::setprecision(4) 
            << cx_n << " " << cy_n << " " << w_n << " " << h_n;
        yolo_linhas.push_back(oss.str());

        std::cout << std::setw(4) << CLASSE << std::setw(9) << std::fixed << std::setprecision(4)
                  << cx_n << std::setw(9) << cy_n << std::setw(9) << w_n << std::setw(9) << h_n << "\n";

        cv::rectangle(img_bbox, cv::Point(x0, y0), cv::Point(x0 + w, y0 + h), 
                     cv::Scalar(0, 255, 0), 4);
        cv::putText(img_bbox, "moeda", cv::Point(x0 + 8, y0 + 60),
                    cv::FONT_HERSHEY_SIMPLEX, 3.8, cv::Scalar(255, 0, 0), 5, cv::LINE_AA);
    }

    // Exportar arquivo de anotação no formato YOLO
    std::ofstream f("moedas.txt");
    for (size_t j = 0; j < yolo_linhas.size(); j++) {
        if (j > 0) f << "\n";
        f << yolo_linhas[j];
    }
    f.close();
    std::cout << "\nAnotação salva em moedas.txt\n";

    // Convert result back to mm::Image
    mm::Image img_bbox_mm(img_bbox.rows, img_bbox.cols, img_bbox.channels());
    std::memcpy(img_bbox_mm.data.data(), img_bbox.data, img_bbox_mm.data.size());

    mm::show(
        std::vector<mm::Image>{img_coins_gray, img_final, img_bbox_mm},
        MM_OUT,
        std::vector<std::string>{"Original", "Segmentação Final", "Bounding Boxes (formato YOLO)"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_coins_gray, "tmp/fig_04_bbox_0.png");
mm::write(img_final, "tmp/fig_04_bbox_1.png");
mm::write(img_bbox, "tmp/fig_04_bbox_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_bbox.cpp


In [69]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_bbox.cpp -o tmp/fig_04_bbox -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_bbox \
  && test -f "tmp/fig_04_bbox.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_bbox.png"

 cls     cx_n     cy_n      w_n      h_n  ← formato YOLO
------------------------------------------------------
   0   0.7753   0.1102   0.2880   0.2117
   0   0.4784   0.0984   0.2359   0.1828
   0   0.1331   0.1225   0.2255   0.1637
   0   0.4464   0.3217   0.2469   0.1816
   0   0.7523   0.3471   0.2807   0.2074
   0   0.1664   0.3812   0.2745   0.2016
   0   0.4862   0.6104   0.3474   0.2598
   0   0.8115   0.6154   0.2760   0.2012
   0   0.1724   0.6023   0.2250   0.1711
   0   0.1893   0.8258   0.2536   0.1922
   0   0.7763   0.8652   0.2255   0.1734
   0   0.4854   0.8953   0.2750   0.2039

Anotação salva em moedas.txt


[1] Original
[2] Segmentação Final
[3] Bounding Boxes (formato YOLO)


In [70]:
try:
    mm.show(
        [
            mm.read("tmp/fig_04_bbox_0.png"),
            mm.read("tmp/fig_04_bbox_1.png"),
            mm.read("tmp/fig_04_bbox_2.png"),
        ],
        titles=[
            'Original',
            'Segmentação Final',
            'Bounding Boxes (formato YOLO)',
        ],
        cols=3,
        figsize=(18, 6),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_04_bbox_0.png (ver a versao Python)")

<Figure size 2700x900 with 3 Axes>

**Figura 4.29:** *Bounding boxes* derivadas dos componentes conexos sobrepostas à imagem original. As anotações são exportadas no formato YOLO (*classe cx cy w h*), com coordenadas normalizadas para o intervalo [0,1].


El formato YOLO almacena cada objeto en una línea que contiene cinco campos:

$$
\texttt{clase}\;\;\texttt{cx}\;\;\texttt{cy}\;\;\texttt{w}\;\;\texttt{h}
$$

donde $(cx,cy)$ representa el centro de la *bounding box* y $(w,h)$ sus dimensiones. Todos los valores geométricos se normalizan al intervalo $[0,1]$ con respecto al ancho y al alto de la imagen. La **clase** es un identificador entero asociado a una categoría definida por el conjunto de datos (por ejemplo, `0 → moneda`). Cuando hay múltiples categorías —como moneda de oro (`0`), moneda de plata (`1`) y disco plástico (`2`)— basta con asignar el identificador correspondiente a cada objeto antes de la exportación, manteniendo exactamente el mismo formato de anotación. En el ejemplo anterior, esta información se almacenó en el archivo `monedas.txt`.

El flujo presentado en este capítulo —segmentación → etiquetado → extracción de *bounding boxes*— corresponde conceptualmente a la etapa de **anotación** (*labeling*) empleada en la construcción de conjuntos de entrenamiento para detectores modernos. Herramientas especializadas, como Label Studio y Roboflow, automatizan este proceso en imágenes complejas, pero la lógica fundamental sigue siendo la misma: asociar a cada objeto una región de interés y una clase. En escenarios controlados, con fondo uniforme y objetos bien separados, las técnicas morfológicas pueden incluso generar anotaciones automáticamente o servir como punto de partida para el etiquetado manual, reduciendo significativamente el esfuerzo de construcción del conjunto de datos. En aplicaciones reales más complejas, sin embargo, la validación humana sigue siendo necesaria para garantizar la calidad de las anotaciones.

> ### 📝 Evaluación: IoU (*Intersection over Union*)
>
> Una forma sencilla de evaluar la calidad de una segmentación consiste en compararla con una máscara de referencia (*ground truth*). La métrica más utilizada para este propósito es la **IoU** (*Intersection over Union*):
>
> <a id="eq-04-iou"></a>
$$
> \text{IoU}=
> \frac{|A\cap B|}
> {|A\cup B|}
> \tag{4.16}
$$

>
> donde $A$ representa la segmentación producida por el algoritmo y $B$ la segmentación de referencia.
>
> El valor de la IoU varía entre 0 y 1. Cuanto mayor sea el valor, mayor será la superposición entre las máscaras. Una IoU igual a 1 indica una correspondencia perfecta entre la segmentación obtenida y la referencia.
>
> La misma métrica también se utiliza ampliamente en la detección de objetos, aplicándose a las *bounding boxes* previstas y anotadas. En esta área, valores de IoU superiores a 0,5 se adoptan frecuentemente como criterio mínimo para considerar una detección correcta.

> ### 💡 Más Allá de la Morfología
>
> Las técnicas estudiadas en este capítulo segmentan objetos explorando la conectividad espacial, las operaciones morfológicas y el relieve topográfico. Existen, sin embargo, enfoques alternativos basados en la agrupación de características, como el algoritmo *k-means*, los modelos de mezcla Gaussiana (GMM) y métodos más recientes basados en aprendizaje profundo. Estas técnicas se retomarán en la Parte II del libro, dedicada a la Visión Computacional.

## 4.6 Resumen

En este capítulo se presentaron las principales técnicas de segmentación y morfología matemática, concluyendo el estudio del PDI en el dominio espacial.

* **Preprocesamiento y umbralización:** La combinación entre ecualización adaptativa CLAHE y el método de Otsu mostró ser eficaz para inducir separación bimodal en el histograma y simplificar la binarización de imágenes con iluminación no uniforme.
* **Erosión y dilatación:** Operadores morfológicos fundamentales basados en la búsqueda de mínimos y máximos locales en una vecindad definida por el elemento estructurante $B$. Son operadores duales por el complemento y fueron implementados mediante las funciones `mm::ero` y `mm::dil`.
* **Apertura y cierre:** Composiciones de erosión y dilatación que permiten eliminar ruidos, suavizar contornos y rellenar pequeñas lagunas, preservando la estructura global de los objetos.
* **Reconstrucción morfológica:** Proceso geodésico iterativo que propaga un marcador dentro de los límites impuestos por una máscara, constituyendo la base de operadores como `mm::clohole` y `mm::edgeoff`.
* **Pipeline de limpieza binaria:** Flujo consolidado compuesto por CLAHE → Otsu → apertura → `mm::clohole` → apertura restringida → `mm::edgeoff`, produciendo máscaras adecuadas para análisis cuantitativo.
* **Morfología en tonos de gris:** Extensión algebraica basada en mínimos y máximos ponderados, viabilizando operadores como gradiente morfológico y filtros *top-hat* para realce de estructuras locales.
* **Transformada de Distancia y Watershed:** La Transformada de Distancia permitió generar marcadores automáticos para el algoritmo *watershed*, posibilitando la separación de objetos adyacentes o parcialmente superpuestos.
* **Componentes conexos y descriptores:** La rotulación de regiones (`mm::label0`) y la extracción de contornos permitieron calcular descriptores geométricos como área, centroide, perímetro, circularidad y *bounding boxes*.
* **Conexión con Visión Computacional Moderna:** Las *bounding boxes* extraídas por morfología fueron exportadas en el formato YOLO, evidenciando el vínculo entre técnicas clásicas de segmentación y sistemas modernos de detección de objetos.

El Capítulo 5 introducirá técnicas de procesamiento en el **dominio de la frecuencia**, abordando la **Transformada de Fourier**, filtrado espectral y los fundamentos de la **compresión de imágenes**, incluyendo DCT, JPEG y *wavelets*.

## 4.7 🤖 Uso de Gemini Notebook como Tutor Complementario

En esta edición, se incentiva el uso de **Gemini Notebook** como herramienta complementaria de aprendizaje. Basado en inteligencia artificial, el sistema utiliza exclusivamente los documentos proporcionados por el autor como fuente de conocimiento, produciendo respuestas alineadas con el contenido y el enfoque adoptado a lo largo de este capítulo.

> ### ❗ 🎓 Estudia con el Tutor Inteligente
>
> [🚀 ACCEDER A Gemini Notebook: CAPÍTULO 04](https://notebooklm.google.com/notebook/5dafcbfa-ad58-44f9-9707-4f761b0a6c70)
>
> #### 🌐 Idioma y Lenguaje de Programación
>
> El proyecto de este capítulo en Gemini Notebook fue construido únicamente con el texto en **portugués** y los ejemplos de código en **Python**. Si estás estudiando con la edición en inglés o francés, o siguiendo la ruta en C++, las respuestas del tutor pueden no corresponder exactamente a la versión que estás leyendo.
>
> #### ⚠️ Aviso sobre Contenido Generado por IA
>
> Aunque es una herramienta valiosa de apoyo al estudio, Gemini Notebook puede eventualmente producir respuestas incompletas, imprecisas o incorrectas. Se recomienda validar la información consultando el material del capítulo, libros, artículos científicos y otras fuentes académicas confiables. Siempre que sea posible, ejecuta y experimenta con los ejemplos prácticos presentados a lo largo del texto para consolidar la comprensión de los conceptos.

## 4.8 Lista de Ejercicios

1. **(10%)** Implemente manualmente el criterio de Otsu sin utilizar `mm::threshold`. Calcule la varianza entre clases $\sigma_B^2(T)$ para todos los umbrales $T \in [0,255]$ usando `mm::hist`, identifique el umbral óptimo $T^*$ y compare el resultado con el valor obtenido por OpenCV. Grafique $\sigma_B^2$ en función de $T$ y resalte el punto de máximo.

2. **(15%)** Aplique umbralización adaptativa con bloques de tamaño 11, 31 y 51 a una imagen que contenga iluminación no uniforme. Compare los resultados con la umbralización global de Otsu y discuta las ventajas y limitaciones de cada enfoque.

3. **(15%)** Ejecute el *pipeline* de watershed de la imagen de monedas variando el umbral aplicado a la Transformada de Distancia ($0.3$, $0.5$ y $0.7$ veces el valor máximo). Explique cómo este parámetro influye en la generación de los marcadores, la separación de objetos adyacentes y la ocurrencia de sobre-segmentación.

4. **(15%)** Utilizando `mm::drawImg`, construya una demostración visual paso a paso de la erosión de una imagen binaria 7×7 con elemento estructurante cuadrado 3×3. Para cada posición analizada, indique si el elemento estructurante está completamente contenido en el objeto y justifique el valor asignado al píxel de salida.

5. **(15%)** Demuestre experimentalmente la dualidad entre erosión y dilatación verificando la identidad $(A \ominus B)^c = A^c \oplus \hat{B}$
   utilizando `mm::ero`, `mm::dil` y `mm::bnot`. Calcule la diferencia píxel a píxel entre los dos lados de la ecuación y presente el resultado utilizando `mm::histImg` o una visualización equivalente.

6. **(15%)** Implemente manualmente el gradiente morfológico utilizando únicamente `mm::ero` y `mm::dil`, comparando el resultado con `mm::gradm(img, B)`. Evalúe el efecto de diferentes elementos estructurantes (cuadrado 3×3, disco 5×5 y línea 1×9) sobre la detección de bordes.

7. **(15%)** Construya un *pipeline* completo para el conteo y clasificación de monedas por tamaño (pequeña, mediana y grande) utilizando área y circularidad como descriptores. Genere una máscara de referencia (*ground truth*) manualmente y calcule la métrica IoU (*Intersection over Union*) para evaluar la calidad de la segmentación. Presente los resultados en una tabla y mediante visualizaciones producidas con `mm::show`.

## Referencias del Capítulo

La fundamentación teórica de este capítulo se basa en las siguientes obras:

* Gonzalez (2018) para los conceptos de segmentación, umbralización de Otsu, Transformada de Distancia, *watershed*, morfología matemática y descriptores de forma.
* Matheron (1975) y Serra (1982) para la fundamentación teórica original, formulación algebraica y desarrollo de la Morfología Matemática.
* Szeliski (2022) para segmentación basada en regiones, etiquetado de componentes conexos, *watershed* basado en marcadores y evaluación de segmentación mediante la métrica IoU.
* Bradski (2008) para la utilización práctica de la biblioteca OpenCV, incluyendo funciones como `mm::dist`, `mm::watershed`, `mm::label0` y extracción de contornos.
* Redmon (2016) para la introducción a los detectores modernos de la familia YOLO y su relación con descriptores geométricos como *bounding boxes* extraídas por segmentación.
* Singh (2024) para la construcción de laberintos complejos basados en ciclos hamiltonianos sobre mosaicos cuasicristalinos, utilizados como ejemplo de aplicación de la Transformada de Distancia Geodésica y de algoritmos de búsqueda de caminos.
* Zampirolli (2025) para la implementación de los operadores morfológicos, transformadas geodésicas y resolución de laberintos por propagación de distancias en dominios restringidos.

------------------------------------------------------------------------


<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.es/cap04/cap04.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 4.9 💻 **Parte Práctica con Ejercicios de Programación**


Esta lista transforma los conceptos del Capítulo 4 en una ruta práctica de segmentación y morfología matemática. Los EPs comienzan con umbralización y avanzan hasta etiquetado y descriptores de componentes, siempre con matrices pequeñas para que cada píxel pueda verificarse a mano.

> ### ❗ Regla común de los EPs morfológicos
>
> En las operaciones con vecindad, **no haga padding**. Para cada píxel, evalúe únicamente las posiciones del elemento estructurante que caen dentro del dominio de la imagen. Esta es la misma idea de las implementaciones didácticas en `morph.py`, como `mm::dil0`, `mm::ero0`, `mm::dil1` y `mm::label0`: la vecindad se recorta por el dominio válido de la imagen.

### 🎯 Objetivo de este cuaderno

El cuaderno permite desarrollar, validar, organizar y probar soluciones de **Ejercicios de Programación (EPs)** en entornos interactivos, como Colab, con los mismos casos de prueba de Moodle, copiándolos allí solo al momento de registrar la nota oficial.

#### *Download*

Descargue `morph.py` y `testsuite.py` ejecutando la celda a continuación:

In [71]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefatos de build del trayecto C++ (.cpp, binario, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# El kernel es Python incluso en el trayecto C++: `mm` (morph.py) es usado por los
# simuladores, por la exhibición de las figuras que el binario C++ genera y por el
# estado mm::Image entre celdas. cpp=True descarga también el trayecto compilado
# (morph.hpp + stb_image*.h), usado en el #include de las celdas %%writefile *.cpp.
import config
config.setup(testsuite=True, cpp=True)
from morph import mm
from testsuite import TestSuite

✅ Entorno listo. Morph: 1.1.9 | OpenCV: 5.0.0 | TestSuite: 1.1.2


#### Ejecutando las pruebas
Para evaluar las pruebas, ejecute `TestSuite("EP04_01.extensión").run()` en una nueva celda, reemplazando la extensión por la del lenguaje utilizado (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). El sistema descarga los casos de prueba de GitHub, ejecuta el programa y calcula la nota automáticamente.

Para probar código Python directamente, sin guardar archivo, use `run_code(codigo)` pasando el código como *string* en una variable `codigo`:

```python
codigo = """
from morph import mm
# 4 ... su código aquí ...
"""
TestSuite("EP04_01").run_code(codigo)
```

### 4.0.1 EP04_01 🎚️ Umbralización Global por Umbral Fijo

En los **escáneres de documentos** y en los **sistemas de lectura de códigos de barras**, la primera etapa del procesamiento siempre consiste en separar lo que es "objeto" (tinta, texto, barras) de lo que es "fondo" (papel, embalaje). La **umbralización global** hace exactamente esto: compara cada píxel con un único umbral $T$ y decide, en tiempo real, si pertenece a la clase clara o a la clase oscura. Es el operador de segmentación más simple — y aun así, está detrás de buena parte de los *pipelines* industriales de inspección visual.
Ver en [Figura 4.30](#fig-04-sim-ep0401-limiar) una simulación de este EP.

#### 4.0.1.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Umbral:** Leer el entero $T$ (umbral de decisión).
3. **Datos:** Leer los valores enteros de la matriz original fila a fila.
4. **Mapeo:** Para cada píxel $p$, calcular el nuevo valor mediante la ecuación:

$$
p' =
\begin{cases}
255, & \text{si } p > T \\
0, & \text{si } p \le T
\end{cases}
$$
5. **Salida:** Mostrar la matriz binarizada con dimensiones $L \times C$.

#### 4.0.1.2 📌 Restricciones Computacionales

* **Binarización:** La salida contiene **solo** los valores $0$ o $255$.
* **Comparación estricta:** El criterio usa $> T$ (los píxeles iguales a $T$ se convierten en fondo).
* **Tipo:** El resultado final debe ser entero.
* **Observación:** Este EP sigue la convención de OpenCV (`cv2.THRESL_BINARY`): solo los píxeles con valor **mayor que** $T$ se convierten en blancos (`255`); los píxeles con valor **igual a** $T$ permanecen negros (`0`).

#### 4.0.1.3 🧠 Fundamentación Teórica

| Parámetro | Tipo | Impacto Visual |
|-----------|------|----------------|
| **$T$ pequeño** | Entero | La mayoría de los píxeles se vuelven blancos |
| **$T$ grande**  | Entero | La mayoría de los píxeles se vuelven negros |
| **$T$ bien elegido** | Entero | Separa nítidamente objeto y fondo |

#### 4.0.1.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $T$.
* Líneas siguientes: Elementos enteros de la matriz original.

**Salida:**

* Matriz binarizada en $L$ filas y $C$ columnas, valores $0$ o $255$ separados por espacio.

#### 4.0.1.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 2<br>4<br>100<br>0 99 100 180<br>255 30 120 80 | 0 0 0 255<br>255 0 255 0 | $T=100$: solo los píxeles con valor mayor que 100 se vuelven blancos; <br>por eso, 99 y 100 se vuelven negros. |
| 1<br>3<br>0<br>0 50 255 | 0 255 255 | $T=0$: solo los píxeles con valor estrictamente mayor que 0 se vuelven blancos. |

In [72]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0401-limiar" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎚️ Simulador EP04_01: Umbralización Global</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = (p > T) ? 255 : 0</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">👆 Haga clic en una celda de <b>Entrada Original</b> para oscurecer el píxel (−30) y haga clic con el botón derecho para aclarar (+30). Ajuste el umbral T para la binarización.</p>

    <!-- Controle do Limiar T -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#2980b9;">T (Umbral)</label>
        <span id="sim_ep0401_vl_t" style="font-family:monospace;font-size:12px;font-weight:700;color:#2980b9;">128</span>
      </div>
      <input type="range" id="sim_ep0401_sl_t" min="0" max="255" step="1" value="128" style="width:100%;cursor:pointer;">
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Resultado Binarizado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original (Clicable)</span>
        <div id="sim_ep0401_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0401_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Resultado Binarizado -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado Binarizado (p')</span>
        <div id="sim_ep0401_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0401_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Restablecer Umbral (T = 128)</button>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0401_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula aplicada: <b>(p > 128) ? 255 : 0</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0401(root){
    if (!root || root.dataset.simEp0401Init) return;
    root.dataset.simEp0401Init = "1";

    var slT      = root.querySelector('#sim_ep0401_sl_t');
    var vlT      = root.querySelector('#sim_ep0401_vl_t');
    var gridOrig = root.querySelector('#sim_ep0401_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0401_grid_new');
    var debugDiv = root.querySelector('#sim_ep0401_debug');

    var btnNew   = root.querySelector('#sim_ep0401_btnNew');
    var btnReset = root.querySelector('#sim_ep0401_btnReset');

    var pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });

    function renderOrig() {
      gridOrig.innerHTML = '';
      pixels.forEach(function(p, idx) {
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;cursor:pointer;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        cellO.title = 'Clique esquerdo: escurece (-30) | Botão direito: clareia (+30)';

        cellO.addEventListener('click', function(e) {
          e.preventDefault();
          pixels[idx] = Math.max(0, pixels[idx] - 30);
          render();
        });

        cellO.addEventListener('contextmenu', function(e) {
          e.preventDefault();
          pixels[idx] = Math.min(255, pixels[idx] + 30);
          render();
        });

        gridOrig.appendChild(cellO);
      });
    }

    function render() {
      var T = parseInt(slT.value, 10) || 0;
      vlT.textContent = T;
      debugDiv.innerHTML = 'Fórmula aplicada: <b>(p > ' + T + ') ? 255 : 0</b>';

      renderOrig();
      gridNew.innerHTML = '';

      pixels.forEach(function(p) {
        var res = (p > T) ? 255 : 0;
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    slT.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });
      render();
    });

    btnReset.addEventListener('click', function() {
      slT.value = '128';
      render();
    });

    render();
  }

  function tryInitSimEP0401(){
    var root = document.getElementById('sim-ep0401-limiar');
    if (root) initSimEP0401(root); else setTimeout(tryInitSimEP0401, 200);
  }
  tryInitSimEP0401();
})();
</script>
</div>
""")

**Figura 4.30:** Simulador EP04_01: Umbralización Global por Umbral Fijo (p


<figure id="fig-04-sim-ep0401-limiar">
  <img src="imagens/fig-04-sim-ep0401-limiar.png" alt=" Simulador EP04_01: Umbralización Global por Umbral Fijo (p' = (p > T) ? 255 : 0) " style="max-width:80%" />
  <figcaption><strong>Figura 4.30:</strong>  Simulador EP04_01: Umbralización Global por Umbral Fijo (p' = (p > T) ? 255 : 0) </figcaption>
</figure>

In [73]:
%%writefile EP04_01.cpp
// your solution

Overwriting EP04_01.cpp


In [74]:
TestSuite("EP04_01.cpp").run()

### 4.0.2 EP04_02 📊 Umbralización Automática de Otsu

Elegir manualmente el umbral $T$ funciona cuando la iluminación es estable, pero en **microscopía digital** y en **inspección de láminas de sangre**, cada muestra tiene un contraste diferente — un umbral fijo fallaría de imagen en imagen. El **método de Otsu** resuelve esto encontrando, por sí solo, el umbral que **maximiza la separación estadística** entre las dos clases de píxeles, haciendo la segmentación automática y adaptativa.
Ver en [Figura 4.31](#fig-04-sim-ep0402-otsu) una simulación de este EP.

#### 4.0.2.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer los valores enteros de la matriz original fila por fila.
3. **Histograma:** Construir el histograma $h[i]$, $i=0,\dots,255$, contando cuántos píxeles tienen valor $i$.
4. **Búsqueda del umbral:** Para cada candidato $T$ de $1$ a $255$, calcular la **varianza entre clases**:
$$
\sigma_B^2(T) = \frac{n_0 \cdot n_1}{N^2}\,(m_0 - m_1)^2
$$
donde $n_0,n_1$ son las cantidades de píxeles con valor $<T$ y $\geq T$, $m_0,m_1$ son sus medias, y $N=L\times C$.

5. **Elección:** El umbral óptimo $T^*$ es el que maximiza $\sigma_B^2(T)$ (en caso de empate, mantener el **primero** encontrado).
6. **Aplicación:** Binarizar la imagen usando $T^*$, aplicando:
$$
p' =
\begin{cases}
255, & \text{si } p > T^* \\
0, & \text{si } p \le T^*
\end{cases}
$$

#### 4.0.2.2 📌 Restricciones Computacionales

* **Candidatos válidos:** Ignorar $T$ que deje $n_0=0$ o $n_1=0$ (clase vacía).
* **Empate:** Mantener siempre el **primer** $T$ que alcanzó el valor máximo de $\sigma_B^2$.
* **Tipo:** $T^*$ y la matriz de salida deben ser enteros.
* **Convención OpenCV:** La binarización sigue `cv2.THRESL_BINARY`; los píxeles con valor exactamente igual a $T^*$ se vuelven negros.

#### 4.0.2.3 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto |
|----------|-------------|---------|
| **$\sigma_B^2(T)$ alta** | Clases bien separadas en $T$ | $T$ es un buen candidato a umbral |
| **Histograma bimodal** | Dos "picos" distintos | Otsu encuentra el valle entre ellos |
| **Histograma unimodal** | Un único "pico" | Otsu aún elige *algún* $T$, pero la segmentación es poco fiable |

#### 4.0.2.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos enteros de la matriz original.

**Salida:**

* Matriz binarizada en $L$ filas y $C$ columnas, valores $0$ o $255$.

#### 4.0.2.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 4<br>4<br>12 12 12 200<br>12 12 200 200<br>12 200 200 200<br>200 200 200 200 | 0 0 0 255<br>0 0 255 255<br>0 255 255 255<br>255 255 255 255 | Histograma bimodal claro: 12 y 200 |
| 1<br>2<br>10 250 | 0 250 | Solo dos valores: $T^*$ queda en el mayor |

In [75]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0402-otsu" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📊 Simulador EP04_02: Otsu Automático</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">T* = argmax σ²_B(T)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">👆 Clic izquierdo oscurece (−25) y clic con el botón derecho aclara (+25) los píxeles de la entrada. Observe el umbral óptimo T* ajustarse dinámicamente al histograma.</p>

    <!-- Painel do Histograma e T* -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;">
      <div id="sim_ep0402_hist" style="display:flex;align-items:flex-end;gap:2px;height:100px;margin-bottom:8px;border-bottom:1px solid #e4dcc8;padding-bottom:2px;"></div>
      <p id="sim_ep0402_info" style="text-align:center;font-size:11.5px;font-family:monospace;font-weight:700;color:#26241d;margin:0;">T* = −</p>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Clicável vs Resultado Otsu -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original (Clicable)</span>
        <div id="sim_ep0402_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado Otsu -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado Otsu (p')</span>
        <div id="sim_ep0402_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botão de Nova Imagem -->
    <div style="text-align:center;">
      <button id="sim_ep0402_btnNew" style="padding:6px 14px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen (Dos Grupos)</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0402(root){
    if (!root || root.dataset.simEp0402Init) return;
    root.dataset.simEp0402Init = "1";

    var gridOrig = root.querySelector('#sim_ep0402_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0402_grid_new');
    var info     = root.querySelector('#sim_ep0402_info');
    var hist     = root.querySelector('#sim_ep0402_hist');
    var btnNew   = root.querySelector('#sim_ep0402_btnNew');

    var pixels = [];

    function generate() {
      var c1 = 20 + Math.floor(Math.random() * 40);
      var c2 = 180 + Math.floor(Math.random() * 60);
      pixels = [];
      for (var i = 0; i < 16; i++) {
        var base = (Math.random() < 0.5) ? c1 : c2;
        pixels.push(Math.max(0, Math.min(255, base + Math.floor(Math.random() * 16 - 8))));
      }
    }

    function otsu(pix) {
      var histArr = new Array(256).fill(0);
      pix.forEach(function(p){ histArr[p]++; });
      var N = pix.length, bestVar = -1, bestT = 0;
      var total = pix.reduce(function(a, b){ return a + b; }, 0);

      for (var T = 1; T < 256; T++) {
        var n0 = 0, s0 = 0;
        for (var i = 0; i < T; i++) {
          n0 += histArr[i];
          s0 += i * histArr[i];
        }
        var n1 = N - n0, s1 = total - s0;
        if (n0 === 0 || n1 === 0) continue;
        var m0 = s0 / n0, m1 = s1 / n1;
        var v = (n0 * n1) * (m0 - m1) * (m0 - m1) / (N * N);
        if (v > bestVar) {
          bestVar = v;
          bestT = T;
        }
      }
      return bestT;
    }

    function renderOrig() {
      gridOrig.innerHTML = '';
      pixels.forEach(function(p, idx) {
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;cursor:pointer;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        cellO.title = 'Clique esquerdo: escurece (-25) | Botão direito: clareia (+25)';

        cellO.addEventListener('click', function(e) {
          e.preventDefault();
          pixels[idx] = Math.max(0, pixels[idx] - 25);
          render();
        });

        cellO.addEventListener('contextmenu', function(e) {
          e.preventDefault();
          pixels[idx] = Math.min(255, pixels[idx] + 25);
          render();
        });

        gridOrig.appendChild(cellO);
      });
    }

    function renderHist(T) {
      hist.innerHTML = '';
      var histArr = new Array(256).fill(0);
      pixels.forEach(function(p){ histArr[p]++; });
      var maxH = Math.max.apply(null, histArr);

      for (var i = 0; i < 256; i += 4) {
        var h = (histArr[i] / (maxH || 1)) * 100;
        var bar = document.createElement('div');
        var col = (i >= T) ? '#2980b9' : '#8a8371';
        bar.style.cssText = 'flex:1;height:' + h + '%;background:' + col + ';border-radius:2px 2px 0 0;';
        hist.appendChild(bar);
      }
    }

    function render() {
      var T = otsu(pixels);
      info.innerHTML = 'T* encontrado = <b>' + T + '</b>';
      renderOrig();
      renderHist(T);

      gridNew.innerHTML = '';
      pixels.forEach(function(p) {
        var res = (p > T) ? 255 : 0;
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0402(){
    var root = document.getElementById('sim-ep0402-otsu');
    if (root) initSimEP0402(root); else setTimeout(tryInitSimEP0402, 200);
  }
  tryInitSimEP0402();
})();
</script>
</div>
""")

**Figura 4.31:** Simulador EP04_02: Limiarización Automática de Otsu (T* = argmax σ²_B(T))


<figure id="fig-04-sim-ep0402-otsu">
  <img src="imagens/fig-04-sim-ep0402-otsu.png" alt=" Simulador EP04_02: Limiarización Automática de Otsu (T* = argmax σ²_B(T)) " style="max-width:80%" />
  <figcaption><strong>Figura 4.31:</strong>  Simulador EP04_02: Limiarización Automática de Otsu (T* = argmax σ²_B(T)) </figcaption>
</figure>

In [76]:
%%writefile EP04_02.cpp
// your solution

Overwriting EP04_02.cpp


In [77]:
TestSuite("EP04_02.cpp").run()

### 4.0.3 EP04_03 🌱 Dilatación Binaria Plana (mm.dil0)

En **microscopía de partículas** y en **OCR de placas de automóvil desgastadas**, los trazos finos o discontinuos deben "engrosarse" para que el reconocimiento funcione. La **dilatación morfológica** hace exactamente eso: expande regiones claras usando un elemento estructurante $B$ — la misma operación implementada en `morph.py` como `mm::dil0(f, B)`, usada cuando $B$ es **plano** (sin pesos, solo $0$/$1$).
Ver en [Figura 4.32](#fig-04-sim-ep0403-dilatacao) una simulación de este EP.

#### 4.0.3.1 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (filas) y $C$ (columnas) de $f$.
2. **Dimensiones de $B$:** Leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** Leer la matriz $B$ con valores $0$ o $1$, fila a fila.
4. **Datos:** Leer la matriz $f$ (la imagen original), fila a fila.
5. **Reflexión:** Construir $B_{ref}$, la versión de $B$ reflejada en $180°$ (filas y columnas invertidas) — exactamente como hace `mm::dil0` internamente.
6. **Vecindad sin padding:** Para cada píxel $(y,x)$, recorrer las posiciones $(by,bx)$ de $B_{ref}$ centradas en $(y,x)$, usando el desplazamiento
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Descartar** todo $(v_y,v_x)$ fuera de $[0,L)\times[0,C)$ — **no rellenar con ceros**.
7. **Mapeo:** Calcular cada píxel de salida como el **máximo** entre $f(y,x)$ y todos los $f(v_y,v_x)$ válidos cuya posición correspondiente en $B_{ref}$ vale $1$:
$$
g(y,x) = \max\Big(f(y,x),\ \max_{\substack{(v_y,v_x)\ \text{válido}\\ B_{ref}(by,bx)=1}} f(v_y,v_x)\Big)
$$
8. **Salida:** Mostrar la matriz $g$ con dimensiones $L \times C$.

#### 4.0.3.2 📌 Restricciones Computacionales

* **Sin padding:** Nunca inventar vecinos fuera de la imagen; usar solo los que existen realmente.
* **Reflexión obligatoria:** $B$ debe reflejarse antes de aplicarse (es lo que diferencia `mm::dil0` de una simple búsqueda de máximo).
* **Robustez de borde:** Si ninguna posición válida de $B_{ref}=1$ cae dentro del dominio para un píxel dado, este **mantiene su valor original**.

#### 4.0.3.3 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Dilatación** | $g \geq f$ siempre (extensiva) | Las regiones claras crecen, los huecos oscuros se encogen |
| **$B$ mayor** | Vecindad más amplia | Crecimiento más agresivo |
| **Reflexión de $B$** | $B_{ref}(y,x) = B(-y,-x)$ | Garantiza la definición formal de Minkowski de la dilatación |

#### 4.0.3.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros ($0$ o $1$) de la matriz $B$.
* Siguientes $L$ líneas: elementos enteros de la matriz $f$.

**Salida:**

* Matriz $g$ en $L$ filas y $C$ columnas, valores enteros separados por espacio.

#### 4.0.3.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>0 0 0<br>0 9 0<br>0 0 0 | 0 9 0<br>9 9 9<br>0 9 0 | $B$ en cruz simétrico: punto aislado se expande en cruz |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 200 200 200 80 | $B$ horizontal: cada píxel "atrae" el máximo de los vecinos de la fila |

In [78]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0403-dilatacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🌱 Simulador EP04_03: Dilatación Plana (mm.dil0)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ⊕ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Cambie el elemento estructurante B (o seleccione los preajustes) y haga clic en las celdas de la imagen original f para encender o apagar píxeles.</p>

    <!-- Painel do Elemento Estruturante B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Elemento Estructurante B (Clic para Alternar 0/1)</span>
      <div id="sim_ep0403_grid_B" style="display:grid;grid-template-columns:repeat(3, 38px);gap:4px;justify-content:center;margin-bottom:12px;user-select:none;"></div>
      
      <!-- Presets de B -->
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
        <button id="sim_ep0403_btnCross" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">➕ Cruz</button>
        <button id="sim_ep0403_btnBox" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬛ Cuadro</button>
        <button id="sim_ep0403_btnDiag" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⤫ Diagonal</button>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Original f vs Dilatada g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagen Original f (5×5)</span>
        <div id="sim_ep0403_grid_orig" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0403_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Imagem Dilatada g -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Dilatada g (f ⊕ B)</span>
        <div id="sim_ep0403_grid_new" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0403_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(y,x) = máximo sobre vecinos válidos de B reflejado
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0403(root){
    if (!root || root.dataset.simEp0403Init) return;
    root.dataset.simEp0403Init = "1";

    var gB      = root.querySelector('#sim_ep0403_grid_B');
    var gO      = root.querySelector('#sim_ep0403_grid_orig');
    var gN      = root.querySelector('#sim_ep0403_grid_new');
    var debugDiv= root.querySelector('#sim_ep0403_debug');

    var btnNew   = root.querySelector('#sim_ep0403_btnNew');
    var btnCross = root.querySelector('#sim_ep0403_btnCross');
    var btnBox   = root.querySelector('#sim_ep0403_btnBox');
    var btnDiag  = root.querySelector('#sim_ep0403_btnDiag');

    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
    var L = 5, C = 5, pixels = [];

    function generate() {
      pixels = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) {
          row.push((Math.random() < 0.78) ? 0 : 1);
        }
        pixels.push(row);
      }
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) {
        out.push(M[i].slice().reverse());
      }
      return out;
    }

    function dilate(f, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var Bref = reflect(Bm);
      var g = [];
      for (var y = 0; y < L; y++) g.push(f[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && f[vy][vx] > g[y][x]) {
                g[y][x] = f[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(byy, bxx){
            var c = document.createElement('div');
            c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;';
            c.style.background = B[byy][bxx] ? '#16a085' : '#fafaf7';
            c.style.color = B[byy][bxx] ? '#ffffff' : '#8a8371';
            c.textContent = B[byy][bxx];

            c.addEventListener('click', function(){
              B[byy][bxx] = 1 - B[byy][bxx];
              renderB();
              render();
            });
            gB.appendChild(c);
          })(by, bx);
        }
      }
    }

    function render() {
      var g = dilate(pixels, B);
      gO.innerHTML = '';
      gN.innerHTML = '';

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var p = pixels[yy][xx] ? 255 : 30;
            var fgO = p > 128 ? '#000000' : '#ffffff';
            var cO = document.createElement('div');
            cO.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgO + ';box-sizing:border-box;';
            cO.textContent = pixels[yy][xx];

            cO.addEventListener('click', function(){
              pixels[yy][xx] = 1 - pixels[yy][xx];
              render();
            });
            gO.appendChild(cO);
          })(y, x);

          var r = g[y][x] ? 255 : 30;
          var fgN = r > 128 ? '#000000' : '#ffffff';
          var cN = document.createElement('div');
          cN.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgN + ';box-sizing:border-box;';
          cN.textContent = g[y][x];
          gN.appendChild(cN);
        }
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnCross.addEventListener('click', function(){
      B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
      renderB();
      render();
    });

    btnBox.addEventListener('click', function(){
      B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
      renderB();
      render();
    });

    btnDiag.addEventListener('click', function(){
      B = [[1, 0, 1], [0, 1, 0], [1, 0, 1]];
      renderB();
      render();
    });

    generate();
    renderB();
    render();
  }

  function tryInitSimEP0403(){
    var root = document.getElementById('sim-ep0403-dilatacao');
    if (root) initSimEP0403(root); else setTimeout(tryInitSimEP0403, 200);
  }
  tryInitSimEP0403();
})();
</script>
</div>
""")

**Figura 4.32:** Simulador EP04_03: Dilatación Binaria Plana (g = f ⊕ B)


<figure id="fig-04-sim-ep0403-dilatacao">
  <img src="imagens/fig-04-sim-ep0403-dilatacao.png" alt=" Simulador EP04_03: Dilatación Binaria Plana (g = f ⊕ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.32:</strong>  Simulador EP04_03: Dilatación Binaria Plana (g = f ⊕ B) </figcaption>
</figure>

In [79]:
%%writefile EP04_03.cpp
// your solution

Overwriting EP04_03.cpp


In [80]:
TestSuite("EP04_03.cpp").run()

### 4.0.4 EP04_04 🪨 Erosión Binaria Plana (mm.ero0)

Si la dilatación engrosa, la **erosión** afina. En **sistemas de conteo de células**, se utiliza para **separar células que se tocan**: al "comer" los bordes de cada región, las conexiones finas entre objetos desaparecen incluso antes de que se realice cualquier conteo. En `morph.py`, esta es la operación `mm::ero0(f, B)` — la **dual** exacta de la dilatación, y la única de las dos que **no** refleja el elemento estructurante.
Ver en [Figura 4.33](#fig-04-sim-ep0404-erosao) una simulación de este EP.

#### 4.0.4.1 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (filas) y $C$ (columnas) de $f$.
2. **Dimensiones de $B$:** Leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** Leer la matriz $B$ con valores $0$ o $1$, fila a fila.
4. **Datos:** Leer la matriz $f$ (la imagen original), fila a fila.
5. **Vecindad sin padding (¡sin reflexión!):** Para cada píxel $(y,x)$, recorrer las posiciones $(by,bx)$ de $B$ **en el orden original** (sin reflejar), usando el mismo desplazamiento del EP04_03:
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$

**Descartar** todo $(v_y,v_x)$ fuera de $[0,L)\times[0,C)$.
6. **Mapeo:** Calcular cada píxel de salida como el **mínimo** entre $f(y,x)$ y todos los $f(v_y,v_x)$ válidos cuya posición correspondiente en $B$ vale $1$:
$$
g(y,x) = \min\Big(f(y,x),\ \min_{\substack{(v_y,v_x)\ \text{válido}\\ B(by,bx)=1}} f(v_y,v_x)\Big)
$$
7. **Salida:** Mostrar la matriz $g$ con dimensiones $L \times C$.

#### 4.0.4.2 📌 Restricciones Computacionales

* **Sin reflexión:** A diferencia de la dilatación, $B$ se usa **exactamente como se lee** — reflejarlo aquí sería un error conceptual grave.
* **Sin padding:** Los vecinos fuera de la imagen simplemente se ignoran, nunca se tratan como $0$.
* **Robustez de borde:** Si ninguna posición válida de $B=1$ cae dentro del dominio, el píxel mantiene su valor original.

#### 4.0.4.3 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Erosión** | $g \leq f$ siempre (anti-extensiva) | Las regiones claras se encogen, el ruido puntual desaparece |
| **Dualidad** | $\text{ero}(f,B) = -\text{dil}(-f, B_{ref})$ | La erosión y la dilatación son "espejos" matemáticos |
| **$B$ más grande** | Erosión más agresiva | Los objetos finos desaparecen por completo |

#### 4.0.4.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros ($0$ o $1$) de la matriz $B$.
* Siguientes $L$ líneas: elementos enteros de la matriz $f$.

**Salida:**

* Matriz $g$ en $L$ filas y $C$ columnas, valores enteros separados por espacios.

#### 4.0.4.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>9 9 9<br>9 0 9<br>9 9 9 | 9 0 9<br>0 0 0<br>9 0 9 | El "agujero" central (0) se propaga en cruz |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 10 5 5 80 | $B$ horizontal: cada píxel "extrae" el mínimo de los vecinos de la fila |

In [81]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0404-erosao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪨 Simulador EP04_04: Erosión Plana (mm.ero0)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ⊖ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Alterne el elemento estructurante B (o seleccione los presets) y haga clic en las celdas de la imagen original f para encender o apagar píxeles.</p>

    <!-- Painel do Elemento Estruturante B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Elemento Estructurante B (Clic para Alternar 0/1)</span>
      <div id="sim_ep0404_grid_B" style="display:grid;grid-template-columns:repeat(3, 38px);gap:4px;justify-content:center;margin-bottom:12px;user-select:none;"></div>
      
      <!-- Presets de B -->
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
        <button id="sim_ep0404_btnCross" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">➕ Cruz</button>
        <button id="sim_ep0404_btnBox" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬛ Caja</button>
        <button id="sim_ep0404_btnDiag" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⤫ Diagonal</button>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Original f vs Erodida g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagen Original f (5×5)</span>
        <div id="sim_ep0404_grid_orig" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0404_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Imagem Erodida g -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Erosionada g (f ⊖ B)</span>
        <div id="sim_ep0404_grid_new" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0404_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(y,x) = mínimo sobre vecinos válidos de B (sin reflejar)
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0404(root){
    if (!root || root.dataset.simEp0404Init) return;
    root.dataset.simEp0404Init = "1";

    var gB      = root.querySelector('#sim_ep0404_grid_B');
    var gO      = root.querySelector('#sim_ep0404_grid_orig');
    var gN      = root.querySelector('#sim_ep0404_grid_new');
    var debugDiv= root.querySelector('#sim_ep0404_debug');

    var btnNew   = root.querySelector('#sim_ep0404_btnNew');
    var btnCross = root.querySelector('#sim_ep0404_btnCross');
    var btnBox   = root.querySelector('#sim_ep0404_btnBox');
    var btnDiag  = root.querySelector('#sim_ep0404_btnDiag');

    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
    var L = 5, C = 5, pixels = [];

    function generate() {
      pixels = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) {
          row.push((Math.random() < 0.78) ? 1 : 0);
        }
        pixels.push(row);
      }
    }

    function erode(f, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(f[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && f[vy][vx] < g[y][x]) {
                g[y][x] = f[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(byy, bxx){
            var c = document.createElement('div');
            c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;';
            c.style.background = B[byy][bxx] ? '#c0392b' : '#fafaf7';
            c.style.color = B[byy][bxx] ? '#ffffff' : '#8a8371';
            c.textContent = B[byy][bxx];

            c.addEventListener('click', function(){
              B[byy][bxx] = 1 - B[byy][bxx];
              renderB();
              render();
            });
            gB.appendChild(c);
          })(by, bx);
        }
      }
    }

    function render() {
      var g = erode(pixels, B);
      gO.innerHTML = '';
      gN.innerHTML = '';

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var p = pixels[yy][xx] ? 255 : 30;
            var fgO = p > 128 ? '#000000' : '#ffffff';
            var cO = document.createElement('div');
            cO.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgO + ';box-sizing:border-box;';
            cO.textContent = pixels[yy][xx];

            cO.addEventListener('click', function(){
              pixels[yy][xx] = 1 - pixels[yy][xx];
              render();
            });
            gO.appendChild(cO);
          })(y, x);

          var r = g[y][x] ? 255 : 30;
          var fgN = r > 128 ? '#000000' : '#ffffff';
          var cN = document.createElement('div');
          cN.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgN + ';box-sizing:border-box;';
          cN.textContent = g[y][x];
          gN.appendChild(cN);
        }
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnCross.addEventListener('click', function(){
      B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
      renderB();
      render();
    });

    btnBox.addEventListener('click', function(){
      B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
      renderB();
      render();
    });

    btnDiag.addEventListener('click', function(){
      B = [[1, 0, 1], [0, 1, 0], [1, 0, 1]];
      renderB();
      render();
    });

    generate();
    renderB();
    render();
  }

  function tryInitSimEP0404(){
    var root = document.getElementById('sim-ep0404-erosao');
    if (root) initSimEP0404(root); else setTimeout(tryInitSimEP0404, 200);
  }
  tryInitSimEP0404();
})();
</script>
</div>
""")

**Figura 4.33:** Simulador EP04_04: Erosión Binaria Plana (g = f ⊖ B)


<figure id="fig-04-sim-ep0404-erosao">
  <img src="imagens/fig-04-sim-ep0404-erosao.png" alt=" Simulador EP04_04: Erosión Binaria Plana (g = f ⊖ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.33:</strong>  Simulador EP04_04: Erosión Binaria Plana (g = f ⊖ B) </figcaption>
</figure>

In [82]:
%%writefile EP04_04.cpp
// your solution

Overwriting EP04_04.cpp


In [83]:
TestSuite("EP04_04.cpp").run()

### 4.0.5 EP04_05 🧹 Apertura Morfológica (Eliminación de Ruido)

Las imágenes capturadas por **sensores de bajo costo**, como los de drones agrícolas, suelen venir salpicadas de pequeños puntos de ruido — píxeles aislados que no representan nada real. Aplicar erosión seguida de dilatación con el **mismo** elemento estructurante produce la **apertura**: esta "limpia" puntos y protuberancias finas, pero devuelve al objeto principal prácticamente su tamaño original. Es la combinación clásica utilizada en **preprocesamiento de imágenes de satélite** antes de cualquier conteo de área plantada.
Ver en [Figura 4.34](#fig-04-sim-ep0405-abertura) una simulación de este EP.

#### 4.0.5.1 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (filas) y $C$ (columnas) de $f$.
2. **Dimensiones de $B$:** Leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** Leer la matriz $B$ con valores $0$ o $1$, fila a fila.
4. **Datos:** Leer la matriz binaria $f$ (valores $0$ o $1$), fila a fila.
5. **Erosión:** Calcular $e = f \ominus B$, usando exactamente el algoritmo del EP04_04 (sin reflejar $B$, sin padding).
6. **Dilatación:** Calcular $g = e \oplus B$, usando exactamente el algoritmo del EP04_03 (reflejando $B$, sin padding) — pero ahora aplicado sobre $e$, no sobre $f$.
7. **Salida:** Mostrar la matriz resultante $g$ (la **apertura** de $f$ por $B$) con dimensiones $L \times C$.

#### 4.0.5.2 📌 Restricciones Computacionales

* **Orden fijo:** Es **siempre** erosión primero, luego dilatación — el orden inverso define otro operador (cierre, del próximo EP).
* **Mismo $B$:** El elemento estructurante utilizado en la erosión y en la dilatación debe ser idéntico.
* **Sin padding en ninguna de las dos etapas.**

#### 4.0.5.3 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Antiextensividad** | $g \subseteq f$ siempre | La apertura nunca crea un píxel nuevo, solo elimina |
| **Idempotencia** | $\text{apertura}(\text{apertura}(f)) = \text{apertura}(f)$ | Aplicar de nuevo no cambia nada más |
| **Puntos aislados** | Menores que $B$ | Son completamente eliminados |
| **Núcleo del objeto** | Mayor que $B$ | Se recupera casi intacto mediante la dilatación final |

#### 4.0.5.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros ($0$ o $1$) de la matriz $B$.
* Siguientes $L$ líneas: elementos enteros ($0$ o $1$) de la matriz $f$.

**Salida:**

* Matriz resultante en $L$ filas y $C$ columnas, valores $0$ o $1$.

#### 4.0.5.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 7<br>7<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0<br>0 1 0 0 0 1 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 1 0<br>0 0 0 0 0 0 0<br>0 1 0 0 0 0 1 | 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 | Los puntos aislados y la protuberancia fina desaparecen; el cuadrado central sobrevive |

In [84]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0405-abertura" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧹 Simulador EP04_05: Apertura Morfológica</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = (f ⊖ B) ⊕ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Haz clic en las celdas de <b>f original</b> para encender o apagar píxeles (¡crea tu propio ruido de fondo!) y ajusta el tamaño del elemento estructurante B.</p>

    <!-- Controle do Tamanho de B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#8e44ad;">Tamaño de B (Caja n×n)</label><br>
      <input type="range" id="sim_ep0405_sl_n" min="3" max="5" step="2" value="3" style="width:60%;cursor:pointer;accent-color:#8e44ad;margin-top:6px;">
      <span id="sim_ep0405_vl_n" style="font-family:monospace;font-size:12px;font-weight:700;color:#8e44ad;margin-left:8px;">3×3</span>
    </div>

    <!-- Pipeline em 3 Colunas: f original vs e (Erosão) vs g (Abertura Final) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(180px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original (Clicable)</span>
        <div id="sim_ep0405_grid_f" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- e = f ⊖ B -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">e = f ⊖ B (Erosión)</span>
        <div id="sim_ep0405_grid_e" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- g = e ⊕ B -->
      <div style="background:#fafaf7;border:2px solid #8e44ad;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">g = e ⊕ B (Apertura)</span>
        <div id="sim_ep0405_grid_g" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botões de Ação -->
    <div style="text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0405_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen (Con Ruido)</button>
      <button id="sim_ep0405_btnClear" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">🧹 Limpiar Todo</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0405(root){
    if (!root || root.dataset.simEp0405Init) return;
    root.dataset.simEp0405Init = "1";

    var slN     = root.querySelector('#sim_ep0405_sl_n');
    var vlN     = root.querySelector('#sim_ep0405_vl_n');
    var gF      = root.querySelector('#sim_ep0405_grid_f');
    var gE      = root.querySelector('#sim_ep0405_grid_e');
    var gG      = root.querySelector('#sim_ep0405_grid_g');
    var btnNew  = root.querySelector('#sim_ep0405_btnNew');
    var btnClear= root.querySelector('#sim_ep0405_btnClear');

    var L = 7, C = 7, f = [];

    function generate() {
      f = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) row.push(0);
        f.push(row);
      }
      for (var y = 2; y < 5; y++) {
        for (var x = 2; x < 5; x++) f[y][x] = 1;
      }
      for (var k = 0; k < 3; k++) {
        var ry = Math.floor(Math.random() * L), rx = Math.floor(Math.random() * C);
        if (f[ry][rx] === 0 && (ry < 1 || ry > 5 || rx < 1 || rx > 5)) f[ry][rx] = 1;
      }
    }

    function clearAll() {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) f[y][x] = 0;
      }
    }

    function box(n) {
      var B = [];
      for (var i = 0; i < n; i++) {
        var row = [];
        for (var j = 0; j < n; j++) row.push(1);
        B.push(row);
      }
      return B;
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) out.push(M[i].slice().reverse());
      return out;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function dilate(img, Bm) {
      var Bref = reflect(Bm);
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] > g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paintStatic(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:28px;height:28px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] === 1 ? '#8e44ad' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintEditable(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:28px;height:28px;border-radius:4px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = img[yy][xx] === 1 ? '#8e44ad' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            grid.appendChild(c);
          })(y, x);
        }
      }
    }

    function render() {
      var n = parseInt(slN.value, 10) || 3;
      vlN.textContent = n + '×' + n;
      var B = box(n);
      var e = erode(f, B);
      var g = dilate(e, B);

      paintEditable(gF, f);
      paintStatic(gE, e);
      paintStatic(gG, g);
    }

    slN.addEventListener('input', render);

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnClear.addEventListener('click', function(){
      clearAll();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0405(){
    var root = document.getElementById('sim-ep0405-abertura');
    if (root) initSimEP0405(root); else setTimeout(tryInitSimEP0405, 200);
  }
  tryInitSimEP0405();
})();
</script>
</div>
""")

**Figura 4.34:** Simulador EP04_05: Apertura Morfológica (g = (f ⊖ B) ⊕ B)


<figure id="fig-04-sim-ep0405-abertura">
  <img src="imagens/fig-04-sim-ep0405-abertura.png" alt=" Simulador EP04_05: Apertura Morfológica (g = (f ⊖ B) ⊕ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.34:</strong>  Simulador EP04_05: Apertura Morfológica (g = (f ⊖ B) ⊕ B) </figcaption>
</figure>

In [85]:
%%writefile EP04_05.cpp
// your solution

Overwriting EP04_05.cpp


In [86]:
TestSuite("EP04_05.cpp").run()

### 4.0.6 EP04_06 🧩 Cierre Morfológico (Relleno de Huecos)

En la **digitalización de huellas dactilares**, los surcos de la piel a veces se ven interrumpidos por suciedad o sequedad, creando pequeñas fallas en la curva continua que debería existir. El **cierre** —dilatación seguida de erosión con el mismo elemento estructurante— es el operador dual de la apertura: **rellena huecos pequeños y entrantes estrechos**, sin alterar significativamente el contorno externo del objeto. Es el paso estándar antes de extraer el esqueleto de una huella dactilar.
Ver en [Figura 4.35](#fig-04-sim-ep0406-fechamento) una simulación de este EP.

#### 4.0.6.1 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (líneas) y $C$ (columnas) de $f$.
2. **Dimensiones de $B$:** Leer los enteros $L_B$ (líneas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** Leer la matriz $B$ con valores $0$ o $1$, línea por línea.
4. **Datos:** Leer la matriz binaria $f$ (valores $0$ o $1$), línea por línea.
5. **Dilatación:** Calcular $d = f \oplus B$, usando exactamente el algoritmo del EP04_03 (reflejando $B$, sin padding).
6. **Erosión:** Calcular $g = d \ominus B$, usando exactamente el algoritmo del EP04_04 (sin reflejar $B$, sin padding) — ahora aplicado sobre $d$, no sobre $f$.
7. **Salida:** Mostrar la matriz resultante $g$ (el **cierre** de $f$ por $B$) con dimensiones $L \times C$.

#### 4.0.6.2 📌 Restricciones Computacionales

* **Orden fijo:** Es **siempre** dilatación primero, luego erosión — el orden inverso es la apertura del EP04_05.
* **Mismo $B$:** El elemento estructurante usado en la dilatación y en la erosión debe ser idéntico.
* **Sin padding en ninguna de las dos etapas.**

#### 4.0.6.3 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Extensividad** | $g \supseteq f$ siempre | El cierre nunca elimina píxeles, solo añade |
| **Idempotencia** | $\text{cierre}(\text{cierre}(f)) = \text{cierre}(f)$ | Aplicarlo de nuevo no cambia nada más |
| **Huecos pequeños** | Menores que $B$ | Se rellenan completamente |
| **Dualidad** | $\text{cierre}(f) = \overline{\text{apertura}(\bar f)}$ | Es la apertura aplicada al "negativo" de la imagen |

#### 4.0.6.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros ($0$ o $1$) de la matriz $B$.
* Siguientes $L$ líneas: elementos enteros ($0$ o $1$) de la matriz $f$.

**Salida:**

* Matriz resultante en $L$ líneas y $C$ columnas, valores $0$ o $1$.

#### 4.0.6.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 8<br>8<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 0 1 1 0 0<br>0 0 1 1 0 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 | 0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0 | Los dos huecos internos no adyacentes se rellenan totalmente |

In [87]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0406-fechamento" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧩 Simulador EP04_06: Cierre Morfológico</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = (f ⊕ B) ⊖ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Haz clic en las celdas de <b>f original</b> para encender o apagar píxeles (¡rellena huecos internos!) y ajusta el tamaño del elemento estructurante B.</p>

    <!-- Controle do Tamanho de B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#2c7a7b;">Tamaño de B (Cuadro n×n)</label><br>
      <input type="range" id="sim_ep0406_sl_n" min="3" max="5" step="2" value="3" style="width:60%;cursor:pointer;accent-color:#2c7a7b;margin-top:6px;">
      <span id="sim_ep0406_vl_n" style="font-family:monospace;font-size:12px;font-weight:700;color:#2c7a7b;margin-left:8px;">3×3</span>
    </div>

    <!-- Pipeline em 3 Colunas: f original vs d (Dilatação) vs g (Fechamento Final) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(170px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original (Clickeable)</span>
        <div id="sim_ep0406_grid_f" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- d = f ⊕ B -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">d = f ⊕ B (Dilatación)</span>
        <div id="sim_ep0406_grid_d" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- g = d ⊖ B -->
      <div style="background:#fafaf7;border:2px solid #2c7a7b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2c7a7b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">g = d ⊖ B (Cierre)</span>
        <div id="sim_ep0406_grid_g" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botões de Ação -->
    <div style="text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0406_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen (Con Huecos)</button>
      <button id="sim_ep0406_btnClear" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">🧹 Limpiar Todo</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0406(root){
    if (!root || root.dataset.simEp0406Init) return;
    root.dataset.simEp0406Init = "1";

    var slN     = root.querySelector('#sim_ep0406_sl_n');
    var vlN     = root.querySelector('#sim_ep0406_vl_n');
    var gF      = root.querySelector('#sim_ep0406_grid_f');
    var gD      = root.querySelector('#sim_ep0406_grid_d');
    var gG      = root.querySelector('#sim_ep0406_grid_g');
    var btnNew  = root.querySelector('#sim_ep0406_btnNew');
    var btnClear= root.querySelector('#sim_ep0406_btnClear');

    var L = 8, C = 8, f = [];

    function generate() {
      f = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) row.push(0);
        f.push(row);
      }
      for (var y = 1; y < 7; y++) {
        for (var x = 2; x < 6; x++) f[y][x] = 1;
      }
      f[3][3] = 0;
      f[4][4] = 0;
    }

    function clearAll() {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) f[y][x] = 0;
      }
    }

    function box(n) {
      var B = [];
      for (var i = 0; i < n; i++) {
        var row = [];
        for (var j = 0; j < n; j++) row.push(1);
        B.push(row);
      }
      return B;
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) out.push(M[i].slice().reverse());
      return out;
    }

    function dilate(img, Bm) {
      var Bref = reflect(Bm);
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] > g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paintStatic(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] === 1 ? '#2c7a7b' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintEditable(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = img[yy][xx] === 1 ? '#2c7a7b' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            grid.appendChild(c);
          })(y, x);
        }
      }
    }

    function render() {
      var n = parseInt(slN.value, 10) || 3;
      vlN.textContent = n + '×' + n;
      var B = box(n);
      var d = dilate(f, B);
      var g = erode(d, B);

      paintEditable(gF, f);
      paintStatic(gD, d);
      paintStatic(gG, g);
    }

    slN.addEventListener('input', render);

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnClear.addEventListener('click', function(){
      clearAll();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0406(){
    var root = document.getElementById('sim-ep0406-fechamento');
    if (root) initSimEP0406(root); else setTimeout(tryInitSimEP0406, 200);
  }
  tryInitSimEP0406();
})();
</script>
</div>
""")

**Figura 4.35:** Simulador EP04_06: Cierre Morfológico (g = (f ⊕ B) ⊖ B)


<figure id="fig-04-sim-ep0406-fechamento">
  <img src="imagens/fig-04-sim-ep0406-fechamento.png" alt=" Simulador EP04_06: Cierre Morfológico (g = (f ⊕ B) ⊖ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.35:</strong>  Simulador EP04_06: Cierre Morfológico (g = (f ⊕ B) ⊖ B) </figcaption>
</figure>

In [88]:
%%writefile EP04_06.cpp
// your solution

Overwriting EP04_06.cpp


In [89]:
TestSuite("EP04_06.cpp").run()

### 4.0.7 EP04_07 ⛰️ Dilatación y Erosión con Pesos (mm.dil1 / mm.ero1)

Hasta ahora, el elemento estructurante solo decía "este vecino cuenta" o "no cuenta" — pero en **modelos digitales de elevación** (usados en SIG y en planificación de drenaje urbano), cada vecino debería tener un **peso diferente** dependiendo de la distancia o de la dirección del relieve. Las versiones **ponderadas** de la dilatación y de la erosión, implementadas en `morph.py` como `mm::dil1(f, b)` y `mm::ero1(f, b)`, suman (o restan) el peso de cada vecino antes de tomar el máximo (o mínimo) — generalizando todo lo realizado en los EPs anteriores.
Ver en [Figura 4.36](#fig-04-sim-ep0407-pesos) una simulación de este EP.

#### 4.0.7.1 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (filas) y $C$ (columnas) de $f$.
2. **Dimensiones de $b$:** Leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante ponderado.
3. **Pesos:** Leer la matriz $b$ de pesos **enteros** (pueden ser negativos, cero o positivos), fila a fila.
4. **Datos:** Leer la matriz $f$ (la imagen original), fila a fila.
5. **Vecindario sin padding:** Para cada píxel $(y,x)$, recorrer **todas** las posiciones $(by,bx)$ de $b$ (no solo donde valdría $1$ — aquí **todo** peso participa), usando el mismo desplazamiento de los EPs anteriores:
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Descartar** todo $(v_y,v_x)$ fuera de $[0,L)\times[0,C)$.
6. **Dilatación ponderada:** Calcular
$$
g_{dil}(y,x) = \max\Big(f(y,x),\ \max_{(v_y,v_x)\ \text{válido}} \big(f(v_y,v_x) + b(by,bx)\big)\Big)
$$
7. **Erosión ponderada:** Calcular, **usando el mismo $b$ y sin reflejar**:
$$
g_{ero}(y,x) = \min\Big(f(y,x),\ \min_{(v_y,v_x)\ \text{válido}} \big(f(v_y,v_x) - b(by,bx)\big)\Big)
$$
8. **Salida:** Mostrar **primero** la matriz $g_{dil}$ completa, y **después** la matriz $g_{ero}$ completa.

#### 4.0.7.2 📌 Restricciones Computacionales

* **Ninguna de las dos refleja $b$** — la versión ponderada no usa reflexión, incluso en la dilatación (diferente de `mm::dil0`).
* **Todos los pesos participan:** No existe aquí el filtro "$B=1$"; incluso el peso $0$ entra en la cuenta.
* **Sin padding:** los vecinos fuera de la imagen se ignoran, nunca se rellenan virtualmente.
* **Tipo:** La salida puede contener valores negativos o mayores que $255$ — **no** hay *clipping* en este EP.
* **Consejo:** Para eliminar mensajes de desbordamiento al superar los límites del tipo uint8, incluir al inicio del código:
```python
import warnings
warnings.filterwarnings("ignore")
```

#### 4.0.7.3 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Peso positivo** | "Empuja" el valor del vecino hacia arriba en la dilatación | Simula relieve que asciende en esa dirección |
| **Peso negativo** | Reduce la contribución del vecino | Simula distancia o atenuación direccional |
| **Dualidad ponderada** | $\text{ero1}(f,b) = -\text{dil1}(-f,b)$ | La simetría entre las dos operaciones se mantiene incluso con pesos |

#### 4.0.7.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros (pueden ser negativos) de la matriz $b$.
* Siguientes $L$ líneas: elementos enteros de la matriz $f$.

**Salida:**

* Primero la matriz $g_{dil}$ en $L$ líneas y $C$ columnas.
* A continuación la matriz $g_{ero}$ en $L$ líneas y $C$ columnas.

#### 4.0.7.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 2 1<br>0 1 0<br>10 20 30<br>40 50 60<br>70 80 90 | 50 60 61<br>80 90 91<br>81 91 92<br>8 9 19<br>9 10 20<br>39 40 50 | Peso central $2$ acelera el crecimiento en la dilatación y la contracción en la erosión |

In [90]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0407-pesos" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">⛰️ Simulador EP04_07: Pesos en el Elemento Estructurante</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">dil1 / ero1</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajuste los pesos del elemento estructurante b con los controles deslizantes y observe el efecto de la dilatación y erosión con pesos sobre la matriz f.</p>

    <!-- Painel dos Pesos b -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#d35400;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Pesos b (Ajuste los Sliders por Celda)</span>
      <div id="sim_ep0407_grid_b" style="display:grid;grid-template-columns:repeat(3, 70px);gap:8px;justify-content:center;user-select:none;"></div>
    </div>

    <!-- Comparativo em 3 Colunas: f original vs dil1 vs ero1 -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(180px, 1fr));gap:14px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original</span>
        <div id="sim_ep0407_grid_f" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- dil1(f,b) -->
      <div style="background:#fafaf7;border:1px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">dil1(f, b) (Dilatación)</span>
        <div id="sim_ep0407_grid_d" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ero1(f,b) -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">ero1(f, b) (Erosión)</span>
        <div id="sim_ep0407_grid_e" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0407(root){
    if (!root || root.dataset.simEp0407Init) return;
    root.dataset.simEp0407Init = "1";

    var gB = root.querySelector('#sim_ep0407_grid_b');
    var gF = root.querySelector('#sim_ep0407_grid_f');
    var gD = root.querySelector('#sim_ep0407_grid_d');
    var gE = root.querySelector('#sim_ep0407_grid_e');

    var b = [[0, 1, 0], [1, 2, 1], [0, 1, 0]];
    var f = [[10, 20, 30], [40, 50, 60], [70, 80, 90]];
    var L = 3, C = 3;

    function compute() {
      var oy = -3 / 2 + 0.5, ox = -3 / 2 + 0.5;
      var dil = f.map(function(r){ return r.slice(); });
      var ero = f.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < 3; by++) {
            for (var bx = 0; bx < 3; bx++) {
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C) {
                var cd = f[vy][vx] + b[by][bx];
                if (cd > dil[y][x]) dil[y][x] = cd;
                var ce = f[vy][vx] - b[by][bx];
                if (ce < ero[y][x]) ero[y][x] = ce;
              }
            }
          }
        }
      }
      return { dil: dil, ero: ero };
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(row, col){
            var wrap = document.createElement('div');
            wrap.style.cssText = 'display:flex;flex-direction:column;align-items:center;background:#fafaf7;border:1px solid #e4dcc8;border-radius:6px;padding:4px;box-sizing:border-box;';

            var val = document.createElement('div');
            val.style.cssText = 'font-family:monospace;font-weight:700;font-size:11px;color:#d35400;margin-bottom:2px;';
            val.textContent = b[row][col];

            var sl = document.createElement('input');
            sl.type = 'range';
            sl.min = '-5';
            sl.max = '5';
            sl.step = '1';
            sl.value = b[row][col];
            sl.style.cssText = 'width:56px;cursor:pointer;accent-color:#d35400;';

            sl.addEventListener('input', function(){
              b[row][col] = parseInt(sl.value, 10);
              val.textContent = b[row][col];
              renderAll();
            });

            wrap.appendChild(val);
            wrap.appendChild(sl);
            gB.appendChild(wrap);
          })(by, bx);
        }
      }
    }

    function paint(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var v = img[y][x];
          var inten = Math.min(255, Math.max(0, v));
          var fg = inten > 128 ? '#000000' : '#ffffff';
          c.style.cssText = 'width:52px;height:42px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + inten + ',' + inten + ',' + inten + ');color:' + fg + ';box-sizing:border-box;';
          c.textContent = v;
          grid.appendChild(c);
        }
      }
    }

    function renderAll() {
      var res = compute();
      paint(gF, f);
      paint(gD, res.dil);
      paint(gE, res.ero);
    }

    renderB();
    renderAll();
  }

  function tryInitSimEP0407(){
    var root = document.getElementById('sim-ep0407-pesos');
    if (root) initSimEP0407(root); else setTimeout(tryInitSimEP0407, 200);
  }
  tryInitSimEP0407();
})();
</script>
</div>
""")

**Figura 4.36:** Simulador EP04_07: Dilatación y Erosión con Pesos (mm.dil1 / mm.ero1)


<figure id="fig-04-sim-ep0407-pesos">
  <img src="imagens/fig-04-sim-ep0407-pesos.png" alt=" Simulador EP04_07: Dilatación y Erosión con Pesos (mm.dil1 / mm.ero1) " style="max-width:80%" />
  <figcaption><strong>Figura 4.36:</strong>  Simulador EP04_07: Dilatación y Erosión con Pesos (mm.dil1 / mm.ero1) </figcaption>
</figure>

In [91]:
%%writefile EP04_07.cpp
// your solution

Overwriting EP04_07.cpp


In [92]:
TestSuite("EP04_07.cpp").run()

### 4.0.8 EP04_08 🌋 Gradiente morfológico, Top-hat y Black-hat

En la **inspección automática de placas de circuito**, tres preguntas aparecen todo el tiempo: ¿dónde están los **bordes** de los componentes? ¿Qué **detalles claros y pequeños** (como puntos de soldadura) se destacan del fondo? ¿Qué **reentrancias oscuras** (como fisuras) esconde el fondo? Un único par erosión/dilatación responde a las tres: el **gradiente morfológico** evidencia contornos, el **top-hat** revela picos estrechos, y el **black-hat** revela valles estrechos — tres herramientas, una sola vecindad.
Ver en [Figura 4.37](#fig-04-sim-ep0408-gradiente) una simulación de este EP.

#### 4.0.8.1 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (filas) y $C$ (columnas) de $f$.
2. **Dimensiones de $B$:** Leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** Leer la matriz $B$ con valores $0$ o $1$, línea a línea.
4. **Datos:** Leer la matriz $f$ (la imagen original, en tonos de gris), línea a línea.
5. **Operadores de base:** Calcular, exactamente como en los EPs 04_03 a 04_06:
   * $d = f \oplus B$ (dilatación),
   * $e = f \ominus B$ (erosión),
   * $\text{abertura} = e \oplus B$,
   * $\text{fechamento} = d \ominus B$.
6. **Gradiente morfológico:** $\text{grad}(y,x) = d(y,x) - e(y,x)$.
7. **Top-hat:** $\text{tophat}(y,x) = f(y,x) - \text{abertura}(y,x)$.
8. **Black-hat:** $\text{blackhat}(y,x) = \text{fechamento}(y,x) - f(y,x)$.
9. **Salida:** Mostrar, **en este orden**, las tres matrices completas: gradiente, top-hat, black-hat.

#### 4.0.8.2 📌 Restricciones Computacionales

* **Sin padding en ninguna etapa intermedia** — dilatación, erosión, apertura y cierre siguen las mismas reglas de vecindad de los EPs anteriores.
* **No hay *clipping*:** las tres salidas pueden contener cualquier valor entero (el gradiente es siempre $\geq 0$, pero top-hat y black-hat también).
* **Reutilización:** $d$ y $e$ deben calcularse **una única vez** y reutilizarse para montar apertura, cierre y gradiente.

#### 4.0.8.3 🧠 Fundamentación Teórica

| Operador | Fórmula | Qué revela |
|----------|---------|----------------|
| **Gradiente** | $d - e$ | Bordes: cero en regiones planas, alto en las transiciones |
| **Top-hat** | $f - \text{abertura}(f)$ | Elementos **claros y finos**, más pequeños que $B$ |
| **Black-hat** | $\text{fechamento}(f) - f$ | Elementos **oscuros y finos**, más pequeños que $B$ |

#### 4.0.8.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros ($0$ o $1$) de la matriz $B$.
* Siguientes $L$ líneas: elementos enteros de la matriz $f$.

**Salida:**

* Matriz gradiente en $L$ filas y $C$ columnas.
* Matriz top-hat en $L$ filas y $C$ columnas.
* Matriz black-hat en $L$ filas y $C$ columnas.

#### 4.0.8.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 9<br>9<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 80 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 2 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10 | (gradiente: halo $3\times3=70$ en torno a $(2,2)$ y halo $3\times3=8$ en torno a $(6,6)$, resto $0$)<br>(top-hat: único $70$ en $(2,2)$, resto $0$)<br>(black-hat: único $8$ en $(6,6)$, resto $0$) | Pico aislado se convierte en top-hat; valle aislado se convierte en black-hat; ambos aparecen en el gradiente |

In [93]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0408-gradiente" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🌋 Simulador EP04_08: Gradiente / Top-hat / Black-hat</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">3 operadores, 1 vecindad</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Agregue picos o valles en la matriz f y observe el comportamiento simultáneo de los operadores de gradiente, top-hat y black-hat.</p>

    <!-- Botões de Ação -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0408_btn_pico" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #b9770e;background:#fef5e7;color:#b9770e;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">☀️ Agregar Pico</button>
      <button id="sim_ep0408_btn_vale" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">🕳️ Agregar Valle</button>
      <button id="sim_ep0408_btn_reset" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">↩ Limpiar Todo</button>
    </div>

    <!-- Comparativo em 4 Colunas: f, Gradiente, Top-hat, Black-hat -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(140px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- Matriz f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">f (Entrada)</span>
        <div id="sim_ep0408_grid_f" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Gradiente -->
      <div style="background:#fafaf7;border:1px solid #8e44ad;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Gradiente</span>
        <div id="sim_ep0408_grid_grad" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Top-hat -->
      <div style="background:#fafaf7;border:1px solid #d35400;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#d35400;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Top-hat</span>
        <div id="sim_ep0408_grid_th" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Black-hat -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Black-hat</span>
        <div id="sim_ep0408_grid_bh" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0408(root){
    if (!root || root.dataset.simEp0408Init) return;
    root.dataset.simEp0408Init = "1";

    var gF   = root.querySelector('#sim_ep0408_grid_f');
    var gGrad= root.querySelector('#sim_ep0408_grid_grad');
    var gTh  = root.querySelector('#sim_ep0408_grid_th');
    var gBh  = root.querySelector('#sim_ep0408_grid_bh');

    var btnPico  = root.querySelector('#sim_ep0408_btn_pico');
    var btnVale  = root.querySelector('#sim_ep0408_btn_vale');
    var btnReset = root.querySelector('#sim_ep0408_btn_reset');

    var L = 9, C = 9, f = [], B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];

    function resetMatrix() {
      f = Array.from({ length: L }, function(){ return new Array(C).fill(10); });
    }

    function morph(img, Bm, mode) {
      var Bref = mode === 'dil' ? Bm.slice().reverse().map(function(r){ return r.slice().reverse(); }) : Bm;
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C) {
                if (mode === 'dil' && img[vy][vx] > g[y][x]) g[y][x] = img[vy][vx];
                if (mode === 'ero' && img[vy][vx] < g[y][x]) g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paint(grid, img, cmin, cmax, hue) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var v = img[y][x];
          var t = cmax > cmin ? (v - cmin) / (cmax - cmin) : 0;
          var c = document.createElement('div');
          c.style.cssText = 'width:20px;height:20px;border-radius:3px;box-sizing:border-box;';
          c.style.background = v === 0 ? '#fafaf7' : hue;
          c.style.opacity = v === 0 ? '1' : (0.35 + 0.65 * Math.min(1, t));
          grid.appendChild(c);
        }
      }
    }

    function render() {
      var d = morph(f, B, 'dil');
      var e = morph(f, B, 'ero');
      var ab = morph(e, B, 'dil');
      var fc = morph(d, B, 'ero');

      var grad = f.map(function(r, y){ return r.map(function(_, x){ return d[y][x] - e[y][x]; }); });
      var th   = f.map(function(r, y){ return r.map(function(v, x){ return v - ab[y][x]; }); });
      var bh   = f.map(function(r, y){ return r.map(function(v, x){ return fc[y][x] - v; }); });

      var maxF = Math.max.apply(null, f.map(function(r){ return Math.max.apply(null, r); }));
      var maxG = Math.max.apply(null, grad.map(function(r){ return Math.max.apply(null, r); }));
      var maxTh = Math.max.apply(null, th.map(function(r){ return Math.max.apply(null, r); }));
      var maxBh = Math.max.apply(null, bh.map(function(r){ return Math.max.apply(null, r); }));

      paint(gF, f, 10, maxF || 1, '#7f8c8d');
      paint(gGrad, grad, 0, Math.max(1, maxG), '#8e44ad');
      paint(gTh, th, 0, Math.max(1, maxTh), '#d35400');
      paint(gBh, bh, 0, Math.max(1, maxBh), '#2980b9');
    }

    btnPico.addEventListener('click', function(){
      var y = 2 + Math.floor(Math.random() * 5), x = 2 + Math.floor(Math.random() * 5);
      f[y][x] = Math.min(255, f[y][x] + 60 + Math.floor(Math.random() * 30));
      render();
    });

    btnVale.addEventListener('click', function(){
      var y = 2 + Math.floor(Math.random() * 5), x = 2 + Math.floor(Math.random() * 5);
      f[y][x] = Math.max(0, f[y][x] - 8 - Math.floor(Math.random() * 4));
      render();
    });

    btnReset.addEventListener('click', function(){
      resetMatrix();
      f[2][2] = 80;
      f[6][6] = 2;
      render();
    });

    resetMatrix();
    f[2][2] = 80;
    f[6][6] = 2;
    render();
  }

  function tryInitSimEP0408(){
    var root = document.getElementById('sim-ep0408-gradiente');
    if (root) initSimEP0408(root); else setTimeout(tryInitSimEP0408, 200);
  }
  tryInitSimEP0408();
})();
</script>
</div>
""")

**Figura 4.37:** Simulador EP04_08: Gradiente Morfológico, Top-hat y Black-hat


<figure id="fig-04-sim-ep0408-gradiente">
  <img src="imagens/fig-04-sim-ep0408-gradiente.png" alt=" Simulador EP04_08: Gradiente Morfológico, Top-hat y Black-hat " style="max-width:80%" />
  <figcaption><strong>Figura 4.37:</strong>  Simulador EP04_08: Gradiente Morfológico, Top-hat y Black-hat </figcaption>
</figure>

In [94]:
%%writefile EP04_08.cpp
// your solution

Overwriting EP04_08.cpp


In [95]:
TestSuite("EP04_08.cpp").run()

### 4.0.9 EP04_09 🗺️ Transformada de Distancia y el "Centro" del Objeto

En **robótica móvil**, al planificar una ruta dentro de un pasillo, el robot quiere saber no solo *dónde* hay espacio libre, sino también **qué tan lejos** está cada punto libre de la pared más cercana. Las rutas más seguras tienden a pasar por el "centro" del pasillo, lejos de los obstáculos.

La **transformada de distancia morfológica** asigna a cada píxel un valor que representa su distancia hasta el borde más cercano, según la métrica definida por el elemento estructurante. Los píxeles cercanos al borde reciben valores bajos, mientras que los píxeles más internos reciben valores mayores. El píxel de valor máximo corresponde a la región más protegida del objeto, frecuentemente asociada a su centro morfológico.

Ver en [Figura 4.38](#fig-04-sim-ep0409-distancia) una simulación de este EP.

#### 4.0.9.1 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** leer los enteros $L$ (filas) y $C$ (columnas) de la imagen $f$.
2. **Dimensiones de $B$:** leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** leer la matriz $b$, que contiene el valor $0$ en el centro y valores negativos en las demás posiciones.
4. **Imagen:** leer la matriz binaria $f$ (valores $0$ o $1$), fila a fila.
5. **Preparación:** multiplicar la imagen por $L\times C$, asegurando que los píxeles internos tengan un valor inicial suficientemente alto para la propagación de las distancias.
6. **Transformada de distancia:** calcular la matriz de distancias utilizando el método `mm::dist1(f,b)`.
7. **Salida:** mostrar la matriz resultante de la transformada de distancia.

#### 4.0.9.2 📌 Restricciones Computacionales

* Utilizar la implementación de erosión ponderada proporcionada por la biblioteca.
* El elemento estructurante puede contener valores negativos arbitrarios.
* La transformada debe obtenerse mediante la aplicación iterativa de erosiones ponderadas hasta alcanzar un punto fijo.

**⚠️ Nota Crucial sobre Lectura de Matrices:** Como el elemento estructurante puede contener valores enteros negativos (por ejemplo, `-1` y `-99`), **no utilice la función `mm::readImg` para leer la matriz $b$**. Esa función convierte los datos al tipo `uint8`, provocando *underflow* y corrompiendo los valores negativos. Lea las $L_B$ filas de $b$ manualmente utilizando el tipo estándar `int`. La imagen $f$ puede seguir leyéndose normalmente con `mm::readImg`.

#### 4.0.9.3 🧠 Fundamentación Teórica

| Concepto                            | Significado                                                                       | Impacto Visual                               |
| ----------------------------------- | --------------------------------------------------------------------------------- | -------------------------------------------- |
| **$\text{dist}(y,x)$**              | Distancia morfológica hasta el borde más cercano según la métrica definida por $b$ | Los píxeles más internos reciben valores mayores |
| **Valor máximo**                    | Píxel más distante del borde                                                      | Aproxima el centro morfológico del objeto      |
| **Elemento estructurante ponderado** | Define los costos de desplazamiento entre píxeles vecinos                         | Determina la métrica de distancia utilizada   |
| **Objetos finos**                   | Regiones estrechas del objeto                                                       | Producen valores bajos de distancia          |

#### 4.0.9.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: entero $L$.
* Línea 2: entero $C$.
* Línea 3: entero $L_B$.
* Línea 4: entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros de la matriz $b$.
* Siguientes $L$ líneas: elementos binarios ($0$ o $1$) de la matriz $f$.

⚠️ **Nota de implementación:** Los elementos de la matriz $f$ (0 o 1) deben multiplicarse por **255** para generar una imagen binaria adecuada ($0$ y $255$) antes de aplicar la Transformada de Distancia (TD).

**Salida:**

* Matriz de la transformada de distancia en $L$ filas y $C$ columnas.

#### 4.0.9.5 📌 Ejemplo

| Entrada                                                                                                                                                          | Salida                                                                                                 | Observación                              |
| ---------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------- | --------------------------------------- |
| 5<br>9<br>3<br>3<br>-99 -1 -99<br>-1 0 -1<br>-99 -1 -99<br>0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | 0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 2 2 2 2 2 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | Resultado de la transformada de distancia. |

**Nota:** el valor `-99` actúa como una aproximación práctica de $-\infty$, impidiendo la propagación por las diagonales. De esta forma, solo los vecinos horizontal y vertical contribuyen a la distancia, produciendo la distancia de Manhattan.

In [96]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0409-distancia" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🗺️ Simulador EP04_09: Transformada de Distancia</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Capas de Erosión</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Haz clic en las celdas para dibujar tu propio objeto o selecciona una forma predefinida para calcular el mapa de distancias en cascada.</p>

    <!-- Grade f Original Clicável -->
    <div style="display:flex;justify-content:center;margin-bottom:14px;">
      <div id="sim_ep0409_grid_f" style="display:grid;grid-template-columns:repeat(9, 32px);gap:2px;user-select:none;"></div>
    </div>

    <!-- Botões de Formas Predefinidas -->
    <div style="text-align:center;margin-bottom:14px;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0409_btn_corredor" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">📐 Corredor</button>
      <button id="sim_ep0409_btn_disco" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬤ Disco</button>
      <button id="sim_ep0409_btn_l" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">📏 Forma en L</button>
    </div>

    <!-- Título do Mapa de Distâncias -->
    <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Mapa de Distancias Calculado</span>

    <!-- Grade de Distâncias -->
    <div style="display:flex;justify-content:center;">
      <div id="sim_ep0409_grid_dist" style="display:grid;grid-template-columns:repeat(9, 32px);gap:2px;user-select:none;"></div>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0409(root){
    if (!root || root.dataset.simEp0409Init) return;
    root.dataset.simEp0409Init = "1";

    var gF = root.querySelector('#sim_ep0409_grid_f');
    var gD = root.querySelector('#sim_ep0409_grid_dist');

    var L = 5, C = 9, f = [];
    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];

    function setCorredor() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 4; y++) {
        for (var x = 1; x < 8; x++) f[y][x] = 1;
      }
    }

    function setDisco() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var cy = 2, cx = 4;
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (Math.pow(y - cy, 2) + Math.pow((x - cx) * 0.6, 2) <= 4) f[y][x] = 1;
        }
      }
    }

    function setL() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 4; y++) {
        for (var x = 1; x < 3; x++) f[y][x] = 1;
      }
      for (var y = 2; y < 4; y++) {
        for (var x = 1; x < 8; x++) f[y][x] = 1;
      }
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function sameMatrix(a, b) {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (a[y][x] !== b[y][x]) return false;
        }
      }
      return true;
    }

    function render() {
      gF.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:32px;height:32px;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = f[yy][xx] ? '#16a085' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            gF.appendChild(c);
          })(y, x);
        }
      }

      var dist = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var atual = f.map(function(r){ return r.slice(); });
      var nivel = 0;

      while (atual.some(function(r){ return r.some(function(v){ return v === 1; }); })) {
        nivel++;
        for (var y = 0; y < L; y++) {
          for (var x = 0; x < C; x++) {
            if (atual[y][x] === 1) dist[y][x] = nivel;
          }
        }
        var prox = erode(atual, B);
        if (sameMatrix(prox, atual)) break;
        atual = prox;
      }

      gD.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var v = dist[y][x];
          var t = v / (nivel || 1);
          var inten = Math.round(220 - t * 170);

          c.style.cssText = 'width:32px;height:32px;border-radius:6px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;font-family:monospace;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = v === 0 ? '#fafaf7' : 'rgb(' + (inten - 60) + ',' + inten + ',' + (inten - 30) + ')';
          c.style.color = v > 0 ? '#ffffff' : '#8a8371';
          c.textContent = v || '';
          gD.appendChild(c);
        }
      }
    }

    root.querySelector('#sim_ep0409_btn_corredor').addEventListener('click', function(){ setCorredor(); render(); });
    root.querySelector('#sim_ep0409_btn_disco').addEventListener('click', function(){ setDisco(); render(); });
    root.querySelector('#sim_ep0409_btn_l').addEventListener('click', function(){ setL(); render(); });

    setCorredor();
    render();
  }

  function tryInitSimEP0409(){
    var root = document.getElementById('sim-ep0409-distancia');
    if (root) initSimEP0409(root); else setTimeout(tryInitSimEP0409, 200);
  }
  tryInitSimEP0409();
})();
</script>
</div>
""")

**Figura 4.38:** Simulador EP04_09: Transformada de Distância (Camadas de Erosión)


<figure id="fig-04-sim-ep0409-distancia">
  <img src="imagens/fig-04-sim-ep0409-distancia.png" alt=" Simulador EP04_09: Transformada de Distância (Camadas de Erosión) " style="max-width:80%" />
  <figcaption><strong>Figura 4.38:</strong>  Simulador EP04_09: Transformada de Distância (Camadas de Erosión) </figcaption>
</figure>

In [97]:
%%writefile EP04_09.cpp
// your solution

Overwriting EP04_09.cpp


In [98]:
TestSuite("EP04_09.cpp").run()

### 4.0.10 EP04_10 🪙 Separación de *Blobs*, Etiquetado y Descriptores

En una **línea de producción de monedas**, es común que las piezas se toquen entre sí en la cinta transportadora, formando una única mancha conectada en la imagen — un conteo ingenuo erraría el total. La solución clásica combina operaciones morfológicas y análisis de conectividad: primero una **erosión** reduce o rompe conexiones frágiles entre objetos, y luego el **etiquetado de componentes conectados** separa cada objeto en una región distinta. Finalmente, **descriptores geométricos** (área y caja delimitadora) resumen cada componente encontrado.

Ver en [Figura 4.39](#fig-04-sim-ep0410-rotulacao) una simulación de este EP.

#### 4.0.10.1 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** leer los enteros $L$ (filas) y $C$ (columnas) de $f$.

2. **Dimensiones de $B$:** leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.

3. **Elemento estructurante:** leer la matriz $B$, que contiene valores $0$ o $1$, fila a fila.

4. **Datos:** leer la matriz binaria $f$ (valores $0$ o $1$), fila a fila.

5. **Separación:** calcular
   $$
   f_{ero} = f \ominus B
   $$
   usando erosión binaria plana (como en el EP04_04), eliminando conexiones frágiles entre objetos.

6. **Etiquetado:** sobre $f_{ero}$, identificar componentes conectados usando conectividad definida por la vecindad $B$. El etiquetado debe seguir un barrido *raster*: al encontrar un píxel $1$ aún no etiquetado, asignar una nueva etiqueta entera creciente a partir de 1 y propagar esa etiqueta a toda la región conectada.

7. **Descriptores:** para cada etiqueta $k$, calcular:

   * **Área:** número de píxeles pertenecientes a la etiqueta;
   * **Caja delimitadora:** $$(y_{min}, x_{min}, y_{max}, x_{max})$$

8. **Salida:** mostrar el número total de etiquetas y, a continuación, una línea por etiqueta en el formato:
   $$
   k,\ \text{área},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
   $$

#### 4.0.10.2 📌 Restricciones Computacionales

* La erosión debe aplicarse antes del etiquetado.
* La conectividad es fija y está definida por la vecindad anterior.
* El elemento estructurante $B$ no interfiere en la conectividad del etiquetado.
* Sin *padding* en ninguna etapa.
* El orden de las etiquetas sigue la primera detección en el barrido *raster*.

#### 4.0.10.3 🧠 Fundamentación Teórica

| Concepto           | Significado                                   | Impacto                                              |
| ------------------ | --------------------------------------------- | ---------------------------------------------------- |
| Puente fino        | Conexión estrecha entre objetos               | Puede ser eliminado por la erosión morfológica       |
| Conectividad       | Definida por el conjunto $$\mathcal{N}(y,x)$$ | Determina qué píxeles pertenecen al mismo componente |
| Área               | Número de píxeles por componente              | Estimación directa del tamaño del objeto             |
| Caja delimitadora  | Extensión espacial de la etiqueta             | Resumen geométrico del componente                    |

#### 4.0.10.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: entero $L$
* Línea 2: entero $C$
* Línea 3: entero $L_B$
* Línea 4: entero $C_B$
* Siguientes $L_B$ líneas: matriz $B$
* Siguientes $L$ líneas: matriz $f$

**Salida:**

* Línea 1: número total de etiquetas encontradas
* Líneas siguientes:
  $$
  k,\ \text{área},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
  $$

In [99]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0410-rotulacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪙 Simulador EP04_10: Monedas Pegadas → Separadas → Contadas</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">erosión + etiqueta + descriptores</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajusta el grosor del puente entre las monedas y observa cómo la erosión morfológica separa los objetos para el conteo y la extracción de descriptores (área y cuadro delimitador).</p>

    <!-- Controle de Espessura da Ponte -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#b9770e;">Grosor del Puente entre las Monedas</label><br>
      <input type="range" id="sim_ep0410_sl_p" min="1" max="3" step="1" value="1" style="width:60%;cursor:pointer;accent-color:#b9770e;margin-top:6px;">
      <span id="sim_ep0410_vl_p" style="font-family:monospace;font-size:12px;font-weight:700;color:#b9770e;margin-left:8px;">1 píxel</span>
    </div>

    <!-- Comparativo Lado a Lado: f original vs Rótulos Pós-Erosão -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original (ligadas) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original (Pegadas)</span>
        <div id="sim_ep0410_grid_f" style="display:grid;grid-template-columns:repeat(10, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Após Erosão + Rótulos -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Tras Erosión + Etiquetas</span>
        <div id="sim_ep0410_grid_lab" style="display:grid;grid-template-columns:repeat(10, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Painel Informativo / Descritores -->
    <div id="sim_ep0410_info" style="background:#fef5e7;border:1px solid #f8c471;border-radius:8px;padding:10px 14px;font-size:11px;color:#7d5a00;text-align:center;line-height:1.5;"></div>

  </div>
</div>

<script>
(function(){
  function initSimEP0410(root){
    if (!root || root.dataset.simEp0410Init) return;
    root.dataset.simEp0410Init = "1";

    var slP  = root.querySelector('#sim_ep0410_sl_p');
    var vlP  = root.querySelector('#sim_ep0410_vl_p');
    var gF   = root.querySelector('#sim_ep0410_grid_f');
    var gL   = root.querySelector('#sim_ep0410_grid_lab');
    var info = root.querySelector('#sim_ep0410_info');

    var L = 7, C = 10;
    var B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
    var palette = ['#e74c3c', '#27ae60', '#2980b9', '#8e44ad', '#d35400'];

    function buildF(p) {
      var f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 6; y++) {
        for (var x = 1; x < 4; x++) f[y][x] = 1;
      }
      for (var y = 1; y < 6; y++) {
        for (var x = 6; x < 9; x++) f[y][x] = 1;
      }
      var midRow = 3;
      for (var dy = 0; dy < p; dy++) {
        var ry = midRow - Math.floor(p / 2) + dy;
        for (var x = 4; x < 6; x++) f[ry][x] = 1;
      }
      return f;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function labelK8(img) {
      var labels = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var dirs = [[-1, -1], [-1, 0], [-1, 1], [0, -1], [0, 1], [1, -1], [1, 0], [1, 1]];
      var cur = 0, desc = [];

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (img[y][x] === 1 && labels[y][x] === 0) {
            cur++;
            var stack = [[y, x]];
            labels[y][x] = cur;
            var area = 0, miny = y, maxy = y, minx = x, maxx = x;

            while (stack.length) {
              var cell = stack.pop();
              var cy = cell[0], cx = cell[1];
              area++;
              if (cy < miny) miny = cy;
              if (cy > maxy) maxy = cy;
              if (cx < minx) minx = cx;
              if (cx > maxx) maxx = cx;

              dirs.forEach(function(d){
                var ny = cy + d[0], nx = cx + d[1];
                if (ny >= 0 && ny < L && nx >= 0 && nx < C && img[ny][nx] === 1 && labels[ny][nx] === 0) {
                  labels[ny][nx] = cur;
                  stack.push([ny, nx]);
                }
              });
            }
            desc.push({k: cur, area: area, miny: miny, minx: minx, maxy: maxy, maxx: maxx});
          }
        }
      }
      return {labels: labels, desc: desc};
    }

    function paintBin(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] ? '#b9770e' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintLabels(grid, labels) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var k = labels[y][x];
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;color:#ffffff;box-sizing:border-box;';
          c.style.background = k > 0 ? palette[(k - 1) % palette.length] : '#fafaf7';
          c.textContent = k > 0 ? k : '';
          grid.appendChild(c);
        }
      }
    }

    function render() {
      var p = parseInt(slP.value, 10) || 1;
      vlP.textContent = p + ' px';
      var f = buildF(p);
      var fe = erode(f, B);
      var res = labelK8(fe);

      paintBin(gF, f);
      paintLabels(gL, res.labels);

      var txt = '<b>' + res.desc.length + ' objeto(s) detectado(s) após a erosão.</b><br>';
      res.desc.forEach(function(d){
        txt += 'Rótulo ' + d.k + ': área = ' + d.area + ', bbox = (' + d.miny + ',' + d.minx + ') → (' + d.maxy + ',' + d.maxx + ')<br>';
      });
      if (res.desc.length < 2) {
        txt += '<i>A ponte ainda é espessa demais para a erosão 3×3 — as moedas continuam fundidas em 1 só objeto.</i>';
      }
      info.innerHTML = txt;
    }

    slP.addEventListener('input', render);

    render();
  }

  function tryInitSimEP0410(){
    var root = document.getElementById('sim-ep0410-rotulacao');
    if (root) initSimEP0410(root); else setTimeout(tryInitSimEP0410, 200);
  }
  tryInitSimEP0410();
})();
</script>
</div>
""")

**Figura 4.39:** Simulador EP04_10: Separación de Blobs, Etiquetado y Descriptores


<figure id="fig-04-sim-ep0410-rotulacao">
  <img src="imagens/fig-04-sim-ep0410-rotulacao.png" alt=" Simulador EP04_10: Separación de Blobs, Etiquetado y Descriptores " style="max-width:80%" />
  <figcaption><strong>Figura 4.39:</strong>  Simulador EP04_10: Separación de Blobs, Etiquetado y Descriptores </figcaption>
</figure>

In [100]:
%%writefile EP04_10.cpp
// your solution

Overwriting EP04_10.cpp


In [101]:
TestSuite("EP04_10.cpp").run()

## Referências do Capítulo


BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

LOTUFO, R.A.; ZAMPIROLLI, F.A. **Fast multidimensional parallel Euclidean distance transform based on mathematical morphology**. 2001.

MATHERON, G. **Random Sets and Integral Geometry**. New York, John Wiley \& Sons, 1975.

REDMON, Joseph *et al*. **You Only Look Once: Unified, Real-Time Object Detection**. 2016.

SERRA, Jean. **Image Analysis and Mathematical Morphology**. London, Academic Press, 1982.

SINGH, S.; LLOYD, J.; FLICKER, F. **Hamiltonian Cycles on Ammann-Beenker Tilings**. 2024.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

ZAMPIROLLI, Francisco de Assis *et al*. **Teaching Hands-On Digital Image Processing with morph.py: Methods and Comprehensive Results**. 2025.

*Referência não encontrada para: staticmethod*